# Step300 Endpoint-0.20 — Eval100 Exhaustive Probability Export

Runs a fixed 100-sample validation evaluation from the verified step300 checkpoint.

Outputs:
- exhaustive raw probabilities for later adaptive decoding/fallback experiments
- a compact readable decoding table
- the current base 10:30:60 result

Every possible fixed-first, fixed-last, and ordered endpoint pair is scored and cached.

KV-cache safety:
- validates the the first eval sample against full recomputation
- checks full24, fixed-first, fixed-last, and middle scoring
- aborts before eval100 if score/probability tolerances or top1 agreement fail

KV implementation update:
- uses the exact mRoPE-aware cache logic from the full-24 training notebook
- handles rope_deltas, 3-axis position_ids, cache_position, and mutable-cache crop
- verifies the the first sample before running eval100

Equivalence gate update:
- raw log-score offsets are logged but do not fail by themselves
- validation gates on centered score differences, softmax probabilities, and top1
- this matches the quantities that actually affect decoding

Clean eval-cache safety update:
- uses a new output directory and cache file
- previous failed/partial eval records are not reused
- raw and centered score differences are diagnostics only
- hard gate uses softmax probability difference <= 0.002 and top1 agreement

Final validation policy:
- checks only the first eval sample
- hard gate: top1 agreement and max softmax probability difference <= 0.003
- then immediately runs eval100
- uses a fresh v4 output directory/cache


In [1]:
# 1) Install dependencies, then restart runtime once.
# After restart, run this cell again and continue.
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_pilot_c_deps_installed")

if Path("/content").exists() and not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "huggingface_hub",
        "hf_xet",
        "modelscope",
        "jedi",
        "pandas==2.2.2",
        "safetensors>=0.4.5",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
elif Path("/content").exists():
    print("Dependencies already installed. Continue.")
else:
    print("Local environment detected. Skipping Colab dependency install.")


Dependencies already installed. Continue.


In [2]:
# 2) Setup: Drive, data, model cache, run paths
from google.colab import drive
from pathlib import Path
import os
import shutil

drive_root = Path("/content/drive")
if drive_root.exists() and not os.path.ismount(str(drive_root)) and any(drive_root.iterdir()):
    print("Removing local pre-mount /content/drive contents:", sorted(str(p) for p in drive_root.iterdir())[:20])
    shutil.rmtree(drive_root)
drive_root.mkdir(parents=True, exist_ok=True)
drive.mount("/content/drive")

import ast
import copy
import gc
import glob
import itertools
import json
import math
import random
import re
import subprocess
import zipfile
from datetime import datetime

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, TrainerCallback, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

SNU_ROOT = Path("/content/drive/MyDrive/SNU_AI_Challenge")
assert SNU_ROOT.exists(), SNU_ROOT

ZIP_PATH = SNU_ROOT / "snuaichallenge.zip"
DATA_DIR = Path("/content/snuaichallenge_data")
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

if not TRAIN_CSV.exists() or not TRAIN_IMAGE_DIR.is_dir():
    print("Extracting:", ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

assert TRAIN_CSV.exists(), TRAIN_CSV
assert TEST_CSV.exists(), TEST_CSV
assert TRAIN_IMAGE_DIR.is_dir(), TRAIN_IMAGE_DIR
assert TEST_IMAGE_DIR.is_dir(), TEST_IMAGE_DIR

MODEL_REPO_ID = "Qwen/Qwen2-VL-7B-Instruct"
USE_MODELSCOPE_BASE_MODEL = False
DRIVE_MODEL_DIR = SNU_ROOT / "model_cache/Qwen2-VL-7B-Instruct"
LOCAL_MODEL_DIR = Path("/content/Qwen2-VL-7B-Instruct")


def print_runtime_storage():
    print("Storage check for /content:")
    try:
        subprocess.run(["df", "-h", "/content"], check=False)
    except Exception as exc:
        total, used, free = shutil.disk_usage("/content")
        print(f"/content free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB ({exc})")

    try:
        meminfo = {}
        with open("/proc/meminfo", "r", encoding="utf-8") as handle:
            for line in handle:
                key, value = line.split(":", 1)
                meminfo[key] = int(value.strip().split()[0]) / (1024 ** 2)
        print(f"RAM available: {meminfo.get('MemAvailable', 0):.1f} GB / total: {meminfo.get('MemTotal', 0):.1f} GB")
    except Exception as exc:
        print("RAM check skipped:", exc)

    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"GPU memory free: {free / (1024 ** 3):.1f} GB / total: {total / (1024 ** 3):.1f} GB")


def model_cache_is_complete(model_dir):
    model_dir = Path(model_dir)
    if not (model_dir / "config.json").exists():
        return False
    has_weight = (model_dir / "model.safetensors.index.json").exists() or bool(list(model_dir.glob("*.safetensors")))
    has_processor = any((model_dir / name).exists() for name in ["preprocessor_config.json", "processor_config.json", "tokenizer.json", "tokenizer_config.json"])
    return bool(has_weight and has_processor)


def copy_drive_cache_to_local():
    if not model_cache_is_complete(DRIVE_MODEL_DIR):
        raise FileNotFoundError(f"Drive model cache is incomplete or missing: {DRIVE_MODEL_DIR}")

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using existing local model:", LOCAL_MODEL_DIR)
        return

    print("Copying base model from Drive to Colab local disk...")
    print("  from:", DRIVE_MODEL_DIR)
    print("  to  :", LOCAL_MODEL_DIR)
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR


def download_base_model_to_local_and_cache():
    print("Base model cache not found. Downloading with Hugging Face snapshot_download to local disk first:")
    print("repo:", MODEL_REPO_ID)
    if LOCAL_MODEL_DIR.exists() and not model_cache_is_complete(LOCAL_MODEL_DIR):
        shutil.rmtree(LOCAL_MODEL_DIR)
    from huggingface_hub import snapshot_download
    hf_token = os.environ.get("HF_TOKEN")
    model_dir = snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=str(LOCAL_MODEL_DIR),
        token=hf_token,
        max_workers=8,
    )
    print("Downloaded:", model_dir)
    assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR

    DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    shutil.copytree(LOCAL_MODEL_DIR, tmp)
    if DRIVE_MODEL_DIR.exists():
        shutil.rmtree(DRIVE_MODEL_DIR)
    os.replace(tmp, DRIVE_MODEL_DIR)
    print("Saved to Drive:", DRIVE_MODEL_DIR)


def ensure_base_model_path():
    print_runtime_storage()

    if model_cache_is_complete(LOCAL_MODEL_DIR):
        print("Using local model:", LOCAL_MODEL_DIR)
    elif model_cache_is_complete(DRIVE_MODEL_DIR):
        copy_drive_cache_to_local()
    elif not USE_MODELSCOPE_BASE_MODEL:
        download_base_model_to_local_and_cache()
    else:
        print("Base model cache not found. Downloading via ModelScope:", MODEL_REPO_ID)
        from modelscope import snapshot_download as modelscope_snapshot_download
        model_dir = modelscope_snapshot_download(MODEL_REPO_ID, cache_dir="/content/modelscope_cache")
        if LOCAL_MODEL_DIR.exists():
            shutil.rmtree(LOCAL_MODEL_DIR)
        shutil.copytree(model_dir, LOCAL_MODEL_DIR)
        assert model_cache_is_complete(LOCAL_MODEL_DIR), LOCAL_MODEL_DIR
        DRIVE_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
        tmp = Path(str(DRIVE_MODEL_DIR) + ".tmp")
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(LOCAL_MODEL_DIR, tmp)
        if DRIVE_MODEL_DIR.exists():
            shutil.rmtree(DRIVE_MODEL_DIR)
        os.replace(tmp, DRIVE_MODEL_DIR)
        print("Saved to Drive:", DRIVE_MODEL_DIR)

    print_runtime_storage()
    print("Using local model:", LOCAL_MODEL_DIR)
    return str(LOCAL_MODEL_DIR)


MODEL_ID = ensure_base_model_path()
MODEL_LOCAL_FILES_ONLY = True

OUTPUT_ROOT = SNU_ROOT / "qwen2vl_7b_multitask_bipair_conditional_v1"
# Set this to a specific run id when needed. If None, the latest run is used.
PILOT_RUN_ID = "20260721_011312"  # base run only for shared config

def resolve_run_root(output_root, run_id=None):
    runs_root = output_root / "runs"
    if run_id is not None:
        run_root = runs_root / run_id
        assert run_root.is_dir(), run_root
        return run_root
    candidates = sorted([p for p in runs_root.iterdir() if p.is_dir()])
    if not candidates:
        raise RuntimeError(f"No runs found under {runs_root}")
    return candidates[-1]

RUN_ROOT = resolve_run_root(OUTPUT_ROOT, PILOT_RUN_ID)
RUN_ID = RUN_ROOT.name
OUTPUT_DIR = (
    SNU_ROOT
    / "qwen2vl_7b_multitask_bipair_conditional_v1"
    / "runs"
    / "20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics"
    / "full24_order_first_last_calibrated_pairwise_metrics"
)
EVAL_DIR = OUTPUT_DIR / "eval_step300_exhaustive"
BEST_ADAPTER_DIR = (
    OUTPUT_DIR / "checkpoints" / "checkpoint-1200-full24-step-300"
)
SUBMIT_PATH = EVAL_DIR / "submission_step300_10_30_60.csv"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
assert BEST_ADAPTER_DIR.is_dir(), BEST_ADAPTER_DIR
for required_name in ["adapter_config.json", "training_state.json"]:
    assert (BEST_ADAPTER_DIR / required_name).exists(), BEST_ADAPTER_DIR / required_name
with open(BEST_ADAPTER_DIR / "training_state.json", "r", encoding="utf-8") as handle:
    STEP300_STATE = json.load(handle)
assert int(STEP300_STATE["step"]) == 300, STEP300_STATE
assert float(STEP300_STATE["loss_weights"]["first_ce"]) == 0.20, STEP300_STATE
assert float(STEP300_STATE["loss_weights"]["last_ce"]) == 0.20, STEP300_STATE
print("Using verified step300 endpoint-0.20 adapter:", BEST_ADAPTER_DIR)

SEED = 42
VALID_RATIO = 0.10
TRAIN_ROWS = None
VALID_ROWS = None
FULL_EVAL_ROWS = 150
SELECTED_CHECKPOINTS = ["checkpoint-1200-full24-step-300"]
SCORE_BATCH_SIZE = 8
NEXT_TOKEN_BATCH_SIZE = 16
RUN_DIRECT_GENERATION = False
USE_KV_CACHE_SEQUENCE_SCORING = True

TASK_RATIOS = {
    "order": 0.30,
    "pairwise": 0.25,
    "first": 0.10,
    "last": 0.10,
    "fixed_first": 0.10,
    "fixed_last": 0.10,
    "fixed_endpoints": 0.05,
}
TASK_LOSS_WEIGHTS = {task: 1.0 for task in TASK_RATIOS}

LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
MAX_TRAIN_STEPS = -1
SAVE_STEPS = 100
LOGGING_STEPS = 20

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
PAIR_INDICES = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
PERMUTATIONS = list(itertools.permutations([1, 2, 3, 4]))

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("run root:", RUN_ROOT)
print("output:", OUTPUT_DIR)
print("model:", MODEL_ID)


Mounted at /content/drive
Extracting: /content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip
Storage check for /content:
RAM available: 80.7 GB / total: 83.5 GB
GPU memory free: 39.1 GB / total: 39.5 GB
Copying base model from Drive to Colab local disk...
  from: /content/drive/MyDrive/SNU_AI_Challenge/model_cache/Qwen2-VL-7B-Instruct
  to  : /content/Qwen2-VL-7B-Instruct
Storage check for /content:
RAM available: 80.4 GB / total: 83.5 GB
GPU memory free: 39.1 GB / total: 39.5 GB
Using local model: /content/Qwen2-VL-7B-Instruct
Using verified step300 endpoint-0.20 adapter: /content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/checkpoints/checkpoint-1200-full24-step-300
run root: /content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260721_011312
output: /content/drive/MyDrive/SNU_AI_Challen

In [3]:
# 3) Data split, prompts, metrics, and model helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result


def order_to_sequence(answer):
    return [input_index + 1 for input_index, _ in sorted(enumerate(answer), key=lambda item: item[1])]


def compact_order(order):
    return " ".join(str(int(value)) for value in order)


def parse_compact_order(text, expected_len=4):
    values = [int(x) for x in re.findall(r"[1-4]", str(text))]
    if len(values) != expected_len or len(set(values)) != expected_len:
        return None
    return values


def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [str(Path(image_root) / sample_id / str(row[f"Input_{i}"])) for i in range(1, 5)]


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()


def base_record(row_index, row):
    answer = [int(value) for value in row["Answer_list"]]
    order = order_to_sequence(answer)
    return {
        "row_index": int(row_index),
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order,
        "image_paths": row_image_paths(row, TRAIN_IMAGE_DIR),
    }


def pair_target_for_order(order, a, b):
    ranks = {frame: idx for idx, frame in enumerate(order)}
    return "A" if ranks[int(a)] < ranks[int(b)] else "B"


def build_record_pools(dataframe):
    pools = {task: [] for task in TASK_RATIOS}
    for row_index, row in dataframe.iterrows():
        base = base_record(row_index, row)
        order = base["order"]

        item = copy.deepcopy(base)
        item.update({"task_type": "order", "target": compact_order(order)})
        pools["order"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "first", "target": str(order[0])})
        pools["first"].append(item)

        item = copy.deepcopy(base)
        item.update({"task_type": "last", "target": str(order[-1])})
        pools["last"].append(item)

        for i, j in PAIR_INDICES:
            a, b = i + 1, j + 1
            for left, right in [(a, b), (b, a)]:
                item = copy.deepcopy(base)
                item.update({
                    "task_type": "pairwise",
                    "pair": [left, right],
                    "image_paths": [base["image_paths"][left - 1], base["image_paths"][right - 1]],
                    "target": pair_target_for_order(order, left, right),
                })
                pools["pairwise"].append(item)

        first = order[0]
        remaining = [x for x in order if x != first]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_first", "fixed_first": first, "target": compact_order(remaining)})
        pools["fixed_first"].append(item)

        last = order[-1]
        remaining = [x for x in order if x != last]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_last", "fixed_last": last, "target": compact_order(remaining)})
        pools["fixed_last"].append(item)

        middle = [x for x in order if x not in {order[0], order[-1]}]
        item = copy.deepcopy(base)
        item.update({"task_type": "fixed_endpoints", "fixed_first": order[0], "fixed_last": order[-1], "target": compact_order(middle)})
        pools["fixed_endpoints"].append(item)
    return pools


def sample_records(records, count, rng):
    indices = rng.integers(0, len(records), size=count)
    return [records[int(index)] for index in indices]


def build_balanced_records(dataframe):
    pools = build_record_pools(dataframe)
    base_total = int(math.ceil(len(pools["order"]) / TASK_RATIOS["order"]))
    rng = np.random.default_rng(SEED)
    merged = []
    distribution = {}
    for task, ratio in TASK_RATIOS.items():
        count = max(1, int(round(base_total * ratio)))
        records = sample_records(pools[task], count, rng)
        merged.extend(records)
        distribution[task] = {"pool": len(pools[task]), "sampled": len(records)}
        print(task, distribution[task])
    rng.shuffle(merged)
    return merged, pools, distribution


train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
train_df["Id"] = train_df["Id"].astype(str)
test_df["Id"] = test_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

split_root = SNU_ROOT / "id_splits" / "qwen2vl_lgt_order_refine_20260714_003635"
if (split_root / "train_ids.json").exists() and (split_root / "validation_ids.json").exists():
    train_ids = set(str(x) for x in json.load(open(split_root / "train_ids.json", "r", encoding="utf-8")))
    valid_ids = set(str(x) for x in json.load(open(split_root / "validation_ids.json", "r", encoding="utf-8")))
else:
    unique_ids = train_df["Id"].unique().copy()
    rng = np.random.default_rng(SEED)
    rng.shuffle(unique_ids)
    valid_size = max(1, int(len(unique_ids) * VALID_RATIO))
    valid_ids = set(unique_ids[:valid_size])
    train_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(train_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
if TRAIN_ROWS is not None:
    training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
if VALID_ROWS is not None:
    validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)

train_records, train_pools, task_distribution = build_balanced_records(training_df)
valid_pools = build_record_pools(validation_df)

run_config = {
    "experiment": "qwen2vl_7b_multitask_bipair_conditional_v1",
    "run_id": RUN_ID,
    "model_repo_id": MODEL_REPO_ID,
    "model_id": MODEL_ID,
    "output_dir": str(OUTPUT_DIR),
    "task_ratios": TASK_RATIOS,
    "task_loss_weights": TASK_LOSS_WEIGHTS,
    "task_distribution": task_distribution,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target_modules": LORA_TARGET_MODULES},
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "max_train_steps": MAX_TRAIN_STEPS,
    "save_steps": SAVE_STEPS,
    "seed": SEED,
    "train_rows": len(training_df),
    "validation_rows": len(validation_df),
}
with open(RUN_ROOT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)
with open(OUTPUT_DIR / "task_distribution.json", "w", encoding="utf-8") as f:
    json.dump(task_distribution, f, ensure_ascii=False, indent=2)

print("train/valid/test:", len(training_df), len(validation_df), len(test_df))
print("train records:", len(train_records))


# Free stale objects before loading the 7B base model. This matters when a previous
# cell was interrupted during shard loading in the same Colab runtime.
for _name in ["trainer", "model", "base_model", "processor"]:
    if _name in globals():
        del globals()[_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory before model load: {free / (1024 ** 3):.1f} GB free / {total / (1024 ** 3):.1f} GB total")


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
    local_files_only=MODEL_LOCAL_FILES_ONLY,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)


def task_instruction(example):
    return globals()["task_instruction_train"](example) if "task_instruction_train" in globals() else task_instruction_impl(example)


def task_instruction_impl(example):
    sentence = example["sentence"]
    task_type = example["task_type"]
    if task_type == "pairwise":
        return f"Caption:\n{sentence}\n\nThe two candidate images are labeled A and B in the presented order.\nWhich image occurs earlier in the story timeline?\nAnswer only A or B."
    if task_type == "first":
        return f"Caption:\n{sentence}\n\nWhich image is the first scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "last":
        return f"Caption:\n{sentence}\n\nWhich image is the last scene in the story? Answer only one frame number from 1 to 4."
    if task_type == "order":
        return f"Caption:\n{sentence}\n\nOrder all four images from earliest to latest in the story.\nAnswer only four frame numbers separated by spaces, for example: 1 2 3 4."
    if task_type == "fixed_first":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_first']} is fixed as the first scene.\nOrder the remaining frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    if task_type == "fixed_last":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_last']} is fixed as the last scene.\nOrder the remaining frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    if task_type == "fixed_endpoints":
        return f"Caption:\n{sentence}\n\nFrame {example['fixed_first']} is fixed as the first scene.\nFrame {example['fixed_last']} is fixed as the last scene.\nOrder the remaining middle frames from earliest to latest.\nAnswer only the remaining frame numbers separated by spaces."
    raise ValueError(task_type)


def make_messages(example, include_answer=False):
    content = []
    for idx, _ in enumerate(example["image_paths"], start=1):
        label = "A" if example["task_type"] == "pairwise" and idx == 1 else "B" if example["task_type"] == "pairwise" and idx == 2 else str(idx)
        content.append({"type": "text", "text": f"\nImage {label}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + task_instruction_impl(example)})
    messages = [{"role": "user", "content": content}]
    if include_answer:
        messages.append({"role": "assistant", "content": str(example["target"])})
    return messages


def checkpoint_name(path):
    return Path(path).name


def find_checkpoint_dirs():
    dirs = []
    for path in OUTPUT_DIR.iterdir():
        if path.name.startswith("checkpoint-") and (path / "adapter_config.json").exists():
            dirs.append(path)
    final_dir = OUTPUT_DIR / "final_adapter"
    if (final_dir / "adapter_config.json").exists():
        dirs.append(final_dir)
    def sort_key(path):
        match = re.findall(r"checkpoint-(\d+)", str(path))
        return int(match[-1]) if match else 10**9
    return sorted(dict.fromkeys(dirs), key=sort_key)


EVAL_MODEL = None
CURRENT_ADAPTER_NAME = None


def adapter_name_for(adapter_dir):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", checkpoint_name(adapter_dir))


def load_eval_model(adapter_dir):
    global EVAL_MODEL, CURRENT_ADAPTER_NAME
    adapter_dir = str(adapter_dir)
    adapter_name = adapter_name_for(adapter_dir)

    def load_fresh():
        base = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            torch_dtype=torch.float16,
            device_map="auto",
            local_files_only=MODEL_LOCAL_FILES_ONLY,
            trust_remote_code=True,
        low_cpu_mem_usage=True,
        )
        from peft import PeftModel
        return PeftModel.from_pretrained(base, adapter_dir, adapter_name=adapter_name, is_trainable=False)

    if EVAL_MODEL is None:
        EVAL_MODEL = load_fresh()
    else:
        try:
            if CURRENT_ADAPTER_NAME is not None and hasattr(EVAL_MODEL, "delete_adapter"):
                EVAL_MODEL.delete_adapter(CURRENT_ADAPTER_NAME)
            EVAL_MODEL.load_adapter(adapter_dir, adapter_name=adapter_name, is_trainable=False)
            EVAL_MODEL.set_adapter(adapter_name)
        except Exception as exc:
            print("Adapter switch failed; reloading base model:", repr(exc))
            del EVAL_MODEL
            gc.collect()
            torch.cuda.empty_cache()
            EVAL_MODEL = load_fresh()

    CURRENT_ADAPTER_NAME = adapter_name
    EVAL_MODEL.eval()
    if hasattr(EVAL_MODEL, "generation_config"):
        EVAL_MODEL.generation_config.do_sample = False
        EVAL_MODEL.generation_config.temperature = None
        EVAL_MODEL.generation_config.top_p = None
        EVAL_MODEL.generation_config.top_k = None
        EVAL_MODEL.generation_config.num_beams = 1
    return EVAL_MODEL

def model_device(active_model):
    return next(active_model.parameters()).device


def single_token_id(value):
    ids = processor.tokenizer.encode(str(value), add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{value!r} tokenized to {ids}")
    return ids[0]


DIGIT_TOKEN_IDS = {digit: single_token_id(str(digit)) for digit in [1, 2, 3, 4]}
AB_TOKEN_IDS = {"A": single_token_id("A"), "B": single_token_id("B")}


def make_eval_example(row, task_type, pair=None, fixed_first=None, fixed_last=None, image_root=TRAIN_IMAGE_DIR):
    answer = [int(value) for value in row.get("Answer_list", [1, 2, 3, 4])]
    image_paths = row_image_paths(row, image_root)
    example = {
        "sample_id": str(row["Id"]),
        "sentence": "" if pd.isna(row["Sentence"]) else str(row["Sentence"]),
        "answer": answer,
        "order": order_to_sequence(answer),
        "image_paths": image_paths,
        "task_type": task_type,
        "target": "1",
    }
    if task_type == "pairwise":
        a, b = pair
        example["pair"] = [a, b]
        example["image_paths"] = [image_paths[a - 1], image_paths[b - 1]]
    if fixed_first is not None:
        example["fixed_first"] = int(fixed_first)
    if fixed_last is not None:
        example["fixed_last"] = int(fixed_last)
    return example


order {'pool': 8582, 'sampled': 8582}
pairwise {'pool': 102984, 'sampled': 7152}
first {'pool': 8582, 'sampled': 2861}
last {'pool': 8582, 'sampled': 2861}
fixed_first {'pool': 8582, 'sampled': 2861}
fixed_last {'pool': 8582, 'sampled': 2861}
fixed_endpoints {'pool': 8582, 'sampled': 1430}
train/valid/test: 8582 953 819
train records: 28608
GPU memory before model load: 39.1 GB free / 39.5 GB total


In [4]:
# 4) Scoring and conditional sequential decoding
@torch.no_grad()
def _batch_to_device(inputs, active_model):
    return {key: value.to(model_device(active_model)) if torch.is_tensor(value) else value for key, value in inputs.items()}


@torch.no_grad()
def score_next_token_examples(active_model, examples, token_ids, batch_size=None):
    batch_size = int(batch_size or NEXT_TOKEN_BATCH_SIZE)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    ids = list(token_ids.values())
    results = []
    try:
        for start_index in range(0, len(examples), batch_size):
            batch_examples = examples[start_index:start_index + batch_size]
            texts = [
                processor.apply_chat_template(
                    make_messages(example, include_answer=False),
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for example in batch_examples
            ]
            batch_images = [[load_rgb(path) for path in example["image_paths"]] for example in batch_examples]
            inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = _batch_to_device(inputs, active_model)
            outputs = active_model(**inputs)
            for row_index in range(len(batch_examples)):
                last_pos = int(inputs["attention_mask"][row_index].sum().item()) - 1
                logits = outputs.logits[row_index, last_pos]
                probs = torch.softmax(logits[ids].float(), dim=-1).detach().cpu().numpy()
                results.append({key: float(prob) for key, prob in zip(token_ids.keys(), probs)})
    finally:
        processor.tokenizer.padding_side = old_padding_side
    return results


@torch.no_grad()
def score_next_token_candidates(active_model, example, token_ids):
    return score_next_token_examples(active_model, [example], token_ids, batch_size=1)[0]


@torch.no_grad()
def generate_text(active_model, example, max_new_tokens=16):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        text = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example["image_paths"]]
        inputs = processor(text=[text], images=[images], return_tensors="pt")
        inputs = _batch_to_device(inputs, active_model)
        generated = active_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
        output = processor.tokenizer.batch_decode(generated[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        return output.strip()
    finally:
        processor.tokenizer.padding_side = old_padding_side


@torch.no_grad()
def _score_sequence_candidates_recompute(active_model, example, candidate_orders, batch_size=None):
    batch_size = int(batch_size or SCORE_BATCH_SIZE)
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"
    try:
        prompt = processor.apply_chat_template(make_messages(example, include_answer=False), tokenize=False, add_generation_prompt=True)
        images = [load_rgb(path) for path in example["image_paths"]]
        prompt_inputs = processor(text=[prompt], images=[images], return_tensors="pt")
        prompt_len = int(prompt_inputs["attention_mask"][0].sum().item())
        scores = {}
        orders = list(candidate_orders)
        for start_index in range(0, len(orders), batch_size):
            batch_orders = orders[start_index:start_index + batch_size]
            texts = [prompt + compact_order(order) for order in batch_orders]
            batch_images = [images for _ in batch_orders]
            inputs = processor(text=texts, images=batch_images, padding=True, return_tensors="pt")
            inputs = _batch_to_device(inputs, active_model)
            outputs = active_model(**inputs)
            for row_index, order in enumerate(batch_orders):
                input_ids = inputs["input_ids"][row_index]
                attention_len = int(inputs["attention_mask"][row_index].sum().item())
                target_len = attention_len - prompt_len
                target_ids = input_ids[prompt_len:prompt_len + target_len]
                logits = outputs.logits[row_index, prompt_len - 1:prompt_len - 1 + target_len]
                log_probs = torch.log_softmax(logits.float(), dim=-1)
                scores[" ".join(map(str, order))] = float(log_probs.gather(1, target_ids[:, None]).mean().item())
        return scores
    finally:
        processor.tokenizer.padding_side = old_padding_side



def target_ids_after_prompt(prompt, answer_text, device):
    """Extract exact continuation token IDs after the already-tokenized prompt."""
    tokenizer = processor.tokenizer
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(prompt + answer_text, add_special_tokens=False)["input_ids"]

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise RuntimeError(
            "Prompt/answer boundary tokenization changed. "
            f"prompt_tail={prompt_ids[-10:]}, "
            f"full_prefix_tail={full_ids[max(0, len(prompt_ids)-10):len(prompt_ids)]}"
        )

    target_ids = full_ids[len(prompt_ids):]
    if not target_ids:
        raise RuntimeError(f"No target tokens for answer: {answer_text!r}")

    return torch.tensor(target_ids, dtype=torch.long, device=device)


def build_qwen2vl_continuation_position_ids(
    prompt_attention_mask,
    rope_deltas,
    continuation_length,
    device,
):
    """Build Qwen2-VL 3-axis mRoPE positions for text continuation tokens."""
    if rope_deltas is None:
        raise RuntimeError("Prompt output has no rope_deltas.")

    batch_size = prompt_attention_mask.shape[0]
    prompt_text_lengths = prompt_attention_mask.long().sum(dim=1).to(device)
    offsets = torch.arange(
        continuation_length,
        dtype=torch.long,
        device=device,
    )

    text_positions = prompt_text_lengths[:, None] + offsets[None, :]
    rope_deltas = rope_deltas.to(device).reshape(batch_size, 1)
    mrope_positions = text_positions + rope_deltas

    return (
        mrope_positions
        .unsqueeze(0)
        .expand(3, -1, -1)
        .contiguous()
    )


@torch.no_grad()
def _score_sequence_candidates_with_cache(active_model, example, candidate_orders):
    """Verified Qwen2-VL mRoPE-aware shared-prefix KV-cache scoring."""
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "right"

    prompt_cache = None
    base_cache_length = None

    try:
        device = model_device(active_model)
        prompt = processor.apply_chat_template(
            make_messages(example, include_answer=False),
            tokenize=False,
            add_generation_prompt=True,
        )
        images = [load_rgb(path) for path in example["image_paths"]]

        prompt_inputs = processor(
            text=[prompt],
            images=[images],
            return_tensors="pt",
        )
        prompt_inputs = _batch_to_device(prompt_inputs, active_model)

        prompt_outputs = active_model(
            **prompt_inputs,
            use_cache=True,
            return_dict=True,
        )

        prompt_cache = prompt_outputs.past_key_values
        if prompt_cache is None:
            raise RuntimeError("Model did not return past_key_values.")
        if not hasattr(prompt_cache, "get_seq_length"):
            raise RuntimeError(
                f"Cache lacks get_seq_length(): {type(prompt_cache)}"
            )
        if not hasattr(prompt_cache, "crop"):
            raise RuntimeError(
                f"Cache lacks crop(): {type(prompt_cache)}"
            )

        base_cache_length = int(prompt_cache.get_seq_length())
        prompt_attention_mask = prompt_inputs["attention_mask"]
        prompt_len = int(prompt_attention_mask[0].sum().item())
        first_logits = prompt_outputs.logits[0, prompt_len - 1]
        prompt_rope_deltas = getattr(prompt_outputs, "rope_deltas", None)

        scores = {}

        for order in candidate_orders:
            # The cache object is mutable; reset it before each candidate.
            prompt_cache.crop(base_cache_length)

            answer_text = compact_order(order)
            target_ids = target_ids_after_prompt(
                prompt,
                answer_text,
                device,
            )
            candidate_logits = [first_logits]

            if target_ids.numel() > 1:
                previous_ids = target_ids[:-1].unsqueeze(0)

                continuation_mask = torch.ones(
                    (1, previous_ids.shape[1]),
                    dtype=prompt_attention_mask.dtype,
                    device=device,
                )
                attention_mask = torch.cat(
                    [prompt_attention_mask, continuation_mask],
                    dim=1,
                )

                cache_position = torch.arange(
                    base_cache_length,
                    base_cache_length + previous_ids.shape[1],
                    dtype=torch.long,
                    device=device,
                )

                position_ids = build_qwen2vl_continuation_position_ids(
                    prompt_attention_mask=prompt_attention_mask,
                    rope_deltas=prompt_rope_deltas,
                    continuation_length=previous_ids.shape[1],
                    device=device,
                )

                continuation_outputs = active_model(
                    input_ids=previous_ids,
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    past_key_values=prompt_cache,
                    cache_position=cache_position,
                    use_cache=True,
                    return_dict=True,
                    pixel_values=None,
                    pixel_values_videos=None,
                    image_grid_thw=None,
                    video_grid_thw=None,
                )

                candidate_logits.extend(
                    continuation_outputs.logits[0, token_index]
                    for token_index in range(
                        continuation_outputs.logits.shape[1]
                    )
                )

            logits = torch.stack(
                candidate_logits[:target_ids.numel()],
                dim=0,
            )
            token_log_probs = (
                torch.log_softmax(logits.float(), dim=-1)
                .gather(1, target_ids[:, None])
                .squeeze(1)
            )

            scores[" ".join(map(str, order))] = float(
                token_log_probs.mean().item()
            )

        return scores

    finally:
        if (
            prompt_cache is not None
            and base_cache_length is not None
            and hasattr(prompt_cache, "crop")
        ):
            prompt_cache.crop(base_cache_length)

        processor.tokenizer.padding_side = old_padding_side

@torch.no_grad()
def score_sequence_candidates(active_model, example, candidate_orders, batch_size=None):
    if USE_KV_CACHE_SEQUENCE_SCORING:
        try:
            return _score_sequence_candidates_with_cache(active_model, example, list(candidate_orders))
        except Exception as exc:
            print("KV-cache sequence scoring failed; falling back to recompute scoring:", repr(exc))
    return _score_sequence_candidates_recompute(active_model, example, candidate_orders, batch_size=batch_size)


def softmax_scores(scores, temperature=1.0):
    keys = list(scores.keys())
    values = np.array([scores[k] for k in keys], dtype=np.float64) / temperature
    values = values - values.max()
    probs = np.exp(values)
    probs = probs / probs.sum()
    return {k: float(v) for k, v in zip(keys, probs)}


def parse_order_key(key):
    return tuple(int(x) for x in str(key).split())


def pair_logit(p, eps=1e-6):
    p = min(max(float(p), eps), 1.0 - eps)
    return math.log(p / (1.0 - p))


def combine_bidirectional_pair(p_forward, p_reverse, eps=1e-6):
    combined_logit = 0.5 * (pair_logit(p_forward, eps) - pair_logit(p_reverse, eps))
    return 1.0 / (1.0 + math.exp(-combined_logit))


def normalized(scores):
    total = sum(max(float(v), 0.0) for v in scores.values())
    if total <= 0:
        return {int(k): 1.0 / len(scores) for k in scores}
    return {int(k): max(float(v), 0.0) / total for k, v in scores.items()}


def position_marginal(order_probs, position, candidates):
    out = {int(c): 0.0 for c in candidates}
    for key, prob in order_probs.items():
        order = parse_order_key(key)
        if order[position] in out:
            out[order[position]] += float(prob)
    return normalized(out)


def endpoint_probs_from_pair(pair_probs, candidates, mode):
    candidates = [int(x) for x in candidates]
    if mode == "first":
        return normalized({i: sum(pair_probs[f"{i}>{j}"]["combined_prob"] for j in candidates if j != i) for i in candidates})
    return normalized({i: sum(pair_probs[f"{j}>{i}"]["combined_prob"] for j in candidates if j != i) for i in candidates})


def fuse_three_raw(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair):
    candidates = sorted(set(order_signal) | set(endpoint_signal) | set(pair_signal))
    return {
        int(i): (
            w_order * order_signal.get(i, 0.0)
            + w_endpoint * endpoint_signal.get(i, 0.0)
            + w_pair * pair_signal.get(i, 0.0)
        )
        for i in candidates
    }


def fuse_three(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair):
    return normalized(fuse_three_raw(order_signal, endpoint_signal, pair_signal, w_order, w_endpoint, w_pair))


def normalize_endpoint_slots(first_scores, last_scores):
    raw = {f"first:{int(k)}": max(float(v), 0.0) for k, v in first_scores.items()}
    raw.update({f"last:{int(k)}": max(float(v), 0.0) for k, v in last_scores.items()})
    total = sum(raw.values())
    if total <= 0:
        return {key: 1.0 / len(raw) for key in raw}
    return {key: value / total for key, value in raw.items()}


def top_with_margin(scores):
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_value = float(ranked[0][1])
    margin = top_value - (float(ranked[1][1]) if len(ranked) > 1 else 0.0)
    return int(ranked[0][0]), top_value, margin


def score_sample(active_model, row, image_root=TRAIN_IMAGE_DIR, has_gold=True):
    gold_order = order_to_sequence(row["Answer_list"]) if has_gold else None
    sample_id = str(row["Id"])

    endpoint_examples = [
        make_eval_example(row, "first", image_root=image_root),
        make_eval_example(row, "last", image_root=image_root),
    ]
    first_probs, last_probs = score_next_token_examples(active_model, endpoint_examples, DIGIT_TOKEN_IDS, batch_size=2)
    first_probs = {str(k): v for k, v in first_probs.items()}
    last_probs = {str(k): v for k, v in last_probs.items()}

    pair_requests = []
    pair_examples = []
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        for left, right in [(a, b), (b, a)]:
            pair_requests.append((left, right))
            pair_examples.append(make_eval_example(row, "pairwise", pair=(left, right), image_root=image_root))
    pair_outputs = score_next_token_examples(active_model, pair_examples, AB_TOKEN_IDS, batch_size=NEXT_TOKEN_BATCH_SIZE)
    pair_lookup = {pair: output for pair, output in zip(pair_requests, pair_outputs)}

    pair_probs = {}
    for i, j in PAIR_INDICES:
        a, b = i + 1, j + 1
        p_forward = pair_lookup[(a, b)]["A"]
        p_reverse = pair_lookup[(b, a)]["A"]
        combined = combine_bidirectional_pair(p_forward, p_reverse)
        pair_probs[f"{a}>{b}"] = {"forward_prob": float(p_forward), "reverse_prob": float(p_reverse), "combined_prob": float(combined), "swap_inconsistency": float(abs(p_forward + p_reverse - 1.0))}
        pair_probs[f"{b}>{a}"] = {"forward_prob": float(1.0 - p_forward), "reverse_prob": float(1.0 - p_reverse), "combined_prob": float(1.0 - combined), "swap_inconsistency": float(abs(p_forward + p_reverse - 1.0))}

    order_scores = score_sequence_candidates(active_model, make_eval_example(row, "order", image_root=image_root), PERMUTATIONS)
    order_probs = softmax_scores(order_scores)
    best_order_key = max(order_scores, key=order_scores.get)
    teacher_forced_order = list(parse_order_key(best_order_key))

    if RUN_DIRECT_GENERATION:
        direct_text = generate_text(active_model, make_eval_example(row, "order", image_root=image_root))
        direct_order = parse_compact_order(direct_text, 4)
    else:
        direct_text = best_order_key
        direct_order = teacher_forced_order

    return {
        "sample_id": sample_id,
        "gold_order": gold_order,
        "direct_order": direct_order,
        "direct_text": direct_text,
        "first_probs": first_probs,
        "last_probs": last_probs,
        "bidirectional_pair_probs": pair_probs,
        "order_24_scores": order_scores,
        "order_24_probs": order_probs,
    }


def conditional_decode(active_model, row, scored, image_root=TRAIN_IMAGE_DIR):
    frames = [1, 2, 3, 4]
    order_probs = scored["order_24_probs"]
    pair_probs = scored["bidirectional_pair_probs"]
    order_first = position_marginal(order_probs, 0, frames)
    order_last = position_marginal(order_probs, 3, frames)
    first_head = normalized({int(k): v for k, v in scored["first_probs"].items()})
    last_head = normalized({int(k): v for k, v in scored["last_probs"].items()})
    pair_first = endpoint_probs_from_pair(pair_probs, frames, "first")
    pair_last = endpoint_probs_from_pair(pair_probs, frames, "last")
    raw_first = fuse_three_raw(order_first, first_head, pair_first, 0.45, 0.20, 0.35)
    raw_last = fuse_three_raw(order_last, last_head, pair_last, 0.45, 0.20, 0.35)
    fused_first = normalized(raw_first)
    fused_last = normalized(raw_last)
    endpoint_slot_probs = normalize_endpoint_slots(raw_first, raw_last)
    first_candidate, first_top1, first_margin = top_with_margin(fused_first)
    last_candidate, last_top1, last_margin = top_with_margin(fused_last)
    first_priority = endpoint_slot_probs[f"first:{first_candidate}"]
    last_priority = endpoint_slot_probs[f"last:{last_candidate}"]

    if first_priority >= last_priority:
        first = first_candidate
        remaining = [x for x in frames if x != first]
        candidates = list(itertools.permutations(remaining))
        cond_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_first", fixed_first=first, image_root=image_root), candidates)
        cond_probs = softmax_scores(cond_scores)
        conditional_order_last = position_marginal(cond_probs, -1, remaining)
        conditional_fused = fuse_three(conditional_order_last, normalized({i: float(scored["last_probs"][str(i)]) for i in remaining}), endpoint_probs_from_pair(pair_probs, remaining, "last"), 0.45, 0.15, 0.40)
        last = max(conditional_fused, key=conditional_fused.get)
        fixed_first_selected = True
    else:
        last = last_candidate
        remaining = [x for x in frames if x != last]
        candidates = list(itertools.permutations(remaining))
        cond_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_last", fixed_last=last, image_root=image_root), candidates)
        cond_probs = softmax_scores(cond_scores)
        conditional_order_first = position_marginal(cond_probs, 0, remaining)
        conditional_fused = fuse_three(conditional_order_first, normalized({i: float(scored["first_probs"][str(i)]) for i in remaining}), endpoint_probs_from_pair(pair_probs, remaining, "first"), 0.45, 0.15, 0.40)
        first = max(conditional_fused, key=conditional_fused.get)
        fixed_first_selected = False

    middle = [x for x in frames if x not in {first, last}]
    middle_candidates = list(itertools.permutations(middle))
    middle_scores = score_sequence_candidates(active_model, make_eval_example(row, "fixed_endpoints", fixed_first=first, fixed_last=last, image_root=image_root), middle_candidates)
    middle_probs = softmax_scores(middle_scores)
    a, b = middle
    order_a_before_b = middle_probs.get(f"{a} {b}", 0.0)
    pair_a_before_b = pair_probs[f"{a}>{b}"]["combined_prob"]
    score_a_before_b = 0.60 * order_a_before_b + 0.40 * pair_a_before_b
    middle_order = [a, b] if score_a_before_b >= 0.5 else [b, a]
    final_order = [first, *middle_order, last]
    scored.update({
        "raw_fused_first": raw_first,
        "raw_fused_last": raw_last,
        "fused_first": fused_first,
        "fused_last": fused_last,
        "endpoint_slot_probs": endpoint_slot_probs,
        "first_top1": first_top1,
        "last_top1": last_top1,
        "first_margin": first_margin,
        "last_margin": last_margin,
        "first_priority": first_priority,
        "last_priority": last_priority,
        "first_endpoint_selected": fixed_first_selected,
        "fixed_endpoint": first if fixed_first_selected else last,
        "selected_other_endpoint": last if fixed_first_selected else first,
        "conditional_order_6_probs": cond_probs,
        "conditional_middle_probs": middle_probs,
        "final_order": final_order,
    })
    return scored


def order_metric_row(pred_order, gold_order):
    if pred_order is None or gold_order is None:
        return {"exact": 0.0, "first": 0.0, "last": 0.0, "both_endpoints": 0.0, "position": 0.0, "relative_pair": 0.0}
    ranks_p = {x: i for i, x in enumerate(pred_order)}
    ranks_g = {x: i for i, x in enumerate(gold_order)}
    return {
        "exact": float(pred_order == gold_order),
        "first": float(pred_order[0] == gold_order[0]),
        "last": float(pred_order[-1] == gold_order[-1]),
        "both_endpoints": float(pred_order[0] == gold_order[0] and pred_order[-1] == gold_order[-1]),
        "position": float(np.mean([p == g for p, g in zip(pred_order, gold_order)])),
        "relative_pair": float(np.mean([(ranks_p[a] < ranks_p[b]) == (ranks_g[a] < ranks_g[b]) for a, b in itertools.combinations([1, 2, 3, 4], 2)])),
    }


In [5]:

# 5) Exhaustive probability export helpers
FRAMES = [1, 2, 3, 4]
BASE_WEIGHTS = {"order": 0.10, "endpoint": 0.30, "pairwise": 0.60}
EPS = 1e-12

def norm_dict(values):
    values = {int(k): max(float(v), 0.0) for k, v in values.items()}
    total = sum(values.values())
    if total <= 0:
        return {k: 1.0 / len(values) for k in values}
    return {k: v / total for k, v in values.items()}

def entropy_stats(values):
    probs = np.array(list(norm_dict(values).values()), dtype=np.float64)
    entropy = float(-(probs * np.log(probs + EPS)).sum())
    normalized_entropy = float(entropy / np.log(len(probs))) if len(probs) > 1 else 0.0
    sorted_probs = np.sort(probs)[::-1]
    top1 = float(sorted_probs[0])
    top2 = float(sorted_probs[1]) if len(sorted_probs) > 1 else 0.0
    return top1, top2, float(top1 - top2), entropy, normalized_entropy

def argmax_key(values):
    return int(max(values, key=values.get))

def order_position_marginal(order_probs, position):
    result = {frame: 0.0 for frame in FRAMES}
    for order_key, probability in order_probs.items():
        order = [int(x) for x in str(order_key).split()]
        result[order[position]] += float(probability)
    return norm_dict(result)

def endpoint_pair_from_order(order_probs):
    joint = {(first, last): 0.0 for first in FRAMES for last in FRAMES if first != last}
    for order_key, probability in order_probs.items():
        order = [int(x) for x in str(order_key).split()]
        joint[(order[0], order[-1])] += float(probability)
    total = sum(joint.values())
    return {key: value / total for key, value in joint.items()}

def pair_endpoint_distributions(pair_probs):
    early = {}
    late = {}
    for frame in FRAMES:
        early[frame] = sum(float(pair_probs[f"{frame}>{other}"]["combined_prob"]) for other in FRAMES if other != frame)
        late[frame] = sum(float(pair_probs[f"{other}>{frame}"]["combined_prob"]) for other in FRAMES if other != frame)
    return norm_dict(early), norm_dict(late)

def combine_endpoint_distribution(order_dist, endpoint_dist, pair_dist, weights=BASE_WEIGHTS):
    raw = {
        frame: (
            weights["order"] * float(order_dist[frame])
            + weights["endpoint"] * float(endpoint_dist[frame])
            + weights["pairwise"] * float(pair_dist[frame])
        )
        for frame in FRAMES
    }
    return norm_dict(raw)

def binary_entropy(p):
    p = min(max(float(p), EPS), 1.0 - EPS)
    h = -(p * math.log(p) + (1.0 - p) * math.log(1.0 - p))
    return float(h), float(h / math.log(2.0))

def flatten_dict(prefix, values, out):
    for key, value in values.items():
        out[f"{prefix}_{key}"] = float(value)

def score_all_conditionals(active_model, row, scored, image_root):
    """Score all fixed-first, fixed-last, and fixed-endpoint alternatives.

    This ensures later decoding can choose any endpoint without rerunning the model.
    """
    exhaustive = {
        "fixed_first": {},
        "fixed_last": {},
        "fixed_endpoints_middle": {},
    }

    # Every possible fixed first -> all 3! remaining orders.
    for first in FRAMES:
        remaining = [x for x in FRAMES if x != first]
        candidates = list(itertools.permutations(remaining))
        scores = score_sequence_candidates(
            active_model,
            make_eval_example(row, "fixed_first", fixed_first=first, image_root=image_root),
            candidates,
        )
        exhaustive["fixed_first"][str(first)] = softmax_scores(scores)

    # Every possible fixed last -> all 3! remaining orders.
    for last in FRAMES:
        remaining = [x for x in FRAMES if x != last]
        candidates = list(itertools.permutations(remaining))
        scores = score_sequence_candidates(
            active_model,
            make_eval_example(row, "fixed_last", fixed_last=last, image_root=image_root),
            candidates,
        )
        exhaustive["fixed_last"][str(last)] = softmax_scores(scores)

    # Every possible ordered endpoint pair -> the two possible middle orders.
    for first in FRAMES:
        for last in FRAMES:
            if first == last:
                continue
            middle = [x for x in FRAMES if x not in {first, last}]
            candidates = list(itertools.permutations(middle))
            scores = score_sequence_candidates(
                active_model,
                make_eval_example(
                    row,
                    "fixed_endpoints",
                    fixed_first=first,
                    fixed_last=last,
                    image_root=image_root,
                ),
                candidates,
            )
            exhaustive["fixed_endpoints_middle"][f"{first}>{last}"] = softmax_scores(scores)
    return exhaustive

def build_raw_row(record, has_gold):
    row = {"sample_id": record["sample_id"]}
    if has_gold:
        row["gold_order"] = " ".join(map(str, record["gold_order"]))
        row["gold_first"] = int(record["gold_order"][0])
        row["gold_last"] = int(record["gold_order"][-1])

    order_probs = {str(k): float(v) for k, v in record["order_24_probs"].items()}
    for order_key, probability in order_probs.items():
        row[f"p_order_{order_key.replace(' ', '')}"] = probability

    # All four order position marginals.
    order_positions = {}
    for position in range(4):
        dist = order_position_marginal(order_probs, position)
        order_positions[position + 1] = dist
        flatten_dict(f"p_order_pos{position + 1}_frame", dist, row)
        top1, top2, margin, entropy, hn = entropy_stats(dist)
        row[f"order_pos{position + 1}_prediction"] = argmax_key(dist)
        row[f"order_pos{position + 1}_top1_probability"] = top1
        row[f"order_pos{position + 1}_top2_probability"] = top2
        row[f"order_pos{position + 1}_margin"] = margin
        row[f"order_pos{position + 1}_entropy"] = entropy
        row[f"order_pos{position + 1}_normalized_entropy"] = hn

    endpoint_first = norm_dict(record["first_probs"])
    endpoint_last = norm_dict(record["last_probs"])
    flatten_dict("p_endpoint_first", endpoint_first, row)
    flatten_dict("p_endpoint_last", endpoint_last, row)
    for name, dist in [("endpoint_first", endpoint_first), ("endpoint_last", endpoint_last)]:
        top1, top2, margin, entropy, hn = entropy_stats(dist)
        row[f"{name}_prediction"] = argmax_key(dist)
        row[f"{name}_top1_probability"] = top1
        row[f"{name}_top2_probability"] = top2
        row[f"{name}_margin"] = margin
        row[f"{name}_entropy"] = entropy
        row[f"{name}_normalized_entropy"] = hn

    endpoint_joint = endpoint_pair_from_order(order_probs)
    for (first, last), probability in endpoint_joint.items():
        row[f"p_endpoint_pair_{first}_{last}"] = probability
    joint_probs = {f"{a}>{b}": p for (a, b), p in endpoint_joint.items()}
    jt1, jt2, jmargin, je, jhn = entropy_stats(joint_probs)
    joint_pred = max(endpoint_joint, key=endpoint_joint.get)
    row["endpoint_pair_prediction"] = f"{joint_pred[0]}>{joint_pred[1]}"
    row["endpoint_pair_top1_probability"] = jt1
    row["endpoint_pair_margin"] = jmargin
    row["endpoint_pair_entropy"] = je
    row["endpoint_pair_normalized_entropy"] = jhn

    pair_probs = record["bidirectional_pair_probs"]
    for a, b in itertools.combinations(FRAMES, 2):
        row[f"p_pair_{a}_before_{b}"] = float(pair_probs[f"{a}>{b}"]["combined_prob"])
        row[f"p_pair_{b}_before_{a}"] = float(pair_probs[f"{b}>{a}"]["combined_prob"])
        row[f"pair_swap_inconsistency_{a}_{b}"] = float(pair_probs[f"{a}>{b}"]["swap_inconsistency"])

    pair_first, pair_last = pair_endpoint_distributions(pair_probs)
    flatten_dict("p_pairwise_first", pair_first, row)
    flatten_dict("p_pairwise_last", pair_last, row)
    for name, dist in [("pairwise_first", pair_first), ("pairwise_last", pair_last)]:
        top1, top2, margin, entropy, hn = entropy_stats(dist)
        row[f"{name}_prediction"] = argmax_key(dist)
        row[f"{name}_top1_probability"] = top1
        row[f"{name}_top2_probability"] = top2
        row[f"{name}_margin"] = margin
        row[f"{name}_entropy"] = entropy
        row[f"{name}_normalized_entropy"] = hn

    # Base 10:30:60 endpoint decisions.
    fused_first = combine_endpoint_distribution(order_positions[1], endpoint_first, pair_first)
    fused_last = combine_endpoint_distribution(order_positions[4], endpoint_last, pair_last)
    flatten_dict("p_base_fused_first", fused_first, row)
    flatten_dict("p_base_fused_last", fused_last, row)
    row["base_first_prediction"] = argmax_key(fused_first)
    row["base_last_prediction"] = argmax_key(fused_last)

    # Save every conditional distribution without assuming which endpoint is selected later.
    exhaustive = record["exhaustive_conditionals"]
    for fixed_first, dist in exhaustive["fixed_first"].items():
        for suffix_order, probability in dist.items():
            row[f"p_cond_fixed_first_{fixed_first}_suffix_{suffix_order.replace(' ', '')}"] = float(probability)
        # Marginal last for each possible fixed first.
        last_dist = {frame: 0.0 for frame in FRAMES if frame != int(fixed_first)}
        for suffix_order, probability in dist.items():
            suffix = [int(x) for x in suffix_order.split()]
            last_dist[suffix[-1]] += float(probability)
        last_dist = norm_dict(last_dist)
        flatten_dict(f"p_cond_last_given_first_{fixed_first}", last_dist, row)
        _, _, margin, entropy, hn = entropy_stats(last_dist)
        row[f"cond_last_given_first_{fixed_first}_prediction"] = argmax_key(last_dist)
        row[f"cond_last_given_first_{fixed_first}_margin"] = margin
        row[f"cond_last_given_first_{fixed_first}_normalized_entropy"] = hn

    for fixed_last, dist in exhaustive["fixed_last"].items():
        for prefix_order, probability in dist.items():
            row[f"p_cond_fixed_last_{fixed_last}_prefix_{prefix_order.replace(' ', '')}"] = float(probability)
        first_dist = {frame: 0.0 for frame in FRAMES if frame != int(fixed_last)}
        for prefix_order, probability in dist.items():
            prefix = [int(x) for x in prefix_order.split()]
            first_dist[prefix[0]] += float(probability)
        first_dist = norm_dict(first_dist)
        flatten_dict(f"p_cond_first_given_last_{fixed_last}", first_dist, row)
        _, _, margin, entropy, hn = entropy_stats(first_dist)
        row[f"cond_first_given_last_{fixed_last}_prediction"] = argmax_key(first_dist)
        row[f"cond_first_given_last_{fixed_last}_margin"] = margin
        row[f"cond_first_given_last_{fixed_last}_normalized_entropy"] = hn

    # Every endpoint pair's two possible middle orders.
    for endpoint_key, dist in exhaustive["fixed_endpoints_middle"].items():
        first, last = map(int, endpoint_key.split(">"))
        middle = [x for x in FRAMES if x not in {first, last}]
        a, b = middle
        p_ab = float(dist.get(f"{a} {b}", 0.0))
        p_ba = float(dist.get(f"{b} {a}", 0.0))
        total = p_ab + p_ba
        p_ab = p_ab / total if total > 0 else 0.5
        p_ba = 1.0 - p_ab
        row[f"p_middle_order_given_{first}_{last}_{a}{b}"] = p_ab
        row[f"p_middle_order_given_{first}_{last}_{b}{a}"] = p_ba
        row[f"p_middle_pairwise_given_{first}_{last}_{a}{b}"] = float(pair_probs[f"{a}>{b}"]["combined_prob"])
        row[f"p_middle_pairwise_given_{first}_{last}_{b}{a}"] = float(pair_probs[f"{b}>{a}"]["combined_prob"])
        _, hn_order = binary_entropy(p_ab)
        _, hn_pair = binary_entropy(float(pair_probs[f"{a}>{b}"]["combined_prob"]))
        row[f"middle_order_normalized_entropy_given_{first}_{last}"] = hn_order
        row[f"middle_pairwise_normalized_entropy_given_{first}_{last}"] = hn_pair

    return row

def build_readable_row(raw):
    out = {
        "sample_id": raw["sample_id"],
        "gold_order": raw.get("gold_order", ""),
        "gold_first": raw.get("gold_first", ""),
        "gold_last": raw.get("gold_last", ""),
    }
    # First: exactly the three requested methods.
    for prefix, source in [
        ("first_order", "order_pos1"),
        ("first_endpoint", "endpoint_first"),
        ("first_pairwise", "pairwise_first"),
        ("last_order", "order_pos4"),
        ("last_endpoint", "endpoint_last"),
        ("last_pairwise", "pairwise_last"),
    ]:
        out[f"{prefix}_prediction"] = raw[f"{source}_prediction"]
        out[f"{prefix}_probability"] = raw[f"{source}_top1_probability"]
        out[f"{prefix}_margin"] = raw[f"{source}_margin"]
        out[f"{prefix}_normalized_entropy"] = raw[f"{source}_normalized_entropy"]

    first_preds = [out["first_order_prediction"], out["first_endpoint_prediction"], out["first_pairwise_prediction"]]
    last_preds = [out["last_order_prediction"], out["last_endpoint_prediction"], out["last_pairwise_prediction"]]
    out["first_threeway"] = " | ".join(map(str, first_preds))
    out["last_threeway"] = " | ".join(map(str, last_preds))
    out["first_threeway_all_agree"] = int(len(set(first_preds)) == 1)
    out["last_threeway_all_agree"] = int(len(set(last_preds)) == 1)

    # Base endpoint decisions using 10:30:60, shown as the current default only.
    first = int(raw["base_first_prediction"])
    last = int(raw["base_last_prediction"])
    if first == last:
        # Resolve collision using the stronger fused probability and the best non-colliding alternative.
        first_probs = {i: raw[f"p_base_fused_first_{i}"] for i in FRAMES}
        last_probs = {i: raw[f"p_base_fused_last_{i}"] for i in FRAMES}
        if first_probs[first] >= last_probs[last]:
            last = max((i for i in FRAMES if i != first), key=lambda i: last_probs[i])
        else:
            first = max((i for i in FRAMES if i != last), key=lambda i: first_probs[i])

    out["base_first_10_30_60"] = first
    out["base_last_10_30_60"] = last

    # Show both possible sequential choices, so the later decoder can choose first-first or last-first.
    out["conditional_last_given_selected_first_prediction"] = raw[f"cond_last_given_first_{first}_prediction"]
    out["conditional_last_given_selected_first_margin"] = raw[f"cond_last_given_first_{first}_margin"]
    out["conditional_last_given_selected_first_normalized_entropy"] = raw[f"cond_last_given_first_{first}_normalized_entropy"]
    out["conditional_first_given_selected_last_prediction"] = raw[f"cond_first_given_last_{last}_prediction"]
    out["conditional_first_given_selected_last_margin"] = raw[f"cond_first_given_last_{last}_margin"]
    out["conditional_first_given_selected_last_normalized_entropy"] = raw[f"cond_first_given_last_{last}_normalized_entropy"]

    # Readable middle comparison for the current base endpoint pair.
    middle = [x for x in FRAMES if x not in {first, last}]
    a, b = middle
    out["middle_frames"] = f"{a},{b}"
    out["middle_order_ab"] = f"{a}>{b}"
    out["middle_order_probability_ab"] = raw[f"p_middle_order_given_{first}_{last}_{a}{b}"]
    out["middle_order_probability_ba"] = raw[f"p_middle_order_given_{first}_{last}_{b}{a}"]
    out["middle_order_normalized_entropy"] = raw[f"middle_order_normalized_entropy_given_{first}_{last}"]
    out["middle_pairwise_probability_ab"] = raw[f"p_middle_pairwise_given_{first}_{last}_{a}{b}"]
    out["middle_pairwise_probability_ba"] = raw[f"p_middle_pairwise_given_{first}_{last}_{b}{a}"]
    out["middle_pairwise_normalized_entropy"] = raw[f"middle_pairwise_normalized_entropy_given_{first}_{last}"]

    # Default 10:30:60 endpoint + pair-priority middle prediction.
    order_p = out["middle_order_probability_ab"]
    pair_p = out["middle_pairwise_probability_ab"]
    middle_ab = (0.30 * order_p + 0.70 * pair_p) >= 0.5
    middle_order = [a, b] if middle_ab else [b, a]
    final_order = [first, *middle_order, last]
    out["base_prediction_10_30_60"] = " ".join(map(str, final_order))
    if raw.get("gold_order"):
        out["base_exact"] = int(out["base_prediction_10_30_60"] == raw["gold_order"])
    return out


# ------------------------------------------------------------------
# Lightweight KV-cache equivalence check (first sample only)
# ------------------------------------------------------------------
KV_CHECK_SAMPLES = 1
KV_RAW_SCORE_WARN = 5e-2
KV_CENTERED_SCORE_WARN = 5e-2
KV_PROB_ATOL = 3e-3

def _candidate_key(candidate):
    if isinstance(candidate, str):
        return candidate
    return " ".join(map(str, candidate))

def _scores_to_ordered_array(scores, candidates):
    return np.array(
        [float(scores[_candidate_key(candidate)]) for candidate in candidates],
        dtype=np.float64,
    )

def _softmax_array(scores):
    shifted = scores - np.max(scores)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum()

def verify_kv_cache_for_example(active_model, row, image_root):
    """Compare cached and full-recompute scoring on representative candidate sets.

    Checks:
    - all 24 complete orders
    - one fixed-first set of 6 candidates
    - one fixed-last set of 6 candidates
    - one fixed-endpoints middle set of 2 candidates
    """
    checks = []

    def run_check(name, example, candidates):
        cached_scores = _score_sequence_candidates_with_cache(
            active_model,
            example,
            candidates,
        )
        recompute_scores = _score_sequence_candidates_recompute(
            active_model,
            example,
            candidates,
        )

        cached_array = _scores_to_ordered_array(cached_scores, candidates)
        recompute_array = _scores_to_ordered_array(recompute_scores, candidates)
        cached_probs = _softmax_array(cached_array)
        recompute_probs = _softmax_array(recompute_array)

        raw_score_diff = float(np.max(np.abs(cached_array - recompute_array)))

        # A nearly constant score offset does not affect softmax or decoding.
        cached_centered = cached_array - cached_array.mean()
        recompute_centered = recompute_array - recompute_array.mean()
        centered_score_diff = float(
            np.max(np.abs(cached_centered - recompute_centered))
        )

        prob_diff = float(np.max(np.abs(cached_probs - recompute_probs)))
        cached_rank = np.argsort(-cached_array)
        recompute_rank = np.argsort(-recompute_array)

        result = {
            "name": name,
            "candidate_count": len(candidates),
            "max_abs_raw_score_diff": raw_score_diff,
            "max_abs_centered_score_diff": centered_score_diff,
            "max_abs_probability_diff": prob_diff,
            "top1_same": bool(cached_rank[0] == recompute_rank[0]),
            "full_ranking_same": bool(np.array_equal(cached_rank, recompute_rank)),
        }
        checks.append(result)

        # Print diagnostics for every representative candidate set.
        print("KV diagnostics:", result)

        # Raw and centered score differences are diagnostics only.
        # They may include a near-common offset that does not alter softmax decoding.
        if raw_score_diff > KV_RAW_SCORE_WARN:
            print(
                f"Warning: large raw score difference in {name}: "
                f"{raw_score_diff:.6g}"
            )
        if centered_score_diff > KV_CENTERED_SCORE_WARN:
            print(
                f"Warning: large centered score difference in {name}: "
                f"{centered_score_diff:.6g}"
            )

        # Hard gates use only quantities that directly affect decoding.
        if prob_diff > KV_PROB_ATOL:
            raise RuntimeError(
                f"KV-cache probability mismatch in {name}: "
                f"max_abs_probability_diff={prob_diff:.6g} > {KV_PROB_ATOL}"
            )
        if not result["top1_same"]:
            raise RuntimeError(f"KV-cache top1 mismatch in {name}")

    # 24 full orders
    full_candidates = list(itertools.permutations(FRAMES))
    run_check(
        "full24",
        make_eval_example(row, "order", image_root=image_root),
        full_candidates,
    )

    # Representative fixed-first
    fixed_first = 1
    remaining = [x for x in FRAMES if x != fixed_first]
    run_check(
        "fixed_first_1",
        make_eval_example(
            row,
            "fixed_first",
            fixed_first=fixed_first,
            image_root=image_root,
        ),
        list(itertools.permutations(remaining)),
    )

    # Representative fixed-last
    fixed_last = 4
    remaining = [x for x in FRAMES if x != fixed_last]
    run_check(
        "fixed_last_4",
        make_eval_example(
            row,
            "fixed_last",
            fixed_last=fixed_last,
            image_root=image_root,
        ),
        list(itertools.permutations(remaining)),
    )

    # Representative middle pair
    first, last = 1, 4
    middle = [x for x in FRAMES if x not in {first, last}]
    run_check(
        "fixed_endpoints_1_4",
        make_eval_example(
            row,
            "fixed_endpoints",
            fixed_first=first,
            fixed_last=last,
            image_root=image_root,
        ),
        list(itertools.permutations(middle)),
    )

    return checks


# 6) KV-cache 동일성 검증 — 첫 샘플 1개

전체 평가를 시작하기 전에 대표 후보군에서 KV-cache 점수와 전체 재계산 점수를 비교합니다.

- full24 후보 24개
- fixed-first 후보 6개
- fixed-last 후보 6개
- fixed-endpoints middle 후보 2개

점수·확률 차이가 허용 범위를 넘거나 top1이 달라지면 평가를 중단합니다.


In [6]:

# Load model once for KV validation and the subsequent eval100 run.
active_model = load_eval_model(BEST_ADAPTER_DIR)

kv_validation_rows = validation_df.sample(
    n=min(100, len(validation_df)),
    random_state=SEED + 1,
).reset_index(drop=True).head(KV_CHECK_SAMPLES)

kv_check_rows = []
for sample_index, (_, row) in enumerate(
    tqdm(
        kv_validation_rows.iterrows(),
        total=len(kv_validation_rows),
        desc="KV-cache equivalence check (first sample only)",
    ),
    start=1,
):
    sample_results = verify_kv_cache_for_example(
        active_model,
        row,
        image_root=TRAIN_IMAGE_DIR,
    )
    for result in sample_results:
        kv_check_rows.append({
            "sample_index": sample_index,
            "sample_id": str(row["Id"]),
            **result,
        })

kv_check_df = pd.DataFrame(kv_check_rows)
display(kv_check_df)

KV_CHECK_CSV = EVAL_DIR / "kv_cache_equivalence_first3_mrope_first1_v4.csv"
kv_check_df.to_csv(KV_CHECK_CSV, index=False)

print("KV-cache equivalence check (first sample only) passed.")
print("saved:", KV_CHECK_CSV)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


KV-cache equivalence check (first sample only):   0%|          | 0/1 [00:00<?, ?it/s]

KV diagnostics: {'name': 'full24', 'candidate_count': 24, 'max_abs_raw_score_diff': 0.011048078536987305, 'max_abs_centered_score_diff': 0.00985618432362867, 'max_abs_probability_diff': 0.0008513089808503405, 'top1_same': True, 'full_ranking_same': False}
KV diagnostics: {'name': 'fixed_first_1', 'candidate_count': 6, 'max_abs_raw_score_diff': 0.019457340240478516, 'max_abs_centered_score_diff': 0.015970289707183838, 'max_abs_probability_diff': 0.0013241569370930906, 'top1_same': True, 'full_ranking_same': True}
KV diagnostics: {'name': 'fixed_last_4', 'candidate_count': 6, 'max_abs_raw_score_diff': 0.0037285685539245605, 'max_abs_centered_score_diff': 0.0030620073278746496, 'max_abs_probability_diff': 0.0007765079709198264, 'top1_same': True, 'full_ranking_same': True}
KV diagnostics: {'name': 'fixed_endpoints_1_4', 'candidate_count': 2, 'max_abs_raw_score_diff': 0.0030313804745674133, 'max_abs_centered_score_diff': 0.0015157051384449005, 'max_abs_probability_diff': 0.0007417971930905

,sample_index,sample_id,name,candidate_count,max_abs_raw_score_diff,max_abs_centered_score_diff,max_abs_probability_diff,top1_same,full_ranking_same
0,1,x6AwMw,full24,24,0.011048,0.009856,0.000851,True,False
1,1,x6AwMw,fixed_first_1,6,0.019457,0.015970,0.001324,True,True
2,1,x6AwMw,fixed_last_4,6,0.003729,0.003062,0.000777,True,True
3,1,x6AwMw,fixed_endpoints_1_4,2,0.003031,0.001516,0.000742,True,True


KV-cache equivalence check (first sample only) passed.
saved: /content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/kv_cache_equivalence_first3_mrope_first1_v4.csv


In [7]:

# 7) Run fixed eval100, save exhaustive raw + readable + base metrics
OUTPUT_RUN_DIR = EVAL_DIR / "eval100_step300_endpoint020_mrope_first1_v4"
OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)
CACHE_JSONL = OUTPUT_RUN_DIR / "eval100_records_incremental_mrope_first1_v4.jsonl"
RAW_CSV = OUTPUT_RUN_DIR / "eval100_raw_all_probabilities_mrope_first1_v4.csv"
READABLE_CSV = OUTPUT_RUN_DIR / "eval100_readable_decoding_mrope_first1_v4.csv"
SUMMARY_CSV = OUTPUT_RUN_DIR / "eval100_base_10_30_60_summary_mrope_first1_v4.csv"

full_rows = validation_df.sample(
    n=min(100, len(validation_df)),
    random_state=SEED + 1,
).reset_index(drop=True)

cached = {}
if CACHE_JSONL.exists():
    with CACHE_JSONL.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                cached[str(record["sample_id"])] = record
print("cached samples:", len(cached))

if "active_model" not in globals():
    raise RuntimeError(
        "Run the KV-cache equivalence-check cell first. "
        "The validated model instance is reused for eval100."
    )
if "kv_check_df" not in globals() or kv_check_df.empty:
    raise RuntimeError("KV-cache equivalence check (first sample only) result is missing.")
records = []
for _, row in tqdm(full_rows.iterrows(), total=len(full_rows), desc="step300 eval100 exhaustive"):
    sample_id = str(row["Id"])
    if sample_id in cached:
        record = cached[sample_id]
    else:
        scored = score_sample(active_model, row, image_root=TRAIN_IMAGE_DIR, has_gold=True)
        scored["exhaustive_conditionals"] = score_all_conditionals(
            active_model, row, scored, image_root=TRAIN_IMAGE_DIR
        )
        record = scored
        with CACHE_JSONL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    records.append(record)

raw_rows = [build_raw_row(record, has_gold=True) for record in records]
raw_df = pd.DataFrame(raw_rows)
readable_df = pd.DataFrame([build_readable_row(row) for row in raw_rows])
raw_df.to_csv(RAW_CSV, index=False)
readable_df.to_csv(READABLE_CSV, index=False)

summary = {
    "checkpoint": str(BEST_ADAPTER_DIR),
    "rows": len(readable_df),
    "weights_order": 0.10,
    "weights_endpoint": 0.30,
    "weights_pairwise": 0.60,
    "base_exact": float(readable_df["base_exact"].mean()),
    "first_threeway_all_agree_rate": float(readable_df["first_threeway_all_agree"].mean()),
    "last_threeway_all_agree_rate": float(readable_df["last_threeway_all_agree"].mean()),
}
pd.DataFrame([summary]).to_csv(SUMMARY_CSV, index=False)
display(pd.DataFrame([summary]))
display(readable_df.head(20))
print("saved raw:", RAW_CSV)
print("saved readable:", READABLE_CSV)
print("saved summary:", SUMMARY_CSV)


cached samples: 0


step300 eval100 exhaustive:   0%|          | 0/100 [00:00<?, ?it/s]

ValueError: invalid literal for int() with base 10: '1>2'

In [13]:
# Patch build_raw_row so endpoint probability keys become integers

_original_build_raw_row = build_raw_row

def build_raw_row(record, has_gold):
    # Make a shallow copy so the cached JSON record is not modified in place
    fixed_record = dict(record)

    fixed_record["first_probs"] = {
        int(key): float(value)
        for key, value in record["first_probs"].items()
    }

    fixed_record["last_probs"] = {
        int(key): float(value)
        for key, value in record["last_probs"].items()
    }

    return _original_build_raw_row(
        fixed_record,
        has_gold=has_gold,
    )

print("Patched endpoint probability key types.")

Patched endpoint probability key types.


In [15]:
# ============================================================
# 복구 셀: 기존 100개 캐시를 재추론 없이 CSV로 변환
# - 기존 재귀에 빠진 build_raw_row()는 사용하지 않음
# - build_raw_row_clean()을 새로 정의
# ============================================================

import json
import math
import itertools
import numpy as np
import pandas as pd


def norm_dict_clean(values):
    """키 유형은 유지하고 값만 정규화."""
    cleaned = {
        key: max(float(value), 0.0)
        for key, value in values.items()
    }

    total = sum(cleaned.values())

    if total <= 0:
        return {
            key: 1.0 / len(cleaned)
            for key in cleaned
        }

    return {
        key: value / total
        for key, value in cleaned.items()
    }


def entropy_stats_clean(values):
    normalized = norm_dict_clean(values)

    probs = np.array(
        list(normalized.values()),
        dtype=np.float64,
    )

    entropy = float(
        -(probs * np.log(probs + 1e-12)).sum()
    )

    normalized_entropy = (
        float(entropy / np.log(len(probs)))
        if len(probs) > 1
        else 0.0
    )

    sorted_probs = np.sort(probs)[::-1]

    top1 = float(sorted_probs[0])
    top2 = (
        float(sorted_probs[1])
        if len(sorted_probs) > 1
        else 0.0
    )

    return (
        top1,
        top2,
        float(top1 - top2),
        entropy,
        normalized_entropy,
    )


def argmax_key_clean(values):
    return max(values, key=values.get)


def flatten_dict_clean(prefix, values, output):
    for key, value in values.items():
        output[f"{prefix}_{key}"] = float(value)


def build_raw_row_clean(record, has_gold=True):
    output = {
        "sample_id": str(record["sample_id"]),
    }

    if has_gold:
        gold_order = [
            int(value)
            for value in record["gold_order"]
        ]

        output["gold_order"] = " ".join(
            map(str, gold_order)
        )
        output["gold_first"] = gold_order[0]
        output["gold_last"] = gold_order[-1]

    # --------------------------------------------------------
    # 1. 24개 전체 order 확률
    # --------------------------------------------------------
    order_probs = {
        str(key): float(value)
        for key, value in record["order_24_probs"].items()
    }

    for order_key, probability in order_probs.items():
        compact_key = order_key.replace(" ", "")
        output[f"p_order_{compact_key}"] = probability

    # --------------------------------------------------------
    # 2. 각 자리별 order marginal
    # --------------------------------------------------------
    order_position_distributions = {}

    for position in range(4):
        distribution = {
            frame: 0.0
            for frame in FRAMES
        }

        for order_key, probability in order_probs.items():
            order = [
                int(value)
                for value in order_key.split()
            ]

            distribution[order[position]] += float(
                probability
            )

        distribution = norm_dict_clean(distribution)
        order_position_distributions[position + 1] = (
            distribution
        )

        flatten_dict_clean(
            f"p_order_pos{position + 1}_frame",
            distribution,
            output,
        )

        (
            top1,
            top2,
            margin,
            entropy,
            normalized_entropy,
        ) = entropy_stats_clean(distribution)

        output[
            f"order_pos{position + 1}_prediction"
        ] = int(argmax_key_clean(distribution))

        output[
            f"order_pos{position + 1}_top1_probability"
        ] = top1

        output[
            f"order_pos{position + 1}_top2_probability"
        ] = top2

        output[
            f"order_pos{position + 1}_margin"
        ] = margin

        output[
            f"order_pos{position + 1}_entropy"
        ] = entropy

        output[
            f"order_pos{position + 1}_normalized_entropy"
        ] = normalized_entropy

    # --------------------------------------------------------
    # 3. 직접 물어본 endpoint first / last
    # JSON에서 "1" 형태로 복원된 키를 int로 변경
    # --------------------------------------------------------
    endpoint_first = norm_dict_clean({
        int(key): float(value)
        for key, value in record["first_probs"].items()
    })

    endpoint_last = norm_dict_clean({
        int(key): float(value)
        for key, value in record["last_probs"].items()
    })

    for name, distribution in [
        ("endpoint_first", endpoint_first),
        ("endpoint_last", endpoint_last),
    ]:
        flatten_dict_clean(
            f"p_{name}",
            distribution,
            output,
        )

        (
            top1,
            top2,
            margin,
            entropy,
            normalized_entropy,
        ) = entropy_stats_clean(distribution)

        output[f"{name}_prediction"] = int(
            argmax_key_clean(distribution)
        )
        output[f"{name}_top1_probability"] = top1
        output[f"{name}_top2_probability"] = top2
        output[f"{name}_margin"] = margin
        output[f"{name}_entropy"] = entropy
        output[
            f"{name}_normalized_entropy"
        ] = normalized_entropy

    # --------------------------------------------------------
    # 4. Full-order에서 계산한 endpoint pair 12개
    # --------------------------------------------------------
    endpoint_joint = {
        (first, last): 0.0
        for first in FRAMES
        for last in FRAMES
        if first != last
    }

    for order_key, probability in order_probs.items():
        order = [
            int(value)
            for value in order_key.split()
        ]

        endpoint_joint[
            (order[0], order[-1])
        ] += float(probability)

    endpoint_joint = norm_dict_clean(endpoint_joint)

    for (first, last), probability in (
        endpoint_joint.items()
    ):
        output[
            f"p_endpoint_pair_{first}_{last}"
        ] = float(probability)

    joint_for_entropy = {
        f"{first}>{last}": probability
        for (first, last), probability
        in endpoint_joint.items()
    }

    (
        joint_top1,
        joint_top2,
        joint_margin,
        joint_entropy,
        joint_normalized_entropy,
    ) = entropy_stats_clean(joint_for_entropy)

    predicted_pair = argmax_key_clean(endpoint_joint)

    output["endpoint_pair_prediction"] = (
        f"{predicted_pair[0]}>{predicted_pair[1]}"
    )
    output[
        "endpoint_pair_top1_probability"
    ] = joint_top1
    output["endpoint_pair_margin"] = joint_margin
    output["endpoint_pair_entropy"] = joint_entropy
    output[
        "endpoint_pair_normalized_entropy"
    ] = joint_normalized_entropy

    # --------------------------------------------------------
    # 5. Pairwise 원본 및 first/last 분포
    # --------------------------------------------------------
    pair_probs = record["bidirectional_pair_probs"]

    for frame_a, frame_b in itertools.combinations(
        FRAMES,
        2,
    ):
        output[
            f"p_pair_{frame_a}_before_{frame_b}"
        ] = float(
            pair_probs[
                f"{frame_a}>{frame_b}"
            ]["combined_prob"]
        )

        output[
            f"p_pair_{frame_b}_before_{frame_a}"
        ] = float(
            pair_probs[
                f"{frame_b}>{frame_a}"
            ]["combined_prob"]
        )

        output[
            f"pair_swap_inconsistency_{frame_a}_{frame_b}"
        ] = float(
            pair_probs[
                f"{frame_a}>{frame_b}"
            ].get("swap_inconsistency", 0.0)
        )

    pairwise_first_raw = {}
    pairwise_last_raw = {}

    for frame in FRAMES:
        pairwise_first_raw[frame] = sum(
            float(
                pair_probs[
                    f"{frame}>{other}"
                ]["combined_prob"]
            )
            for other in FRAMES
            if other != frame
        )

        pairwise_last_raw[frame] = sum(
            float(
                pair_probs[
                    f"{other}>{frame}"
                ]["combined_prob"]
            )
            for other in FRAMES
            if other != frame
        )

    pairwise_first = norm_dict_clean(
        pairwise_first_raw
    )
    pairwise_last = norm_dict_clean(
        pairwise_last_raw
    )

    for name, distribution in [
        ("pairwise_first", pairwise_first),
        ("pairwise_last", pairwise_last),
    ]:
        flatten_dict_clean(
            f"p_{name}",
            distribution,
            output,
        )

        (
            top1,
            top2,
            margin,
            entropy,
            normalized_entropy,
        ) = entropy_stats_clean(distribution)

        output[f"{name}_prediction"] = int(
            argmax_key_clean(distribution)
        )
        output[f"{name}_top1_probability"] = top1
        output[f"{name}_top2_probability"] = top2
        output[f"{name}_margin"] = margin
        output[f"{name}_entropy"] = entropy
        output[
            f"{name}_normalized_entropy"
        ] = normalized_entropy

    # --------------------------------------------------------
    # 6. 기본 10:30:60 endpoint 융합
    # --------------------------------------------------------
    fused_first = norm_dict_clean({
        frame: (
            0.10
            * float(
                order_position_distributions[1][frame]
            )
            + 0.30 * float(endpoint_first[frame])
            + 0.60 * float(pairwise_first[frame])
        )
        for frame in FRAMES
    })

    fused_last = norm_dict_clean({
        frame: (
            0.10
            * float(
                order_position_distributions[4][frame]
            )
            + 0.30 * float(endpoint_last[frame])
            + 0.60 * float(pairwise_last[frame])
        )
        for frame in FRAMES
    })

    flatten_dict_clean(
        "p_base_fused_first",
        fused_first,
        output,
    )

    flatten_dict_clean(
        "p_base_fused_last",
        fused_last,
        output,
    )

    output["base_first_prediction"] = int(
        argmax_key_clean(fused_first)
    )
    output["base_last_prediction"] = int(
        argmax_key_clean(fused_last)
    )

    # --------------------------------------------------------
    # 7. 모든 fixed-first 조건부
    # --------------------------------------------------------
    exhaustive = record["exhaustive_conditionals"]

    for fixed_first, distribution in (
        exhaustive["fixed_first"].items()
    ):
        fixed_first_int = int(fixed_first)

        last_distribution = {
            frame: 0.0
            for frame in FRAMES
            if frame != fixed_first_int
        }

        for suffix_order, probability in (
            distribution.items()
        ):
            compact_suffix = suffix_order.replace(
                " ",
                "",
            )

            output[
                f"p_cond_fixed_first_{fixed_first}_suffix_{compact_suffix}"
            ] = float(probability)

            suffix = [
                int(value)
                for value in suffix_order.split()
            ]

            last_distribution[
                suffix[-1]
            ] += float(probability)

        last_distribution = norm_dict_clean(
            last_distribution
        )

        flatten_dict_clean(
            f"p_cond_last_given_first_{fixed_first}",
            last_distribution,
            output,
        )

        (
            _,
            _,
            margin,
            _,
            normalized_entropy,
        ) = entropy_stats_clean(last_distribution)

        output[
            f"cond_last_given_first_{fixed_first}_prediction"
        ] = int(
            argmax_key_clean(last_distribution)
        )

        output[
            f"cond_last_given_first_{fixed_first}_margin"
        ] = margin

        output[
            f"cond_last_given_first_{fixed_first}_normalized_entropy"
        ] = normalized_entropy

    # --------------------------------------------------------
    # 8. 모든 fixed-last 조건부
    # --------------------------------------------------------
    for fixed_last, distribution in (
        exhaustive["fixed_last"].items()
    ):
        fixed_last_int = int(fixed_last)

        first_distribution = {
            frame: 0.0
            for frame in FRAMES
            if frame != fixed_last_int
        }

        for prefix_order, probability in (
            distribution.items()
        ):
            compact_prefix = prefix_order.replace(
                " ",
                "",
            )

            output[
                f"p_cond_fixed_last_{fixed_last}_prefix_{compact_prefix}"
            ] = float(probability)

            prefix = [
                int(value)
                for value in prefix_order.split()
            ]

            first_distribution[
                prefix[0]
            ] += float(probability)

        first_distribution = norm_dict_clean(
            first_distribution
        )

        flatten_dict_clean(
            f"p_cond_first_given_last_{fixed_last}",
            first_distribution,
            output,
        )

        (
            _,
            _,
            margin,
            _,
            normalized_entropy,
        ) = entropy_stats_clean(first_distribution)

        output[
            f"cond_first_given_last_{fixed_last}_prediction"
        ] = int(
            argmax_key_clean(first_distribution)
        )

        output[
            f"cond_first_given_last_{fixed_last}_margin"
        ] = margin

        output[
            f"cond_first_given_last_{fixed_last}_normalized_entropy"
        ] = normalized_entropy

    # --------------------------------------------------------
    # 9. 12개 endpoint 조합별 middle 두 순서
    # --------------------------------------------------------
    for endpoint_key, distribution in (
        exhaustive[
            "fixed_endpoints_middle"
        ].items()
    ):
        first, last = [
            int(value)
            for value in endpoint_key.split(">")
        ]

        middle = [
            frame
            for frame in FRAMES
            if frame not in {first, last}
        ]

        frame_a, frame_b = middle

        key_ab = f"{frame_a} {frame_b}"
        key_ba = f"{frame_b} {frame_a}"

        probability_ab = float(
            distribution.get(key_ab, 0.0)
        )
        probability_ba = float(
            distribution.get(key_ba, 0.0)
        )

        total = probability_ab + probability_ba

        if total <= 0:
            probability_ab = 0.5
            probability_ba = 0.5
        else:
            probability_ab /= total
            probability_ba /= total

        output[
            f"p_middle_order_given_{first}_{last}_{frame_a}{frame_b}"
        ] = probability_ab

        output[
            f"p_middle_order_given_{first}_{last}_{frame_b}{frame_a}"
        ] = probability_ba

        pairwise_ab = float(
            pair_probs[
                f"{frame_a}>{frame_b}"
            ]["combined_prob"]
        )

        pairwise_ba = float(
            pair_probs[
                f"{frame_b}>{frame_a}"
            ]["combined_prob"]
        )

        output[
            f"p_middle_pairwise_given_{first}_{last}_{frame_a}{frame_b}"
        ] = pairwise_ab

        output[
            f"p_middle_pairwise_given_{first}_{last}_{frame_b}{frame_a}"
        ] = pairwise_ba

        order_entropy = -(
            probability_ab
            * math.log(probability_ab + 1e-12)
            + probability_ba
            * math.log(probability_ba + 1e-12)
        ) / math.log(2.0)

        pair_entropy = -(
            pairwise_ab
            * math.log(pairwise_ab + 1e-12)
            + pairwise_ba
            * math.log(pairwise_ba + 1e-12)
        ) / math.log(2.0)

        output[
            f"middle_order_normalized_entropy_given_{first}_{last}"
        ] = float(order_entropy)

        output[
            f"middle_pairwise_normalized_entropy_given_{first}_{last}"
        ] = float(pair_entropy)

    return output


# ------------------------------------------------------------
# 기존 JSONL 캐시 100개 다시 읽기
# ------------------------------------------------------------
cached = {}

with CACHE_JSONL.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        cached[str(record["sample_id"])] = record

print("Recovered cached samples:", len(cached))


# 기존 eval100과 동일한 샘플 순서
full_rows = validation_df.sample(
    n=min(100, len(validation_df)),
    random_state=SEED + 1,
).reset_index(drop=True)

records = []

for _, row in full_rows.iterrows():
    sample_id = str(row["Id"])

    if sample_id not in cached:
        raise RuntimeError(
            f"Missing cached sample: {sample_id}"
        )

    records.append(cached[sample_id])

print("Recovered records:", len(records))


# ------------------------------------------------------------
# 재추론 없이 CSV 생성
# ------------------------------------------------------------
raw_rows = [
    build_raw_row_clean(
        record,
        has_gold=True,
    )
    for record in records
]

raw_df = pd.DataFrame(raw_rows)

readable_df = pd.DataFrame([
    build_readable_row(raw_row)
    for raw_row in raw_rows
])

RAW_RECOVERED_CSV = (
    OUTPUT_RUN_DIR
    / "eval100_raw_all_probabilities_recovered.csv"
)

READABLE_RECOVERED_CSV = (
    OUTPUT_RUN_DIR
    / "eval100_readable_decoding_recovered.csv"
)

SUMMARY_RECOVERED_CSV = (
    OUTPUT_RUN_DIR
    / "eval100_base_10_30_60_summary_recovered.csv"
)

raw_df.to_csv(
    RAW_RECOVERED_CSV,
    index=False,
)

readable_df.to_csv(
    READABLE_RECOVERED_CSV,
    index=False,
)

summary = {
    "checkpoint": str(BEST_ADAPTER_DIR),
    "rows": len(readable_df),
    "weights_order": 0.10,
    "weights_endpoint": 0.30,
    "weights_pairwise": 0.60,
    "base_exact": float(
        readable_df["base_exact"].mean()
    ),
    "first_threeway_all_agree_rate": float(
        readable_df[
            "first_threeway_all_agree"
        ].mean()
    ),
    "last_threeway_all_agree_rate": float(
        readable_df[
            "last_threeway_all_agree"
        ].mean()
    ),
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    SUMMARY_RECOVERED_CSV,
    index=False,
)

display(summary_df)
display(readable_df.head(20))

print("saved raw:", RAW_RECOVERED_CSV)
print("saved readable:", READABLE_RECOVERED_CSV)
print("saved summary:", SUMMARY_RECOVERED_CSV)

Recovered cached samples: 100
Recovered records: 100


,checkpoint,rows,weights_order,weights_endpoint,weights_pairwise,base_exact,first_threeway_all_agree_rate,last_threeway_all_agree_rate
0,/content/drive/MyDrive/SNU_AI_Challenge/qwen2v...,100,0.1,0.3,0.6,0.4,0.66,0.57


,sample_id,gold_order,gold_first,gold_last,first_order_prediction,first_order_probability,first_order_margin,first_order_normalized_entropy,first_endpoint_prediction,first_endpoint_probability,...,middle_frames,middle_order_ab,middle_order_probability_ab,middle_order_probability_ba,middle_order_normalized_entropy,middle_pairwise_probability_ab,middle_pairwise_probability_ba,middle_pairwise_normalized_entropy,base_prediction_10_30_60,base_exact
0,x6AwMw,2 3 4 1,2,1,3,0.351291,0.020689,0.950733,2,0.499723,...,"1,3",1>3,0.076222,0.923778,0.388724,0.000057,0.999943,0.000891,2 3 1 4,0
1,jOilDx,1 3 2 4,1,4,1,0.307579,0.060925,0.993211,1,0.367963,...,"3,4",3>4,0.667735,0.332265,0.917224,0.443600,0.556400,0.990802,1 3 4 2,0
2,B6W1Qk,2 1 3 4,2,4,2,0.288027,0.004673,0.976693,2,0.587708,...,"1,3",1>3,0.960171,0.039829,0.241508,0.999460,0.000540,0.006640,2 1 3 4,1
3,AolL8G,4 2 3 1,4,1,4,0.291385,0.039219,0.996198,4,0.729967,...,"1,2",1>2,0.466208,0.533792,0.996703,0.447460,0.552540,0.992020,4 2 1 3,0
4,MjIq4T,3 4 2 1,3,1,3,0.384042,0.100854,0.952950,3,0.997747,...,"2,4",2>4,0.019517,0.980483,0.138719,0.000804,0.999196,0.009425,3 4 2 1,1
5,NjCw77,2 3 4 1,2,1,2,0.435086,0.195882,0.933022,2,0.999958,...,"3,4",3>4,0.971864,0.028136,0.184956,0.998968,0.001032,0.011728,2 3 4 1,1
6,1SOLbE,1 2 3 4,1,4,4,0.333785,0.078599,0.982392,4,0.925279,...,"1,3",1>3,0.595176,0.404824,0.973703,0.529263,0.470737,0.997528,4 1 3 2,0
7,rht7VJ,2 1 4 3,2,3,4,0.365567,0.102730,0.969132,4,0.945400,...,"1,2",1>2,0.280349,0.719651,0.855927,0.007012,0.992988,0.060257,4 2 1 3,0
8,tUsgQQ,1 4 2 3,1,3,4,0.361988,0.086418,0.964554,4,0.950439,...,"1,2",1>2,0.933433,0.066567,0.352979,0.997834,0.002166,0.022290,4 1 2 3,0
9,TWfXii,2 1 4 3,2,3,4,0.273017,0.007776,0.997846,2,0.301944,...,"1,2",1>2,0.208579,0.791421,0.738757,0.564098,0.435902,0.988112,3 2 1 4,0


saved raw: /content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval100_raw_all_probabilities_recovered.csv
saved readable: /content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval100_readable_decoding_recovered.csv
saved summary: /content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval100_base_10_30_60_summary_r

In [17]:
# ============================================================
# 현재 캐시 200개를 재추론 없이 CSV로 복구
# 전제:
# - build_raw_row_clean() 함수가 이미 위 셀에서 정의되어 있음
# - CACHE_JSONL, OUTPUT_RUN_DIR, validation_df, SEED 등이 살아 있음
# ============================================================

import json
import pandas as pd


TARGET_ROWS = 200


# ------------------------------------------------------------
# 1. JSONL 캐시 다시 읽기
# ------------------------------------------------------------
cached = {}

with CACHE_JSONL.open(
    "r",
    encoding="utf-8",
) as handle:
    for line in handle:
        if not line.strip():
            continue

        record = json.loads(line)
        cached[str(record["sample_id"])] = record

print("Recovered cached samples:", len(cached))


# ------------------------------------------------------------
# 2. 기존과 같은 seed로 앞 200개 샘플 순서 복원
# ------------------------------------------------------------
eval_rows_200 = validation_df.sample(
    n=min(TARGET_ROWS, len(validation_df)),
    random_state=SEED + 1,
).reset_index(drop=True)

records_200 = []
missing_ids = []

for _, row in eval_rows_200.iterrows():
    sample_id = str(row["Id"])

    if sample_id not in cached:
        missing_ids.append(sample_id)
        continue

    records_200.append(cached[sample_id])

if missing_ids:
    raise RuntimeError(
        f"Missing {len(missing_ids)} cached samples: "
        f"{missing_ids[:10]}"
    )

print("Recovered records:", len(records_200))


# ------------------------------------------------------------
# 3. 재귀에 꼬인 build_raw_row()는 절대 사용하지 않음
# ------------------------------------------------------------
if "build_raw_row_clean" not in globals():
    raise RuntimeError(
        "build_raw_row_clean() 함수가 없습니다. "
        "이전에 제공한 clean 함수 정의 셀을 먼저 실행하세요."
    )

raw_rows_200 = [
    build_raw_row_clean(
        record,
        has_gold=True,
    )
    for record in records_200
]

raw_df_200 = pd.DataFrame(raw_rows_200)

readable_df_200 = pd.DataFrame([
    build_readable_row(raw_row)
    for raw_row in raw_rows_200
])

print("Raw rows:", len(raw_df_200))
print("Readable rows:", len(readable_df_200))


# ------------------------------------------------------------
# 4. 200개용 파일 경로
# ------------------------------------------------------------
RAW_CSV_200 = (
    OUTPUT_RUN_DIR
    / "eval200_raw_all_probabilities_recovered.csv"
)

READABLE_CSV_200 = (
    OUTPUT_RUN_DIR
    / "eval200_readable_decoding_recovered.csv"
)

SUMMARY_CSV_200 = (
    OUTPUT_RUN_DIR
    / "eval200_base_10_30_60_summary_recovered.csv"
)


# ------------------------------------------------------------
# 5. CSV 저장
# ------------------------------------------------------------
raw_df_200.to_csv(
    RAW_CSV_200,
    index=False,
)

readable_df_200.to_csv(
    READABLE_CSV_200,
    index=False,
)


# ------------------------------------------------------------
# 6. 기본 10:30:60 요약
# ------------------------------------------------------------
required_columns = [
    "base_exact",
    "first_threeway_all_agree",
    "last_threeway_all_agree",
]

missing_columns = [
    column
    for column in required_columns
    if column not in readable_df_200.columns
]

if missing_columns:
    raise RuntimeError(
        f"Readable CSV에 필요한 컬럼이 없습니다: "
        f"{missing_columns}"
    )

summary_200 = {
    "checkpoint": str(BEST_ADAPTER_DIR),
    "rows": len(readable_df_200),

    "weights_order": 0.10,
    "weights_endpoint": 0.30,
    "weights_pairwise": 0.60,

    "base_exact": float(
        readable_df_200[
            "base_exact"
        ].mean()
    ),

    "first_threeway_all_agree_rate": float(
        readable_df_200[
            "first_threeway_all_agree"
        ].mean()
    ),

    "last_threeway_all_agree_rate": float(
        readable_df_200[
            "last_threeway_all_agree"
        ].mean()
    ),
}

summary_df_200 = pd.DataFrame(
    [summary_200]
)

summary_df_200.to_csv(
    SUMMARY_CSV_200,
    index=False,
)


# ------------------------------------------------------------
# 7. 결과 확인
# ------------------------------------------------------------
display(summary_df_200)
display(readable_df_200.head(20))

print()
print("saved raw:")
print(RAW_CSV_200)

print()
print("saved readable:")
print(READABLE_CSV_200)

print()
print("saved summary:")
print(SUMMARY_CSV_200)

Recovered cached samples: 200
Recovered records: 200
Raw rows: 200
Readable rows: 200


,checkpoint,rows,weights_order,weights_endpoint,weights_pairwise,base_exact,first_threeway_all_agree_rate,last_threeway_all_agree_rate
0,/content/drive/MyDrive/SNU_AI_Challenge/qwen2v...,200,0.1,0.3,0.6,0.37,0.645,0.575


,sample_id,gold_order,gold_first,gold_last,first_order_prediction,first_order_probability,first_order_margin,first_order_normalized_entropy,first_endpoint_prediction,first_endpoint_probability,...,middle_frames,middle_order_ab,middle_order_probability_ab,middle_order_probability_ba,middle_order_normalized_entropy,middle_pairwise_probability_ab,middle_pairwise_probability_ba,middle_pairwise_normalized_entropy,base_prediction_10_30_60,base_exact
0,x6AwMw,2 3 4 1,2,1,3,0.351291,0.020689,0.950733,2,0.499723,...,"1,3",1>3,0.076222,0.923778,0.388724,0.000057,0.999943,0.000891,2 3 1 4,0
1,jOilDx,1 3 2 4,1,4,1,0.307579,0.060925,0.993211,1,0.367963,...,"3,4",3>4,0.667735,0.332265,0.917224,0.443600,0.556400,0.990802,1 3 4 2,0
2,B6W1Qk,2 1 3 4,2,4,2,0.288027,0.004673,0.976693,2,0.587708,...,"1,3",1>3,0.960171,0.039829,0.241508,0.999460,0.000540,0.006640,2 1 3 4,1
3,AolL8G,4 2 3 1,4,1,4,0.291385,0.039219,0.996198,4,0.729967,...,"1,2",1>2,0.466208,0.533792,0.996703,0.447460,0.552540,0.992020,4 2 1 3,0
4,MjIq4T,3 4 2 1,3,1,3,0.384042,0.100854,0.952950,3,0.997747,...,"2,4",2>4,0.019517,0.980483,0.138719,0.000804,0.999196,0.009425,3 4 2 1,1
5,NjCw77,2 3 4 1,2,1,2,0.435086,0.195882,0.933022,2,0.999958,...,"3,4",3>4,0.971864,0.028136,0.184956,0.998968,0.001032,0.011728,2 3 4 1,1
6,1SOLbE,1 2 3 4,1,4,4,0.333785,0.078599,0.982392,4,0.925279,...,"1,3",1>3,0.595176,0.404824,0.973703,0.529263,0.470737,0.997528,4 1 3 2,0
7,rht7VJ,2 1 4 3,2,3,4,0.365567,0.102730,0.969132,4,0.945400,...,"1,2",1>2,0.280349,0.719651,0.855927,0.007012,0.992988,0.060257,4 2 1 3,0
8,tUsgQQ,1 4 2 3,1,3,4,0.361988,0.086418,0.964554,4,0.950439,...,"1,2",1>2,0.933433,0.066567,0.352979,0.997834,0.002166,0.022290,4 1 2 3,0
9,TWfXii,2 1 4 3,2,3,4,0.273017,0.007776,0.997846,2,0.301944,...,"1,2",1>2,0.208579,0.791421,0.738757,0.564098,0.435902,0.988112,3 2 1 4,0



saved raw:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_raw_all_probabilities_recovered.csv

saved readable:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_readable_decoding_recovered.csv

saved summary:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_base_10_30_60_summar

In [21]:
# ============================================================
# Eval200 전용
# 최종 10:30:60 First / Last 분포 + confidence + priority 추가
# 전제:
# - raw_df_200
# - readable_df_200
# - OUTPUT_RUN_DIR
# 가 이미 존재
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime


if "raw_df_200" not in globals():
    raise RuntimeError("raw_df_200이 없습니다.")

if "readable_df_200" not in globals():
    raise RuntimeError("readable_df_200이 없습니다.")


raw_final = raw_df_200.copy()
readable_final = readable_df_200.copy()

FRAMES_LOCAL = [1, 2, 3, 4]
EPS = 1e-12


def normalize_four(values):
    arr = np.asarray(values, dtype=np.float64)
    arr = np.clip(arr, 0.0, None)

    total = arr.sum()

    if total <= 0:
        return np.full(4, 0.25, dtype=np.float64)

    return arr / total


def confidence_stats(probabilities):
    probabilities = normalize_four(probabilities)

    sorted_probs = np.sort(probabilities)[::-1]

    top1 = float(sorted_probs[0])
    top2 = float(sorted_probs[1])
    margin = float(top1 - top2)

    entropy = float(
        -np.sum(
            probabilities
            * np.log(probabilities + EPS)
        )
    )

    normalized_entropy = float(
        entropy / np.log(4.0)
    )

    prediction = int(
        np.argmax(probabilities) + 1
    )

    return {
        "prediction": prediction,
        "top1_probability": top1,
        "top2_probability": top2,
        "margin": margin,
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
    }


# ------------------------------------------------------------
# 필요한 원본 확률 컬럼 검증
# ------------------------------------------------------------
required_columns = []

for frame in FRAMES_LOCAL:
    required_columns.extend([
        f"p_order_pos1_frame_{frame}",
        f"p_order_pos4_frame_{frame}",
        f"p_endpoint_first_{frame}",
        f"p_endpoint_last_{frame}",
        f"p_pairwise_first_{frame}",
        f"p_pairwise_last_{frame}",
    ])

missing_columns = [
    column
    for column in required_columns
    if column not in raw_final.columns
]

if missing_columns:
    raise RuntimeError(
        "최종 first/last 계산에 필요한 컬럼이 없습니다:\n"
        + "\n".join(missing_columns)
    )


# ------------------------------------------------------------
# 샘플별 최종 10:30:60 분포 계산
# ------------------------------------------------------------
derived_rows = []

for _, row in raw_final.iterrows():

    order_first = np.array([
        float(row[f"p_order_pos1_frame_{frame}"])
        for frame in FRAMES_LOCAL
    ])

    order_last = np.array([
        float(row[f"p_order_pos4_frame_{frame}"])
        for frame in FRAMES_LOCAL
    ])

    endpoint_first = np.array([
        float(row[f"p_endpoint_first_{frame}"])
        for frame in FRAMES_LOCAL
    ])

    endpoint_last = np.array([
        float(row[f"p_endpoint_last_{frame}"])
        for frame in FRAMES_LOCAL
    ])

    pairwise_first = np.array([
        float(row[f"p_pairwise_first_{frame}"])
        for frame in FRAMES_LOCAL
    ])

    pairwise_last = np.array([
        float(row[f"p_pairwise_last_{frame}"])
        for frame in FRAMES_LOCAL
    ])

    # 최종 10:30:60
    final_first = normalize_four(
        0.10 * order_first
        + 0.30 * endpoint_first
        + 0.60 * pairwise_first
    )

    final_last = normalize_four(
        0.10 * order_last
        + 0.30 * endpoint_last
        + 0.60 * pairwise_last
    )

    first_stats = confidence_stats(final_first)
    last_stats = confidence_stats(final_last)

    derived = {
        "sample_id": str(row["sample_id"]),
    }

    for index, frame in enumerate(FRAMES_LOCAL):
        derived[f"p_final_first_{frame}"] = float(
            final_first[index]
        )
        derived[f"p_final_last_{frame}"] = float(
            final_last[index]
        )

    derived.update({
        "final_first_prediction":
            first_stats["prediction"],

        "final_first_top1_probability":
            first_stats["top1_probability"],

        "final_first_top2_probability":
            first_stats["top2_probability"],

        "final_first_margin":
            first_stats["margin"],

        "final_first_entropy":
            first_stats["entropy"],

        "final_first_normalized_entropy":
            first_stats["normalized_entropy"],

        "final_last_prediction":
            last_stats["prediction"],

        "final_last_top1_probability":
            last_stats["top1_probability"],

        "final_last_top2_probability":
            last_stats["top2_probability"],

        "final_last_margin":
            last_stats["margin"],

        "final_last_entropy":
            last_stats["entropy"],

        "final_last_normalized_entropy":
            last_stats["normalized_entropy"],
    })

    # 양수면 first 쪽이 더 확신 높음
    derived["final_first_minus_last_top1_probability"] = (
        first_stats["top1_probability"]
        - last_stats["top1_probability"]
    )

    derived["final_first_minus_last_margin"] = (
        first_stats["margin"]
        - last_stats["margin"]
    )

    # 양수면 first entropy가 더 낮아서 first가 더 확신 높음
    derived["final_first_confidence_entropy_advantage"] = (
        last_stats["normalized_entropy"]
        - first_stats["normalized_entropy"]
    )

    derived["final_endpoint_priority_by_probability"] = (
        "first"
        if first_stats["top1_probability"]
           > last_stats["top1_probability"]
        else "last"
        if last_stats["top1_probability"]
           > first_stats["top1_probability"]
        else "tie"
    )

    derived["final_endpoint_priority_by_margin"] = (
        "first"
        if first_stats["margin"] > last_stats["margin"]
        else "last"
        if last_stats["margin"] > first_stats["margin"]
        else "tie"
    )

    derived["final_endpoint_priority_by_entropy"] = (
        "first"
        if first_stats["normalized_entropy"]
           < last_stats["normalized_entropy"]
        else "last"
        if last_stats["normalized_entropy"]
           < first_stats["normalized_entropy"]
        else "tie"
    )

    probability_vote = (
        1
        if first_stats["top1_probability"]
           > last_stats["top1_probability"]
        else -1
        if last_stats["top1_probability"]
           > first_stats["top1_probability"]
        else 0
    )

    margin_vote = (
        1
        if first_stats["margin"] > last_stats["margin"]
        else -1
        if last_stats["margin"] > first_stats["margin"]
        else 0
    )

    entropy_vote = (
        1
        if first_stats["normalized_entropy"]
           < last_stats["normalized_entropy"]
        else -1
        if last_stats["normalized_entropy"]
           < first_stats["normalized_entropy"]
        else 0
    )

    vote_sum = (
        probability_vote
        + margin_vote
        + entropy_vote
    )

    derived["final_endpoint_priority_vote_sum"] = vote_sum

    derived["final_endpoint_priority_majority"] = (
        "first"
        if vote_sum > 0
        else "last"
        if vote_sum < 0
        else "tie"
    )

    derived_rows.append(derived)


derived_df = pd.DataFrame(derived_rows)


# ------------------------------------------------------------
# raw에 추가
# ------------------------------------------------------------
new_columns = [
    column
    for column in derived_df.columns
    if column != "sample_id"
]

raw_final = raw_final.drop(
    columns=[
        column
        for column in new_columns
        if column in raw_final.columns
    ],
    errors="ignore",
)

raw_final = raw_final.merge(
    derived_df,
    on="sample_id",
    how="left",
    validate="one_to_one",
)


# ------------------------------------------------------------
# readable에 추가
# ------------------------------------------------------------
readable_final = readable_final.drop(
    columns=[
        column
        for column in new_columns
        if column in readable_final.columns
    ],
    errors="ignore",
)

readable_final = readable_final.merge(
    derived_df,
    on="sample_id",
    how="left",
    validate="one_to_one",
)


# ------------------------------------------------------------
# readable 컬럼 순서 정리
# ------------------------------------------------------------
identity_columns = [
    column
    for column in [
        "sample_id",
        "gold_order",
        "gold_first",
        "gold_last",
    ]
    if column in readable_final.columns
]

full_columns = [
    column
    for column in [
        "full_order_top1_probability",
        "full_order_top2_probability",
        "full_order_margin",
        "full_order_entropy",
        "full_order_normalized_entropy",
    ]
    if column in readable_final.columns
]

final_endpoint_columns = [
    "p_final_first_1",
    "p_final_first_2",
    "p_final_first_3",
    "p_final_first_4",
    "final_first_prediction",
    "final_first_top1_probability",
    "final_first_top2_probability",
    "final_first_margin",
    "final_first_entropy",
    "final_first_normalized_entropy",

    "p_final_last_1",
    "p_final_last_2",
    "p_final_last_3",
    "p_final_last_4",
    "final_last_prediction",
    "final_last_top1_probability",
    "final_last_top2_probability",
    "final_last_margin",
    "final_last_entropy",
    "final_last_normalized_entropy",

    "final_first_minus_last_top1_probability",
    "final_first_minus_last_margin",
    "final_first_confidence_entropy_advantage",

    "final_endpoint_priority_by_probability",
    "final_endpoint_priority_by_margin",
    "final_endpoint_priority_by_entropy",
    "final_endpoint_priority_vote_sum",
    "final_endpoint_priority_majority",
]

final_endpoint_columns = [
    column
    for column in final_endpoint_columns
    if column in readable_final.columns
]

used_columns = set(
    identity_columns
    + full_columns
    + final_endpoint_columns
)

remaining_columns = [
    column
    for column in readable_final.columns
    if column not in used_columns
]

readable_final = readable_final[
    identity_columns
    + full_columns
    + final_endpoint_columns
    + remaining_columns
]


# ------------------------------------------------------------
# 기존과 안 겹치는 새 파일명
# ------------------------------------------------------------
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RAW_FINAL_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_step300_endpoint020_"
        f"final103060_raw_{timestamp}.csv"
    )
)

READABLE_FINAL_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_step300_endpoint020_"
        f"final103060_readable_{timestamp}.csv"
    )
)


raw_final.to_csv(
    RAW_FINAL_PATH,
    index=False,
)

readable_final.to_csv(
    READABLE_FINAL_PATH,
    index=False,
)


# 기존 변수에도 반영
raw_df_200 = raw_final
readable_df_200 = readable_final


# ------------------------------------------------------------
# 확인
# ------------------------------------------------------------
display_columns = [
    column
    for column in [
        "sample_id",
        "gold_order",

        "full_order_margin",
        "full_order_normalized_entropy",

        "final_first_prediction",
        "final_first_top1_probability",
        "final_first_margin",
        "final_first_normalized_entropy",

        "final_last_prediction",
        "final_last_top1_probability",
        "final_last_margin",
        "final_last_normalized_entropy",

        "final_first_minus_last_margin",
        "final_first_confidence_entropy_advantage",
        "final_endpoint_priority_majority",
    ]
    if column in readable_df_200.columns
]

display(
    readable_df_200[
        display_columns
    ].head(20)
)

print()
print("Eval200 final 10:30:60 RAW saved:")
print(RAW_FINAL_PATH)

print()
print("Eval200 final 10:30:60 READABLE saved:")
print(READABLE_FINAL_PATH)

,sample_id,gold_order,full_order_margin,full_order_normalized_entropy,final_first_prediction,final_first_top1_probability,final_first_margin,final_first_normalized_entropy,final_last_prediction,final_last_top1_probability,final_last_margin,final_last_normalized_entropy,final_first_minus_last_margin,final_first_confidence_entropy_advantage,final_endpoint_priority_majority
0,x6AwMw,2 3 4 1,0.004828,0.919593,2,0.479608,0.091215,0.767003,4,0.437728,0.000756,0.745782,0.090459,-0.021221,first
1,jOilDx,1 3 2 4,0.000075,0.995850,1,0.316980,0.024570,0.976788,2,0.308185,0.037132,0.989132,-0.012562,0.012344,first
2,B6W1Qk,2 1 3 4,0.014451,0.952003,2,0.472080,0.090023,0.778376,4,0.637185,0.406894,0.706918,-0.316871,-0.071458,last
3,AolL8G,4 2 3 1,0.000116,0.997393,4,0.466769,0.242054,0.913711,3,0.342776,0.100525,0.982759,0.141529,0.069047,first
4,MjIq4T,3 4 2 1,0.099490,0.898505,3,0.636281,0.406019,0.683047,1,0.649566,0.423138,0.663183,-0.017119,-0.019864,last
5,NjCw77,2 3 4 1,0.147000,0.913713,2,0.643230,0.425088,0.707777,1,0.615920,0.361147,0.689006,0.063941,-0.018771,first
6,1SOLbE,1 2 3 4,0.002175,0.977186,4,0.590254,0.385190,0.740532,2,0.544306,0.359462,0.843898,0.025728,0.103367,first
7,rht7VJ,2 1 4 3,0.028693,0.956753,4,0.617414,0.373423,0.702320,3,0.644402,0.425384,0.681821,-0.051961,-0.020499,last
8,tUsgQQ,1 4 2 3,0.025274,0.952219,4,0.563438,0.265030,0.736763,3,0.509925,0.157069,0.777729,0.107961,0.040965,first
9,TWfXii,2 1 4 3,0.001901,0.993056,3,0.292420,0.016132,0.988351,4,0.496781,0.289656,0.893266,-0.273523,-0.095085,last



Eval200 final 10:30:60 RAW saved:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_step300_endpoint020_final103060_raw_20260724_082958.csv

Eval200 final 10:30:60 READABLE saved:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_step300_endpoint020_final103060_readable_20260724_082958.csv


In [22]:
# ============================================================
# Eval200 전용:
# 우선 선택 endpoint 이후 남은 endpoint의 3-way 비교 추가
#
# First 먼저 선택 시:
#   conditional order last | endpoint last | pairwise last
#
# Last 먼저 선택 시:
#   conditional order first | endpoint first | pairwise first
#
# 전제:
# - raw_df_200
# - readable_df_200
# - final_first_prediction / final_last_prediction
# - final_endpoint_priority_majority
# 가 이미 생성되어 있음
# ============================================================

import numpy as np
import pandas as pd
from datetime import datetime


if "raw_df_200" not in globals():
    raise RuntimeError("raw_df_200이 없습니다.")

if "readable_df_200" not in globals():
    raise RuntimeError("readable_df_200이 없습니다.")


raw_stage2 = raw_df_200.copy()
readable_stage2 = readable_df_200.copy()

FRAMES_LOCAL = [1, 2, 3, 4]
EPS = 1e-12


def normalize_variable_distribution(values):
    values = np.asarray(values, dtype=np.float64)
    values = np.clip(values, 0.0, None)

    total = float(values.sum())

    if total <= 0:
        return np.full(
            len(values),
            1.0 / len(values),
            dtype=np.float64,
        )

    return values / total


def distribution_stats(
    frame_candidates,
    probabilities,
):
    probabilities = normalize_variable_distribution(
        probabilities
    )

    order = np.argsort(
        probabilities
    )[::-1]

    top1_index = int(order[0])
    top2_index = (
        int(order[1])
        if len(order) > 1
        else int(order[0])
    )

    top1 = float(probabilities[top1_index])
    top2 = (
        float(probabilities[top2_index])
        if len(probabilities) > 1
        else 0.0
    )

    entropy = float(
        -np.sum(
            probabilities
            * np.log(probabilities + EPS)
        )
    )

    normalized_entropy = (
        float(
            entropy / np.log(len(probabilities))
        )
        if len(probabilities) > 1
        else 0.0
    )

    return {
        "prediction": int(
            frame_candidates[top1_index]
        ),
        "top1_probability": top1,
        "top2_probability": top2,
        "margin": float(top1 - top2),
        "entropy": entropy,
        "normalized_entropy": (
            normalized_entropy
        ),
    }


def choose_priority(row):
    priority = str(
        row.get(
            "final_endpoint_priority_majority",
            "",
        )
    )

    if priority in {"first", "last"}:
        return priority

    # Majority가 tie면 margin으로 결정
    first_margin = float(
        row["final_first_margin"]
    )
    last_margin = float(
        row["final_last_margin"]
    )

    if first_margin > last_margin:
        return "first"

    if last_margin > first_margin:
        return "last"

    # margin도 같으면 entropy가 낮은 쪽
    first_entropy = float(
        row[
            "final_first_normalized_entropy"
        ]
    )
    last_entropy = float(
        row[
            "final_last_normalized_entropy"
        ]
    )

    if first_entropy <= last_entropy:
        return "first"

    return "last"


derived_rows = []


for _, row in raw_stage2.iterrows():

    sample_id = str(row["sample_id"])
    priority = choose_priority(row)

    output = {
        "sample_id": sample_id,
        "stage2_fixed_endpoint_type": priority,
    }

    # ========================================================
    # A. First를 먼저 선택
    # 남은 Last를 3개 방식으로 비교
    # ========================================================
    if priority == "first":

        fixed_first = int(
            row["final_first_prediction"]
        )

        remaining_frames = [
            frame
            for frame in FRAMES_LOCAL
            if frame != fixed_first
        ]

        output[
            "stage2_fixed_endpoint_value"
        ] = fixed_first

        output[
            "stage2_remaining_endpoint_type"
        ] = "last"

        # 1) 조건부 order:
        # P(last | selected first)
        conditional_order_probabilities = [
            float(
                row[
                    f"p_cond_last_given_first_"
                    f"{fixed_first}_{frame}"
                ]
            )
            for frame in remaining_frames
        ]

        conditional_order_stats = (
            distribution_stats(
                remaining_frames,
                conditional_order_probabilities,
            )
        )

        # 2) 직접 last head
        endpoint_probabilities = [
            float(
                row[
                    f"p_endpoint_last_{frame}"
                ]
            )
            for frame in remaining_frames
        ]

        endpoint_probabilities = (
            normalize_variable_distribution(
                endpoint_probabilities
            )
        )

        endpoint_stats = distribution_stats(
            remaining_frames,
            endpoint_probabilities,
        )

        # 3) pairwise last
        pairwise_probabilities = [
            float(
                row[
                    f"p_pairwise_last_{frame}"
                ]
            )
            for frame in remaining_frames
        ]

        pairwise_probabilities = (
            normalize_variable_distribution(
                pairwise_probabilities
            )
        )

        pairwise_stats = distribution_stats(
            remaining_frames,
            pairwise_probabilities,
        )

        for frame, probability in zip(
            remaining_frames,
            normalize_variable_distribution(
                conditional_order_probabilities
            ),
        ):
            output[
                f"p_stage2_conditional_order_last_"
                f"{frame}"
            ] = float(probability)

        for frame, probability in zip(
            remaining_frames,
            endpoint_probabilities,
        ):
            output[
                f"p_stage2_endpoint_last_{frame}"
            ] = float(probability)

        for frame, probability in zip(
            remaining_frames,
            pairwise_probabilities,
        ):
            output[
                f"p_stage2_pairwise_last_{frame}"
            ] = float(probability)

    # ========================================================
    # B. Last를 먼저 선택
    # 남은 First를 3개 방식으로 비교
    # ========================================================
    else:

        fixed_last = int(
            row["final_last_prediction"]
        )

        remaining_frames = [
            frame
            for frame in FRAMES_LOCAL
            if frame != fixed_last
        ]

        output[
            "stage2_fixed_endpoint_value"
        ] = fixed_last

        output[
            "stage2_remaining_endpoint_type"
        ] = "first"

        # 1) 조건부 order:
        # P(first | selected last)
        conditional_order_probabilities = [
            float(
                row[
                    f"p_cond_first_given_last_"
                    f"{fixed_last}_{frame}"
                ]
            )
            for frame in remaining_frames
        ]

        conditional_order_stats = (
            distribution_stats(
                remaining_frames,
                conditional_order_probabilities,
            )
        )

        # 2) 직접 first head
        endpoint_probabilities = [
            float(
                row[
                    f"p_endpoint_first_{frame}"
                ]
            )
            for frame in remaining_frames
        ]

        endpoint_probabilities = (
            normalize_variable_distribution(
                endpoint_probabilities
            )
        )

        endpoint_stats = distribution_stats(
            remaining_frames,
            endpoint_probabilities,
        )

        # 3) pairwise first
        pairwise_probabilities = [
            float(
                row[
                    f"p_pairwise_first_{frame}"
                ]
            )
            for frame in remaining_frames
        ]

        pairwise_probabilities = (
            normalize_variable_distribution(
                pairwise_probabilities
            )
        )

        pairwise_stats = distribution_stats(
            remaining_frames,
            pairwise_probabilities,
        )

        for frame, probability in zip(
            remaining_frames,
            normalize_variable_distribution(
                conditional_order_probabilities
            ),
        ):
            output[
                f"p_stage2_conditional_order_first_"
                f"{frame}"
            ] = float(probability)

        for frame, probability in zip(
            remaining_frames,
            endpoint_probabilities,
        ):
            output[
                f"p_stage2_endpoint_first_{frame}"
            ] = float(probability)

        for frame, probability in zip(
            remaining_frames,
            pairwise_probabilities,
        ):
            output[
                f"p_stage2_pairwise_first_{frame}"
            ] = float(probability)


    # ========================================================
    # 공통 결과 저장
    # ========================================================
    output.update({
        "stage2_conditional_order_prediction":
            conditional_order_stats[
                "prediction"
            ],

        "stage2_conditional_order_top1_probability":
            conditional_order_stats[
                "top1_probability"
            ],

        "stage2_conditional_order_top2_probability":
            conditional_order_stats[
                "top2_probability"
            ],

        "stage2_conditional_order_margin":
            conditional_order_stats[
                "margin"
            ],

        "stage2_conditional_order_entropy":
            conditional_order_stats[
                "entropy"
            ],

        "stage2_conditional_order_normalized_entropy":
            conditional_order_stats[
                "normalized_entropy"
            ],

        "stage2_endpoint_prediction":
            endpoint_stats[
                "prediction"
            ],

        "stage2_endpoint_top1_probability":
            endpoint_stats[
                "top1_probability"
            ],

        "stage2_endpoint_top2_probability":
            endpoint_stats[
                "top2_probability"
            ],

        "stage2_endpoint_margin":
            endpoint_stats[
                "margin"
            ],

        "stage2_endpoint_entropy":
            endpoint_stats[
                "entropy"
            ],

        "stage2_endpoint_normalized_entropy":
            endpoint_stats[
                "normalized_entropy"
            ],

        "stage2_pairwise_prediction":
            pairwise_stats[
                "prediction"
            ],

        "stage2_pairwise_top1_probability":
            pairwise_stats[
                "top1_probability"
            ],

        "stage2_pairwise_top2_probability":
            pairwise_stats[
                "top2_probability"
            ],

        "stage2_pairwise_margin":
            pairwise_stats[
                "margin"
            ],

        "stage2_pairwise_entropy":
            pairwise_stats[
                "entropy"
            ],

        "stage2_pairwise_normalized_entropy":
            pairwise_stats[
                "normalized_entropy"
            ],
    })

    # 요청한 3|1|2 형식
    output[
        "stage2_threeway_predictions"
    ] = (
        f"{conditional_order_stats['prediction']}"
        f"|{endpoint_stats['prediction']}"
        f"|{pairwise_stats['prediction']}"
    )

    prediction_values = [
        conditional_order_stats[
            "prediction"
        ],
        endpoint_stats[
            "prediction"
        ],
        pairwise_stats[
            "prediction"
        ],
    ]

    output[
        "stage2_threeway_all_agree"
    ] = int(
        len(set(prediction_values)) == 1
    )

    # 세 방식 중 2개 이상 일치하는 값
    counts = {
        value: prediction_values.count(value)
        for value in set(prediction_values)
    }

    majority_prediction = max(
        counts,
        key=counts.get,
    )

    if counts[majority_prediction] >= 2:
        output[
            "stage2_threeway_majority_prediction"
        ] = int(majority_prediction)
    else:
        output[
            "stage2_threeway_majority_prediction"
        ] = np.nan

    output[
        "stage2_threeway_pattern"
    ] = (
        "C=E=P"
        if len(set(prediction_values)) == 1
        else "C=E!=P"
        if prediction_values[0]
           == prediction_values[1]
        else "C=P!=E"
        if prediction_values[0]
           == prediction_values[2]
        else "E=P!=C"
        if prediction_values[1]
           == prediction_values[2]
        else "ALL_DIFFERENT"
    )

    derived_rows.append(output)


stage2_df = pd.DataFrame(
    derived_rows
)


# ============================================================
# Raw에 병합
# ============================================================
stage2_columns = [
    column
    for column in stage2_df.columns
    if column != "sample_id"
]

raw_stage2 = raw_stage2.drop(
    columns=[
        column
        for column in stage2_columns
        if column in raw_stage2.columns
    ],
    errors="ignore",
)

raw_stage2 = raw_stage2.merge(
    stage2_df,
    on="sample_id",
    how="left",
    validate="one_to_one",
)


# ============================================================
# Readable에 병합
# ============================================================
readable_stage2 = readable_stage2.drop(
    columns=[
        column
        for column in stage2_columns
        if column in readable_stage2.columns
    ],
    errors="ignore",
)

readable_stage2 = readable_stage2.merge(
    stage2_df,
    on="sample_id",
    how="left",
    validate="one_to_one",
)


# ============================================================
# Readable 컬럼 순서:
# full → final first/last → stage2 → 나머지
# ============================================================
identity_columns = [
    column
    for column in [
        "sample_id",
        "gold_order",
        "gold_first",
        "gold_last",
    ]
    if column in readable_stage2.columns
]

full_columns = [
    column
    for column in [
        "full_order_top1_probability",
        "full_order_top2_probability",
        "full_order_margin",
        "full_order_entropy",
        "full_order_normalized_entropy",
    ]
    if column in readable_stage2.columns
]

final_columns = [
    column
    for column in readable_stage2.columns
    if (
        column.startswith(
            "p_final_first_"
        )
        or column.startswith(
            "p_final_last_"
        )
        or column.startswith(
            "final_first_"
        )
        or column.startswith(
            "final_last_"
        )
        or column.startswith(
            "final_endpoint_"
        )
    )
]

stage2_ordered_columns = [
    column
    for column in [
        "stage2_fixed_endpoint_type",
        "stage2_fixed_endpoint_value",
        "stage2_remaining_endpoint_type",

        "stage2_threeway_predictions",
        "stage2_threeway_pattern",
        "stage2_threeway_all_agree",
        "stage2_threeway_majority_prediction",

        "stage2_conditional_order_prediction",
        "stage2_conditional_order_top1_probability",
        "stage2_conditional_order_top2_probability",
        "stage2_conditional_order_margin",
        "stage2_conditional_order_entropy",
        "stage2_conditional_order_normalized_entropy",

        "stage2_endpoint_prediction",
        "stage2_endpoint_top1_probability",
        "stage2_endpoint_top2_probability",
        "stage2_endpoint_margin",
        "stage2_endpoint_entropy",
        "stage2_endpoint_normalized_entropy",

        "stage2_pairwise_prediction",
        "stage2_pairwise_top1_probability",
        "stage2_pairwise_top2_probability",
        "stage2_pairwise_margin",
        "stage2_pairwise_entropy",
        "stage2_pairwise_normalized_entropy",
    ]
    if column in readable_stage2.columns
]

stage2_probability_columns = [
    column
    for column in readable_stage2.columns
    if column.startswith("p_stage2_")
]

used = set(
    identity_columns
    + full_columns
    + final_columns
    + stage2_ordered_columns
    + stage2_probability_columns
)

remaining_columns = [
    column
    for column in readable_stage2.columns
    if column not in used
]

readable_stage2 = readable_stage2[
    identity_columns
    + full_columns
    + final_columns
    + stage2_ordered_columns
    + stage2_probability_columns
    + remaining_columns
]


# ============================================================
# 새 파일명으로 저장
# ============================================================
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

RAW_STAGE2_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_step300_endpoint020_"
        f"sequential_stage2_threeway_raw_{timestamp}.csv"
    )
)

READABLE_STAGE2_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_step300_endpoint020_"
        f"sequential_stage2_threeway_readable_{timestamp}.csv"
    )
)


raw_stage2.to_csv(
    RAW_STAGE2_PATH,
    index=False,
)

readable_stage2.to_csv(
    READABLE_STAGE2_PATH,
    index=False,
)


raw_df_200 = raw_stage2
readable_df_200 = readable_stage2


# ============================================================
# 확인
# ============================================================
display_columns = [
    column
    for column in [
        "sample_id",
        "gold_order",
        "final_endpoint_priority_majority",
        "stage2_fixed_endpoint_type",
        "stage2_fixed_endpoint_value",
        "stage2_remaining_endpoint_type",
        "stage2_threeway_predictions",
        "stage2_threeway_pattern",
        "stage2_threeway_majority_prediction",
        "stage2_conditional_order_margin",
        "stage2_endpoint_margin",
        "stage2_pairwise_margin",
    ]
    if column in readable_df_200.columns
]

display(
    readable_df_200[
        display_columns
    ].head(30)
)

print()
print("Eval200 sequential stage2 RAW saved:")
print(RAW_STAGE2_PATH)

print()
print("Eval200 sequential stage2 READABLE saved:")
print(READABLE_STAGE2_PATH)

,sample_id,gold_order,final_endpoint_priority_majority,stage2_fixed_endpoint_type,stage2_fixed_endpoint_value,stage2_remaining_endpoint_type,stage2_threeway_predictions,stage2_threeway_pattern,stage2_threeway_majority_prediction,stage2_conditional_order_margin,stage2_endpoint_margin,stage2_pairwise_margin
0,x6AwMw,2 3 4 1,first,first,2,last,1|4|1,C=P!=E,1.0,0.067293,0.215264,0.102041
1,jOilDx,1 3 2 4,first,first,1,last,2|4|2,C=P!=E,2.0,0.001407,0.108210,0.105366
2,B6W1Qk,2 1 3 4,last,last,4,first,1|2|2,E=P!=C,2.0,0.094441,0.185288,0.057461
3,AolL8G,4 2 3 1,first,first,4,last,3|3|3,C=E=P,3.0,0.021685,0.433334,0.037065
4,MjIq4T,3 4 2 1,last,last,1,first,3|3|3,C=E=P,3.0,0.189524,0.995548,0.162162
5,NjCw77,2 3 4 1,first,first,2,last,1|1|1,C=E=P,1.0,0.460444,0.999988,0.076162
6,1SOLbE,1 2 3 4,first,first,4,last,2|2|2,C=E=P,2.0,0.298289,0.730464,0.265194
7,rht7VJ,2 1 4 3,last,last,3,first,4|4|4,C=E=P,4.0,0.154981,0.892470,0.159290
8,tUsgQQ,1 4 2 3,first,first,4,last,2|3|2,C=P!=E,2.0,0.193550,0.897131,0.177268
9,TWfXii,2 1 4 3,last,last,4,first,2|2|3,C=E!=P,2.0,0.051512,0.045145,0.009546



Eval200 sequential stage2 RAW saved:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_step300_endpoint020_sequential_stage2_threeway_raw_20260724_083413.csv

Eval200 sequential stage2 READABLE saved:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_step300_endpoint020_sequential_stage2_threeway_readable_20260724_083413.csv


In [25]:
# ============================================================
# Eval200: hierarchical entropy-driven adaptive fallback
#
# Gate:
#   full_order_normalized_entropy >= 0.98
#
# Stage 1:
#   order first / endpoint first / pairwise first
#
# Stage 2 after fixing first:
#   conditional-order last / endpoint last / pairwise last
#
# Middle after fixing endpoints:
#   conditional-order middle / pairwise middle
#
# 불일치 시 normalized entropy가 가장 낮은 branch를 hard-select
# ============================================================

import math
from datetime import datetime

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------
FULL_ENTROPY_GATE = 0.98
FRAMES_LOCAL = [1, 2, 3, 4]
EPS = 1e-12

# middle에서 entropy가 완전히 같을 때의 기본 선택
MIDDLE_TIE_PREFER = "pairwise"

if "raw_df_200" not in globals():
    raise RuntimeError("raw_df_200이 없습니다.")

adaptive_source = raw_df_200.copy()

required_base_columns = [
    "sample_id",
    "gold_order",
    "full_order_normalized_entropy",

    "order_pos1_prediction",
    "order_pos1_normalized_entropy",

    "endpoint_first_prediction",
    "endpoint_first_normalized_entropy",

    "pairwise_first_prediction",
    "pairwise_first_normalized_entropy",
]

missing = [
    column
    for column in required_base_columns
    if column not in adaptive_source.columns
]

if missing:
    raise RuntimeError(
        "필요한 컬럼이 없습니다:\n"
        + "\n".join(missing)
    )


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def normalize_distribution(frame_to_value):
    cleaned = {
        int(frame): max(float(value), 0.0)
        for frame, value in frame_to_value.items()
    }

    total = sum(cleaned.values())

    if total <= 0:
        uniform = 1.0 / len(cleaned)
        return {
            frame: uniform
            for frame in cleaned
        }

    return {
        frame: value / total
        for frame, value in cleaned.items()
    }


def distribution_stats(frame_to_probability):
    distribution = normalize_distribution(
        frame_to_probability
    )

    sorted_items = sorted(
        distribution.items(),
        key=lambda item: item[1],
        reverse=True,
    )

    prediction = int(sorted_items[0][0])
    top1 = float(sorted_items[0][1])
    top2 = (
        float(sorted_items[1][1])
        if len(sorted_items) > 1
        else 0.0
    )

    probabilities = np.array(
        list(distribution.values()),
        dtype=np.float64,
    )

    entropy = float(
        -np.sum(
            probabilities
            * np.log(probabilities + EPS)
        )
    )

    normalized_entropy = (
        float(
            entropy / np.log(len(probabilities))
        )
        if len(probabilities) > 1
        else 0.0
    )

    return {
        "distribution": distribution,
        "prediction": prediction,
        "top1_probability": top1,
        "top2_probability": top2,
        "margin": float(top1 - top2),
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
    }


def choose_lowest_entropy_branch(branches):
    """
    branches:
    {
        "order": {
            "prediction": ...,
            "normalized_entropy": ...
        },
        ...
    }
    """
    return min(
        branches,
        key=lambda name: (
            float(
                branches[name][
                    "normalized_entropy"
                ]
            ),
            name,
        ),
    )


def threeway_pattern(predictions):
    order_pred = int(predictions["order"])
    endpoint_pred = int(predictions["endpoint"])
    pairwise_pred = int(predictions["pairwise"])

    if (
        order_pred
        == endpoint_pred
        == pairwise_pred
    ):
        return "O=E=P"

    if order_pred == endpoint_pred:
        return "O=E!=P"

    if order_pred == pairwise_pred:
        return "O=P!=E"

    if endpoint_pred == pairwise_pred:
        return "E=P!=O"

    return "ALL_DIFFERENT"


def parse_order(order_value):
    if isinstance(order_value, str):
        cleaned = (
            order_value
            .replace("[", "")
            .replace("]", "")
            .replace(",", " ")
        )

        return [
            int(value)
            for value in cleaned.split()
        ]

    return [
        int(value)
        for value in order_value
    ]


def order_to_text(order):
    return " ".join(
        map(str, order)
    )


# ------------------------------------------------------------
# 2. 선택한 first를 조건으로 한 last 3-way 계산
# ------------------------------------------------------------
def calculate_second_stage(row, fixed_first):
    candidates = [
        frame
        for frame in FRAMES_LOCAL
        if frame != fixed_first
    ]

    # A. Conditional order: P(last | first)
    conditional_distribution = {
        frame: float(
            row[
                f"p_cond_last_given_first_"
                f"{fixed_first}_{frame}"
            ]
        )
        for frame in candidates
    }

    conditional_stats = distribution_stats(
        conditional_distribution
    )

    # B. Direct last endpoint head
    endpoint_distribution = {
        frame: float(
            row[
                f"p_endpoint_last_{frame}"
            ]
        )
        for frame in candidates
    }

    endpoint_stats = distribution_stats(
        endpoint_distribution
    )

    # C. Pairwise last
    pairwise_distribution = {
        frame: float(
            row[
                f"p_pairwise_last_{frame}"
            ]
        )
        for frame in candidates
    }

    pairwise_stats = distribution_stats(
        pairwise_distribution
    )

    branches = {
        "order": conditional_stats,
        "endpoint": endpoint_stats,
        "pairwise": pairwise_stats,
    }

    predictions = {
        name: stats["prediction"]
        for name, stats in branches.items()
    }

    all_agree = (
        len(set(predictions.values())) == 1
    )

    if all_agree:
        selected_branch = "all_agree"
        selected_prediction = int(
            predictions["order"]
        )
    else:
        selected_branch = (
            choose_lowest_entropy_branch(
                branches
            )
        )

        selected_prediction = int(
            branches[selected_branch][
                "prediction"
            ]
        )

    return {
        "branches": branches,
        "predictions": predictions,
        "pattern": threeway_pattern(
            predictions
        ),
        "all_agree": all_agree,
        "selected_branch": selected_branch,
        "selected_prediction": (
            selected_prediction
        ),
    }


# ------------------------------------------------------------
# 3. 최종 endpoints를 조건으로 middle 계산
# ------------------------------------------------------------
def calculate_middle_stage(row, first, last):
    middle_frames = [
        frame
        for frame in FRAMES_LOCAL
        if frame not in {first, last}
    ]

    frame_a, frame_b = middle_frames

    # Conditional order middle
    p_order_ab = float(
        row[
            f"p_middle_order_given_"
            f"{first}_{last}_{frame_a}{frame_b}"
        ]
    )

    p_order_ba = float(
        row[
            f"p_middle_order_given_"
            f"{first}_{last}_{frame_b}{frame_a}"
        ]
    )

    order_stats = distribution_stats({
        0: p_order_ab,
        1: p_order_ba,
    })

    order_ab_selected = (
        order_stats["prediction"] == 0
    )

    # Pairwise middle
    p_pair_ab = float(
        row[
            f"p_middle_pairwise_given_"
            f"{first}_{last}_{frame_a}{frame_b}"
        ]
    )

    p_pair_ba = float(
        row[
            f"p_middle_pairwise_given_"
            f"{first}_{last}_{frame_b}{frame_a}"
        ]
    )

    pairwise_stats = distribution_stats({
        0: p_pair_ab,
        1: p_pair_ba,
    })

    pairwise_ab_selected = (
        pairwise_stats["prediction"] == 0
    )

    order_prediction_text = (
        f"{frame_a}>{frame_b}"
        if order_ab_selected
        else f"{frame_b}>{frame_a}"
    )

    pairwise_prediction_text = (
        f"{frame_a}>{frame_b}"
        if pairwise_ab_selected
        else f"{frame_b}>{frame_a}"
    )

    agree = (
        order_prediction_text
        == pairwise_prediction_text
    )

    if agree:
        selected_branch = "all_agree"
        selected_ab = order_ab_selected

    elif (
        order_stats["normalized_entropy"]
        < pairwise_stats["normalized_entropy"]
    ):
        selected_branch = "order"
        selected_ab = order_ab_selected

    elif (
        pairwise_stats["normalized_entropy"]
        < order_stats["normalized_entropy"]
    ):
        selected_branch = "pairwise"
        selected_ab = pairwise_ab_selected

    else:
        selected_branch = (
            MIDDLE_TIE_PREFER
        )

        selected_ab = (
            pairwise_ab_selected
            if MIDDLE_TIE_PREFER == "pairwise"
            else order_ab_selected
        )

    selected_middle = (
        [frame_a, frame_b]
        if selected_ab
        else [frame_b, frame_a]
    )

    return {
        "frames": middle_frames,
        "order_prediction": (
            order_prediction_text
        ),
        "pairwise_prediction": (
            pairwise_prediction_text
        ),
        "agree": agree,
        "selected_branch": selected_branch,
        "selected_middle": selected_middle,

        "order_margin": (
            order_stats["margin"]
        ),
        "order_normalized_entropy": (
            order_stats[
                "normalized_entropy"
            ]
        ),

        "pairwise_margin": (
            pairwise_stats["margin"]
        ),
        "pairwise_normalized_entropy": (
            pairwise_stats[
                "normalized_entropy"
            ]
        ),
    }


# ------------------------------------------------------------
# 4. 샘플별 adaptive fallback
# ------------------------------------------------------------
result_rows = []

for _, row in adaptive_source.iterrows():
    sample_id = str(row["sample_id"])
    gold_order = parse_order(
        row["gold_order"]
    )

    full_entropy = float(
        row[
            "full_order_normalized_entropy"
        ]
    )

    fallback_target = (
        full_entropy
        >= FULL_ENTROPY_GATE
    )

    # 기존 baseline 예측
    baseline_text = None

    for candidate_column in [
        "base_prediction_10_30_60",
        "base_final_order",
        "prediction",
    ]:
        if candidate_column in row.index:
            value = row[candidate_column]

            if pd.notna(value):
                baseline_text = (
                    order_to_text(
                        parse_order(value)
                    )
                )
                break

    # --------------------------------------------------------
    # Stage 1: first 3-way
    # --------------------------------------------------------
    first_branches = {
        "order": {
            "prediction": int(
                row[
                    "order_pos1_prediction"
                ]
            ),
            "normalized_entropy": float(
                row[
                    "order_pos1_normalized_entropy"
                ]
            ),
            "margin": float(
                row["order_pos1_margin"]
            ),
        },

        "endpoint": {
            "prediction": int(
                row[
                    "endpoint_first_prediction"
                ]
            ),
            "normalized_entropy": float(
                row[
                    "endpoint_first_normalized_entropy"
                ]
            ),
            "margin": float(
                row["endpoint_first_margin"]
            ),
        },

        "pairwise": {
            "prediction": int(
                row[
                    "pairwise_first_prediction"
                ]
            ),
            "normalized_entropy": float(
                row[
                    "pairwise_first_normalized_entropy"
                ]
            ),
            "margin": float(
                row["pairwise_first_margin"]
            ),
        },
    }

    first_predictions = {
        name: values["prediction"]
        for name, values in first_branches.items()
    }

    first_all_agree = (
        len(set(first_predictions.values()))
        == 1
    )

    first_pattern = threeway_pattern(
        first_predictions
    )

    if first_all_agree:
        first_selected_branch = (
            "all_agree"
        )

        selected_first = int(
            first_predictions["order"]
        )

    else:
        first_selected_branch = (
            choose_lowest_entropy_branch(
                first_branches
            )
        )

        selected_first = int(
            first_branches[
                first_selected_branch
            ]["prediction"]
        )

    # --------------------------------------------------------
    # Stage 2: selected first 조건의 last
    # --------------------------------------------------------
    second_stage = calculate_second_stage(
        row,
        selected_first,
    )

    selected_last = int(
        second_stage[
            "selected_prediction"
        ]
    )

    # 안전장치
    if selected_last == selected_first:
        raise RuntimeError(
            f"{sample_id}: selected first and last "
            f"are both {selected_first}"
        )

    # --------------------------------------------------------
    # Middle
    # --------------------------------------------------------
    middle_stage = calculate_middle_stage(
        row,
        first=selected_first,
        last=selected_last,
    )

    adaptive_order = [
        selected_first,
        *middle_stage[
            "selected_middle"
        ],
        selected_last,
    ]

    adaptive_text = order_to_text(
        adaptive_order
    )

    # Gate 밖에서는 baseline 유지.
    # baseline 컬럼이 없다면 현재 adaptive 결과를 사용.
    if not fallback_target:
        final_prediction_text = (
            baseline_text
            if baseline_text is not None
            else adaptive_text
        )

        decision_type = (
            "baseline_not_gated"
        )
    else:
        final_prediction_text = (
            adaptive_text
        )

        if first_all_agree:
            if second_stage["all_agree"]:
                decision_type = (
                    "gated_endpoints_agree_"
                    "middle_only"
                )
            else:
                decision_type = (
                    "gated_first_agree_"
                    "second_entropy_select"
                )
        else:
            decision_type = (
                "gated_first_entropy_select"
            )

    baseline_exact = (
        int(
            baseline_text
            == order_to_text(gold_order)
        )
        if baseline_text is not None
        else np.nan
    )

    adaptive_exact = int(
        adaptive_text
        == order_to_text(gold_order)
    )

    final_exact = int(
        final_prediction_text
        == order_to_text(gold_order)
    )

    result_rows.append({
        "sample_id": sample_id,
        "gold_order": (
            order_to_text(gold_order)
        ),

        "full_order_normalized_entropy": (
            full_entropy
        ),
        "fallback_target": int(
            fallback_target
        ),

        "baseline_prediction": (
            baseline_text
        ),
        "adaptive_prediction_if_gated": (
            adaptive_text
        ),
        "final_prediction": (
            final_prediction_text
        ),

        "baseline_exact": baseline_exact,
        "adaptive_exact_if_gated": (
            adaptive_exact
        ),
        "final_exact": final_exact,

        "decision_type": decision_type,

        # Stage 1
        "first_threeway_predictions": (
            f"{first_predictions['order']}"
            f"|{first_predictions['endpoint']}"
            f"|{first_predictions['pairwise']}"
        ),
        "first_threeway_pattern": (
            first_pattern
        ),
        "first_all_agree": int(
            first_all_agree
        ),
        "first_selected_branch": (
            first_selected_branch
        ),
        "selected_first": (
            selected_first
        ),

        "first_order_entropy": (
            first_branches["order"][
                "normalized_entropy"
            ]
        ),
        "first_endpoint_entropy": (
            first_branches["endpoint"][
                "normalized_entropy"
            ]
        ),
        "first_pairwise_entropy": (
            first_branches["pairwise"][
                "normalized_entropy"
            ]
        ),

        # Stage 2
        "second_threeway_predictions": (
            f"{second_stage['predictions']['order']}"
            f"|{second_stage['predictions']['endpoint']}"
            f"|{second_stage['predictions']['pairwise']}"
        ),
        "second_threeway_pattern": (
            second_stage["pattern"]
        ),
        "second_all_agree": int(
            second_stage["all_agree"]
        ),
        "second_selected_branch": (
            second_stage[
                "selected_branch"
            ]
        ),
        "selected_last": selected_last,

        "second_order_entropy": (
            second_stage["branches"][
                "order"
            ]["normalized_entropy"]
        ),
        "second_endpoint_entropy": (
            second_stage["branches"][
                "endpoint"
            ]["normalized_entropy"]
        ),
        "second_pairwise_entropy": (
            second_stage["branches"][
                "pairwise"
            ]["normalized_entropy"]
        ),

        # Middle
        "middle_order_prediction": (
            middle_stage[
                "order_prediction"
            ]
        ),
        "middle_pairwise_prediction": (
            middle_stage[
                "pairwise_prediction"
            ]
        ),
        "middle_agree": int(
            middle_stage["agree"]
        ),
        "middle_selected_branch": (
            middle_stage[
                "selected_branch"
            ]
        ),
        "middle_order_entropy": (
            middle_stage[
                "order_normalized_entropy"
            ]
        ),
        "middle_pairwise_entropy": (
            middle_stage[
                "pairwise_normalized_entropy"
            ]
        ),
    })


adaptive_result_df = pd.DataFrame(
    result_rows
)


# ------------------------------------------------------------
# 5. 평가 요약
# ------------------------------------------------------------
gated_df = adaptive_result_df[
    adaptive_result_df[
        "fallback_target"
    ] == 1
].copy()

changed_mask = (
    adaptive_result_df[
        "baseline_prediction"
    ].notna()
    &
    (
        adaptive_result_df[
            "baseline_prediction"
        ]
        !=
        adaptive_result_df[
            "final_prediction"
        ]
    )
)

wrong_to_right = int(
    (
        changed_mask
        & (
            adaptive_result_df[
                "baseline_exact"
            ] == 0
        )
        & (
            adaptive_result_df[
                "final_exact"
            ] == 1
        )
    ).sum()
)

right_to_wrong = int(
    (
        changed_mask
        & (
            adaptive_result_df[
                "baseline_exact"
            ] == 1
        )
        & (
            adaptive_result_df[
                "final_exact"
            ] == 0
        )
    ).sum()
)

summary = {
    "rows": len(adaptive_result_df),
    "full_entropy_gate": (
        FULL_ENTROPY_GATE
    ),
    "fallback_count": int(
        adaptive_result_df[
            "fallback_target"
        ].sum()
    ),
    "fallback_rate": float(
        adaptive_result_df[
            "fallback_target"
        ].mean()
    ),

    "baseline_exact": (
        float(
            adaptive_result_df[
                "baseline_exact"
            ].mean()
        )
        if adaptive_result_df[
            "baseline_exact"
        ].notna().any()
        else np.nan
    ),

    "adaptive_final_exact": float(
        adaptive_result_df[
            "final_exact"
        ].mean()
    ),

    "gated_baseline_exact": (
        float(
            gated_df[
                "baseline_exact"
            ].mean()
        )
        if (
            len(gated_df) > 0
            and gated_df[
                "baseline_exact"
            ].notna().any()
        )
        else np.nan
    ),

    "gated_adaptive_exact": (
        float(
            gated_df[
                "final_exact"
            ].mean()
        )
        if len(gated_df) > 0
        else np.nan
    ),

    "changed_count": int(
        changed_mask.sum()
    ),
    "wrong_to_right": wrong_to_right,
    "right_to_wrong": right_to_wrong,
    "net_exact_change": (
        wrong_to_right
        - right_to_wrong
    ),

    "first_all_agree_rate_in_gate": (
        float(
            gated_df[
                "first_all_agree"
            ].mean()
        )
        if len(gated_df) > 0
        else np.nan
    ),

    "second_all_agree_rate_in_gate": (
        float(
            gated_df[
                "second_all_agree"
            ].mean()
        )
        if len(gated_df) > 0
        else np.nan
    ),
}

summary_df = pd.DataFrame(
    [summary]
)


# ------------------------------------------------------------
# 6. decision 유형별 성능
# ------------------------------------------------------------
decision_summary_df = (
    adaptive_result_df
    .groupby(
        "decision_type",
        dropna=False,
    )
    .agg(
        rows=("sample_id", "size"),
        baseline_exact=(
            "baseline_exact",
            "mean",
        ),
        final_exact=(
            "final_exact",
            "mean",
        ),
        changed=(
            "baseline_prediction",
            lambda series: np.nan,
        ),
    )
    .reset_index()
)

# changed는 별도 계산
decision_changed = (
    adaptive_result_df
    .assign(
        changed=(
            adaptive_result_df[
                "baseline_prediction"
            ].notna()
            &
            (
                adaptive_result_df[
                    "baseline_prediction"
                ]
                !=
                adaptive_result_df[
                    "final_prediction"
                ]
            )
        )
    )
    .groupby(
        "decision_type"
    )["changed"]
    .sum()
)

decision_summary_df["changed"] = (
    decision_summary_df[
        "decision_type"
    ].map(decision_changed)
)


# ------------------------------------------------------------
# 7. 저장
# ------------------------------------------------------------
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

DETAIL_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_entropy098_"
        "hierarchical_adaptive_detail_"
        f"{timestamp}.csv"
    )
)

SUMMARY_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_entropy098_"
        "hierarchical_adaptive_summary_"
        f"{timestamp}.csv"
    )
)

DECISION_SUMMARY_PATH = (
    OUTPUT_RUN_DIR
    / (
        "eval200_entropy098_"
        "hierarchical_adaptive_by_decision_"
        f"{timestamp}.csv"
    )
)

adaptive_result_df.to_csv(
    DETAIL_PATH,
    index=False,
)

summary_df.to_csv(
    SUMMARY_PATH,
    index=False,
)

decision_summary_df.to_csv(
    DECISION_SUMMARY_PATH,
    index=False,
)


display(summary_df)
display(decision_summary_df)

display(
    adaptive_result_df[
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "fallback_target",
            "first_threeway_predictions",
            "first_selected_branch",
            "selected_first",
            "second_threeway_predictions",
            "second_selected_branch",
            "selected_last",
            "middle_selected_branch",
            "baseline_prediction",
            "final_prediction",
            "baseline_exact",
            "final_exact",
            "decision_type",
        ]
    ].head(50)
)

print()
print("detail:")
print(DETAIL_PATH)

print()
print("summary:")
print(SUMMARY_PATH)

print()
print("decision summary:")
print(DECISION_SUMMARY_PATH)

,rows,full_entropy_gate,fallback_count,fallback_rate,baseline_exact,adaptive_final_exact,gated_baseline_exact,gated_adaptive_exact,changed_count,wrong_to_right,right_to_wrong,net_exact_change,first_all_agree_rate_in_gate,second_all_agree_rate_in_gate
0,200,0.98,70,0.35,0.37,0.375,0.1,0.114286,30,5,4,1,0.514286,0.328571


,decision_type,rows,baseline_exact,final_exact,changed
0,baseline_not_gated,130,0.515385,0.515385,0
1,gated_endpoints_agree_middle_only,12,0.166667,0.166667,0
2,gated_first_agree_second_entropy_select,24,0.125000,0.166667,9
3,gated_first_entropy_select,34,0.058824,0.058824,21


,sample_id,gold_order,full_order_normalized_entropy,fallback_target,first_threeway_predictions,first_selected_branch,selected_first,second_threeway_predictions,second_selected_branch,selected_last,middle_selected_branch,baseline_prediction,final_prediction,baseline_exact,final_exact,decision_type
0,x6AwMw,2 3 4 1,0.919593,0,3|2|2,endpoint,2,1|4|1,endpoint,4,all_agree,2 3 1 4,2 3 1 4,0,0,baseline_not_gated
1,jOilDx,1 3 2 4,0.995850,1,1|1|1,all_agree,1,2|4|2,endpoint,4,pairwise,1 3 4 2,1 3 2 4,0,1,gated_first_agree_second_entropy_select
2,B6W1Qk,2 1 3 4,0.952003,0,2|2|2,all_agree,2,4|4|4,all_agree,4,all_agree,2 1 3 4,2 1 3 4,1,1,baseline_not_gated
3,AolL8G,4 2 3 1,0.997393,1,4|4|4,all_agree,4,3|3|3,all_agree,3,all_agree,4 2 1 3,4 2 1 3,0,0,gated_endpoints_agree_middle_only
4,MjIq4T,3 4 2 1,0.898505,0,3|3|3,all_agree,3,1|1|1,all_agree,1,all_agree,3 4 2 1,3 4 2 1,1,1,baseline_not_gated
5,NjCw77,2 3 4 1,0.913713,0,2|2|2,all_agree,2,1|1|1,all_agree,1,all_agree,2 3 4 1,2 3 4 1,1,1,baseline_not_gated
6,1SOLbE,1 2 3 4,0.977186,0,4|4|4,all_agree,4,2|2|2,all_agree,2,all_agree,4 1 3 2,4 1 3 2,0,0,baseline_not_gated
7,rht7VJ,2 1 4 3,0.956753,0,4|4|4,all_agree,4,3|3|3,all_agree,3,all_agree,4 2 1 3,4 2 1 3,0,0,baseline_not_gated
8,tUsgQQ,1 4 2 3,0.952219,0,4|4|1,endpoint,4,2|3|2,endpoint,3,all_agree,4 1 2 3,4 1 2 3,0,0,baseline_not_gated
9,TWfXii,2 1 4 3,0.993056,1,4|2|3,pairwise,3,1|4|4,endpoint,4,order,3 2 1 4,3 2 1 4,0,0,gated_first_entropy_select



detail:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_entropy098_hierarchical_adaptive_detail_20260724_092619.csv

summary:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint020_mrope_first1_v4/eval200_entropy098_hierarchical_adaptive_summary_20260724_092619.csv

decision summary:
/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_7b_multitask_bipair_conditional_v1/runs/20260723_full24_order_first_last_from_checkpoint1200_v2_pairwise_metrics/full24_order_first_last_calibrated_pairwise_metrics/eval_step300_exhaustive/eval100_step300_endpoint02

In [26]:
# ============================================================
# Eval200: priority-preserving conservative fallback
#
# 핵심 변경:
# 1. 무조건 first부터 시작하지 않음
# 2. 더 확실한 endpoint를 10:30:60 fused 결과로 고정
# 3. 남은 endpoint는 entropy hard-select 대신 확률 융합
# 4. middle은 order 0.3 + pairwise 0.7
# 5. consensus가 충분할 때만 baseline 변경
# ============================================================

import re
import numpy as np
import pandas as pd


FULL_ENTROPY_GATE = 0.98

# 남은 endpoint 선택:
# conditional order / direct endpoint / pairwise
STAGE2_WEIGHTS = {
    "conditional": 0.10,
    "endpoint": 0.30,
    "pairwise": 0.60,
}

# middle 선택
MIDDLE_ORDER_WEIGHT = 0.30
MIDDLE_PAIRWISE_WEIGHT = 0.70

FRAMES = [1, 2, 3, 4]
EPS = 1e-12


# ------------------------------------------------------------
# 0. 입력 준비
# ------------------------------------------------------------
if "raw_df_200" not in globals():
    raise RuntimeError("raw_df_200이 없습니다.")

source_df = raw_df_200.copy()
source_df["sample_id"] = source_df["sample_id"].astype(str)


# baseline이 raw에 없다면 readable에서 연결
if (
    "base_prediction_10_30_60"
    not in source_df.columns
):
    if "readable_df_200" not in globals():
        raise RuntimeError(
            "baseline prediction이 없고 "
            "readable_df_200도 없습니다."
        )

    baseline_lookup = (
        readable_df_200[
            [
                "sample_id",
                "base_prediction_10_30_60",
            ]
        ]
        .copy()
    )

    baseline_lookup["sample_id"] = (
        baseline_lookup["sample_id"]
        .astype(str)
    )

    source_df = source_df.merge(
        baseline_lookup,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def parse_order(value):
    numbers = [
        int(number)
        for number in re.findall(
            r"[1-4]",
            str(value),
        )
    ]

    if (
        len(numbers) != 4
        or sorted(numbers) != FRAMES
    ):
        raise ValueError(
            f"잘못된 order: {value!r}"
        )

    return numbers


def order_text(order):
    return " ".join(
        map(str, order)
    )


def normalize_dict(values):
    cleaned = {
        int(key): max(float(value), 0.0)
        for key, value in values.items()
    }

    total = sum(cleaned.values())

    if total <= 0:
        probability = 1.0 / len(cleaned)
        return {
            key: probability
            for key in cleaned
        }

    return {
        key: value / total
        for key, value in cleaned.items()
    }


def argmax_dict(values):
    return int(
        max(
            values,
            key=values.get,
        )
    )


def endpoint_priority(row):
    """
    앞 셀에서 계산한 first/last confidence priority 사용.
    """

    priority = str(
        row.get(
            "final_endpoint_priority_majority",
            "",
        )
    )

    if priority in {"first", "last"}:
        return priority

    # fallback: fused endpoint entropy가 낮은 쪽
    first_entropy = float(
        row[
            "final_first_normalized_entropy"
        ]
    )

    last_entropy = float(
        row[
            "final_last_normalized_entropy"
        ]
    )

    return (
        "first"
        if first_entropy <= last_entropy
        else "last"
    )


def fixed_endpoint_support(
    row,
    endpoint_type,
    prediction,
):
    """
    fused endpoint prediction을
    order / endpoint / pairwise 중 몇 개가 지지하는지.
    """

    if endpoint_type == "first":
        branch_predictions = [
            int(row["order_pos1_prediction"]),
            int(row["endpoint_first_prediction"]),
            int(row["pairwise_first_prediction"]),
        ]

    else:
        branch_predictions = [
            int(row["order_pos4_prediction"]),
            int(row["endpoint_last_prediction"]),
            int(row["pairwise_last_prediction"]),
        ]

    support = sum(
        prediction == branch_prediction
        for branch_prediction
        in branch_predictions
    )

    return support, branch_predictions


# ------------------------------------------------------------
# 2. 고정 endpoint 이후 나머지 endpoint 선택
# ------------------------------------------------------------
def select_remaining_endpoint(
    row,
    fixed_type,
    fixed_value,
):
    candidates = [
        frame
        for frame in FRAMES
        if frame != fixed_value
    ]

    if fixed_type == "first":
        remaining_type = "last"

        conditional = {
            frame: float(
                row[
                    f"p_cond_last_given_first_"
                    f"{fixed_value}_{frame}"
                ]
            )
            for frame in candidates
        }

        endpoint = {
            frame: float(
                row[
                    f"p_endpoint_last_{frame}"
                ]
            )
            for frame in candidates
        }

        pairwise = {
            frame: float(
                row[
                    f"p_pairwise_last_{frame}"
                ]
            )
            for frame in candidates
        }

    else:
        remaining_type = "first"

        conditional = {
            frame: float(
                row[
                    f"p_cond_first_given_last_"
                    f"{fixed_value}_{frame}"
                ]
            )
            for frame in candidates
        }

        endpoint = {
            frame: float(
                row[
                    f"p_endpoint_first_{frame}"
                ]
            )
            for frame in candidates
        }

        pairwise = {
            frame: float(
                row[
                    f"p_pairwise_first_{frame}"
                ]
            )
            for frame in candidates
        }

    conditional = normalize_dict(conditional)
    endpoint = normalize_dict(endpoint)
    pairwise = normalize_dict(pairwise)

    branch_predictions = {
        "conditional": argmax_dict(conditional),
        "endpoint": argmax_dict(endpoint),
        "pairwise": argmax_dict(pairwise),
    }

    # 서로 다른 head entropy를 비교하지 않고
    # 확률분포 자체를 기존 10:30:60 방식으로 융합
    fused = normalize_dict({
        frame: (
            STAGE2_WEIGHTS["conditional"]
            * conditional[frame]
            +
            STAGE2_WEIGHTS["endpoint"]
            * endpoint[frame]
            +
            STAGE2_WEIGHTS["pairwise"]
            * pairwise[frame]
        )
        for frame in candidates
    })

    prediction = argmax_dict(fused)

    support = sum(
        prediction == branch_prediction
        for branch_prediction
        in branch_predictions.values()
    )

    sorted_probabilities = sorted(
        fused.values(),
        reverse=True,
    )

    margin = float(
        sorted_probabilities[0]
        - sorted_probabilities[1]
    )

    return {
        "remaining_type": remaining_type,
        "prediction": prediction,
        "support": support,
        "margin": margin,
        "branch_predictions": (
            branch_predictions
        ),
        "fused_distribution": fused,
    }


# ------------------------------------------------------------
# 3. endpoint가 정해진 뒤 middle 선택
# ------------------------------------------------------------
def select_middle(
    row,
    first,
    last,
):
    middle = [
        frame
        for frame in FRAMES
        if frame not in {first, last}
    ]

    frame_a, frame_b = middle

    order_distribution = normalize_dict({
        0: float(
            row[
                f"p_middle_order_given_"
                f"{first}_{last}_"
                f"{frame_a}{frame_b}"
            ]
        ),
        1: float(
            row[
                f"p_middle_order_given_"
                f"{first}_{last}_"
                f"{frame_b}{frame_a}"
            ]
        ),
    })

    pairwise_distribution = normalize_dict({
        0: float(
            row[
                f"p_middle_pairwise_given_"
                f"{first}_{last}_"
                f"{frame_a}{frame_b}"
            ]
        ),
        1: float(
            row[
                f"p_middle_pairwise_given_"
                f"{first}_{last}_"
                f"{frame_b}{frame_a}"
            ]
        ),
    })

    fused = normalize_dict({
        choice: (
            MIDDLE_ORDER_WEIGHT
            * order_distribution[choice]
            +
            MIDDLE_PAIRWISE_WEIGHT
            * pairwise_distribution[choice]
        )
        for choice in [0, 1]
    })

    selected_ab = (
        argmax_dict(fused) == 0
    )

    order_prediction = (
        0
        if order_distribution[0]
           >= order_distribution[1]
        else 1
    )

    pairwise_prediction = (
        0
        if pairwise_distribution[0]
           >= pairwise_distribution[1]
        else 1
    )

    branch_agree = (
        order_prediction
        == pairwise_prediction
    )

    selected_middle = (
        [frame_a, frame_b]
        if selected_ab
        else [frame_b, frame_a]
    )

    return {
        "selected_middle": selected_middle,
        "branch_agree": branch_agree,
        "order_prediction": order_prediction,
        "pairwise_prediction": (
            pairwise_prediction
        ),
        "fused_probability": float(
            max(fused.values())
        ),
    }


# ------------------------------------------------------------
# 4. 샘플별 priority-preserving fallback
# ------------------------------------------------------------
result_rows = []

for _, row in source_df.iterrows():
    sample_id = str(row["sample_id"])

    gold = parse_order(
        row["gold_order"]
    )

    baseline = parse_order(
        row[
            "base_prediction_10_30_60"
        ]
    )

    baseline_text = order_text(baseline)
    gold_text = order_text(gold)

    fallback_target = (
        float(
            row[
                "full_order_normalized_entropy"
            ]
        )
        >= FULL_ENTROPY_GATE
    )

    priority = endpoint_priority(row)

    # 더 확실한 endpoint는
    # entropy-selected branch가 아니라
    # 기존 10:30:60 fused prediction으로 고정
    if priority == "first":
        fixed_type = "first"
        fixed_value = int(
            row[
                "final_first_prediction"
            ]
        )

    else:
        fixed_type = "last"
        fixed_value = int(
            row[
                "final_last_prediction"
            ]
        )

    (
        fixed_support,
        fixed_branch_predictions,
    ) = fixed_endpoint_support(
        row,
        fixed_type,
        fixed_value,
    )

    remaining_result = (
        select_remaining_endpoint(
            row,
            fixed_type,
            fixed_value,
        )
    )

    if fixed_type == "first":
        selected_first = fixed_value
        selected_last = int(
            remaining_result["prediction"]
        )

    else:
        selected_last = fixed_value
        selected_first = int(
            remaining_result["prediction"]
        )

    middle_result = select_middle(
        row,
        first=selected_first,
        last=selected_last,
    )

    candidate = [
        selected_first,
        *middle_result[
            "selected_middle"
        ],
        selected_last,
    ]

    candidate_text = order_text(
        candidate
    )

    # --------------------------------------------------------
    # 여러 보수적 정책을 동시에 평가
    # --------------------------------------------------------

    # A. gate이면 항상 적용
    apply_always = fallback_target

    # B. 양쪽 endpoint가 각각 2개 branch 이상 지지를 받을 때
    apply_support_2_2 = (
        fallback_target
        and fixed_support >= 2
        and remaining_result["support"] >= 2
    )

    # C. 먼저 고정한 endpoint는 3-way 완전 일치,
    #    나머지는 2개 이상 지지
    apply_support_3_2 = (
        fallback_target
        and fixed_support == 3
        and remaining_result["support"] >= 2
    )

    # D. 양쪽 모두 3-way 완전 일치
    apply_support_3_3 = (
        fallback_target
        and fixed_support == 3
        and remaining_result["support"] == 3
    )

    predictions = {
        "priority_always": (
            candidate_text
            if apply_always
            else baseline_text
        ),
        "priority_support_2_2": (
            candidate_text
            if apply_support_2_2
            else baseline_text
        ),
        "priority_support_3_2": (
            candidate_text
            if apply_support_3_2
            else baseline_text
        ),
        "priority_support_3_3": (
            candidate_text
            if apply_support_3_3
            else baseline_text
        ),
    }

    output = {
        "sample_id": sample_id,
        "gold_order": gold_text,
        "baseline_prediction": (
            baseline_text
        ),
        "candidate_prediction": (
            candidate_text
        ),
        "fallback_target": int(
            fallback_target
        ),
        "priority": priority,
        "fixed_type": fixed_type,
        "fixed_value": fixed_value,
        "fixed_support": fixed_support,
        "fixed_branch_predictions": (
            "|".join(
                map(
                    str,
                    fixed_branch_predictions,
                )
            )
        ),
        "remaining_type": (
            remaining_result[
                "remaining_type"
            ]
        ),
        "remaining_prediction": (
            remaining_result[
                "prediction"
            ]
        ),
        "remaining_support": (
            remaining_result[
                "support"
            ]
        ),
        "remaining_margin": (
            remaining_result[
                "margin"
            ]
        ),
        "remaining_branch_predictions": (
            "|".join(
                str(
                    remaining_result[
                        "branch_predictions"
                    ][name]
                )
                for name in [
                    "conditional",
                    "endpoint",
                    "pairwise",
                ]
            )
        ),
        "middle_agree": int(
            middle_result[
                "branch_agree"
            ]
        ),
    }

    output["baseline_exact"] = int(
        baseline_text == gold_text
    )

    output["baseline_first_exact"] = int(
        baseline[0] == gold[0]
    )

    output["baseline_last_exact"] = int(
        baseline[-1] == gold[-1]
    )

    output["baseline_both_endpoints_exact"] = int(
        baseline[0] == gold[0]
        and baseline[-1] == gold[-1]
    )

    for name, prediction in predictions.items():
        output[f"{name}_prediction"] = (
            prediction
        )

        output[f"{name}_exact"] = int(
            prediction == gold_text
        )

    result_rows.append(output)


priority_fallback_df = pd.DataFrame(
    result_rows
)


# ------------------------------------------------------------
# 5. 전략별 요약
# ------------------------------------------------------------
strategy_rows = []

baseline_exact = float(
    priority_fallback_df[
        "baseline_exact"
    ].mean()
)

strategy_rows.append({
    "strategy": "baseline",
    "exact": baseline_exact,
    "changed_count": 0,
    "wrong_to_right": 0,
    "right_to_wrong": 0,
    "net_change": 0,
})

for strategy in [
    "priority_always",
    "priority_support_2_2",
    "priority_support_3_2",
    "priority_support_3_3",
]:
    prediction_column = (
        f"{strategy}_prediction"
    )

    exact_column = (
        f"{strategy}_exact"
    )

    changed = (
        priority_fallback_df[
            prediction_column
        ]
        !=
        priority_fallback_df[
            "baseline_prediction"
        ]
    )

    wrong_to_right = int(
        (
            changed
            & (
                priority_fallback_df[
                    "baseline_exact"
                ] == 0
            )
            & (
                priority_fallback_df[
                    exact_column
                ] == 1
            )
        ).sum()
    )

    right_to_wrong = int(
        (
            changed
            & (
                priority_fallback_df[
                    "baseline_exact"
                ] == 1
            )
            & (
                priority_fallback_df[
                    exact_column
                ] == 0
            )
        ).sum()
    )

    strategy_rows.append({
        "strategy": strategy,
        "exact": float(
            priority_fallback_df[
                exact_column
            ].mean()
        ),
        "changed_count": int(
            changed.sum()
        ),
        "wrong_to_right": (
            wrong_to_right
        ),
        "right_to_wrong": (
            right_to_wrong
        ),
        "net_change": (
            wrong_to_right
            - right_to_wrong
        ),
    })


priority_summary_df = (
    pd.DataFrame(strategy_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "changed_count",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. baseline 오류가 endpoint 문제인지 middle 문제인지 확인
# ------------------------------------------------------------
gated_wrong_df = priority_fallback_df[
    (
        priority_fallback_df[
            "fallback_target"
        ] == 1
    )
    &
    (
        priority_fallback_df[
            "baseline_exact"
        ] == 0
    )
].copy()


def classify_baseline_error(row):
    first_correct = (
        row["baseline_first_exact"] == 1
    )

    last_correct = (
        row["baseline_last_exact"] == 1
    )

    if first_correct and last_correct:
        return "both_endpoints_correct_middle_error"

    if first_correct or last_correct:
        return "one_endpoint_wrong"

    return "both_endpoints_wrong"


gated_wrong_df[
    "baseline_error_type"
] = gated_wrong_df.apply(
    classify_baseline_error,
    axis=1,
)

error_type_summary_df = (
    gated_wrong_df[
        "baseline_error_type"
    ]
    .value_counts()
    .rename_axis(
        "baseline_error_type"
    )
    .reset_index(
        name="rows"
    )
)

error_type_summary_df[
    "rate"
] = (
    error_type_summary_df["rows"]
    / len(gated_wrong_df)
)


display(priority_summary_df)

print(
    "\nGated baseline 오류 유형:"
)
display(error_type_summary_df)

print(
    "\n변경된 샘플 예시:"
)

best_strategy = (
    priority_summary_df.loc[
        priority_summary_df[
            "strategy"
        ] != "baseline",
        "strategy",
    ].iloc[0]
)

best_prediction_column = (
    f"{best_strategy}_prediction"
)

display(
    priority_fallback_df[
        priority_fallback_df[
            best_prediction_column
        ]
        !=
        priority_fallback_df[
            "baseline_prediction"
        ]
    ][
        [
            "sample_id",
            "gold_order",
            "baseline_prediction",
            "candidate_prediction",
            "priority",
            "fixed_support",
            "remaining_support",
            "fixed_branch_predictions",
            "remaining_branch_predictions",
            "baseline_exact",
            f"{best_strategy}_exact",
        ]
    ].head(50)
)

,strategy,exact,changed_count,wrong_to_right,right_to_wrong,net_change
0,baseline,0.37,0,0,0,0
1,priority_always,0.37,0,0,0,0
2,priority_support_2_2,0.37,0,0,0,0
3,priority_support_3_2,0.37,0,0,0,0
4,priority_support_3_3,0.37,0,0,0,0



Gated baseline 오류 유형:


,baseline_error_type,rows,rate
0,one_endpoint_wrong,30,0.476190
1,both_endpoints_wrong,29,0.460317
2,both_endpoints_correct_middle_error,4,0.063492



변경된 샘플 예시:


,sample_id,gold_order,baseline_prediction,candidate_prediction,priority,fixed_support,remaining_support,fixed_branch_predictions,remaining_branch_predictions,baseline_exact,priority_always_exact


In [27]:
# ============================================================
# 기존 hierarchical fallback의 endpoint / middle 분해 평가
# 전제: adaptive_result_df가 존재
# ============================================================

import re
import numpy as np
import pandas as pd


FRAMES = [1, 2, 3, 4]


def parse_order(value):
    values = [
        int(number)
        for number in re.findall(
            r"[1-4]",
            str(value),
        )
    ]

    if (
        len(values) != 4
        or sorted(values) != FRAMES
    ):
        raise ValueError(
            f"Invalid order: {value!r} -> {values}"
        )

    return tuple(values)


if "adaptive_result_df" not in globals():
    raise RuntimeError(
        "adaptive_result_df가 없습니다. "
        "기존 hierarchical fallback 셀부터 실행하세요."
    )


diag_df = adaptive_result_df.copy()

adaptive_prediction_column = (
    "adaptive_prediction_if_gated"
    if "adaptive_prediction_if_gated"
       in diag_df.columns
    else "final_prediction"
)


gold_orders = (
    diag_df["gold_order"]
    .map(parse_order)
)

baseline_orders = (
    diag_df["baseline_prediction"]
    .map(parse_order)
)

adaptive_orders = (
    diag_df[adaptive_prediction_column]
    .map(parse_order)
)


def add_metrics(
    frame,
    prefix,
    predictions,
    golds,
):
    frame[f"{prefix}_first_exact"] = [
        int(prediction[0] == gold[0])
        for prediction, gold
        in zip(predictions, golds)
    ]

    frame[f"{prefix}_last_exact"] = [
        int(prediction[-1] == gold[-1])
        for prediction, gold
        in zip(predictions, golds)
    ]

    frame[
        f"{prefix}_both_endpoints_exact"
    ] = [
        int(
            prediction[0] == gold[0]
            and prediction[-1] == gold[-1]
        )
        for prediction, gold
        in zip(predictions, golds)
    ]

    frame[
        f"{prefix}_full_exact"
    ] = [
        int(prediction == gold)
        for prediction, gold
        in zip(predictions, golds)
    ]


add_metrics(
    diag_df,
    "baseline",
    baseline_orders,
    gold_orders,
)

add_metrics(
    diag_df,
    "adaptive",
    adaptive_orders,
    gold_orders,
)


gated_df = diag_df[
    diag_df["fallback_target"] == 1
].copy()


summary_rows = []

for decision_type, group in [
    ("ALL_GATED", gated_df),
    *list(
        gated_df.groupby(
            "decision_type"
        )
    ),
]:
    for decoder in [
        "baseline",
        "adaptive",
    ]:
        both_correct = (
            group[
                f"{decoder}_both_endpoints_exact"
            ] == 1
        )

        summary_rows.append({
            "decision_type": decision_type,
            "decoder": decoder,
            "rows": len(group),

            "first_exact": float(
                group[
                    f"{decoder}_first_exact"
                ].mean()
            ),

            "last_exact": float(
                group[
                    f"{decoder}_last_exact"
                ].mean()
            ),

            "both_endpoints_exact": float(
                group[
                    f"{decoder}_both_endpoints_exact"
                ].mean()
            ),

            "full_exact": float(
                group[
                    f"{decoder}_full_exact"
                ].mean()
            ),

            "correct_endpoint_pair_count": int(
                both_correct.sum()
            ),

            # endpoint가 둘 다 맞은 샘플 중
            # middle 순서까지 맞힌 비율
            "middle_accuracy_given_correct_endpoints": (
                float(
                    group.loc[
                        both_correct,
                        f"{decoder}_full_exact",
                    ].mean()
                )
                if both_correct.any()
                else np.nan
            ),
        })


endpoint_middle_summary_df = pd.DataFrame(
    summary_rows
)

display(
    endpoint_middle_summary_df
)


# ------------------------------------------------------------
# 변경된 샘플에서 endpoint가 실제로 개선됐는지 확인
# ------------------------------------------------------------
changed_df = gated_df[
    gated_df["baseline_prediction"]
    != gated_df[adaptive_prediction_column]
].copy()

changed_df["endpoint_pair_delta"] = (
    changed_df[
        "adaptive_both_endpoints_exact"
    ]
    -
    changed_df[
        "baseline_both_endpoints_exact"
    ]
)

changed_df["full_exact_delta"] = (
    changed_df[
        "adaptive_full_exact"
    ]
    -
    changed_df[
        "baseline_full_exact"
    ]
)


print("변경 샘플의 endpoint / full-order 변화")

display(
    changed_df.groupby(
        [
            "decision_type",
            "endpoint_pair_delta",
            "full_exact_delta",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        [
            "decision_type",
            "endpoint_pair_delta",
            "full_exact_delta",
        ]
    )
)

,decision_type,decoder,rows,first_exact,last_exact,both_endpoints_exact,full_exact,correct_endpoint_pair_count,middle_accuracy_given_correct_endpoints
0,ALL_GATED,baseline,70,0.400000,0.342857,0.157143,0.100000,11,0.636364
1,ALL_GATED,adaptive,70,0.357143,0.357143,0.185714,0.114286,13,0.615385
2,gated_endpoints_agree_middle_only,baseline,12,0.500000,0.583333,0.333333,0.166667,4,0.500000
3,gated_endpoints_agree_middle_only,adaptive,12,0.500000,0.583333,0.333333,0.166667,4,0.500000
4,gated_first_agree_second_entropy_select,baseline,24,0.583333,0.250000,0.208333,0.125000,5,0.600000
5,gated_first_agree_second_entropy_select,adaptive,24,0.541667,0.333333,0.250000,0.166667,6,0.666667
6,gated_first_entropy_select,baseline,34,0.235294,0.323529,0.058824,0.058824,2,1.000000
7,gated_first_entropy_select,adaptive,34,0.176471,0.294118,0.088235,0.058824,3,0.666667


변경 샘플의 endpoint / full-order 변화


,decision_type,endpoint_pair_delta,full_exact_delta,rows
0,gated_first_agree_second_entropy_select,-1,-1,2
1,gated_first_agree_second_entropy_select,0,0,4
2,gated_first_agree_second_entropy_select,1,1,3
3,gated_first_entropy_select,-1,-1,2
4,gated_first_entropy_select,0,0,16
5,gated_first_entropy_select,1,0,1
6,gated_first_entropy_select,1,1,2


In [29]:
# ============================================================
# Joint decoder 후속 진단
#
# 1. entropy threshold별 fallback 성능
# 2. global joint 성능
# 3. prediction order 분포
# 4. gold order별 성능
#
# 전제:
# joint_result_df가 이전 셀에서 생성되어 있어야 함
# ============================================================

import numpy as np
import pandas as pd


if "joint_result_df" not in globals():
    raise RuntimeError(
        "joint_result_df가 없습니다. "
        "24-order joint reranking 셀을 먼저 실행하세요."
    )


df = joint_result_df.copy()


# ------------------------------------------------------------
# 1. Threshold별 fallback 평가
# ------------------------------------------------------------
def evaluate_switch_strategy(
    frame,
    prediction_column,
    use_joint_mask,
    strategy_name,
):
    final_prediction = np.where(
        use_joint_mask,
        frame[prediction_column],
        frame["baseline_prediction"],
    )

    final_exact = (
        final_prediction
        == frame["gold_order"]
    ).astype(int)

    changed = (
        final_prediction
        != frame["baseline_prediction"]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (final_exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (final_exact == 0)
    )

    return {
        "strategy": strategy_name,
        "joint_rows": int(
            use_joint_mask.sum()
        ),
        "joint_rate": float(
            use_joint_mask.mean()
        ),
        "exact": float(
            final_exact.mean()
        ),
        "correct_count": int(
            final_exact.sum()
        ),
        "changed_count": int(
            changed.sum()
        ),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
    }


threshold_rows = []

for decoder in [
    "joint_equal",
    "joint_with_full",
]:
    prediction_column = (
        f"{decoder}_prediction"
    )

    # Entropy threshold fallback
    for threshold in np.arange(
        0.90,
        1.001,
        0.005,
    ):
        use_joint = (
            df[
                "full_order_normalized_entropy"
            ]
            >= threshold
        )

        threshold_rows.append(
            evaluate_switch_strategy(
                frame=df,
                prediction_column=(
                    prediction_column
                ),
                use_joint_mask=use_joint,
                strategy_name=(
                    f"{decoder}_entropy_"
                    f"{threshold:.3f}"
                ),
            )
        )

    # 전체 적용
    threshold_rows.append(
        evaluate_switch_strategy(
            frame=df,
            prediction_column=(
                prediction_column
            ),
            use_joint_mask=pd.Series(
                True,
                index=df.index,
            ),
            strategy_name=(
                f"{decoder}_global"
            ),
        )
    )


threshold_summary_df = (
    pd.DataFrame(threshold_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "changed_count",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print("Threshold / global 전략 비교")
display(
    threshold_summary_df.head(30)
)


# ------------------------------------------------------------
# 2. Gold / prediction 순서 분포 비교
# ------------------------------------------------------------
distribution_frames = []

for name, column in {
    "gold": "gold_order",
    "baseline": "baseline_prediction",
    "full24": "full24_prediction",
    "joint_equal": (
        "joint_equal_prediction"
    ),
    "joint_with_full": (
        "joint_with_full_prediction"
    ),
}.items():
    counts = (
        df[column]
        .value_counts()
        .rename_axis("order")
        .reset_index(name="count")
    )

    counts["rate"] = (
        counts["count"]
        / len(df)
    )

    counts["source"] = name

    distribution_frames.append(
        counts
    )


order_distribution_df = pd.concat(
    distribution_frames,
    ignore_index=True,
)

print("\n예측 순서 분포 상위 15개")
display(
    order_distribution_df
    .sort_values(
        [
            "source",
            "count",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "source",
        group_keys=False,
    )
    .head(15)
)


# ------------------------------------------------------------
# 3. 1 2 3 4 쏠림만 빠르게 확인
# ------------------------------------------------------------
canonical_order = "1 2 3 4"

canonical_summary = []

for name, column in {
    "gold": "gold_order",
    "baseline": "baseline_prediction",
    "full24": "full24_prediction",
    "joint_equal": (
        "joint_equal_prediction"
    ),
    "joint_with_full": (
        "joint_with_full_prediction"
    ),
}.items():
    count = int(
        (
            df[column]
            == canonical_order
        ).sum()
    )

    canonical_summary.append({
        "source": name,
        "canonical_count": count,
        "canonical_rate": (
            count / len(df)
        ),
    })


print("\n'1 2 3 4' 분포")
display(
    pd.DataFrame(
        canonical_summary
    )
)


# ------------------------------------------------------------
# 4. Gold order별 정확도
# ------------------------------------------------------------
gold_order_rows = []

for gold_order, group in df.groupby(
    "gold_order"
):
    row = {
        "gold_order": gold_order,
        "rows": len(group),
    }

    for decoder in [
        "baseline",
        "full24",
        "joint_equal",
        "joint_with_full",
    ]:
        row[f"{decoder}_exact"] = float(
            group[
                f"{decoder}_exact"
            ].mean()
        )

    gold_order_rows.append(row)


gold_order_summary_df = (
    pd.DataFrame(gold_order_rows)
    .sort_values(
        [
            "rows",
            "gold_order",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

print("\nGold order별 decoder 정확도")
display(
    gold_order_summary_df
)

Threshold / global 전략 비교


,strategy,joint_rows,joint_rate,exact,correct_count,changed_count,wrong_to_right,right_to_wrong,net_change
0,joint_with_full_entropy_0.940,138,0.690,0.445,89,73,21,6,15
1,joint_with_full_entropy_0.945,128,0.640,0.445,89,73,21,6,15
2,joint_with_full_entropy_0.950,122,0.610,0.440,88,69,19,5,14
3,joint_with_full_entropy_0.935,148,0.740,0.440,88,74,21,7,14
4,joint_with_full_entropy_0.930,154,0.770,0.440,88,75,21,7,14
5,joint_with_full_entropy_0.925,163,0.815,0.440,88,77,22,8,14
6,joint_equal_entropy_0.940,138,0.690,0.435,87,73,20,7,13
7,joint_equal_entropy_0.945,128,0.640,0.435,87,73,20,7,13
8,joint_with_full_entropy_0.920,173,0.865,0.435,87,78,22,9,13
9,joint_with_full_entropy_0.900,194,0.970,0.435,87,81,23,10,13



예측 순서 분포 상위 15개


,order,count,rate,source
24,1 2 3 4,28,0.140,baseline
25,1 3 2 4,19,0.095,baseline
26,4 2 3 1,15,0.075,baseline
27,4 1 3 2,14,0.070,baseline
28,4 2 1 3,11,0.055,baseline
...,...,...,...,...
106,3 4 2 1,7,0.035,joint_with_full
107,4 1 3 2,6,0.030,joint_with_full
108,3 2 4 1,6,0.030,joint_with_full
109,2 4 3 1,6,0.030,joint_with_full



'1 2 3 4' 분포


,source,canonical_count,canonical_rate
0,gold,32,0.160
1,baseline,28,0.140
2,full24,55,0.275
3,joint_equal,54,0.270
4,joint_with_full,55,0.275



Gold order별 decoder 정확도


,gold_order,rows,baseline_exact,full24_exact,joint_equal_exact,joint_with_full_exact
0,1 2 3 4,32,0.531250,0.750000,0.875000,0.906250
1,1 2 4 3,12,0.500000,0.583333,0.500000,0.500000
2,2 4 3 1,10,0.100000,0.300000,0.400000,0.400000
3,4 2 3 1,10,0.500000,0.500000,0.500000,0.500000
4,1 3 2 4,9,0.444444,0.333333,0.444444,0.444444
5,1 3 4 2,9,0.444444,0.555556,0.444444,0.444444
6,2 3 4 1,9,0.333333,0.555556,0.333333,0.333333
7,3 4 2 1,9,0.333333,0.222222,0.333333,0.333333
8,4 2 1 3,9,0.444444,0.333333,0.333333,0.333333
9,1 4 2 3,8,0.125000,0.000000,0.000000,0.000000


In [30]:
# ============================================================
# Final hybrid diagnostic
#
# baseline:
#   entropy < threshold
#
# joint_with_full:
#   entropy >= threshold
# ============================================================

import numpy as np
import pandas as pd


THRESHOLD = 0.945

if "joint_result_df" not in globals():
    raise RuntimeError(
        "joint_result_df가 없습니다."
    )


hybrid_df = joint_result_df.copy()

hybrid_df["use_joint"] = (
    hybrid_df[
        "full_order_normalized_entropy"
    ]
    >= THRESHOLD
)

hybrid_df["hybrid_prediction"] = np.where(
    hybrid_df["use_joint"],
    hybrid_df[
        "joint_with_full_prediction"
    ],
    hybrid_df[
        "baseline_prediction"
    ],
)

hybrid_df["hybrid_exact"] = (
    hybrid_df["hybrid_prediction"]
    ==
    hybrid_df["gold_order"]
).astype(int)

hybrid_df["changed"] = (
    hybrid_df["hybrid_prediction"]
    !=
    hybrid_df["baseline_prediction"]
)

hybrid_df["wrong_to_right"] = (
    hybrid_df["changed"]
    &
    (hybrid_df["baseline_exact"] == 0)
    &
    (hybrid_df["hybrid_exact"] == 1)
)

hybrid_df["right_to_wrong"] = (
    hybrid_df["changed"]
    &
    (hybrid_df["baseline_exact"] == 1)
    &
    (hybrid_df["hybrid_exact"] == 0)
)


print(
    "Hybrid exact:",
    hybrid_df["hybrid_exact"].mean(),
)

print(
    "Correct:",
    hybrid_df["hybrid_exact"].sum(),
    "/",
    len(hybrid_df),
)

print(
    "Changed:",
    hybrid_df["changed"].sum(),
)

print(
    "Wrong -> right:",
    hybrid_df["wrong_to_right"].sum(),
)

print(
    "Right -> wrong:",
    hybrid_df["right_to_wrong"].sum(),
)


# ------------------------------------------------------------
# 예측 분포
# ------------------------------------------------------------
distribution_rows = []

for decoder, column in {
    "gold": "gold_order",
    "baseline": "baseline_prediction",
    "joint_global": (
        "joint_with_full_prediction"
    ),
    "hybrid_0945": "hybrid_prediction",
}.items():
    counts = (
        hybrid_df[column]
        .value_counts()
        .rename_axis("order")
        .reset_index(name="count")
    )

    counts["rate"] = (
        counts["count"]
        / len(hybrid_df)
    )

    counts["decoder"] = decoder

    distribution_rows.append(counts)


hybrid_distribution_df = pd.concat(
    distribution_rows,
    ignore_index=True,
)

display(
    hybrid_distribution_df[
        hybrid_distribution_df["order"]
        == "1 2 3 4"
    ]
)


# ------------------------------------------------------------
# Gold order별 득실
# ------------------------------------------------------------
order_rows = []

for gold_order, group in hybrid_df.groupby(
    "gold_order"
):
    baseline_correct = int(
        group["baseline_exact"].sum()
    )

    hybrid_correct = int(
        group["hybrid_exact"].sum()
    )

    order_rows.append({
        "gold_order": gold_order,
        "rows": len(group),
        "baseline_correct": baseline_correct,
        "hybrid_correct": hybrid_correct,
        "delta": (
            hybrid_correct
            - baseline_correct
        ),
        "baseline_exact": float(
            group["baseline_exact"].mean()
        ),
        "hybrid_exact": float(
            group["hybrid_exact"].mean()
        ),
    })


hybrid_by_gold_df = (
    pd.DataFrame(order_rows)
    .sort_values(
        [
            "delta",
            "rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(hybrid_by_gold_df)


# ------------------------------------------------------------
# Joint가 특정 답을 출력했을 때 precision
# ------------------------------------------------------------
prediction_rows = []

for prediction, group in hybrid_df.groupby(
    "hybrid_prediction"
):
    correct = int(
        group["hybrid_exact"].sum()
    )

    prediction_rows.append({
        "prediction": prediction,
        "predicted_count": len(group),
        "correct_count": correct,
        "precision": (
            correct / len(group)
        ),
    })


hybrid_prediction_precision_df = (
    pd.DataFrame(prediction_rows)
    .sort_values(
        [
            "predicted_count",
            "precision",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    hybrid_prediction_precision_df
)

Hybrid exact: 0.445
Correct: 89 / 200
Changed: 73
Wrong -> right: 21
Right -> wrong: 6


,order,count,rate,decoder
0,1 2 3 4,32,0.160,gold
24,1 2 3 4,28,0.140,baseline
48,1 2 3 4,55,0.275,joint_global
72,1 2 3 4,55,0.275,hybrid_0945


,gold_order,rows,baseline_correct,hybrid_correct,delta,baseline_exact,hybrid_exact
0,1 2 3 4,32,17,29,12,0.531250,0.906250
1,2 4 3 1,10,1,3,2,0.100000,0.300000
2,3 1 4 2,8,1,2,1,0.125000,0.250000
3,3 1 2 4,6,2,3,1,0.333333,0.500000
4,4 3 1 2,6,2,3,1,0.333333,0.500000
5,1 4 3 2,5,1,2,1,0.200000,0.400000
6,2 3 1 4,5,1,2,1,0.200000,0.400000
7,3 4 1 2,5,0,1,1,0.000000,0.200000
8,1 2 4 3,12,6,6,0,0.500000,0.500000
9,4 2 3 1,10,5,5,0,0.500000,0.500000


,prediction,predicted_count,correct_count,precision
0,1 2 3 4,55,29,0.527273
1,4 2 3 1,10,5,0.500000
2,1 3 2 4,10,4,0.400000
3,1 3 4 2,9,4,0.444444
4,2 3 4 1,9,3,0.333333
5,3 1 2 4,9,3,0.333333
6,4 3 1 2,9,3,0.333333
7,2 3 1 4,8,2,0.250000
8,1 2 4 3,7,6,0.857143
9,3 4 2 1,7,3,0.428571


In [31]:
# ============================================================
# Joint top1-top2 margin + entropy gate sweep
#
# 전제:
# - raw_df_200
# - joint_result_df
# - ORDERS
# - joint_components
# 가 이전 셀에서 생성되어 있어야 함
# ============================================================

import numpy as np
import pandas as pd


FIRST_WEIGHT = 0.495
LAST_WEIGHT = 0.405
FULL_WEIGHT = 0.100
EPS = 1e-12


if "raw_df_200" not in globals():
    raise RuntimeError("raw_df_200이 없습니다.")

if "joint_result_df" not in globals():
    raise RuntimeError(
        "joint_result_df가 없습니다."
    )


def get_joint_score_diagnostics(row):
    order_scores = []

    for order in ORDERS:
        (
            score_from_first,
            score_from_last,
            full_probability,
        ) = joint_components(
            row,
            order,
        )

        score = (
            FIRST_WEIGHT
            * np.log(
                max(
                    score_from_first,
                    EPS,
                )
            )
            +
            LAST_WEIGHT
            * np.log(
                max(
                    score_from_last,
                    EPS,
                )
            )
            +
            FULL_WEIGHT
            * np.log(
                max(
                    full_probability,
                    EPS,
                )
            )
        )

        order_scores.append(
            (
                order,
                float(score),
            )
        )

    order_scores.sort(
        key=lambda item: item[1],
        reverse=True,
    )

    top1_order, top1_score = (
        order_scores[0]
    )

    top2_order, top2_score = (
        order_scores[1]
    )

    # log score 차이
    log_margin = (
        top1_score
        - top2_score
    )

    # exp를 이용한 top1/top2 상대비
    probability_ratio = float(
        np.exp(
            min(
                log_margin,
                50.0,
            )
        )
    )

    return {
        "joint_top1": " ".join(
            map(str, top1_order)
        ),
        "joint_top2": " ".join(
            map(str, top2_order)
        ),
        "joint_top1_score": top1_score,
        "joint_top2_score": top2_score,
        "joint_log_margin": log_margin,
        "joint_top1_top2_ratio": (
            probability_ratio
        ),
    }


diagnostic_rows = []

for _, row in raw_df_200.iterrows():
    diagnostic = (
        get_joint_score_diagnostics(row)
    )

    diagnostic_rows.append({
        "sample_id": str(
            row["sample_id"]
        ),
        **diagnostic,
    })


joint_margin_df = pd.DataFrame(
    diagnostic_rows
)

margin_eval_df = (
    joint_result_df.merge(
        joint_margin_df,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Entropy + margin threshold sweep
# ------------------------------------------------------------
entropy_thresholds = np.arange(
    0.920,
    0.971,
    0.005,
)

margin_thresholds = sorted(
    set(
        np.quantile(
            margin_eval_df[
                "joint_log_margin"
            ],
            np.linspace(
                0.0,
                0.9,
                19,
            ),
        )
    )
)


sweep_rows = []

for entropy_threshold in entropy_thresholds:
    for margin_threshold in margin_thresholds:
        use_joint = (
            (
                margin_eval_df[
                    "full_order_normalized_entropy"
                ]
                >= entropy_threshold
            )
            &
            (
                margin_eval_df[
                    "joint_log_margin"
                ]
                >= margin_threshold
            )
        )

        final_prediction = np.where(
            use_joint,
            margin_eval_df[
                "joint_with_full_prediction"
            ],
            margin_eval_df[
                "baseline_prediction"
            ],
        )

        final_exact = (
            final_prediction
            ==
            margin_eval_df["gold_order"]
        ).astype(int)

        changed = (
            final_prediction
            !=
            margin_eval_df[
                "baseline_prediction"
            ]
        )

        wrong_to_right = (
            changed
            &
            (
                margin_eval_df[
                    "baseline_exact"
                ] == 0
            )
            &
            (final_exact == 1)
        )

        right_to_wrong = (
            changed
            &
            (
                margin_eval_df[
                    "baseline_exact"
                ] == 1
            )
            &
            (final_exact == 0)
        )

        sweep_rows.append({
            "entropy_threshold": (
                entropy_threshold
            ),
            "margin_threshold": (
                margin_threshold
            ),
            "joint_rows": int(
                use_joint.sum()
            ),
            "changed_count": int(
                changed.sum()
            ),
            "exact": float(
                final_exact.mean()
            ),
            "correct_count": int(
                final_exact.sum()
            ),
            "wrong_to_right": int(
                wrong_to_right.sum()
            ),
            "right_to_wrong": int(
                right_to_wrong.sum()
            ),
            "net_change": int(
                wrong_to_right.sum()
                - right_to_wrong.sum()
            ),
        })


entropy_margin_sweep_df = (
    pd.DataFrame(sweep_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
            "changed_count",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


display(
    entropy_margin_sweep_df.head(30)
)


# ------------------------------------------------------------
# 기존 0.945 전략에서
# W->R과 R->W margin 분포 비교
# ------------------------------------------------------------
base_gate = (
    margin_eval_df[
        "full_order_normalized_entropy"
    ]
    >= 0.945
)

base_changed = (
    margin_eval_df[
        "joint_with_full_prediction"
    ]
    !=
    margin_eval_df[
        "baseline_prediction"
    ]
)

margin_eval_df[
    "change_outcome"
] = "unchanged_or_wrong_to_wrong"

margin_eval_df.loc[
    (
        base_gate
        & base_changed
        &
        (
            margin_eval_df[
                "baseline_exact"
            ] == 0
        )
        &
        (
            margin_eval_df[
                "joint_with_full_exact"
            ] == 1
        )
    ),
    "change_outcome",
] = "wrong_to_right"

margin_eval_df.loc[
    (
        base_gate
        & base_changed
        &
        (
            margin_eval_df[
                "baseline_exact"
            ] == 1
        )
        &
        (
            margin_eval_df[
                "joint_with_full_exact"
            ] == 0
        )
    ),
    "change_outcome",
] = "right_to_wrong"


display(
    margin_eval_df[
        base_gate & base_changed
    ]
    .groupby(
        "change_outcome"
    )[
        [
            "full_order_normalized_entropy",
            "joint_log_margin",
            "joint_top1_top2_ratio",
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "median",
            "min",
            "max",
        ]
    )
)

,entropy_threshold,margin_threshold,joint_rows,changed_count,exact,correct_count,wrong_to_right,right_to_wrong,net_change
0,0.940,0.000801,138,73,0.445,89,21,6,15
1,0.945,0.000801,128,73,0.445,89,21,6,15
2,0.950,0.000801,122,69,0.440,88,19,5,14
3,0.935,0.000801,148,74,0.440,88,21,7,14
4,0.930,0.000801,154,75,0.440,88,21,7,14
5,0.925,0.000801,163,77,0.440,88,22,8,14
6,0.920,0.000801,173,78,0.435,87,22,9,13
7,0.955,0.000801,111,65,0.430,86,16,4,12
8,0.940,0.015735,128,63,0.430,86,18,6,12
9,0.945,0.015735,118,63,0.430,86,18,6,12


full_order_normalized_entropy                      \
                                                    count      mean    median   
change_outcome                                                                  
right_to_wrong                                          6  0.973657  0.975712   
unchanged_or_wrong_to_wrong                            46  0.987755  0.993117   
wrong_to_right                                         21  0.973293  0.976736   

                                                joint_log_margin            \
                                  min       max            count      mean   
change_outcome                                                               
right_to_wrong               0.947270  0.997943                6  0.093060   
unchanged_or_wrong_to_wrong  0.945018  0.999043               46  0.114338   
wrong_to_right               0.946662  0.997340               21  0.217700   

                                                           \
                               median       min       max   
change_outcome                                              
right_to_wrong               0.071412  0.036764  0.251534   
unchanged_or_wrong_to_wrong  0.084549  0.001320  0.410501   
wrong_to_right               0.094095  0.000801  0.947239   

                            joint_top1_top2_ratio                      \
                                            count      mean    median   
change_outcome                                                          
right_to_wrong                                  6  1.100612  1.074111   
unchanged_or_wrong_to_wrong                    46  1.126928  1.088228   
wrong_to_right                                 21  1.294621  1.098664   

                                                 
                                  min       max  
change_outcome                                   
right_to_wrong               1.037448  1.285996  
unchanged_or_wrong_to_wrong  1.001321  1.507572  
wrong_to_right               1.000801  2.578581

In [36]:
# ============================================================
# 독립 실행 셀
# 24-order + full pairwise + endpoint + conditional 비교
#
# 재추론 없음
# raw_df_200에 저장된 확률만 사용
#
# 기존 결과 재현 기준:
# baseline                 = 0.370
# previous_joint_global    = 0.435
# previous_joint_hybrid945 = 0.445
# ============================================================

import itertools
import re

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. 독립 namespace 설정
# ------------------------------------------------------------
FE_FRAMES = (1, 2, 3, 4)

FE_ORDERS = list(
    itertools.permutations(FE_FRAMES)
)

FE_EPS = 1e-12
FE_HYBRID_THRESHOLD = 0.945


if "raw_df_200" not in globals():
    raise RuntimeError(
        "raw_df_200이 없습니다. "
        "Eval200 raw 복구 셀을 먼저 실행하세요."
    )


fe_source_df = raw_df_200.copy()
fe_source_df["sample_id"] = (
    fe_source_df["sample_id"]
    .astype(str)
)


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def fe_parse_order(value):
    if isinstance(
        value,
        (list, tuple, np.ndarray),
    ):
        order = tuple(
            int(number)
            for number in value
        )

    else:
        order = tuple(
            int(number)
            for number in re.findall(
                r"[1-4]",
                str(value),
            )
        )

    if (
        len(order) != 4
        or sorted(order) != list(FE_FRAMES)
    ):
        raise ValueError(
            f"잘못된 order: {value!r} -> {order}"
        )

    return order


def fe_order_text(order):
    return " ".join(
        map(str, order)
    )


def fe_safe_log(probability):
    return float(
        np.log(
            max(
                float(probability),
                FE_EPS,
            )
        )
    )


def fe_mean_log(probabilities):
    probabilities = list(probabilities)

    return float(
        np.mean(
            [
                fe_safe_log(probability)
                for probability
                in probabilities
            ]
        )
    )


# ------------------------------------------------------------
# 2. 기존 10:30:60 baseline 정확히 재구성
# ------------------------------------------------------------
def fe_decode_baseline(row):
    first = int(
        row["base_first_prediction"]
    )

    last = int(
        row["base_last_prediction"]
    )

    # first와 last가 동일하면
    # 기존 fused confidence가 더 약한 쪽을 변경
    if first == last:
        first_probs = {
            frame: float(
                row[
                    f"p_base_fused_first_{frame}"
                ]
            )
            for frame in FE_FRAMES
        }

        last_probs = {
            frame: float(
                row[
                    f"p_base_fused_last_{frame}"
                ]
            )
            for frame in FE_FRAMES
        }

        if (
            first_probs[first]
            >= last_probs[last]
        ):
            last = max(
                (
                    frame
                    for frame in FE_FRAMES
                    if frame != first
                ),
                key=lambda frame: (
                    last_probs[frame]
                ),
            )

        else:
            first = max(
                (
                    frame
                    for frame in FE_FRAMES
                    if frame != last
                ),
                key=lambda frame: (
                    first_probs[frame]
                ),
            )

    middle_frames = [
        frame
        for frame in FE_FRAMES
        if frame not in {first, last}
    ]

    middle_a, middle_b = middle_frames

    p_middle_order_ab = float(
        row[
            f"p_middle_order_given_"
            f"{first}_{last}_"
            f"{middle_a}{middle_b}"
        ]
    )

    p_middle_pairwise_ab = float(
        row[
            f"p_middle_pairwise_given_"
            f"{first}_{last}_"
            f"{middle_a}{middle_b}"
        ]
    )

    p_middle_ab = (
        0.30 * p_middle_order_ab
        + 0.70 * p_middle_pairwise_ab
    )

    if p_middle_ab >= 0.5:
        middle = (
            middle_a,
            middle_b,
        )
    else:
        middle = (
            middle_b,
            middle_a,
        )

    return (
        first,
        middle[0],
        middle[1],
        last,
    )


# ------------------------------------------------------------
# 3. Pairwise 방향 확률
# ------------------------------------------------------------
def fe_pair_probability(
    row,
    earlier,
    later,
):
    forward = max(
        float(
            row[
                f"p_pair_{earlier}"
                f"_before_{later}"
            ]
        ),
        0.0,
    )

    backward = max(
        float(
            row[
                f"p_pair_{later}"
                f"_before_{earlier}"
            ]
        ),
        0.0,
    )

    total = forward + backward

    if total <= 0:
        return 0.5

    return float(
        forward / total
    )


# ------------------------------------------------------------
# 4. 후보 전체 순서의 evidence 계산
# ------------------------------------------------------------
def fe_components(row, candidate_order):
    first, second, third, last = (
        candidate_order
    )

    order_key = "".join(
        map(str, candidate_order)
    )

    suffix_key = (
        f"{second}{third}{last}"
    )

    prefix_key = (
        f"{first}{second}{third}"
    )

    # A. 전체 24-order 확률
    order_log = fe_safe_log(
        row[f"p_order_{order_key}"]
    )

    # B. 후보 순서가 요구하는 6개 pairwise 관계
    pair_probabilities = []

    for left_index in range(4):
        for right_index in range(
            left_index + 1,
            4,
        ):
            earlier = candidate_order[
                left_index
            ]

            later = candidate_order[
                right_index
            ]

            pair_probabilities.append(
                fe_pair_probability(
                    row,
                    earlier,
                    later,
                )
            )

    # 관계가 6개이므로 합이 아니라 평균 log 사용
    full_pairwise_log = fe_mean_log(
        pair_probabilities
    )

    # C. Direct endpoint
    endpoint_log = fe_mean_log([
        row[
            f"p_endpoint_first_{first}"
        ],
        row[
            f"p_endpoint_last_{last}"
        ],
    ])

    # D. Pairwise에서 집계된 first / last
    pair_endpoint_log = fe_mean_log([
        row[
            f"p_pairwise_first_{first}"
        ],
        row[
            f"p_pairwise_last_{last}"
        ],
    ])

    # E. first/last 고정 조건부 전체 순서
    conditional_first_probability = float(
        row[
            f"p_cond_fixed_first_"
            f"{first}_suffix_{suffix_key}"
        ]
    )

    conditional_last_probability = float(
        row[
            f"p_cond_fixed_last_"
            f"{last}_prefix_{prefix_key}"
        ]
    )

    conditional_log = fe_mean_log([
        conditional_first_probability,
        conditional_last_probability,
    ])

    return {
        "order_log": order_log,
        "full_pairwise_log": (
            full_pairwise_log
        ),
        "endpoint_log": endpoint_log,
        "pair_endpoint_log": (
            pair_endpoint_log
        ),
        "conditional_log": (
            conditional_log
        ),

        # 기존 joint_with_full 재현용
        "pairwise_first_log": (
            fe_safe_log(
                row[
                    f"p_pairwise_first_{first}"
                ]
            )
        ),
        "endpoint_last_log": (
            fe_safe_log(
                row[
                    f"p_endpoint_last_{last}"
                ]
            )
        ),
        "conditional_first_log": (
            fe_safe_log(
                conditional_first_probability
            )
        ),
        "conditional_last_log": (
            fe_safe_log(
                conditional_last_probability
            )
        ),
    }


# ------------------------------------------------------------
# 5. Decoder별 후보 점수
# ------------------------------------------------------------
def fe_score_candidate(
    components,
    decoder_name,
):
    # 24-order head 단독
    if decoder_name == "full24_order":
        return components["order_log"]

    # 기존 joint_with_full 정확히 재현
    if decoder_name == "previous_joint_with_full":
        return float(
            0.495
            * (
                components[
                    "pairwise_first_log"
                ]
                +
                components[
                    "conditional_first_log"
                ]
            )
            +
            0.405
            * (
                components[
                    "endpoint_last_log"
                ]
                +
                components[
                    "conditional_last_log"
                ]
            )
            +
            0.10
            * components["order_log"]
        )

    # Order + 전체 6개 pairwise
    if decoder_name == "order_full_pairwise":
        return float(
            0.20
            * components["order_log"]
            +
            0.80
            * components[
                "full_pairwise_log"
            ]
        )

    # Order + 전체 pairwise + direct endpoint
    if decoder_name == "order_pairwise_endpoint":
        return float(
            0.15
            * components["order_log"]
            +
            0.60
            * components[
                "full_pairwise_log"
            ]
            +
            0.25
            * components[
                "endpoint_log"
            ]
        )

    # 모든 주요 evidence
    if decoder_name == "all_evidence_joint":
        return float(
            0.10
            * components["order_log"]
            +
            0.35
            * components[
                "full_pairwise_log"
            ]
            +
            0.15
            * components[
                "endpoint_log"
            ]
            +
            0.40
            * components[
                "conditional_log"
            ]
        )

    # 모든 evidence + pairwise first/last 집계
    if (
        decoder_name
        == "all_evidence_plus_pair_endpoint"
    ):
        return float(
            0.10
            * components["order_log"]
            +
            0.30
            * components[
                "full_pairwise_log"
            ]
            +
            0.15
            * components[
                "endpoint_log"
            ]
            +
            0.10
            * components[
                "pair_endpoint_log"
            ]
            +
            0.35
            * components[
                "conditional_log"
            ]
        )

    raise ValueError(
        f"알 수 없는 decoder: {decoder_name}"
    )


FE_DECODERS = [
    "full24_order",
    "previous_joint_with_full",
    "order_full_pairwise",
    "order_pairwise_endpoint",
    "all_evidence_joint",
    "all_evidence_plus_pair_endpoint",
]


# ------------------------------------------------------------
# 6. 24개 전체 순서를 끝까지 경쟁시켜 선택
# ------------------------------------------------------------
def fe_decode(row, decoder_name):
    scored_candidates = []

    for candidate_order in FE_ORDERS:
        components = fe_components(
            row,
            candidate_order,
        )

        score = fe_score_candidate(
            components,
            decoder_name,
        )

        # candidate_order와 order 점수의
        # 이름이 절대 겹치지 않도록 분리
        scored_candidates.append({
            "candidate_order": (
                candidate_order
            ),
            "score": float(score),
        })

    scored_candidates.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    top1 = scored_candidates[0]
    top2 = scored_candidates[1]

    return {
        "prediction": fe_order_text(
            top1["candidate_order"]
        ),
        "top2_prediction": fe_order_text(
            top2["candidate_order"]
        ),
        "margin": float(
            top1["score"]
            - top2["score"]
        ),
    }


# ------------------------------------------------------------
# 7. 전체 샘플 계산
# ------------------------------------------------------------
fe_rows = []

for _, row in fe_source_df.iterrows():
    gold_tuple = fe_parse_order(
        row["gold_order"]
    )

    baseline_tuple = fe_decode_baseline(
        row
    )

    gold_text = fe_order_text(
        gold_tuple
    )

    baseline_text = fe_order_text(
        baseline_tuple
    )

    output = {
        "sample_id": str(
            row["sample_id"]
        ),
        "gold_order": gold_text,
        "baseline_prediction": (
            baseline_text
        ),
        "baseline_exact": int(
            baseline_text == gold_text
        ),
        "full_order_normalized_entropy": (
            float(
                row[
                    "full_order_normalized_entropy"
                ]
            )
        ),
    }

    for decoder_name in FE_DECODERS:
        decoded = fe_decode(
            row,
            decoder_name,
        )

        prediction = decoded[
            "prediction"
        ]

        output[
            f"{decoder_name}_prediction"
        ] = prediction

        output[
            f"{decoder_name}_exact"
        ] = int(
            prediction == gold_text
        )

        output[
            f"{decoder_name}_margin"
        ] = decoded["margin"]

        output[
            f"{decoder_name}_top2"
        ] = decoded[
            "top2_prediction"
        ]

    fe_rows.append(output)


fe_result_df = pd.DataFrame(
    fe_rows
)


# ------------------------------------------------------------
# 8. 평가 함수
# ------------------------------------------------------------
def fe_evaluate(
    frame,
    prediction,
    strategy_name,
):
    prediction = pd.Series(
        prediction,
        index=frame.index,
    )

    exact = (
        prediction
        == frame["gold_order"]
    ).astype(int)

    changed = (
        prediction
        != frame[
            "baseline_prediction"
        ]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (exact == 0)
    )

    return {
        "strategy": strategy_name,
        "exact": float(
            exact.mean()
        ),
        "correct_count": int(
            exact.sum()
        ),
        "changed_count": int(
            changed.sum()
        ),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
        "canonical_1234_count": int(
            (
                prediction
                == "1 2 3 4"
            ).sum()
        ),
    }


# ------------------------------------------------------------
# 9. Global 비교
# ------------------------------------------------------------
fe_global_rows = [
    fe_evaluate(
        fe_result_df,
        fe_result_df[
            "baseline_prediction"
        ],
        "baseline",
    )
]

for decoder_name in FE_DECODERS:
    fe_global_rows.append(
        fe_evaluate(
            fe_result_df,
            fe_result_df[
                f"{decoder_name}_prediction"
            ],
            decoder_name,
        )
    )


fe_global_summary_df = (
    pd.DataFrame(fe_global_rows)
    .sort_values(
        [
            "exact",
            "net_change",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print("Global decoder 비교")

display(
    fe_global_summary_df
)


# ------------------------------------------------------------
# 10. Entropy 0.945 고정 hybrid 비교
# ------------------------------------------------------------
fe_hybrid_mask = (
    fe_result_df[
        "full_order_normalized_entropy"
    ]
    >= FE_HYBRID_THRESHOLD
)


fe_hybrid_rows = []

for decoder_name in FE_DECODERS:
    hybrid_prediction = np.where(
        fe_hybrid_mask,
        fe_result_df[
            f"{decoder_name}_prediction"
        ],
        fe_result_df[
            "baseline_prediction"
        ],
    )

    evaluated = fe_evaluate(
        fe_result_df,
        hybrid_prediction,
        (
            f"{decoder_name}"
            f"_entropy_{FE_HYBRID_THRESHOLD}"
        ),
    )

    evaluated["decoder_rows"] = int(
        fe_hybrid_mask.sum()
    )

    fe_hybrid_rows.append(
        evaluated
    )


fe_hybrid_0945_summary_df = (
    pd.DataFrame(fe_hybrid_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nEntropy 0.945 고정 hybrid 비교"
)

display(
    fe_hybrid_0945_summary_df
)


# ------------------------------------------------------------
# 11. 기존 결과 재현 검증
# ------------------------------------------------------------
fe_baseline_exact = float(
    fe_result_df[
        "baseline_exact"
    ].mean()
)

fe_previous_global_exact = float(
    fe_result_df[
        "previous_joint_with_full_exact"
    ].mean()
)

fe_previous_hybrid_prediction = np.where(
    fe_hybrid_mask,
    fe_result_df[
        "previous_joint_with_full_prediction"
    ],
    fe_result_df[
        "baseline_prediction"
    ],
)

fe_previous_hybrid_exact = float(
    (
        fe_previous_hybrid_prediction
        ==
        fe_result_df["gold_order"]
    ).mean()
)


print("\n기존 결과 재현 확인")
print(
    "baseline:",
    fe_baseline_exact,
    "(expected 0.370)",
)

print(
    "previous joint global:",
    fe_previous_global_exact,
    "(expected 0.435)",
)

print(
    "previous joint hybrid 0.945:",
    fe_previous_hybrid_exact,
    "(expected 0.445)",
)


assert np.isclose(
    fe_baseline_exact,
    0.370,
), (
    "baseline이 0.370과 다릅니다. "
    "raw_df_200이 이전과 다른 데이터인지 확인하세요."
)

assert np.isclose(
    fe_previous_global_exact,
    0.435,
), (
    "previous joint global이 0.435와 다릅니다. "
    "raw_df_200 또는 확률 컬럼이 이전과 다릅니다."
)

assert np.isclose(
    fe_previous_hybrid_exact,
    0.445,
), (
    "previous joint hybrid가 0.445와 다릅니다. "
    "raw_df_200 또는 entropy 값이 이전과 다릅니다."
)


print(
    "\n✅ 기존 결과가 정확히 재현되었습니다."
)

Global decoder 비교


,strategy,exact,correct_count,changed_count,wrong_to_right,right_to_wrong,net_change,canonical_1234_count
0,previous_joint_with_full,0.435,87,81,23,10,13,55
1,all_evidence_joint,0.415,83,49,15,6,9,42
2,all_evidence_plus_pair_endpoint,0.415,83,47,15,6,9,42
3,full24_order,0.405,81,82,21,14,7,55
4,baseline,0.370,74,0,0,0,0,28
5,order_pairwise_endpoint,0.365,73,24,5,6,-1,30
6,order_full_pairwise,0.360,72,80,16,18,-2,24



Entropy 0.945 고정 hybrid 비교


,strategy,exact,correct_count,changed_count,wrong_to_right,right_to_wrong,net_change,canonical_1234_count,decoder_rows
0,previous_joint_with_full_entropy_0.945,0.445,89,73,21,6,15,55,128
1,full24_order_entropy_0.945,0.420,84,73,19,9,10,55,128
2,all_evidence_joint_entropy_0.945,0.405,81,44,12,5,7,42,128
3,all_evidence_plus_pair_endpoint_entropy_0.945,0.405,81,42,12,5,7,42,128
4,order_full_pairwise_entropy_0.945,0.390,78,60,10,6,4,30,128
5,order_pairwise_endpoint_entropy_0.945,0.370,74,21,4,4,0,30,128



기존 결과 재현 확인
baseline: 0.37 (expected 0.370)
previous joint global: 0.435 (expected 0.435)
previous joint hybrid 0.945: 0.445 (expected 0.445)

✅ 기존 결과가 정확히 재현되었습니다.


In [37]:
# ============================================================
# Positional Joint Decoder
#
# 별도 추론 없음:
# raw_df_200에 저장된 확률만 재조합
#
# 핵심:
# - Order → 자리별 확률
# - Pairwise → 자리별 확률
# - 양끝에는 endpoint first/last 추가
# - 24개 완성 순서를 공동 비교
# ============================================================

import itertools
import re
from math import prod

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. 설정
# ------------------------------------------------------------
PJ_FRAMES = (1, 2, 3, 4)

PJ_ORDERS = list(
    itertools.permutations(PJ_FRAMES)
)

PJ_EPS = 1e-12
PJ_FIXED_ENTROPY_THRESHOLD = 0.945


if "raw_df_200" not in globals():
    raise RuntimeError(
        "raw_df_200이 없습니다. "
        "Eval200 raw 복구 셀부터 실행하세요."
    )


pj_source_df = raw_df_200.copy()

pj_source_df["sample_id"] = (
    pj_source_df["sample_id"]
    .astype(str)
)


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def pj_parse_order(value):
    if isinstance(
        value,
        (list, tuple, np.ndarray),
    ):
        order = tuple(
            int(number)
            for number in value
        )

    else:
        order = tuple(
            int(number)
            for number in re.findall(
                r"[1-4]",
                str(value),
            )
        )

    if (
        len(order) != 4
        or sorted(order) != list(PJ_FRAMES)
    ):
        raise ValueError(
            f"잘못된 order: {value!r} -> {order}"
        )

    return order


def pj_order_text(order):
    return " ".join(
        map(str, order)
    )


def pj_safe_log(value):
    return float(
        np.log(
            max(
                float(value),
                PJ_EPS,
            )
        )
    )


def pj_normalize_dict(values):
    cleaned = {
        key: max(float(value), 0.0)
        for key, value in values.items()
    }

    total = sum(cleaned.values())

    if total <= 0:
        uniform = 1.0 / len(cleaned)

        return {
            key: uniform
            for key in cleaned
        }

    return {
        key: value / total
        for key, value in cleaned.items()
    }


# ------------------------------------------------------------
# 2. Baseline 연결
# ------------------------------------------------------------
def pj_find_baseline_lookup():
    # raw_df_200에 이미 있는 경우
    if (
        "base_prediction_10_30_60"
        in pj_source_df.columns
    ):
        return None

    candidates = [
        (
            "fe_result_df",
            globals().get("fe_result_df"),
            "baseline_prediction",
        ),
        (
            "joint_result_df",
            globals().get("joint_result_df"),
            "baseline_prediction",
        ),
        (
            "readable_df_200",
            globals().get("readable_df_200"),
            "base_prediction_10_30_60",
        ),
    ]

    for _, frame, prediction_column in candidates:
        if (
            isinstance(frame, pd.DataFrame)
            and "sample_id" in frame.columns
            and prediction_column in frame.columns
        ):
            lookup = (
                frame[
                    [
                        "sample_id",
                        prediction_column,
                    ]
                ]
                .copy()
                .rename(
                    columns={
                        prediction_column:
                            "pj_baseline_prediction"
                    }
                )
            )

            lookup["sample_id"] = (
                lookup["sample_id"]
                .astype(str)
            )

            return lookup

    return None


if (
    "base_prediction_10_30_60"
    in pj_source_df.columns
):
    pj_source_df[
        "pj_baseline_prediction"
    ] = pj_source_df[
        "base_prediction_10_30_60"
    ]

else:
    pj_baseline_lookup = (
        pj_find_baseline_lookup()
    )

    if pj_baseline_lookup is None:
        raise RuntimeError(
            "기존 baseline prediction을 찾을 수 없습니다.\n"
            "fe_result_df, joint_result_df 또는 "
            "readable_df_200을 먼저 생성하세요."
        )

    pj_source_df = pj_source_df.merge(
        pj_baseline_lookup,
        on="sample_id",
        how="left",
        validate="one_to_one",
    )


if (
    pj_source_df[
        "pj_baseline_prediction"
    ].isna().any()
):
    raise RuntimeError(
        "일부 샘플에 baseline prediction이 없습니다."
    )


# ------------------------------------------------------------
# 3. 필요한 컬럼 확인
# ------------------------------------------------------------
pj_required_columns = {
    "sample_id",
    "gold_order",
    "full_order_normalized_entropy",
    "pj_baseline_prediction",
}


# 24개 order 확률
for pj_order in PJ_ORDERS:
    pj_key = "".join(
        map(str, pj_order)
    )

    pj_required_columns.add(
        f"p_order_{pj_key}"
    )


# endpoint 확률
for pj_frame in PJ_FRAMES:
    pj_required_columns.add(
        f"p_endpoint_first_{pj_frame}"
    )

    pj_required_columns.add(
        f"p_endpoint_last_{pj_frame}"
    )


# pairwise 양방향 확률
for pj_left in PJ_FRAMES:
    for pj_right in PJ_FRAMES:
        if pj_left == pj_right:
            continue

        pj_required_columns.add(
            f"p_pair_{pj_left}"
            f"_before_{pj_right}"
        )


# 양방향 conditional
for pj_order in PJ_ORDERS:
    pj_first, pj_second, pj_third, pj_last = (
        pj_order
    )

    pj_suffix = (
        f"{pj_second}{pj_third}{pj_last}"
    )

    pj_prefix = (
        f"{pj_first}{pj_second}{pj_third}"
    )

    pj_required_columns.add(
        f"p_cond_fixed_first_"
        f"{pj_first}_suffix_{pj_suffix}"
    )

    pj_required_columns.add(
        f"p_cond_fixed_last_"
        f"{pj_last}_prefix_{pj_prefix}"
    )


pj_missing_columns = sorted(
    pj_required_columns
    - set(pj_source_df.columns)
)

if pj_missing_columns:
    print(
        "누락 컬럼 수:",
        len(pj_missing_columns),
    )

    print(
        "\n".join(
            pj_missing_columns[:50]
        )
    )

    raise RuntimeError(
        "필요한 확률 컬럼이 없습니다."
    )


# ------------------------------------------------------------
# 4. Order 24개 확률 → 자리별 확률
#
# 예:
# P_order(frame=2 at position=1)
# =
# 첫 자리가 2인 모든 전체 순서 확률의 합
# ------------------------------------------------------------
def pj_get_order_position_probs(row):
    raw_order_probs = {}

    for order in PJ_ORDERS:
        key = "".join(
            map(str, order)
        )

        raw_order_probs[order] = max(
            float(
                row[f"p_order_{key}"]
            ),
            0.0,
        )

    raw_order_probs = pj_normalize_dict(
        raw_order_probs
    )

    position_probs = {
        position: {
            frame: 0.0
            for frame in PJ_FRAMES
        }
        for position in range(1, 5)
    }

    for order, probability in (
        raw_order_probs.items()
    ):
        for position_index, frame in (
            enumerate(order)
        ):
            position = position_index + 1

            position_probs[
                position
            ][frame] += probability

    # 각 자리에서 네 frame의 합이 1이 되게 정규화
    for position in range(1, 5):
        position_probs[position] = (
            pj_normalize_dict(
                position_probs[position]
            )
        )

    return position_probs


# ------------------------------------------------------------
# 5. Pairwise 방향 확률 정규화
#
# P(a before b)와 P(b before a)의 합이
# 정확히 1이 아닐 수 있으므로 쌍 안에서 정규화
# ------------------------------------------------------------
def pj_pair_before_probability(
    row,
    earlier,
    later,
):
    forward = max(
        float(
            row[
                f"p_pair_{earlier}"
                f"_before_{later}"
            ]
        ),
        0.0,
    )

    backward = max(
        float(
            row[
                f"p_pair_{later}"
                f"_before_{earlier}"
            ]
        ),
        0.0,
    )

    total = forward + backward

    if total <= 0:
        return 0.5

    return float(
        forward / total
    )


# ------------------------------------------------------------
# 6. Pairwise → 각 frame의 자리별 확률
#
# frame f가 r번째라는 것은
# 다른 3개 중 정확히 r-1개가 f보다 앞이라는 뜻.
#
# 예:
# f가 2번째
# = 다른 3개 중 정확히 1개가 f보다 앞
# ------------------------------------------------------------
def pj_get_pairwise_position_probs(row):
    # frame 기준 rank 분포
    per_frame_rank = {
        frame: {
            position: 0.0
            for position in range(1, 5)
        }
        for frame in PJ_FRAMES
    }

    for target_frame in PJ_FRAMES:
        other_frames = [
            frame
            for frame in PJ_FRAMES
            if frame != target_frame
        ]

        # q[other] =
        # other가 target보다 앞일 확률
        q_before_target = {
            other: pj_pair_before_probability(
                row,
                other,
                target_frame,
            )
            for other in other_frames
        }

        # 3개 frame 각각에 대해
        # 앞/뒤 모든 2^3 경우 열거
        for before_flags in itertools.product(
            [0, 1],
            repeat=3,
        ):
            number_before = sum(
                before_flags
            )

            position = (
                number_before + 1
            )

            probability_terms = []

            for other, is_before in zip(
                other_frames,
                before_flags,
            ):
                q = q_before_target[other]

                probability_terms.append(
                    q
                    if is_before
                    else 1.0 - q
                )

            case_probability = prod(
                probability_terms
            )

            per_frame_rank[
                target_frame
            ][position] += (
                case_probability
            )

        # frame 하나의 rank 1~4 합 정규화
        per_frame_rank[
            target_frame
        ] = pj_normalize_dict(
            per_frame_rank[
                target_frame
            ]
        )

    # position 기준 구조로 변환
    position_probs = {
        position: {
            frame: per_frame_rank[
                frame
            ][position]
            for frame in PJ_FRAMES
        }
        for position in range(1, 5)
    }

    # 각 자리에서 네 frame의 합이 1이 되게 정규화
    for position in range(1, 5):
        position_probs[position] = (
            pj_normalize_dict(
                position_probs[position]
            )
        )

    return position_probs


# ------------------------------------------------------------
# 7. Endpoint 확률 정규화
# ------------------------------------------------------------
def pj_get_endpoint_probs(row):
    first_probs = pj_normalize_dict({
        frame: row[
            f"p_endpoint_first_{frame}"
        ]
        for frame in PJ_FRAMES
    })

    last_probs = pj_normalize_dict({
        frame: row[
            f"p_endpoint_last_{frame}"
        ]
        for frame in PJ_FRAMES
    })

    return first_probs, last_probs


# ------------------------------------------------------------
# 8. 자리별 결합 확률
#
# position 1:
# order 0.1 + pairwise 0.6 + first 0.3
#
# position 2/3:
# order 0.4 + pairwise 0.6
#
# position 4:
# order 0.1 + pairwise 0.6 + last 0.3
# ------------------------------------------------------------
def pj_get_blended_position_probs(row):
    order_position_probs = (
        pj_get_order_position_probs(row)
    )

    pair_position_probs = (
        pj_get_pairwise_position_probs(row)
    )

    (
        endpoint_first_probs,
        endpoint_last_probs,
    ) = pj_get_endpoint_probs(row)

    blended = {
        position: {}
        for position in range(1, 5)
    }

    for frame in PJ_FRAMES:
        blended[1][frame] = (
            0.10
            * order_position_probs[1][frame]
            +
            0.60
            * pair_position_probs[1][frame]
            +
            0.30
            * endpoint_first_probs[frame]
        )

        blended[2][frame] = (
            0.40
            * order_position_probs[2][frame]
            +
            0.60
            * pair_position_probs[2][frame]
        )

        blended[3][frame] = (
            0.40
            * order_position_probs[3][frame]
            +
            0.60
            * pair_position_probs[3][frame]
        )

        blended[4][frame] = (
            0.10
            * order_position_probs[4][frame]
            +
            0.60
            * pair_position_probs[4][frame]
            +
            0.30
            * endpoint_last_probs[frame]
        )

    for position in range(1, 5):
        blended[position] = (
            pj_normalize_dict(
                blended[position]
            )
        )

    return {
        "order": order_position_probs,
        "pairwise": pair_position_probs,
        "blended": blended,
        "endpoint_first": (
            endpoint_first_probs
        ),
        "endpoint_last": (
            endpoint_last_probs
        ),
    }


# ------------------------------------------------------------
# 9. 후보 순서별 점수
#
# positional:
# 네 자리 blended 확률의 평균 log
#
# conditional:
# first 고정 suffix + last 고정 prefix 평균 log
#
# full_order:
# 해당 24-order 확률
# ------------------------------------------------------------
def pj_candidate_components(
    row,
    candidate_order,
    distributions,
):
    first, second, third, last = (
        candidate_order
    )

    order_key = "".join(
        map(str, candidate_order)
    )

    suffix = (
        f"{second}{third}{last}"
    )

    prefix = (
        f"{first}{second}{third}"
    )

    blended = distributions[
        "blended"
    ]

    positional_log = float(
        np.mean([
            pj_safe_log(
                blended[1][first]
            ),
            pj_safe_log(
                blended[2][second]
            ),
            pj_safe_log(
                blended[3][third]
            ),
            pj_safe_log(
                blended[4][last]
            ),
        ])
    )

    conditional_first = float(
        row[
            f"p_cond_fixed_first_"
            f"{first}_suffix_{suffix}"
        ]
    )

    conditional_last = float(
        row[
            f"p_cond_fixed_last_"
            f"{last}_prefix_{prefix}"
        ]
    )

    conditional_log = float(
        np.mean([
            pj_safe_log(
                conditional_first
            ),
            pj_safe_log(
                conditional_last
            ),
        ])
    )

    full_order_log = pj_safe_log(
        row[f"p_order_{order_key}"]
    )

    return {
        "positional_log": positional_log,
        "conditional_log": (
            conditional_log
        ),
        "full_order_log": (
            full_order_log
        ),
    }


# ------------------------------------------------------------
# 10. 비교할 공동 디코더
#
# positional_joint:
#   자리별 결합만 사용
#
# positional_cond:
#   자리별 75% + conditional 25%
#
# positional_cond_full:
#   자리별 70% + conditional 20%
#   + full-order 10%
#
# positional_cond_strong:
#   자리별 60% + conditional 30%
#   + full-order 10%
# ------------------------------------------------------------
PJ_DECODER_WEIGHTS = {
    "positional_joint": {
        "positional_log": 1.00,
        "conditional_log": 0.00,
        "full_order_log": 0.00,
    },

    "positional_cond": {
        "positional_log": 0.75,
        "conditional_log": 0.25,
        "full_order_log": 0.00,
    },

    "positional_cond_full": {
        "positional_log": 0.70,
        "conditional_log": 0.20,
        "full_order_log": 0.10,
    },

    "positional_cond_strong": {
        "positional_log": 0.60,
        "conditional_log": 0.30,
        "full_order_log": 0.10,
    },
}


def pj_decode_row(
    row,
    decoder_name,
):
    weights = PJ_DECODER_WEIGHTS[
        decoder_name
    ]

    distributions = (
        pj_get_blended_position_probs(
            row
        )
    )

    scored_candidates = []

    for candidate_order in PJ_ORDERS:
        components = (
            pj_candidate_components(
                row,
                candidate_order,
                distributions,
            )
        )

        total_score = sum(
            weights[key]
            * components[key]
            for key in weights
        )

        scored_candidates.append({
            "candidate_order": (
                candidate_order
            ),
            "score": float(
                total_score
            ),
            **components,
        })

    scored_candidates.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    top1 = scored_candidates[0]
    top2 = scored_candidates[1]

    return {
        "prediction": pj_order_text(
            top1["candidate_order"]
        ),
        "top2_prediction": pj_order_text(
            top2["candidate_order"]
        ),
        "margin": float(
            top1["score"]
            - top2["score"]
        ),
    }


# ------------------------------------------------------------
# 11. 전체 디코딩
# ------------------------------------------------------------
pj_rows = []

for _, row in pj_source_df.iterrows():
    gold_order = pj_order_text(
        pj_parse_order(
            row["gold_order"]
        )
    )

    baseline_prediction = (
        pj_order_text(
            pj_parse_order(
                row[
                    "pj_baseline_prediction"
                ]
            )
        )
    )

    output = {
        "sample_id": str(
            row["sample_id"]
        ),
        "gold_order": gold_order,
        "baseline_prediction": (
            baseline_prediction
        ),
        "baseline_exact": int(
            baseline_prediction
            == gold_order
        ),
        "full_order_normalized_entropy": (
            float(
                row[
                    "full_order_normalized_entropy"
                ]
            )
        ),
    }

    for decoder_name in (
        PJ_DECODER_WEIGHTS
    ):
        decoded = pj_decode_row(
            row,
            decoder_name,
        )

        prediction = decoded[
            "prediction"
        ]

        output[
            f"{decoder_name}_prediction"
        ] = prediction

        output[
            f"{decoder_name}_exact"
        ] = int(
            prediction == gold_order
        )

        output[
            f"{decoder_name}_margin"
        ] = decoded["margin"]

        output[
            f"{decoder_name}_top2"
        ] = decoded[
            "top2_prediction"
        ]

    pj_rows.append(output)


pj_result_df = pd.DataFrame(
    pj_rows
)


# ------------------------------------------------------------
# 12. 평가 함수
# ------------------------------------------------------------
def pj_evaluate(
    frame,
    prediction,
    strategy_name,
):
    prediction = pd.Series(
        prediction,
        index=frame.index,
    )

    exact = (
        prediction
        == frame["gold_order"]
    ).astype(int)

    changed = (
        prediction
        != frame[
            "baseline_prediction"
        ]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (exact == 0)
    )

    canonical_count = int(
        (
            prediction == "1 2 3 4"
        ).sum()
    )

    return {
        "strategy": strategy_name,
        "exact": float(
            exact.mean()
        ),
        "correct_count": int(
            exact.sum()
        ),
        "changed_count": int(
            changed.sum()
        ),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
        "canonical_1234_count": (
            canonical_count
        ),
        "canonical_1234_rate": float(
            canonical_count
            / len(frame)
        ),
    }


# ------------------------------------------------------------
# 13. Global 비교
# ------------------------------------------------------------
pj_global_rows = [
    pj_evaluate(
        pj_result_df,
        pj_result_df[
            "baseline_prediction"
        ],
        "baseline",
    )
]


for decoder_name in (
    PJ_DECODER_WEIGHTS
):
    pj_global_rows.append(
        pj_evaluate(
            pj_result_df,
            pj_result_df[
                f"{decoder_name}_prediction"
            ],
            decoder_name,
        )
    )


# 이전 joint 결과가 존재하면 함께 표시
if (
    "fe_result_df" in globals()
    and isinstance(
        fe_result_df,
        pd.DataFrame,
    )
    and "previous_joint_with_full_prediction"
        in fe_result_df.columns
):
    pj_previous_lookup = (
        fe_result_df[
            [
                "sample_id",
                "previous_joint_with_full_prediction",
            ]
        ]
        .copy()
    )

    pj_previous_lookup["sample_id"] = (
        pj_previous_lookup["sample_id"]
        .astype(str)
    )

    pj_previous_merged = (
        pj_result_df[
            [
                "sample_id",
                "gold_order",
                "baseline_prediction",
                "baseline_exact",
            ]
        ]
        .merge(
            pj_previous_lookup,
            on="sample_id",
            how="left",
            validate="one_to_one",
        )
    )

    pj_global_rows.append(
        pj_evaluate(
            pj_previous_merged,
            pj_previous_merged[
                "previous_joint_with_full_prediction"
            ],
            "previous_joint_with_full",
        )
    )


pj_global_summary_df = (
    pd.DataFrame(pj_global_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print("Global positional joint 비교")

display(
    pj_global_summary_df
)


# ------------------------------------------------------------
# 14. Entropy 0.945 고정 hybrid
# ------------------------------------------------------------
pj_hybrid_mask = (
    pj_result_df[
        "full_order_normalized_entropy"
    ]
    >= PJ_FIXED_ENTROPY_THRESHOLD
)


pj_hybrid_rows = []

for decoder_name in (
    PJ_DECODER_WEIGHTS
):
    final_prediction = np.where(
        pj_hybrid_mask,
        pj_result_df[
            f"{decoder_name}_prediction"
        ],
        pj_result_df[
            "baseline_prediction"
        ],
    )

    evaluated = pj_evaluate(
        pj_result_df,
        final_prediction,
        (
            f"{decoder_name}"
            f"_entropy_"
            f"{PJ_FIXED_ENTROPY_THRESHOLD}"
        ),
    )

    evaluated["decoder_rows"] = int(
        pj_hybrid_mask.sum()
    )

    pj_hybrid_rows.append(
        evaluated
    )


pj_hybrid_0945_summary_df = (
    pd.DataFrame(pj_hybrid_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nEntropy 0.945 고정 hybrid"
)

display(
    pj_hybrid_0945_summary_df
)


# ------------------------------------------------------------
# 15. Entropy threshold sweep
# ------------------------------------------------------------
pj_threshold_rows = []

for decoder_name in (
    PJ_DECODER_WEIGHTS
):
    decoder_prediction = (
        pj_result_df[
            f"{decoder_name}_prediction"
        ]
    )

    for threshold in np.arange(
        0.920,
        0.976,
        0.005,
    ):
        use_decoder = (
            pj_result_df[
                "full_order_normalized_entropy"
            ]
            >= threshold
        )

        final_prediction = np.where(
            use_decoder,
            decoder_prediction,
            pj_result_df[
                "baseline_prediction"
            ],
        )

        evaluated = pj_evaluate(
            pj_result_df,
            final_prediction,
            (
                f"{decoder_name}"
                f"_entropy_{threshold:.3f}"
            ),
        )

        evaluated["decoder"] = (
            decoder_name
        )

        evaluated[
            "entropy_threshold"
        ] = float(threshold)

        evaluated["decoder_rows"] = int(
            use_decoder.sum()
        )

        pj_threshold_rows.append(
            evaluated
        )


pj_threshold_summary_df = (
    pd.DataFrame(pj_threshold_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
            "changed_count",
        ],
        ascending=[
            False,
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nEntropy threshold 상위 결과"
)

display(
    pj_threshold_summary_df.head(30)
)


# ------------------------------------------------------------
# 16. 최고 positional 전략의 변경 샘플
# ------------------------------------------------------------
pj_best_row = (
    pj_threshold_summary_df.iloc[0]
)

pj_best_decoder = str(
    pj_best_row["decoder"]
)

pj_best_threshold = float(
    pj_best_row[
        "entropy_threshold"
    ]
)

pj_best_mask = (
    pj_result_df[
        "full_order_normalized_entropy"
    ]
    >= pj_best_threshold
)

pj_result_df[
    "best_positional_prediction"
] = np.where(
    pj_best_mask,
    pj_result_df[
        f"{pj_best_decoder}_prediction"
    ],
    pj_result_df[
        "baseline_prediction"
    ],
)

pj_result_df[
    "best_positional_exact"
] = (
    pj_result_df[
        "best_positional_prediction"
    ]
    ==
    pj_result_df["gold_order"]
).astype(int)


print(
    "\n최고 positional decoder:",
    pj_best_decoder,
)

print(
    "최고 entropy threshold:",
    pj_best_threshold,
)

print(
    "최고 exact:",
    pj_best_row["exact"],
)


display(
    pj_result_df[
        pj_result_df[
            "best_positional_prediction"
        ]
        !=
        pj_result_df[
            "baseline_prediction"
        ]
    ][
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "baseline_prediction",
            f"{pj_best_decoder}_prediction",
            "best_positional_prediction",
            "baseline_exact",
            "best_positional_exact",
            f"{pj_best_decoder}_margin",
            f"{pj_best_decoder}_top2",
        ]
    ].head(100)
)

Global positional joint 비교


,strategy,exact,correct_count,changed_count,wrong_to_right,right_to_wrong,net_change,canonical_1234_count,canonical_1234_rate
0,previous_joint_with_full,0.435,87,81,23,10,13,55,0.275
1,positional_cond_strong,0.405,81,64,16,9,7,34,0.170
2,positional_cond_full,0.385,77,66,16,13,3,28,0.140
3,baseline,0.370,74,0,0,0,0,28,0.140
4,positional_cond,0.365,73,67,15,16,-1,24,0.120
5,positional_joint,0.330,66,73,13,21,-8,18,0.090



Entropy 0.945 고정 hybrid


,strategy,exact,correct_count,changed_count,wrong_to_right,right_to_wrong,net_change,canonical_1234_count,canonical_1234_rate,decoder_rows
0,positional_cond_strong_entropy_0.945,0.400,80,53,11,5,6,36,0.18,128
1,positional_cond_full_entropy_0.945,0.395,79,51,10,5,5,32,0.16,128
2,positional_cond_entropy_0.945,0.390,78,48,9,5,4,30,0.15,128
3,positional_joint_entropy_0.945,0.375,75,49,7,6,1,26,0.13,128



Entropy threshold 상위 결과


,strategy,exact,correct_count,changed_count,wrong_to_right,right_to_wrong,net_change,canonical_1234_count,canonical_1234_rate,decoder,entropy_threshold,decoder_rows
0,positional_cond_strong_entropy_0.950,0.410,82,50,11,3,8,36,0.180,positional_cond_strong,0.950,122
1,positional_cond_strong_entropy_0.925,0.410,82,60,15,7,8,34,0.170,positional_cond_strong,0.925,163
2,positional_cond_full_entropy_0.950,0.405,81,49,10,3,7,32,0.160,positional_cond_full,0.950,122
3,positional_cond_strong_entropy_0.920,0.405,81,61,15,8,7,34,0.170,positional_cond_strong,0.920,173
4,positional_cond_entropy_0.950,0.400,80,46,9,3,6,30,0.150,positional_cond,0.950,122
5,positional_cond_strong_entropy_0.955,0.400,80,47,9,3,6,36,0.180,positional_cond_strong,0.955,111
6,positional_cond_strong_entropy_0.945,0.400,80,53,11,5,6,36,0.180,positional_cond_strong,0.945,128
7,positional_cond_strong_entropy_0.930,0.400,80,56,12,6,6,35,0.175,positional_cond_strong,0.930,154
8,positional_cond_full_entropy_0.955,0.395,79,46,8,3,5,32,0.160,positional_cond_full,0.955,111
9,positional_cond_full_entropy_0.945,0.395,79,51,10,5,5,32,0.160,positional_cond_full,0.945,128



최고 positional decoder: positional_cond_strong
최고 entropy threshold: 0.9500000000000001
최고 exact: 0.41


,sample_id,gold_order,full_order_normalized_entropy,baseline_prediction,positional_cond_strong_prediction,best_positional_prediction,baseline_exact,best_positional_exact,positional_cond_strong_margin,positional_cond_strong_top2
8,tUsgQQ,1 4 2 3,0.952219,4 1 2 3,4 1 3 2,4 1 3 2,0,0,0.057147,1 4 3 2
9,TWfXii,2 1 4 3,0.993056,3 2 1 4,1 2 3 4,1 2 3 4,0,0,0.017538,3 2 1 4
13,vnbiS3,3 4 2 1,0.995453,1 3 2 4,3 4 1 2,3 4 1 2,0,0,0.103812,3 4 2 1
14,loqSZu,4 3 2 1,0.995321,3 2 1 4,3 4 2 1,3 4 2 1,0,0,0.043830,3 2 1 4
23,o9v7Ns,1 2 3 4,0.968957,1 2 4 3,1 2 3 4,1 2 3 4,0,1,0.542671,1 2 4 3
26,hTyomo,2 4 1 3,0.988230,1 3 2 4,1 2 3 4,1 2 3 4,0,0,0.037136,1 3 4 2
28,BnOxub,1 4 3 2,0.985859,1 4 2 3,1 2 4 3,1 2 4 3,0,0,0.023599,4 1 2 3
36,kDcOFZ,4 2 1 3,0.973892,4 2 1 3,4 2 3 1,4 2 3 1,1,0,0.145878,4 2 1 3
41,hmQV7h,1 4 2 3,0.961252,4 1 2 3,4 3 2 1,4 3 2 1,0,0,0.170930,4 3 1 2
47,RPYON6,3 1 2 4,0.996929,4 1 2 3,4 2 1 3,4 2 1 3,0,0,0.017424,4 1 2 3


In [38]:
# ============================================================
# Symmetric mixed-anchor joint decoding
#
# 네 의도:
# - first anchor:
#     pairwise_first + endpoint_first
# - last anchor:
#     pairwise_last + endpoint_last
# - 각 anchor에 fixed-first / fixed-last conditional을 결합
# - 24개 전체 순서를 마지막까지 공동 비교
# - full-order 확률은 10% 보조
#
# 별도 모델 추론 없음
# ============================================================

import numpy as np
import pandas as pd


MIX_EPS = 1e-12
MIX_ENTROPY_THRESHOLD = 0.945


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def mix_safe_log(value):
    return float(
        np.log(
            max(
                float(value),
                MIX_EPS,
            )
        )
    )


def mix_normalize(values):
    values = {
        key: max(float(value), 0.0)
        for key, value in values.items()
    }

    total = sum(values.values())

    if total <= 0:
        uniform = 1.0 / len(values)

        return {
            key: uniform
            for key in values
        }

    return {
        key: value / total
        for key, value in values.items()
    }


# ------------------------------------------------------------
# 2. Row마다 first/last anchor 분포 생성
#
# 기존 10:30:60에서 order 10%는
# 전체 후보의 full-order 점수로 따로 사용.
#
# 따라서 anchor 내부는:
# pairwise : endpoint = 60 : 30 = 2 : 1
# ------------------------------------------------------------
def get_mixed_anchor_probs(
    row,
    pairwise_weight=2.0 / 3.0,
    endpoint_weight=1.0 / 3.0,
):
    pair_first = mix_normalize({
        frame: row[
            f"p_pairwise_first_{frame}"
        ]
        for frame in FE_FRAMES
    })

    endpoint_first = mix_normalize({
        frame: row[
            f"p_endpoint_first_{frame}"
        ]
        for frame in FE_FRAMES
    })

    pair_last = mix_normalize({
        frame: row[
            f"p_pairwise_last_{frame}"
        ]
        for frame in FE_FRAMES
    })

    endpoint_last = mix_normalize({
        frame: row[
            f"p_endpoint_last_{frame}"
        ]
        for frame in FE_FRAMES
    })

    mixed_first = {
        frame: (
            pairwise_weight
            * pair_first[frame]
            +
            endpoint_weight
            * endpoint_first[frame]
        )
        for frame in FE_FRAMES
    }

    mixed_last = {
        frame: (
            pairwise_weight
            * pair_last[frame]
            +
            endpoint_weight
            * endpoint_last[frame]
        )
        for frame in FE_FRAMES
    }

    return (
        mix_normalize(mixed_first),
        mix_normalize(mixed_last),
    )


# ------------------------------------------------------------
# 3. 후보 순서 점수
# ------------------------------------------------------------
def score_mixed_anchor_candidate(
    row,
    candidate_order,
    mixed_first,
    mixed_last,
    direction_first_weight=0.45,
    direction_last_weight=0.45,
    full_order_weight=0.10,
):
    first, second, third, last = (
        candidate_order
    )

    suffix = f"{second}{third}{last}"
    prefix = f"{first}{second}{third}"

    order_key = "".join(
        map(str, candidate_order)
    )

    first_anchor_probability = (
        mixed_first[first]
    )

    last_anchor_probability = (
        mixed_last[last]
    )

    fixed_first_probability = float(
        row[
            f"p_cond_fixed_first_"
            f"{first}_suffix_{suffix}"
        ]
    )

    fixed_last_probability = float(
        row[
            f"p_cond_fixed_last_"
            f"{last}_prefix_{prefix}"
        ]
    )

    full_order_probability = float(
        row[f"p_order_{order_key}"]
    )

    # log(anchor × conditional)
    first_direction_log = (
        mix_safe_log(
            first_anchor_probability
        )
        +
        mix_safe_log(
            fixed_first_probability
        )
    )

    last_direction_log = (
        mix_safe_log(
            last_anchor_probability
        )
        +
        mix_safe_log(
            fixed_last_probability
        )
    )

    total_score = (
        direction_first_weight
        * first_direction_log
        +
        direction_last_weight
        * last_direction_log
        +
        full_order_weight
        * mix_safe_log(
            full_order_probability
        )
    )

    return {
        "score": float(total_score),
        "first_anchor": float(
            first_anchor_probability
        ),
        "last_anchor": float(
            last_anchor_probability
        ),
        "fixed_first": float(
            fixed_first_probability
        ),
        "fixed_last": float(
            fixed_last_probability
        ),
        "full_order": float(
            full_order_probability
        ),
    }


# ------------------------------------------------------------
# 4. 24개 후보 공동 디코딩
# ------------------------------------------------------------
def decode_mixed_anchor_joint(
    row,
    pairwise_weight=2.0 / 3.0,
    endpoint_weight=1.0 / 3.0,
    direction_first_weight=0.45,
    direction_last_weight=0.45,
    full_order_weight=0.10,
):
    mixed_first, mixed_last = (
        get_mixed_anchor_probs(
            row,
            pairwise_weight=(
                pairwise_weight
            ),
            endpoint_weight=(
                endpoint_weight
            ),
        )
    )

    candidates = []

    for candidate_order in FE_ORDERS:
        component = (
            score_mixed_anchor_candidate(
                row=row,
                candidate_order=(
                    candidate_order
                ),
                mixed_first=mixed_first,
                mixed_last=mixed_last,
                direction_first_weight=(
                    direction_first_weight
                ),
                direction_last_weight=(
                    direction_last_weight
                ),
                full_order_weight=(
                    full_order_weight
                ),
            )
        )

        candidates.append({
            "candidate_order": (
                candidate_order
            ),
            **component,
        })

    candidates.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    top1 = candidates[0]
    top2 = candidates[1]

    return {
        "prediction": fe_order_text(
            top1["candidate_order"]
        ),
        "top2_prediction": fe_order_text(
            top2["candidate_order"]
        ),
        "margin": float(
            top1["score"]
            - top2["score"]
        ),
        "first_anchor": top1[
            "first_anchor"
        ],
        "last_anchor": top1[
            "last_anchor"
        ],
        "fixed_first": top1[
            "fixed_first"
        ],
        "fixed_last": top1[
            "fixed_last"
        ],
    }


# ------------------------------------------------------------
# 5. 비교할 혼합 비율
#
# mixed_60_30:
# 기존 60:30 비율을 정규화한 2:1
#
# mixed_equal:
# pairwise와 endpoint 동일 비중
#
# mixed_pairwise_strong:
# pairwise를 더 강하게
#
# mixed_previous_asym:
# anchor는 섞되 이전 방향 가중치
# 0.495 / 0.405 / 0.10 유지
# ------------------------------------------------------------
MIXED_CONFIGS = {
    "mixed_symmetric_60_30": {
        "pairwise_weight": 2.0 / 3.0,
        "endpoint_weight": 1.0 / 3.0,
        "direction_first_weight": 0.45,
        "direction_last_weight": 0.45,
        "full_order_weight": 0.10,
    },

    "mixed_symmetric_equal": {
        "pairwise_weight": 0.50,
        "endpoint_weight": 0.50,
        "direction_first_weight": 0.45,
        "direction_last_weight": 0.45,
        "full_order_weight": 0.10,
    },

    "mixed_pairwise_strong": {
        "pairwise_weight": 0.75,
        "endpoint_weight": 0.25,
        "direction_first_weight": 0.45,
        "direction_last_weight": 0.45,
        "full_order_weight": 0.10,
    },

    "mixed_previous_direction_weights": {
        "pairwise_weight": 2.0 / 3.0,
        "endpoint_weight": 1.0 / 3.0,
        "direction_first_weight": 0.495,
        "direction_last_weight": 0.405,
        "full_order_weight": 0.10,
    },
}


# ------------------------------------------------------------
# 6. 전체 샘플 계산
# ------------------------------------------------------------
mixed_rows = []

for _, row in fe_source_df.iterrows():
    sample_id = str(
        row["sample_id"]
    )

    original_result = (
        fe_result_df.loc[
            fe_result_df["sample_id"]
            == sample_id
        ]
    )

    if len(original_result) != 1:
        raise RuntimeError(
            f"sample_id 연결 실패: {sample_id}"
        )

    original_result = (
        original_result.iloc[0]
    )

    output = {
        "sample_id": sample_id,
        "gold_order": original_result[
            "gold_order"
        ],
        "baseline_prediction": (
            original_result[
                "baseline_prediction"
            ]
        ),
        "baseline_exact": int(
            original_result[
                "baseline_exact"
            ]
        ),
        "full_order_normalized_entropy": (
            float(
                row[
                    "full_order_normalized_entropy"
                ]
            )
        ),
        "previous_joint_prediction": (
            original_result[
                "previous_joint_with_full_prediction"
            ]
        ),
    }

    for config_name, config in (
        MIXED_CONFIGS.items()
    ):
        decoded = (
            decode_mixed_anchor_joint(
                row,
                **config,
            )
        )

        prediction = decoded[
            "prediction"
        ]

        output[
            f"{config_name}_prediction"
        ] = prediction

        output[
            f"{config_name}_exact"
        ] = int(
            prediction
            == output["gold_order"]
        )

        output[
            f"{config_name}_margin"
        ] = decoded["margin"]

        output[
            f"{config_name}_top2"
        ] = decoded[
            "top2_prediction"
        ]

    mixed_rows.append(output)


mixed_joint_result_df = pd.DataFrame(
    mixed_rows
)


# ------------------------------------------------------------
# 7. 평가 함수
# ------------------------------------------------------------
def evaluate_mixed_prediction(
    frame,
    prediction,
    strategy,
):
    prediction = pd.Series(
        prediction,
        index=frame.index,
    )

    exact = (
        prediction
        == frame["gold_order"]
    ).astype(int)

    changed = (
        prediction
        != frame[
            "baseline_prediction"
        ]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (exact == 0)
    )

    changed_from_previous = (
        prediction
        != frame[
            "previous_joint_prediction"
        ]
    )

    return {
        "strategy": strategy,
        "exact": float(
            exact.mean()
        ),
        "correct_count": int(
            exact.sum()
        ),
        "changed_from_baseline": int(
            changed.sum()
        ),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
        "changed_from_previous_joint": int(
            changed_from_previous.sum()
        ),
        "canonical_1234_count": int(
            (
                prediction
                == "1 2 3 4"
            ).sum()
        ),
    }


# ------------------------------------------------------------
# 8. Global 비교
# ------------------------------------------------------------
mixed_global_rows = [
    evaluate_mixed_prediction(
        mixed_joint_result_df,
        mixed_joint_result_df[
            "baseline_prediction"
        ],
        "baseline",
    ),

    evaluate_mixed_prediction(
        mixed_joint_result_df,
        mixed_joint_result_df[
            "previous_joint_prediction"
        ],
        "previous_joint_with_full",
    ),
]


for config_name in MIXED_CONFIGS:
    mixed_global_rows.append(
        evaluate_mixed_prediction(
            mixed_joint_result_df,
            mixed_joint_result_df[
                f"{config_name}_prediction"
            ],
            config_name,
        )
    )


mixed_global_summary_df = (
    pd.DataFrame(mixed_global_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "Global mixed-anchor joint 비교"
)

display(
    mixed_global_summary_df
)


# ------------------------------------------------------------
# 9. Entropy 0.945 고정 hybrid
# ------------------------------------------------------------
mixed_hybrid_mask = (
    mixed_joint_result_df[
        "full_order_normalized_entropy"
    ]
    >= MIX_ENTROPY_THRESHOLD
)


mixed_hybrid_rows = []


# 기존 previous joint hybrid도 표시
previous_hybrid_prediction = np.where(
    mixed_hybrid_mask,
    mixed_joint_result_df[
        "previous_joint_prediction"
    ],
    mixed_joint_result_df[
        "baseline_prediction"
    ],
)

mixed_hybrid_rows.append(
    evaluate_mixed_prediction(
        mixed_joint_result_df,
        previous_hybrid_prediction,
        "previous_joint_entropy_0.945",
    )
)


for config_name in MIXED_CONFIGS:
    final_prediction = np.where(
        mixed_hybrid_mask,
        mixed_joint_result_df[
            f"{config_name}_prediction"
        ],
        mixed_joint_result_df[
            "baseline_prediction"
        ],
    )

    evaluated = (
        evaluate_mixed_prediction(
            mixed_joint_result_df,
            final_prediction,
            (
                f"{config_name}"
                "_entropy_0.945"
            ),
        )
    )

    evaluated["decoder_rows"] = int(
        mixed_hybrid_mask.sum()
    )

    mixed_hybrid_rows.append(
        evaluated
    )


mixed_hybrid_summary_df = (
    pd.DataFrame(mixed_hybrid_rows)
    .sort_values(
        [
            "exact",
            "net_change",
            "right_to_wrong",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nEntropy 0.945 mixed-anchor hybrid"
)

display(
    mixed_hybrid_summary_df
)

Global mixed-anchor joint 비교


,strategy,exact,correct_count,changed_from_baseline,wrong_to_right,right_to_wrong,net_change,changed_from_previous_joint,canonical_1234_count
0,previous_joint_with_full,0.435,87,81,23,10,13,0,55
1,mixed_pairwise_strong,0.430,86,65,19,7,12,50,48
2,mixed_previous_direction_weights,0.430,86,66,19,7,12,52,48
3,mixed_symmetric_60_30,0.425,85,63,18,7,11,52,48
4,mixed_symmetric_equal,0.395,79,59,12,7,5,46,47
5,baseline,0.370,74,0,0,0,0,81,28



Entropy 0.945 mixed-anchor hybrid


,strategy,exact,correct_count,changed_from_baseline,wrong_to_right,right_to_wrong,net_change,changed_from_previous_joint,canonical_1234_count,decoder_rows
0,previous_joint_entropy_0.945,0.445,89,73,21,6,15,8,55,NaN
1,mixed_pairwise_strong_entropy_0.945,0.430,86,58,16,4,12,52,48,128.0
2,mixed_previous_direction_weights_entropy_0.945,0.430,86,59,16,4,12,54,48,128.0
3,mixed_symmetric_60_30_entropy_0.945,0.425,85,56,15,4,11,54,48,128.0
4,mixed_symmetric_equal_entropy_0.945,0.395,79,54,10,5,5,50,47,128.0


In [39]:
# ============================================================
# Asymmetric anchor weight sweep
#
# first:
#   alpha_first * pairwise_first
#   + (1-alpha_first) * endpoint_first
#
# last:
#   alpha_last * pairwise_last
#   + (1-alpha_last) * endpoint_last
#
# 24개 완성 순서를 모두 공동 디코딩
# 재추론 없음
# ============================================================

import numpy as np
import pandas as pd


ASYM_EPS = 1e-12
ASYM_ENTROPY_THRESHOLD = 0.945


def asym_safe_log(value):
    return float(
        np.log(
            max(float(value), ASYM_EPS)
        )
    )


def asym_normalize(values):
    cleaned = {
        key: max(float(value), 0.0)
        for key, value in values.items()
    }

    total = sum(cleaned.values())

    if total <= 0:
        uniform = 1.0 / len(cleaned)

        return {
            key: uniform
            for key in cleaned
        }

    return {
        key: value / total
        for key, value in cleaned.items()
    }


def asym_get_anchor_sources(row):
    pair_first = asym_normalize({
        frame: row[
            f"p_pairwise_first_{frame}"
        ]
        for frame in FE_FRAMES
    })

    endpoint_first = asym_normalize({
        frame: row[
            f"p_endpoint_first_{frame}"
        ]
        for frame in FE_FRAMES
    })

    pair_last = asym_normalize({
        frame: row[
            f"p_pairwise_last_{frame}"
        ]
        for frame in FE_FRAMES
    })

    endpoint_last = asym_normalize({
        frame: row[
            f"p_endpoint_last_{frame}"
        ]
        for frame in FE_FRAMES
    })

    return {
        "pair_first": pair_first,
        "endpoint_first": endpoint_first,
        "pair_last": pair_last,
        "endpoint_last": endpoint_last,
    }


def asym_decode_row(
    row,
    alpha_first,
    alpha_last,
    first_direction_weight=0.495,
    last_direction_weight=0.405,
    full_order_weight=0.10,
):
    sources = asym_get_anchor_sources(row)

    mixed_first = asym_normalize({
        frame: (
            alpha_first
            * sources["pair_first"][frame]
            +
            (1.0 - alpha_first)
            * sources["endpoint_first"][frame]
        )
        for frame in FE_FRAMES
    })

    mixed_last = asym_normalize({
        frame: (
            alpha_last
            * sources["pair_last"][frame]
            +
            (1.0 - alpha_last)
            * sources["endpoint_last"][frame]
        )
        for frame in FE_FRAMES
    })

    candidate_rows = []

    for candidate_order in FE_ORDERS:
        first, second, third, last = (
            candidate_order
        )

        suffix = f"{second}{third}{last}"
        prefix = f"{first}{second}{third}"

        order_key = "".join(
            map(str, candidate_order)
        )

        # first 후보 확률
        # × first 고정 후 나머지 3장 완성 순서
        first_direction_log = (
            asym_safe_log(
                mixed_first[first]
            )
            +
            asym_safe_log(
                row[
                    f"p_cond_fixed_first_"
                    f"{first}_suffix_{suffix}"
                ]
            )
        )

        # last 후보 확률
        # × last 고정 후 앞의 3장 완성 순서
        last_direction_log = (
            asym_safe_log(
                mixed_last[last]
            )
            +
            asym_safe_log(
                row[
                    f"p_cond_fixed_last_"
                    f"{last}_prefix_{prefix}"
                ]
            )
        )

        full_order_log = asym_safe_log(
            row[f"p_order_{order_key}"]
        )

        total_score = (
            first_direction_weight
            * first_direction_log
            +
            last_direction_weight
            * last_direction_log
            +
            full_order_weight
            * full_order_log
        )

        candidate_rows.append({
            "candidate_order": candidate_order,
            "score": float(total_score),
        })

    # 24개 완성 순서를 전부 점수화한 후
    # 마지막에 한 번만 선택
    candidate_rows.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    top1 = candidate_rows[0]
    top2 = candidate_rows[1]

    return {
        "prediction": fe_order_text(
            top1["candidate_order"]
        ),
        "top2": fe_order_text(
            top2["candidate_order"]
        ),
        "margin": float(
            top1["score"]
            - top2["score"]
        ),
    }


# ------------------------------------------------------------
# 평가용 데이터 연결
# ------------------------------------------------------------
asym_eval_df = (
    fe_result_df[
        [
            "sample_id",
            "gold_order",
            "baseline_prediction",
            "baseline_exact",
            "full_order_normalized_entropy",
            "previous_joint_with_full_prediction",
        ]
    ]
    .copy()
)

asym_eval_df["sample_id"] = (
    asym_eval_df["sample_id"]
    .astype(str)
)


def asym_evaluate(
    frame,
    predictions,
    strategy,
):
    predictions = pd.Series(
        predictions,
        index=frame.index,
    )

    exact = (
        predictions
        == frame["gold_order"]
    ).astype(int)

    changed = (
        predictions
        != frame["baseline_prediction"]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (exact == 0)
    )

    return {
        "strategy": strategy,
        "exact": float(exact.mean()),
        "correct_count": int(exact.sum()),
        "changed_count": int(changed.sum()),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
        "canonical_1234_count": int(
            (
                predictions == "1 2 3 4"
            ).sum()
        ),
    }


# ------------------------------------------------------------
# First와 last 혼합 비율을 별도로 탐색
#
# first는 pairwise가 강해 보이므로 0.5~1.0
# last는 endpoint가 강해 보이므로 0.0~0.5
# ------------------------------------------------------------
alpha_first_values = np.round(
    np.arange(0.50, 1.001, 0.05),
    2,
)

alpha_last_values = np.round(
    np.arange(0.00, 0.501, 0.05),
    2,
)


asym_sweep_rows = []
asym_prediction_cache = {}


# sample_id -> raw row 빠른 연결
asym_raw_lookup = {
    str(row["sample_id"]): row
    for _, row in fe_source_df.iterrows()
}


for alpha_first in alpha_first_values:
    for alpha_last in alpha_last_values:
        predictions = []

        for _, eval_row in (
            asym_eval_df.iterrows()
        ):
            sample_id = str(
                eval_row["sample_id"]
            )

            raw_row = asym_raw_lookup[
                sample_id
            ]

            decoded = asym_decode_row(
                raw_row,
                alpha_first=float(
                    alpha_first
                ),
                alpha_last=float(
                    alpha_last
                ),
            )

            predictions.append(
                decoded["prediction"]
            )

        predictions = pd.Series(
            predictions,
            index=asym_eval_df.index,
        )

        cache_key = (
            float(alpha_first),
            float(alpha_last),
        )

        asym_prediction_cache[
            cache_key
        ] = predictions

        # Global
        global_result = asym_evaluate(
            asym_eval_df,
            predictions,
            (
                f"af_{alpha_first:.2f}"
                f"_al_{alpha_last:.2f}"
                "_global"
            ),
        )

        # Entropy 0.945 hybrid
        use_joint = (
            asym_eval_df[
                "full_order_normalized_entropy"
            ]
            >= ASYM_ENTROPY_THRESHOLD
        )

        hybrid_predictions = np.where(
            use_joint,
            predictions,
            asym_eval_df[
                "baseline_prediction"
            ],
        )

        hybrid_result = asym_evaluate(
            asym_eval_df,
            hybrid_predictions,
            (
                f"af_{alpha_first:.2f}"
                f"_al_{alpha_last:.2f}"
                "_hybrid"
            ),
        )

        asym_sweep_rows.append({
            "alpha_first_pairwise": float(
                alpha_first
            ),
            "alpha_last_pairwise": float(
                alpha_last
            ),

            "global_exact": global_result[
                "exact"
            ],
            "global_correct": global_result[
                "correct_count"
            ],
            "global_wrong_to_right": (
                global_result[
                    "wrong_to_right"
                ]
            ),
            "global_right_to_wrong": (
                global_result[
                    "right_to_wrong"
                ]
            ),
            "global_net": global_result[
                "net_change"
            ],

            "hybrid_exact": hybrid_result[
                "exact"
            ],
            "hybrid_correct": hybrid_result[
                "correct_count"
            ],
            "hybrid_wrong_to_right": (
                hybrid_result[
                    "wrong_to_right"
                ]
            ),
            "hybrid_right_to_wrong": (
                hybrid_result[
                    "right_to_wrong"
                ]
            ),
            "hybrid_net": hybrid_result[
                "net_change"
            ],

            "canonical_1234_count": int(
                (
                    predictions
                    == "1 2 3 4"
                ).sum()
            ),
        })


asym_sweep_df = (
    pd.DataFrame(asym_sweep_rows)
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "First/last 비대칭 anchor sweep 상위 결과"
)

display(
    asym_sweep_df.head(30)
)


# ------------------------------------------------------------
# 기존 previous joint 위치 확인
#
# alpha_first=1.0:
# first는 pairwise만
#
# alpha_last=0.0:
# last는 endpoint만
# ------------------------------------------------------------
print(
    "\n기존 previous joint와 동일한 anchor:"
)

display(
    asym_sweep_df[
        np.isclose(
            asym_sweep_df[
                "alpha_first_pairwise"
            ],
            1.0,
        )
        &
        np.isclose(
            asym_sweep_df[
                "alpha_last_pairwise"
            ],
            0.0,
        )
    ]
)

First/last 비대칭 anchor sweep 상위 결과


,alpha_first_pairwise,alpha_last_pairwise,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count
0,1.00,0.45,0.445,89,25,10,15,0.450,90,22,6,16,54
1,1.00,0.50,0.440,88,25,11,14,0.445,89,21,6,15,51
2,1.00,0.00,0.435,87,23,10,13,0.445,89,21,6,15,55
3,1.00,0.35,0.435,87,23,10,13,0.440,88,20,6,14,54
4,1.00,0.40,0.435,87,23,10,13,0.440,88,20,6,14,54
5,1.00,0.05,0.430,86,22,10,12,0.440,88,20,6,14,54
6,1.00,0.10,0.430,86,22,10,12,0.440,88,20,6,14,54
7,1.00,0.15,0.430,86,22,10,12,0.440,88,20,6,14,54
8,1.00,0.20,0.430,86,22,10,12,0.440,88,20,6,14,54
9,1.00,0.25,0.430,86,22,10,12,0.440,88,20,6,14,54



기존 previous joint와 동일한 anchor:


,alpha_first_pairwise,alpha_last_pairwise,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count
2,1.0,0.0,0.435,87,23,10,13,0.445,89,21,6,15,55


In [40]:
# ============================================================
# Recursive Pairwise Joint Decoder
#
# - 추가 모델 추론 없음
# - 저장된 pairwise / endpoint / conditional / order 확률 사용
# - 24개 완성 순서를 전부 유지
# - 각 단계에서 남은 후보끼리 pairwise 점수를 재정규화
#
# 기준 anchor:
#   first = pairwise_first 100%
#   last  = pairwise_last 45% + endpoint_last 55%
#
# recursive_weight=0이면
# 비대칭 anchor 최고 결과를 재현해야 함:
#   global = 0.445
#   entropy 0.945 hybrid = 0.450
# ============================================================

import itertools
from math import prod

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. 사전 변수 확인
# ------------------------------------------------------------
required_global_names = [
    "fe_source_df",
    "fe_result_df",
    "FE_FRAMES",
    "FE_ORDERS",
    "fe_order_text",
]

missing_global_names = [
    name
    for name in required_global_names
    if name not in globals()
]

if missing_global_names:
    raise RuntimeError(
        "먼저 이전 full-evidence 독립 셀을 실행하세요.\n"
        f"누락 변수: {missing_global_names}"
    )


RC_EPS = 1e-12
RC_ENTROPY_THRESHOLD = 0.945

# 현재 sweep에서 가장 좋았던 anchor
RC_ALPHA_FIRST_PAIRWISE = 1.00
RC_ALPHA_LAST_PAIRWISE = 0.45

# full-order는 기존처럼 10%
RC_FULL_ORDER_WEIGHT = 0.10

# 재귀 pairwise 보조 가중치
RC_RECURSIVE_WEIGHTS = np.round(
    np.arange(0.00, 0.301, 0.025),
    3,
)

# inner:
#   first/last anchor를 제외하고 중간 재귀만 사용
#
# full:
#   first부터 last까지 전체 재귀 경로 사용
RC_RECURSIVE_MODES = [
    "inner",
    "full",
]


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def rc_safe_log(value):
    return float(
        np.log(
            max(float(value), RC_EPS)
        )
    )


def rc_normalize_dict(values):
    cleaned = {
        key: max(float(value), 0.0)
        for key, value in values.items()
    }

    total = sum(cleaned.values())

    if total <= 0:
        uniform = 1.0 / len(cleaned)

        return {
            key: uniform
            for key in cleaned
        }

    return {
        key: value / total
        for key, value in cleaned.items()
    }


# ------------------------------------------------------------
# 2. Pairwise 방향 확률
#
# p(a before b), p(b before a)가
# 정확히 합 1이 아닐 수 있으므로 쌍별 정규화
# ------------------------------------------------------------
def rc_pair_before_probability(
    row,
    earlier,
    later,
):
    forward = max(
        float(
            row[
                f"p_pair_{earlier}"
                f"_before_{later}"
            ]
        ),
        0.0,
    )

    backward = max(
        float(
            row[
                f"p_pair_{later}"
                f"_before_{earlier}"
            ]
        ),
        0.0,
    )

    total = forward + backward

    if total <= 0:
        return 0.5

    return float(
        forward / total
    )


# ------------------------------------------------------------
# 3. 현재 남은 후보 중 "다음 장" 분포
#
# front_raw(x | remaining)
# = product P(x before y)
#           for every other y
#
# 이후 remaining 내부에서 정규화
# ------------------------------------------------------------
def rc_front_distribution(
    row,
    remaining_frames,
):
    remaining_frames = tuple(
        remaining_frames
    )

    raw_scores = {}

    for candidate in remaining_frames:
        other_frames = [
            frame
            for frame in remaining_frames
            if frame != candidate
        ]

        if not other_frames:
            raw_scores[candidate] = 1.0
            continue

        raw_scores[candidate] = prod(
            rc_pair_before_probability(
                row,
                candidate,
                other,
            )
            for other in other_frames
        )

    return rc_normalize_dict(
        raw_scores
    )


# ------------------------------------------------------------
# 4. 현재 남은 후보 중 "마지막 장" 분포
#
# back_raw(x | remaining)
# = product P(y before x)
#           for every other y
# ------------------------------------------------------------
def rc_back_distribution(
    row,
    remaining_frames,
):
    remaining_frames = tuple(
        remaining_frames
    )

    raw_scores = {}

    for candidate in remaining_frames:
        other_frames = [
            frame
            for frame in remaining_frames
            if frame != candidate
        ]

        if not other_frames:
            raw_scores[candidate] = 1.0
            continue

        raw_scores[candidate] = prod(
            rc_pair_before_probability(
                row,
                other,
                candidate,
            )
            for other in other_frames
        )

    return rc_normalize_dict(
        raw_scores
    )


# ------------------------------------------------------------
# 5. 한 완성 순서의 재귀 pairwise 경로 확률
#
# candidate_order = (a, b, c, d)
#
# forward:
#   q(a | abcd)
#   q(b | bcd)
#   q(c | cd)
#
# backward:
#   q(d last | abcd)
#   q(c last | abc)
#   q(b last | ab)
# ------------------------------------------------------------
def rc_recursive_path_components(
    row,
    candidate_order,
):
    candidate_order = tuple(
        candidate_order
    )

    # ------------------------
    # 앞 방향 재귀
    # ------------------------
    forward_probabilities = []

    forward_remaining = list(
        candidate_order
    )

    for position in range(
        len(candidate_order) - 1
    ):
        chosen = candidate_order[
            position
        ]

        distribution = (
            rc_front_distribution(
                row,
                forward_remaining,
            )
        )

        forward_probabilities.append(
            float(distribution[chosen])
        )

        forward_remaining.remove(
            chosen
        )

    # ------------------------
    # 뒤 방향 재귀
    # ------------------------
    backward_probabilities = []

    backward_remaining = list(
        candidate_order
    )

    for position in range(
        len(candidate_order) - 1,
        0,
        -1,
    ):
        chosen = candidate_order[
            position
        ]

        distribution = (
            rc_back_distribution(
                row,
                backward_remaining,
            )
        )

        backward_probabilities.append(
            float(distribution[chosen])
        )

        backward_remaining.remove(
            chosen
        )

    # 예:
    # forward = [first 선택, second 선택, third 선택]
    # backward = [last 선택, third 선택, second 선택]

    forward_full_logs = [
        rc_safe_log(probability)
        for probability
        in forward_probabilities
    ]

    backward_full_logs = [
        rc_safe_log(probability)
        for probability
        in backward_probabilities
    ]

    # first/last anchor와 중복되지 않게
    # 첫 단계와 마지막 단계 제외
    forward_inner_logs = (
        forward_full_logs[1:]
    )

    backward_inner_logs = (
        backward_full_logs[1:]
    )

    # inner는 각 방향 2단계이므로
    # first/last conditional component와 비슷한
    # "두 log factor" 크기로 유지
    recursive_inner_log = float(
        0.5 * sum(forward_inner_logs)
        +
        0.5 * sum(backward_inner_logs)
    )

    # full은 각 방향 3단계.
    # 크기를 inner와 비슷하게 맞추기 위해 2/3 보정
    recursive_full_log = float(
        (2.0 / 3.0)
        * (
            0.5 * sum(forward_full_logs)
            +
            0.5 * sum(backward_full_logs)
        )
    )

    return {
        "forward_probabilities": (
            forward_probabilities
        ),
        "backward_probabilities": (
            backward_probabilities
        ),
        "recursive_inner_log": (
            recursive_inner_log
        ),
        "recursive_full_log": (
            recursive_full_log
        ),
    }


# ------------------------------------------------------------
# 6. First/last anchor 생성
#
# first:
# alpha=1.0 → pairwise_first만
#
# last:
# alpha=0.45
# → pairwise_last 45% + endpoint_last 55%
# ------------------------------------------------------------
def rc_get_anchor_probs(
    row,
    alpha_first_pairwise=(
        RC_ALPHA_FIRST_PAIRWISE
    ),
    alpha_last_pairwise=(
        RC_ALPHA_LAST_PAIRWISE
    ),
):
    pair_first = rc_normalize_dict({
        frame: row[
            f"p_pairwise_first_{frame}"
        ]
        for frame in FE_FRAMES
    })

    endpoint_first = rc_normalize_dict({
        frame: row[
            f"p_endpoint_first_{frame}"
        ]
        for frame in FE_FRAMES
    })

    pair_last = rc_normalize_dict({
        frame: row[
            f"p_pairwise_last_{frame}"
        ]
        for frame in FE_FRAMES
    })

    endpoint_last = rc_normalize_dict({
        frame: row[
            f"p_endpoint_last_{frame}"
        ]
        for frame in FE_FRAMES
    })

    mixed_first = rc_normalize_dict({
        frame: (
            alpha_first_pairwise
            * pair_first[frame]
            +
            (
                1.0
                - alpha_first_pairwise
            )
            * endpoint_first[frame]
        )
        for frame in FE_FRAMES
    })

    mixed_last = rc_normalize_dict({
        frame: (
            alpha_last_pairwise
            * pair_last[frame]
            +
            (
                1.0
                - alpha_last_pairwise
            )
            * endpoint_last[frame]
        )
        for frame in FE_FRAMES
    })

    return mixed_first, mixed_last


# ------------------------------------------------------------
# 7. 후보 순서 점수
#
# 기존 joint:
# first anchor × fixed-first conditional
# last anchor  × fixed-last conditional
#
# 여기에 recursive pairwise를 보조 추가
# ------------------------------------------------------------
def rc_score_candidate(
    row,
    candidate_order,
    mixed_first,
    mixed_last,
    recursive_weight,
    recursive_mode,
):
    first, second, third, last = (
        candidate_order
    )

    suffix = (
        f"{second}{third}{last}"
    )

    prefix = (
        f"{first}{second}{third}"
    )

    order_key = "".join(
        map(str, candidate_order)
    )

    # 기존 first 방향
    first_direction_log = (
        rc_safe_log(
            mixed_first[first]
        )
        +
        rc_safe_log(
            row[
                f"p_cond_fixed_first_"
                f"{first}_suffix_{suffix}"
            ]
        )
    )

    # 기존 last 방향
    last_direction_log = (
        rc_safe_log(
            mixed_last[last]
        )
        +
        rc_safe_log(
            row[
                f"p_cond_fixed_last_"
                f"{last}_prefix_{prefix}"
            ]
        )
    )

    # 재귀 pairwise
    recursive_components = (
        rc_recursive_path_components(
            row,
            candidate_order,
        )
    )

    if recursive_mode == "inner":
        recursive_log = (
            recursive_components[
                "recursive_inner_log"
            ]
        )

    elif recursive_mode == "full":
        recursive_log = (
            recursive_components[
                "recursive_full_log"
            ]
        )

    else:
        raise ValueError(
            f"잘못된 recursive_mode: "
            f"{recursive_mode}"
        )

    full_order_log = rc_safe_log(
        row[f"p_order_{order_key}"]
    )

    # 기존에는:
    # first 0.495
    # last  0.405
    # order 0.100
    #
    # recursive_weight만큼 first/last 몫에서 떼어냄
    available_direction_mass = (
        1.0
        - RC_FULL_ORDER_WEIGHT
        - recursive_weight
    )

    if available_direction_mass < 0:
        raise ValueError(
            "recursive_weight가 너무 큽니다."
        )

    # 기존 first:last 비율
    # 0.495 : 0.405 = 0.55 : 0.45
    first_weight = (
        available_direction_mass
        * 0.55
    )

    last_weight = (
        available_direction_mass
        * 0.45
    )

    total_score = (
        first_weight
        * first_direction_log
        +
        last_weight
        * last_direction_log
        +
        recursive_weight
        * recursive_log
        +
        RC_FULL_ORDER_WEIGHT
        * full_order_log
    )

    return {
        "score": float(total_score),
        "first_direction_log": float(
            first_direction_log
        ),
        "last_direction_log": float(
            last_direction_log
        ),
        "recursive_log": float(
            recursive_log
        ),
        "full_order_log": float(
            full_order_log
        ),
    }


# ------------------------------------------------------------
# 8. 24개 완성 순서 공동 디코딩
# ------------------------------------------------------------
def rc_decode_row(
    row,
    recursive_weight,
    recursive_mode,
):
    mixed_first, mixed_last = (
        rc_get_anchor_probs(row)
    )

    candidate_rows = []

    for candidate_order in FE_ORDERS:
        scored = rc_score_candidate(
            row=row,
            candidate_order=(
                candidate_order
            ),
            mixed_first=mixed_first,
            mixed_last=mixed_last,
            recursive_weight=float(
                recursive_weight
            ),
            recursive_mode=(
                recursive_mode
            ),
        )

        candidate_rows.append({
            "candidate_order": (
                candidate_order
            ),
            **scored,
        })

    # 중간 확정 없음.
    # 24개를 모두 계산한 뒤 마지막에 선택.
    candidate_rows.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    top1 = candidate_rows[0]
    top2 = candidate_rows[1]

    return {
        "prediction": fe_order_text(
            top1["candidate_order"]
        ),
        "top2_prediction": fe_order_text(
            top2["candidate_order"]
        ),
        "margin": float(
            top1["score"]
            - top2["score"]
        ),
    }


# ------------------------------------------------------------
# 9. 평가 데이터 및 raw lookup
# ------------------------------------------------------------
rc_eval_df = (
    fe_result_df[
        [
            "sample_id",
            "gold_order",
            "baseline_prediction",
            "baseline_exact",
            "full_order_normalized_entropy",
        ]
    ]
    .copy()
)

rc_eval_df["sample_id"] = (
    rc_eval_df["sample_id"]
    .astype(str)
)


rc_raw_lookup = {
    str(row["sample_id"]): row
    for _, row in fe_source_df.iterrows()
}


def rc_evaluate(
    frame,
    predictions,
    strategy_name,
):
    predictions = pd.Series(
        predictions,
        index=frame.index,
    )

    exact = (
        predictions
        == frame["gold_order"]
    ).astype(int)

    changed = (
        predictions
        != frame["baseline_prediction"]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (exact == 0)
    )

    return {
        "strategy": strategy_name,
        "exact": float(
            exact.mean()
        ),
        "correct_count": int(
            exact.sum()
        ),
        "changed_count": int(
            changed.sum()
        ),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
        "canonical_1234_count": int(
            (
                predictions
                == "1 2 3 4"
            ).sum()
        ),
    }


# ------------------------------------------------------------
# 10. 재귀 방식 및 가중치 sweep
# ------------------------------------------------------------
rc_sweep_rows = []
rc_prediction_cache = {}


for recursive_mode in (
    RC_RECURSIVE_MODES
):
    for recursive_weight in (
        RC_RECURSIVE_WEIGHTS
    ):
        predictions = []

        for _, eval_row in (
            rc_eval_df.iterrows()
        ):
            sample_id = str(
                eval_row["sample_id"]
            )

            raw_row = rc_raw_lookup[
                sample_id
            ]

            decoded = rc_decode_row(
                row=raw_row,
                recursive_weight=float(
                    recursive_weight
                ),
                recursive_mode=(
                    recursive_mode
                ),
            )

            predictions.append(
                decoded["prediction"]
            )

        predictions = pd.Series(
            predictions,
            index=rc_eval_df.index,
        )

        cache_key = (
            recursive_mode,
            float(recursive_weight),
        )

        rc_prediction_cache[
            cache_key
        ] = predictions

        # ------------------------
        # Global
        # ------------------------
        global_result = rc_evaluate(
            rc_eval_df,
            predictions,
            (
                f"{recursive_mode}"
                f"_rw_{recursive_weight:.3f}"
                "_global"
            ),
        )

        # ------------------------
        # Entropy 0.945 hybrid
        # ------------------------
        use_recursive = (
            rc_eval_df[
                "full_order_normalized_entropy"
            ]
            >= RC_ENTROPY_THRESHOLD
        )

        hybrid_predictions = np.where(
            use_recursive,
            predictions,
            rc_eval_df[
                "baseline_prediction"
            ],
        )

        hybrid_result = rc_evaluate(
            rc_eval_df,
            hybrid_predictions,
            (
                f"{recursive_mode}"
                f"_rw_{recursive_weight:.3f}"
                "_hybrid"
            ),
        )

        rc_sweep_rows.append({
            "recursive_mode": (
                recursive_mode
            ),
            "recursive_weight": float(
                recursive_weight
            ),

            "global_exact": global_result[
                "exact"
            ],
            "global_correct": global_result[
                "correct_count"
            ],
            "global_wrong_to_right": (
                global_result[
                    "wrong_to_right"
                ]
            ),
            "global_right_to_wrong": (
                global_result[
                    "right_to_wrong"
                ]
            ),
            "global_net": global_result[
                "net_change"
            ],

            "hybrid_exact": hybrid_result[
                "exact"
            ],
            "hybrid_correct": hybrid_result[
                "correct_count"
            ],
            "hybrid_wrong_to_right": (
                hybrid_result[
                    "wrong_to_right"
                ]
            ),
            "hybrid_right_to_wrong": (
                hybrid_result[
                    "right_to_wrong"
                ]
            ),
            "hybrid_net": hybrid_result[
                "net_change"
            ],

            "canonical_1234_count": int(
                (
                    predictions
                    == "1 2 3 4"
                ).sum()
            ),
        })


rc_sweep_df = (
    pd.DataFrame(rc_sweep_rows)
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "Recursive pairwise joint 상위 결과"
)

display(
    rc_sweep_df.head(30)
)


# ------------------------------------------------------------
# 11. recursive_weight=0 재현 확인
#
# inner/full 모두 재귀 가중치가 0이면
# 같은 anchor decoder가 되어야 함.
#
# 예상:
# global 0.445
# hybrid 0.450
# ------------------------------------------------------------
print(
    "\nrecursive_weight=0 기준 결과"
)

display(
    rc_sweep_df[
        np.isclose(
            rc_sweep_df[
                "recursive_weight"
            ],
            0.0,
        )
    ]
)


# ------------------------------------------------------------
# 12. 최고 결과의 변경 샘플 확인
# ------------------------------------------------------------
rc_best_row = rc_sweep_df.iloc[0]

rc_best_mode = str(
    rc_best_row["recursive_mode"]
)

rc_best_weight = float(
    rc_best_row["recursive_weight"]
)

rc_best_predictions = (
    rc_prediction_cache[
        (
            rc_best_mode,
            rc_best_weight,
        )
    ]
)

rc_best_hybrid_predictions = np.where(
    rc_eval_df[
        "full_order_normalized_entropy"
    ]
    >= RC_ENTROPY_THRESHOLD,
    rc_best_predictions,
    rc_eval_df[
        "baseline_prediction"
    ],
)

rc_best_result_df = (
    rc_eval_df.copy()
)

rc_best_result_df[
    "recursive_prediction"
] = rc_best_predictions

rc_best_result_df[
    "recursive_hybrid_prediction"
] = rc_best_hybrid_predictions

rc_best_result_df[
    "recursive_hybrid_exact"
] = (
    rc_best_result_df[
        "recursive_hybrid_prediction"
    ]
    ==
    rc_best_result_df[
        "gold_order"
    ]
).astype(int)


print(
    "\n최고 recursive mode:",
    rc_best_mode,
)

print(
    "최고 recursive weight:",
    rc_best_weight,
)

print(
    "최고 global exact:",
    rc_best_row["global_exact"],
)

print(
    "최고 hybrid exact:",
    rc_best_row["hybrid_exact"],
)


display(
    rc_best_result_df[
        rc_best_result_df[
            "recursive_hybrid_prediction"
        ]
        !=
        rc_best_result_df[
            "baseline_prediction"
        ]
    ][
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "baseline_prediction",
            "recursive_prediction",
            "recursive_hybrid_prediction",
            "baseline_exact",
            "recursive_hybrid_exact",
        ]
    ].head(100)
)

Recursive pairwise joint 상위 결과


,recursive_mode,recursive_weight,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count
0,inner,0.000,0.445,89,25,10,15,0.450,90,22,6,16,54
1,full,0.000,0.445,89,25,10,15,0.450,90,22,6,16,54
2,inner,0.025,0.430,86,22,10,12,0.430,86,18,6,12,48
3,inner,0.200,0.415,83,21,12,9,0.430,86,16,4,12,34
4,full,0.200,0.415,83,22,13,9,0.430,86,16,4,12,33
5,full,0.050,0.430,86,23,11,12,0.425,85,17,6,11,42
6,inner,0.050,0.425,85,21,10,11,0.425,85,17,6,11,45
7,full,0.025,0.425,85,21,10,11,0.425,85,17,6,11,46
8,inner,0.150,0.415,83,22,13,9,0.425,85,16,5,11,36
9,inner,0.225,0.410,82,20,12,8,0.425,85,15,4,11,33



recursive_weight=0 기준 결과


,recursive_mode,recursive_weight,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count
0,inner,0.0,0.445,89,25,10,15,0.45,90,22,6,16,54
1,full,0.0,0.445,89,25,10,15,0.45,90,22,6,16,54



최고 recursive mode: inner
최고 recursive weight: 0.0
최고 global exact: 0.445
최고 hybrid exact: 0.45


,sample_id,gold_order,full_order_normalized_entropy,baseline_prediction,recursive_prediction,recursive_hybrid_prediction,baseline_exact,recursive_hybrid_exact
1,jOilDx,1 3 2 4,0.995850,1 3 4 2,1 2 3 4,1 2 3 4,0,0
2,B6W1Qk,2 1 3 4,0.952003,2 1 3 4,1 2 3 4,1 2 3 4,1,0
9,TWfXii,2 1 4 3,0.993056,3 2 1 4,1 2 3 4,1 2 3 4,0,0
13,vnbiS3,3 4 2 1,0.995453,1 3 2 4,1 2 3 4,1 2 3 4,0,0
14,loqSZu,4 3 2 1,0.995321,3 2 1 4,3 1 2 4,3 1 2 4,0,0
...,...,...,...,...,...,...,...,...
186,xA111Z,2 1 3 4,0.995229,4 2 1 3,1 2 3 4,1 2 3 4,0,0
193,x4RTOK,1 4 2 3,0.998593,4 2 1 3,4 1 2 3,4 1 2 3,0,0
194,3jqefc,3 4 2 1,0.997835,1 3 2 4,3 1 2 4,3 1 2 4,0,0
195,TKxZIX,3 1 4 2,0.955277,3 1 2 4,1 3 2 4,1 3 2 4,0,0


In [41]:
# ============================================================
# True Mixed Recursive Joint Decoder
#
# 후보 (a, b, c, d)에 대해:
#
# Forward
# 1) a 선택:
#    order-first + pairwise-first + endpoint-first
#
# 2) b 선택 | prefix=a:
#    order-next + recursive-pairwise-next
#    + fixed-first-conditional-next
#
# 3) c 선택 | prefix=a,b:
#    order-next + recursive-pairwise-next
#    + fixed-first-conditional-next
#
# Backward
# 1) d 선택:
#    order-last + pairwise-last + endpoint-last
#
# 2) c 선택 | suffix=d:
#    order-prev + recursive-pairwise-prev
#    + fixed-last-conditional-prev
#
# 3) b 선택 | suffix=c,d:
#    order-prev + recursive-pairwise-prev
#    + fixed-last-conditional-prev
#
# 24개 완성 순서를 전부 평가한 뒤 마지막에 1개 선택.
# 중간 greedy 확정 없음.
#
# 추가 모델 추론 없음.
# raw_df_200에 저장된 확률만 사용.
# ============================================================

import itertools
import re
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. 기본 설정 및 데이터 연결
# ------------------------------------------------------------
MR_FRAMES = (1, 2, 3, 4)
MR_ORDERS = list(itertools.permutations(MR_FRAMES))
MR_EPS = 1e-12
MR_ENTROPY_THRESHOLD = 0.945


if "fe_source_df" in globals():
    mr_source_df = fe_source_df.copy()

elif "raw_df_200" in globals():
    mr_source_df = raw_df_200.copy()

else:
    raise RuntimeError(
        "fe_source_df 또는 raw_df_200이 없습니다."
    )


if "fe_result_df" not in globals():
    raise RuntimeError(
        "fe_result_df가 없습니다. "
        "기존 결과 재현 셀을 먼저 실행하세요."
    )


mr_source_df["sample_id"] = (
    mr_source_df["sample_id"]
    .astype(str)
)

mr_eval_df = (
    fe_result_df[
        [
            "sample_id",
            "gold_order",
            "baseline_prediction",
            "baseline_exact",
            "full_order_normalized_entropy",
            "previous_joint_with_full_prediction",
        ]
    ]
    .copy()
)

mr_eval_df["sample_id"] = (
    mr_eval_df["sample_id"]
    .astype(str)
)


mr_raw_lookup = {
    str(row["sample_id"]): row
    for _, row in mr_source_df.iterrows()
}


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def mr_safe_log(value):
    return float(
        np.log(
            max(float(value), MR_EPS)
        )
    )


def mr_normalize(values):
    cleaned = {}

    for key, value in values.items():
        try:
            value = float(value)
        except (TypeError, ValueError):
            value = 0.0

        if not np.isfinite(value):
            value = 0.0

        cleaned[key] = max(value, 0.0)

    if not cleaned:
        return {}

    total = float(sum(cleaned.values()))

    if total <= 0:
        uniform = 1.0 / len(cleaned)

        return {
            key: uniform
            for key in cleaned
        }

    return {
        key: value / total
        for key, value in cleaned.items()
    }


def mr_order_text(order):
    return " ".join(
        map(str, order)
    )


# ------------------------------------------------------------
# 2. Pairwise 방향 확률
#
# P(a before b), P(b before a)가 합 1이 아닐 수 있으므로
# 두 방향 안에서 다시 정규화
# ------------------------------------------------------------
def mr_pair_before_probability(
    row,
    earlier,
    later,
):
    forward = max(
        float(
            row[
                f"p_pair_{earlier}"
                f"_before_{later}"
            ]
        ),
        0.0,
    )

    backward = max(
        float(
            row[
                f"p_pair_{later}"
                f"_before_{earlier}"
            ]
        ),
        0.0,
    )

    total = forward + backward

    if total <= 0:
        return 0.5

    return float(forward / total)


# ------------------------------------------------------------
# 3. 여러 분포를 가중 평균
#
# distributions:
# {
#   "order": {frame: prob},
#   "pair":  {frame: prob},
#   "cond":  {frame: prob}
# }
#
# weights:
# {
#   "order": 0.2,
#   "pair":  0.6,
#   "cond":  0.2
# }
# ------------------------------------------------------------
def mr_mix_distributions(
    distributions,
    weights,
    remaining_frames,
):
    remaining_frames = tuple(
        remaining_frames
    )

    mixed = {
        frame: 0.0
        for frame in remaining_frames
    }

    total_weight = 0.0

    for source_name, source_weight in (
        weights.items()
    ):
        source_weight = float(
            source_weight
        )

        if source_weight <= 0:
            continue

        if source_name not in distributions:
            continue

        distribution = distributions[
            source_name
        ]

        if not distribution:
            continue

        restricted = {
            frame: distribution.get(
                frame,
                0.0,
            )
            for frame in remaining_frames
        }

        restricted = mr_normalize(
            restricted
        )

        for frame in remaining_frames:
            mixed[frame] += (
                source_weight
                * restricted[frame]
            )

        total_weight += source_weight

    if total_weight <= 0:
        uniform = 1.0 / len(
            remaining_frames
        )

        return {
            frame: uniform
            for frame in remaining_frames
        }

    mixed = {
        frame: value / total_weight
        for frame, value in mixed.items()
    }

    return mr_normalize(mixed)


# ------------------------------------------------------------
# 4. Full-order 확률 분포
# ------------------------------------------------------------
def mr_get_full_order_probs(row):
    values = {}

    for order in MR_ORDERS:
        key = "".join(
            map(str, order)
        )

        values[order] = row[
            f"p_order_{key}"
        ]

    return mr_normalize(values)


# ------------------------------------------------------------
# 5. Order 기반 forward next 분포
#
# prefix=()
# → 전체 중 첫 장 분포
#
# prefix=(2,)
# → 첫 장이 2라고 가정했을 때 두 번째 분포
#
# prefix=(2,3)
# → 앞이 2,3일 때 세 번째 분포
# ------------------------------------------------------------
def mr_order_forward_distribution(
    full_order_probs,
    prefix,
    remaining_frames,
):
    prefix = tuple(prefix)

    next_position = len(prefix)

    scores = {
        frame: 0.0
        for frame in remaining_frames
    }

    for order, probability in (
        full_order_probs.items()
    ):
        if order[:len(prefix)] != prefix:
            continue

        next_frame = order[
            next_position
        ]

        if next_frame in scores:
            scores[next_frame] += (
                probability
            )

    return mr_normalize(scores)


# ------------------------------------------------------------
# 6. Order 기반 backward previous 분포
#
# suffix=()
# → 전체 중 마지막 장 분포
#
# suffix=(4,)
# → 마지막이 4일 때 바로 앞 장 분포
#
# suffix=(1,4)
# → 끝이 1,4일 때 그 앞 장 분포
# ------------------------------------------------------------
def mr_order_backward_distribution(
    full_order_probs,
    suffix,
    remaining_frames,
):
    suffix = tuple(suffix)

    scores = {
        frame: 0.0
        for frame in remaining_frames
    }

    for order, probability in (
        full_order_probs.items()
    ):
        if len(suffix) > 0:
            if order[-len(suffix):] != suffix:
                continue

        previous_index = (
            len(order)
            - len(suffix)
            - 1
        )

        previous_frame = order[
            previous_index
        ]

        if previous_frame in scores:
            scores[previous_frame] += (
                probability
            )

    return mr_normalize(scores)


# ------------------------------------------------------------
# 7. Pairwise recursive forward 분포
#
# 현재 남은 후보 중 누가 모두보다 앞인가?
#
# raw(x)
# = product P(x before y)
# ------------------------------------------------------------
def mr_pair_forward_distribution(
    row,
    remaining_frames,
):
    remaining_frames = tuple(
        remaining_frames
    )

    scores = {}

    for candidate in remaining_frames:
        others = [
            frame
            for frame in remaining_frames
            if frame != candidate
        ]

        if not others:
            scores[candidate] = 1.0
            continue

        score = 1.0

        for other in others:
            score *= (
                mr_pair_before_probability(
                    row,
                    candidate,
                    other,
                )
            )

        scores[candidate] = score

    return mr_normalize(scores)


# ------------------------------------------------------------
# 8. Pairwise recursive backward 분포
#
# 현재 남은 후보 중 누가 모두보다 뒤인가?
#
# raw(x)
# = product P(y before x)
# ------------------------------------------------------------
def mr_pair_backward_distribution(
    row,
    remaining_frames,
):
    remaining_frames = tuple(
        remaining_frames
    )

    scores = {}

    for candidate in remaining_frames:
        others = [
            frame
            for frame in remaining_frames
            if frame != candidate
        ]

        if not others:
            scores[candidate] = 1.0
            continue

        score = 1.0

        for other in others:
            score *= (
                mr_pair_before_probability(
                    row,
                    other,
                    candidate,
                )
            )

        scores[candidate] = score

    return mr_normalize(scores)


# ------------------------------------------------------------
# 9. 저장된 pairwise-first / pairwise-last 분포
#
# 첫 단계와 마지막 단계에서는 기존에 계산한
# pairwise 요약 확률을 그대로 사용
# ------------------------------------------------------------
def mr_pairwise_first_distribution(row):
    return mr_normalize({
        frame: row[
            f"p_pairwise_first_{frame}"
        ]
        for frame in MR_FRAMES
    })


def mr_pairwise_last_distribution(row):
    return mr_normalize({
        frame: row[
            f"p_pairwise_last_{frame}"
        ]
        for frame in MR_FRAMES
    })


# ------------------------------------------------------------
# 10. Endpoint first / last 분포
# ------------------------------------------------------------
def mr_endpoint_first_distribution(row):
    return mr_normalize({
        frame: row[
            f"p_endpoint_first_{frame}"
        ]
        for frame in MR_FRAMES
    })


def mr_endpoint_last_distribution(row):
    return mr_normalize({
        frame: row[
            f"p_endpoint_last_{frame}"
        ]
        for frame in MR_FRAMES
    })


# ------------------------------------------------------------
# 11. fixed-first conditional 전체 분포
#
# first=f일 때 나머지 3장 순서 6개의 확률
# ------------------------------------------------------------
def mr_fixed_first_order_probs(
    row,
    fixed_first,
):
    remaining = [
        frame
        for frame in MR_FRAMES
        if frame != fixed_first
    ]

    values = {}

    for suffix in itertools.permutations(
        remaining
    ):
        suffix_key = "".join(
            map(str, suffix)
        )

        values[suffix] = row[
            f"p_cond_fixed_first_"
            f"{fixed_first}_suffix_"
            f"{suffix_key}"
        ]

    return mr_normalize(values)


# ------------------------------------------------------------
# 12. fixed-first 기반 next 분포
#
# first=2, prefix_after_first=()
# → P(second | first=2)
#
# first=2, prefix_after_first=(3,)
# → P(third | first=2, second=3)
# ------------------------------------------------------------
def mr_fixed_first_next_distribution(
    fixed_first_probs,
    prefix_after_first,
    remaining_frames,
):
    prefix_after_first = tuple(
        prefix_after_first
    )

    next_position = len(
        prefix_after_first
    )

    scores = {
        frame: 0.0
        for frame in remaining_frames
    }

    for suffix, probability in (
        fixed_first_probs.items()
    ):
        if (
            suffix[
                :len(prefix_after_first)
            ]
            != prefix_after_first
        ):
            continue

        next_frame = suffix[
            next_position
        ]

        if next_frame in scores:
            scores[next_frame] += (
                probability
            )

    return mr_normalize(scores)


# ------------------------------------------------------------
# 13. fixed-last conditional 전체 분포
#
# last=l일 때 앞의 3장 순서 6개의 확률
# ------------------------------------------------------------
def mr_fixed_last_order_probs(
    row,
    fixed_last,
):
    remaining = [
        frame
        for frame in MR_FRAMES
        if frame != fixed_last
    ]

    values = {}

    for prefix in itertools.permutations(
        remaining
    ):
        prefix_key = "".join(
            map(str, prefix)
        )

        values[prefix] = row[
            f"p_cond_fixed_last_"
            f"{fixed_last}_prefix_"
            f"{prefix_key}"
        ]

    return mr_normalize(values)


# ------------------------------------------------------------
# 14. fixed-last 기반 previous 분포
#
# last=4, tail_before_last=()
# → P(third | last=4)
#
# last=4, tail_before_last=(1,)
# → P(second | third=1, last=4)
# ------------------------------------------------------------
def mr_fixed_last_previous_distribution(
    fixed_last_probs,
    tail_before_last,
    remaining_frames,
):
    tail_before_last = tuple(
        tail_before_last
    )

    scores = {
        frame: 0.0
        for frame in remaining_frames
    }

    for prefix, probability in (
        fixed_last_probs.items()
    ):
        if len(tail_before_last) > 0:
            if (
                prefix[-len(tail_before_last):]
                != tail_before_last
            ):
                continue

        previous_index = (
            len(prefix)
            - len(tail_before_last)
            - 1
        )

        previous_frame = prefix[
            previous_index
        ]

        if previous_frame in scores:
            scores[previous_frame] += (
                probability
            )

    return mr_normalize(scores)


# ------------------------------------------------------------
# 15. 한 후보 순서의 실제 재귀 점수
# ------------------------------------------------------------
def mr_score_candidate(
    row,
    candidate_order,
    config,
    return_details=False,
):
    candidate_order = tuple(
        candidate_order
    )

    full_order_probs = (
        mr_get_full_order_probs(row)
    )

    pair_first_probs = (
        mr_pairwise_first_distribution(
            row
        )
    )

    pair_last_probs = (
        mr_pairwise_last_distribution(
            row
        )
    )

    endpoint_first_probs = (
        mr_endpoint_first_distribution(
            row
        )
    )

    endpoint_last_probs = (
        mr_endpoint_last_distribution(
            row
        )
    )

    # ========================================================
    # Forward recursion
    # ========================================================
    forward_factors = []
    forward_details = []

    first_frame = candidate_order[0]

    # Step F1:
    # order-first + pairwise-first + endpoint-first
    order_first_probs = (
        mr_order_forward_distribution(
            full_order_probs,
            prefix=(),
            remaining_frames=MR_FRAMES,
        )
    )

    first_distribution = (
        mr_mix_distributions(
            distributions={
                "order": order_first_probs,
                "pair": pair_first_probs,
                "endpoint": (
                    endpoint_first_probs
                ),
            },
            weights=config[
                "first_anchor_weights"
            ],
            remaining_frames=MR_FRAMES,
        )
    )

    first_probability = float(
        first_distribution[first_frame]
    )

    forward_factors.append(
        first_probability
    )

    forward_details.append({
        "step": "F1_first",
        "chosen": first_frame,
        "probability": first_probability,
        "order": order_first_probs[
            first_frame
        ],
        "pair": pair_first_probs[
            first_frame
        ],
        "endpoint": (
            endpoint_first_probs[
                first_frame
            ]
        ),
    })

    fixed_first_probs = (
        mr_fixed_first_order_probs(
            row,
            fixed_first=first_frame,
        )
    )

    # Step F2, F3
    for position in [1, 2]:
        prefix = candidate_order[
            :position
        ]

        chosen_frame = candidate_order[
            position
        ]

        remaining_frames = [
            frame
            for frame in MR_FRAMES
            if frame not in prefix
        ]

        order_next_probs = (
            mr_order_forward_distribution(
                full_order_probs,
                prefix=prefix,
                remaining_frames=(
                    remaining_frames
                ),
            )
        )

        pair_next_probs = (
            mr_pair_forward_distribution(
                row,
                remaining_frames,
            )
        )

        prefix_after_first = (
            candidate_order[
                1:position
            ]
        )

        conditional_next_probs = (
            mr_fixed_first_next_distribution(
                fixed_first_probs,
                prefix_after_first=(
                    prefix_after_first
                ),
                remaining_frames=(
                    remaining_frames
                ),
            )
        )

        mixed_next_distribution = (
            mr_mix_distributions(
                distributions={
                    "order": (
                        order_next_probs
                    ),
                    "pair": (
                        pair_next_probs
                    ),
                    "conditional": (
                        conditional_next_probs
                    ),
                },
                weights=config[
                    "middle_weights"
                ],
                remaining_frames=(
                    remaining_frames
                ),
            )
        )

        chosen_probability = float(
            mixed_next_distribution[
                chosen_frame
            ]
        )

        forward_factors.append(
            chosen_probability
        )

        forward_details.append({
            "step": f"F{position + 1}",
            "chosen": chosen_frame,
            "probability": (
                chosen_probability
            ),
            "order": order_next_probs[
                chosen_frame
            ],
            "pair": pair_next_probs[
                chosen_frame
            ],
            "conditional": (
                conditional_next_probs[
                    chosen_frame
                ]
            ),
        })

    # ========================================================
    # Backward recursion
    # ========================================================
    backward_factors = []
    backward_details = []

    last_frame = candidate_order[-1]

    # Step B1:
    # order-last + pairwise-last + endpoint-last
    order_last_probs = (
        mr_order_backward_distribution(
            full_order_probs,
            suffix=(),
            remaining_frames=MR_FRAMES,
        )
    )

    last_distribution = (
        mr_mix_distributions(
            distributions={
                "order": order_last_probs,
                "pair": pair_last_probs,
                "endpoint": (
                    endpoint_last_probs
                ),
            },
            weights=config[
                "last_anchor_weights"
            ],
            remaining_frames=MR_FRAMES,
        )
    )

    last_probability = float(
        last_distribution[last_frame]
    )

    backward_factors.append(
        last_probability
    )

    backward_details.append({
        "step": "B1_last",
        "chosen": last_frame,
        "probability": last_probability,
        "order": order_last_probs[
            last_frame
        ],
        "pair": pair_last_probs[
            last_frame
        ],
        "endpoint": endpoint_last_probs[
            last_frame
        ],
    })

    fixed_last_probs = (
        mr_fixed_last_order_probs(
            row,
            fixed_last=last_frame,
        )
    )

    # Step B2, B3
    for position in [2, 1]:
        suffix = candidate_order[
            position + 1:
        ]

        chosen_frame = candidate_order[
            position
        ]

        remaining_frames = [
            frame
            for frame in MR_FRAMES
            if frame not in suffix
        ]

        order_previous_probs = (
            mr_order_backward_distribution(
                full_order_probs,
                suffix=suffix,
                remaining_frames=(
                    remaining_frames
                ),
            )
        )

        pair_previous_probs = (
            mr_pair_backward_distribution(
                row,
                remaining_frames,
            )
        )

        tail_before_last = (
            candidate_order[
                position + 1:-1
            ]
        )

        conditional_previous_probs = (
            mr_fixed_last_previous_distribution(
                fixed_last_probs,
                tail_before_last=(
                    tail_before_last
                ),
                remaining_frames=(
                    remaining_frames
                ),
            )
        )

        mixed_previous_distribution = (
            mr_mix_distributions(
                distributions={
                    "order": (
                        order_previous_probs
                    ),
                    "pair": (
                        pair_previous_probs
                    ),
                    "conditional": (
                        conditional_previous_probs
                    ),
                },
                weights=config[
                    "middle_weights"
                ],
                remaining_frames=(
                    remaining_frames
                ),
            )
        )

        chosen_probability = float(
            mixed_previous_distribution[
                chosen_frame
            ]
        )

        backward_factors.append(
            chosen_probability
        )

        backward_details.append({
            "step": (
                f"B{4 - position}"
            ),
            "chosen": chosen_frame,
            "probability": (
                chosen_probability
            ),
            "order": (
                order_previous_probs[
                    chosen_frame
                ]
            ),
            "pair": (
                pair_previous_probs[
                    chosen_frame
                ]
            ),
            "conditional": (
                conditional_previous_probs[
                    chosen_frame
                ]
            ),
        })

    # 각 방향 모두 3개의 선택 확률
    #
    # 평균 log를 사용해 geometric mean 점수로 변환.
    # 실제 확률 곱과 후보 랭킹은 동일한 방향.
    forward_log = float(
        np.mean([
            mr_safe_log(probability)
            for probability
            in forward_factors
        ])
    )

    backward_log = float(
        np.mean([
            mr_safe_log(probability)
            for probability
            in backward_factors
        ])
    )

    direction_first_weight = float(
        config["direction_first_weight"]
    )

    direction_last_weight = float(
        config["direction_last_weight"]
    )

    direction_total = (
        direction_first_weight
        + direction_last_weight
    )

    direction_first_weight /= (
        direction_total
    )

    direction_last_weight /= (
        direction_total
    )

    total_score = float(
        direction_first_weight
        * forward_log
        +
        direction_last_weight
        * backward_log
    )

    result = {
        "score": total_score,
        "forward_log": forward_log,
        "backward_log": backward_log,
        "forward_product": float(
            np.prod(forward_factors)
        ),
        "backward_product": float(
            np.prod(backward_factors)
        ),
    }

    if return_details:
        result[
            "forward_factors"
        ] = forward_factors

        result[
            "backward_factors"
        ] = backward_factors

        result[
            "forward_details"
        ] = forward_details

        result[
            "backward_details"
        ] = backward_details

    return result


# ------------------------------------------------------------
# 16. 24개 공동 디코딩
# ------------------------------------------------------------
def mr_decode_row(
    row,
    config,
    return_candidates=False,
):
    candidate_rows = []

    for candidate_order in MR_ORDERS:
        scored = mr_score_candidate(
            row=row,
            candidate_order=(
                candidate_order
            ),
            config=config,
            return_details=(
                return_candidates
            ),
        )

        candidate_rows.append({
            "candidate_order": (
                candidate_order
            ),
            "prediction": mr_order_text(
                candidate_order
            ),
            **scored,
        })

    # 중간 확정 없음.
    # 24개 모두 계산한 뒤 마지막 랭킹.
    candidate_rows.sort(
        key=lambda item: item["score"],
        reverse=True,
    )

    top1 = candidate_rows[0]
    top2 = candidate_rows[1]

    output = {
        "prediction": top1[
            "prediction"
        ],
        "top2_prediction": top2[
            "prediction"
        ],
        "margin": float(
            top1["score"]
            - top2["score"]
        ),
        "forward_product": top1[
            "forward_product"
        ],
        "backward_product": top1[
            "backward_product"
        ],
    }

    if return_candidates:
        output["candidates"] = (
            candidate_rows
        )

    return output


# ------------------------------------------------------------
# 17. 실험 설정
#
# Anchor:
# A) 기존 10:60:30
# B) 비대칭 sweep 최고
# C) 부드러운 혼합
#
# Middle:
# 1) pairwise 60%
# 2) 균형
# 3) conditional 강화
#
# 최종 full-order를 별도로 또 더하지 않음.
# 이미 각 단계의 order conditional로 사용되기 때문.
# ------------------------------------------------------------
MR_ANCHOR_PRESETS = {
    "baseline_anchor": {
        "first_anchor_weights": {
            "order": 0.10,
            "pair": 0.60,
            "endpoint": 0.30,
        },
        "last_anchor_weights": {
            "order": 0.10,
            "pair": 0.60,
            "endpoint": 0.30,
        },
    },

    "asym_best_anchor": {
        "first_anchor_weights": {
            "order": 0.00,
            "pair": 1.00,
            "endpoint": 0.00,
        },
        "last_anchor_weights": {
            "order": 0.00,
            "pair": 0.45,
            "endpoint": 0.55,
        },
    },

    "soft_asym_anchor": {
        "first_anchor_weights": {
            "order": 0.05,
            "pair": 0.80,
            "endpoint": 0.15,
        },
        "last_anchor_weights": {
            "order": 0.05,
            "pair": 0.45,
            "endpoint": 0.50,
        },
    },
}


MR_MIDDLE_PRESETS = {
    "middle_pair60": {
        "order": 0.20,
        "pair": 0.60,
        "conditional": 0.20,
    },

    "middle_balanced": {
        "order": 0.25,
        "pair": 0.40,
        "conditional": 0.35,
    },

    "middle_cond50": {
        "order": 0.20,
        "pair": 0.30,
        "conditional": 0.50,
    },

    "middle_order40_pair60": {
        "order": 0.40,
        "pair": 0.60,
        "conditional": 0.00,
    },
}


MR_DIRECTION_PRESETS = {
    "dir_55_45": {
        "direction_first_weight": 0.55,
        "direction_last_weight": 0.45,
    },

    "dir_equal": {
        "direction_first_weight": 0.50,
        "direction_last_weight": 0.50,
    },
}


MR_CONFIGS = {}

for anchor_name, anchor_config in (
    MR_ANCHOR_PRESETS.items()
):
    for middle_name, middle_weights in (
        MR_MIDDLE_PRESETS.items()
    ):
        for direction_name, direction_config in (
            MR_DIRECTION_PRESETS.items()
        ):
            config_name = (
                f"{anchor_name}"
                f"__{middle_name}"
                f"__{direction_name}"
            )

            MR_CONFIGS[config_name] = {
                "first_anchor_weights": (
                    anchor_config[
                        "first_anchor_weights"
                    ]
                ),
                "last_anchor_weights": (
                    anchor_config[
                        "last_anchor_weights"
                    ]
                ),
                "middle_weights": (
                    middle_weights
                ),
                "direction_first_weight": (
                    direction_config[
                        "direction_first_weight"
                    ]
                ),
                "direction_last_weight": (
                    direction_config[
                        "direction_last_weight"
                    ]
                ),
            }


print(
    "실험 config 수:",
    len(MR_CONFIGS),
)


# ------------------------------------------------------------
# 18. 평가 함수
# ------------------------------------------------------------
def mr_evaluate(
    frame,
    predictions,
    strategy_name,
):
    predictions = pd.Series(
        predictions,
        index=frame.index,
    )

    exact = (
        predictions
        == frame["gold_order"]
    ).astype(int)

    changed = (
        predictions
        != frame[
            "baseline_prediction"
        ]
    )

    wrong_to_right = (
        changed
        &
        (frame["baseline_exact"] == 0)
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (frame["baseline_exact"] == 1)
        &
        (exact == 0)
    )

    return {
        "strategy": strategy_name,
        "exact": float(
            exact.mean()
        ),
        "correct_count": int(
            exact.sum()
        ),
        "changed_count": int(
            changed.sum()
        ),
        "wrong_to_right": int(
            wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            right_to_wrong.sum()
        ),
        "net_change": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),
        "canonical_1234_count": int(
            (
                predictions == "1 2 3 4"
            ).sum()
        ),
    }


# ------------------------------------------------------------
# 19. 전체 config 실행
# ------------------------------------------------------------
mr_prediction_cache = {}
mr_summary_rows = []


for config_index, (
    config_name,
    config,
) in enumerate(MR_CONFIGS.items(), start=1):

    print(
        f"[{config_index}/{len(MR_CONFIGS)}] "
        f"{config_name}"
    )

    predictions = []

    for _, eval_row in mr_eval_df.iterrows():
        sample_id = str(
            eval_row["sample_id"]
        )

        raw_row = mr_raw_lookup[
            sample_id
        ]

        decoded = mr_decode_row(
            row=raw_row,
            config=config,
            return_candidates=False,
        )

        predictions.append(
            decoded["prediction"]
        )

    predictions = pd.Series(
        predictions,
        index=mr_eval_df.index,
    )

    mr_prediction_cache[
        config_name
    ] = predictions

    # Global
    global_result = mr_evaluate(
        mr_eval_df,
        predictions,
        config_name,
    )

    # Entropy 0.945 hybrid
    use_recursive = (
        mr_eval_df[
            "full_order_normalized_entropy"
        ]
        >= MR_ENTROPY_THRESHOLD
    )

    hybrid_predictions = np.where(
        use_recursive,
        predictions,
        mr_eval_df[
            "baseline_prediction"
        ],
    )

    hybrid_result = mr_evaluate(
        mr_eval_df,
        hybrid_predictions,
        f"{config_name}_hybrid",
    )

    mr_summary_rows.append({
        "strategy": config_name,

        "global_exact": global_result[
            "exact"
        ],
        "global_correct": global_result[
            "correct_count"
        ],
        "global_wrong_to_right": (
            global_result[
                "wrong_to_right"
            ]
        ),
        "global_right_to_wrong": (
            global_result[
                "right_to_wrong"
            ]
        ),
        "global_net": global_result[
            "net_change"
        ],

        "hybrid_exact": hybrid_result[
            "exact"
        ],
        "hybrid_correct": hybrid_result[
            "correct_count"
        ],
        "hybrid_wrong_to_right": (
            hybrid_result[
                "wrong_to_right"
            ]
        ),
        "hybrid_right_to_wrong": (
            hybrid_result[
                "right_to_wrong"
            ]
        ),
        "hybrid_net": hybrid_result[
            "net_change"
        ],

        "canonical_1234_count": (
            global_result[
                "canonical_1234_count"
            ]
        ),
    })


mr_summary_df = (
    pd.DataFrame(mr_summary_rows)
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nTrue mixed recursive joint 상위 결과"
)

display(
    mr_summary_df.head(30)
)


# ------------------------------------------------------------
# 20. 기존 방식 비교
# ------------------------------------------------------------
previous_global = mr_evaluate(
    mr_eval_df,
    mr_eval_df[
        "previous_joint_with_full_prediction"
    ],
    "previous_joint_global",
)

previous_hybrid_predictions = np.where(
    mr_eval_df[
        "full_order_normalized_entropy"
    ]
    >= MR_ENTROPY_THRESHOLD,
    mr_eval_df[
        "previous_joint_with_full_prediction"
    ],
    mr_eval_df[
        "baseline_prediction"
    ],
)

previous_hybrid = mr_evaluate(
    mr_eval_df,
    previous_hybrid_predictions,
    "previous_joint_hybrid",
)


print("\n기존 joint 비교")
print(
    "previous global:",
    previous_global["exact"],
)
print(
    "previous hybrid:",
    previous_hybrid["exact"],
)


# ------------------------------------------------------------
# 21. 최고 config 결과 저장
# ------------------------------------------------------------
mr_best_row = mr_summary_df.iloc[0]

mr_best_config_name = str(
    mr_best_row["strategy"]
)

mr_best_config = MR_CONFIGS[
    mr_best_config_name
]

mr_best_predictions = (
    mr_prediction_cache[
        mr_best_config_name
    ]
)

mr_best_hybrid_predictions = np.where(
    mr_eval_df[
        "full_order_normalized_entropy"
    ]
    >= MR_ENTROPY_THRESHOLD,
    mr_best_predictions,
    mr_eval_df[
        "baseline_prediction"
    ],
)


mr_best_result_df = (
    mr_eval_df.copy()
)

mr_best_result_df[
    "recursive_prediction"
] = mr_best_predictions

mr_best_result_df[
    "recursive_hybrid_prediction"
] = mr_best_hybrid_predictions

mr_best_result_df[
    "recursive_exact"
] = (
    mr_best_result_df[
        "recursive_prediction"
    ]
    ==
    mr_best_result_df[
        "gold_order"
    ]
).astype(int)

mr_best_result_df[
    "recursive_hybrid_exact"
] = (
    mr_best_result_df[
        "recursive_hybrid_prediction"
    ]
    ==
    mr_best_result_df[
        "gold_order"
    ]
).astype(int)


print(
    "\n최고 config:",
    mr_best_config_name,
)

print(
    "최고 global exact:",
    mr_best_row["global_exact"],
)

print(
    "최고 hybrid exact:",
    mr_best_row["hybrid_exact"],
)

print(
    "canonical 1234:",
    mr_best_row[
        "canonical_1234_count"
    ],
)


display(
    mr_best_result_df[
        mr_best_result_df[
            "recursive_hybrid_prediction"
        ]
        !=
        mr_best_result_df[
            "baseline_prediction"
        ]
    ].head(100)
)


# ------------------------------------------------------------
# 22. 실제로 단계별 혼합이 됐는지 확인
#
# 첫 번째 샘플에서 최고 config의 top-5 후보와
# 각 forward/backward 단계 확률을 출력
# ------------------------------------------------------------
debug_sample_id = str(
    mr_eval_df.iloc[0]["sample_id"]
)

debug_raw_row = mr_raw_lookup[
    debug_sample_id
]

debug_decoded = mr_decode_row(
    row=debug_raw_row,
    config=mr_best_config,
    return_candidates=True,
)


debug_rows = []

for candidate in (
    debug_decoded["candidates"][:5]
):
    debug_rows.append({
        "prediction": candidate[
            "prediction"
        ],
        "score": candidate["score"],
        "forward_product": candidate[
            "forward_product"
        ],
        "backward_product": candidate[
            "backward_product"
        ],
        "forward_factors": candidate[
            "forward_factors"
        ],
        "backward_factors": candidate[
            "backward_factors"
        ],
    })


print(
    "\n단계별 재귀 동작 확인"
)

print(
    "debug sample_id:",
    debug_sample_id,
)

display(
    pd.DataFrame(debug_rows)
)


print("\nTop-1 forward 단계 상세")

display(
    pd.DataFrame(
        debug_decoded[
            "candidates"
        ][0]["forward_details"]
    )
)


print("\nTop-1 backward 단계 상세")

display(
    pd.DataFrame(
        debug_decoded[
            "candidates"
        ][0]["backward_details"]
    )
)

실험 config 수: 24
[1/24] baseline_anchor__middle_pair60__dir_55_45
[2/24] baseline_anchor__middle_pair60__dir_equal
[3/24] baseline_anchor__middle_balanced__dir_55_45
[4/24] baseline_anchor__middle_balanced__dir_equal
[5/24] baseline_anchor__middle_cond50__dir_55_45
[6/24] baseline_anchor__middle_cond50__dir_equal
[7/24] baseline_anchor__middle_order40_pair60__dir_55_45
[8/24] baseline_anchor__middle_order40_pair60__dir_equal
[9/24] asym_best_anchor__middle_pair60__dir_55_45
[10/24] asym_best_anchor__middle_pair60__dir_equal
[11/24] asym_best_anchor__middle_balanced__dir_55_45
[12/24] asym_best_anchor__middle_balanced__dir_equal
[13/24] asym_best_anchor__middle_cond50__dir_55_45
[14/24] asym_best_anchor__middle_cond50__dir_equal
[15/24] asym_best_anchor__middle_order40_pair60__dir_55_45
[16/24] asym_best_anchor__middle_order40_pair60__dir_equal
[17/24] soft_asym_anchor__middle_pair60__dir_55_45
[18/24] soft_asym_anchor__middle_pair60__dir_equal
[19/24] soft_asym_anchor__middle_balanced__

,strategy,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count
0,asym_best_anchor__middle_cond50__dir_equal,0.395,79,17,12,5,0.410,82,12,4,8,31
1,asym_best_anchor__middle_cond50__dir_55_45,0.385,77,16,13,3,0.405,81,11,4,7,31
2,asym_best_anchor__middle_balanced__dir_equal,0.380,76,16,14,2,0.405,81,11,4,7,28
3,soft_asym_anchor__middle_cond50__dir_equal,0.415,83,14,5,9,0.400,80,10,4,6,35
4,asym_best_anchor__middle_balanced__dir_55_45,0.375,75,15,14,1,0.400,80,10,4,6,28
5,soft_asym_anchor__middle_cond50__dir_55_45,0.410,82,13,5,8,0.395,79,9,4,5,34
6,baseline_anchor__middle_cond50__dir_55_45,0.405,81,11,4,7,0.395,79,9,4,5,34
7,soft_asym_anchor__middle_balanced__dir_equal,0.400,80,14,8,6,0.395,79,9,4,5,30
8,baseline_anchor__middle_order40_pair60__dir_55_45,0.385,77,14,11,3,0.395,79,9,4,5,24
9,asym_best_anchor__middle_pair60__dir_equal,0.365,73,16,17,-1,0.395,79,11,6,5,25



기존 joint 비교
previous global: 0.435
previous hybrid: 0.445

최고 config: asym_best_anchor__middle_cond50__dir_equal
최고 global exact: 0.395
최고 hybrid exact: 0.41
canonical 1234: 31


,sample_id,gold_order,baseline_prediction,baseline_exact,full_order_normalized_entropy,previous_joint_with_full_prediction,recursive_prediction,recursive_hybrid_prediction,recursive_exact,recursive_hybrid_exact
8,tUsgQQ,1 4 2 3,4 1 2 3,0,0.952219,4 1 2 3,1 4 3 2,1 4 3 2,0,0
13,vnbiS3,3 4 2 1,1 3 2 4,0,0.995453,1 2 3 4,3 1 2 4,3 1 2 4,0,0
23,o9v7Ns,1 2 3 4,1 2 4 3,0,0.968957,1 2 3 4,1 2 3 4,1 2 3 4,1,1
28,BnOxub,1 4 3 2,1 4 2 3,0,0.985859,1 2 4 3,4 1 2 3,4 1 2 3,0,0
29,xEKSQr,1 3 4 2,1 3 4 2,1,0.947270,4 3 1 2,4 3 1 2,4 3 1 2,0,0
41,hmQV7h,1 4 2 3,4 1 2 3,0,0.961252,4 1 2 3,4 3 2 1,4 3 2 1,0,0
50,zZ6ACe,2 4 3 1,4 2 3 1,0,0.946662,2 4 3 1,2 4 3 1,2 4 3 1,1,1
56,ndnxpJ,2 1 3 4,4 2 3 1,0,0.994273,2 4 3 1,2 4 3 1,2 4 3 1,0,0
57,uwh2UV,1 2 3 4,4 1 3 2,0,0.983804,1 2 3 4,4 1 2 3,4 1 2 3,0,0
58,EhxCB8,1 2 4 3,4 2 1 3,0,0.976512,2 4 1 3,2 4 1 3,2 4 1 3,0,0



단계별 재귀 동작 확인
debug sample_id: x6AwMw


,prediction,score,forward_product,backward_product,forward_factors,backward_factors
0,2 3 4 1,-0.495322,0.250139,0.204704,"[0.49438451069952105, 0.822920551073239, 0.614...","[0.4259823483055841, 0.7449156822904648, 0.645..."
1,2 3 1 4,-0.544468,0.156700,0.243317,"[0.49438451069952105, 0.822920551073239, 0.385...","[0.4986824474602891, 0.7503384393359516, 0.650..."
2,3 2 4 1,-0.669035,0.160342,0.112617,"[0.3389117569334812, 0.8106374471940203, 0.583...","[0.4259823483055841, 0.7449156822904648, 0.354..."
3,3 2 1 4,-0.700287,0.114393,0.130863,"[0.3389117569334812, 0.8106374471940203, 0.416...","[0.4986824474602891, 0.7503384393359516, 0.349..."
4,4 2 3 1,-1.104473,0.063969,0.020703,"[0.13408609696882942, 0.5935654328624145, 0.80...","[0.4259823483055841, 0.11960068629111015, 0.40..."



Top-1 forward 단계 상세


,step,chosen,probability,order,pair,endpoint,conditional
0,F1_first,2,0.494385,0.330602,0.494385,0.499723,NaN
1,F2,3,0.822921,0.682305,0.999939,NaN,0.772956
2,F3,4,0.614835,0.535653,0.804406,NaN,0.532765



Top-1 backward 단계 상세


,step,chosen,probability,order,pair,endpoint,conditional
0,B1_last,1,0.425982,0.389098,0.467382,0.39211,NaN
1,B2,4,0.744916,0.608660,0.999939,NaN,0.646404
2,B3,3,0.645100,0.510193,0.966410,NaN,0.506277


In [42]:
import itertools
from collections import Counter

import numpy as np
import pandas as pd

# ============================================================
# 3가지 앙상블 실험을 한 번에 실행
#
# 1) asym joint Top-K 후보만 recursive로 재랭킹
# 2) previous/asym/recursive/full24 후보 순위 융합
# 3) 디코더 합의가 있을 때만 예측 변경
#
# 추가 모델 추론 없음
# ============================================================

E_FRAMES = tuple(
    globals().get(
        "FE_FRAMES",
        (1, 2, 3, 4),
    )
)

E_ORDERS = list(
    globals().get(
        "FE_ORDERS",
        itertools.permutations(E_FRAMES),
    )
)

E_EPS = 1e-12
E_ENTROPY_THRESHOLD = 0.945

E_RECURSIVE_CONFIG_NAME = (
    "asym_best_anchor"
    "__middle_cond50"
    "__dir_equal"
)


# ------------------------------------------------------------
# 0. 데이터 및 기존 함수 확인
# ------------------------------------------------------------
if "fe_source_df" in globals():
    e_source_df = fe_source_df.copy()

elif "raw_df_200" in globals():
    e_source_df = raw_df_200.copy()

else:
    raise RuntimeError(
        "fe_source_df 또는 raw_df_200이 없습니다."
    )


if "fe_result_df" not in globals():
    raise RuntimeError(
        "fe_result_df가 없습니다."
    )


if (
    "mr_score_candidate" not in globals()
    or "MR_CONFIGS" not in globals()
):
    raise RuntimeError(
        "True Mixed Recursive Joint 셀을 "
        "먼저 실행하세요."
    )


if E_RECURSIVE_CONFIG_NAME not in MR_CONFIGS:
    raise RuntimeError(
        "MR_CONFIGS에 다음 config가 없습니다: "
        f"{E_RECURSIVE_CONFIG_NAME}"
    )


e_recursive_config = MR_CONFIGS[
    E_RECURSIVE_CONFIG_NAME
]


e_source_df["sample_id"] = (
    e_source_df["sample_id"]
    .astype(str)
)


e_eval_df = fe_result_df[
    [
        "sample_id",
        "gold_order",
        "baseline_prediction",
        "baseline_exact",
        "full_order_normalized_entropy",
        "previous_joint_with_full_prediction",
    ]
].copy()


e_eval_df["sample_id"] = (
    e_eval_df["sample_id"]
    .astype(str)
)


e_raw_lookup = {
    str(row["sample_id"]): row
    for _, row in e_source_df.iterrows()
}


# ------------------------------------------------------------
# 1. 공통 함수
# ------------------------------------------------------------
def e_order_text(order):
    return " ".join(
        map(str, order)
    )


def e_safe_log(value):
    return float(
        np.log(
            max(
                float(value),
                E_EPS,
            )
        )
    )


def e_normalize(values):
    cleaned = {}

    for key, value in values.items():
        try:
            value = float(value)

        except (TypeError, ValueError):
            value = 0.0

        if not np.isfinite(value):
            value = 0.0

        cleaned[key] = max(
            value,
            0.0,
        )

    total = float(
        sum(cleaned.values())
    )

    if total <= 0:
        uniform = (
            1.0 / len(cleaned)
        )

        return {
            key: uniform
            for key in cleaned
        }

    return {
        key: value / total
        for key, value
        in cleaned.items()
    }


def e_zscore(values):
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    std = float(
        values.std()
    )

    if std <= E_EPS:
        return np.zeros_like(
            values
        )

    return (
        values - values.mean()
    ) / std


def e_borda(values):
    """
    후보 점수의 순위만 이용.

    최고 점수 후보 = 1.0
    최저 점수 후보 = 0.0
    """
    values = np.asarray(
        values,
        dtype=np.float64,
    )

    ranks = (
        pd.Series(values)
        .rank(
            method="average",
            ascending=False,
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    if len(values) <= 1:
        return np.ones_like(
            values
        )

    return (
        len(values) - ranks
    ) / (
        len(values) - 1
    )


def e_argmax_prediction(
    table,
    score_column,
):
    scores = table[
        score_column
    ].to_numpy(
        dtype=np.float64
    )

    best_position = int(
        np.argmax(scores)
    )

    return table.iloc[
        best_position
    ]["prediction"]


# ------------------------------------------------------------
# 2. Anchor 확률 분포
# ------------------------------------------------------------
def e_anchor_sources(row):
    return {
        "pair_first": e_normalize({
            frame: row[
                f"p_pairwise_first_{frame}"
            ]
            for frame in E_FRAMES
        }),

        "endpoint_first": e_normalize({
            frame: row[
                f"p_endpoint_first_{frame}"
            ]
            for frame in E_FRAMES
        }),

        "pair_last": e_normalize({
            frame: row[
                f"p_pairwise_last_{frame}"
            ]
            for frame in E_FRAMES
        }),

        "endpoint_last": e_normalize({
            frame: row[
                f"p_endpoint_last_{frame}"
            ]
            for frame in E_FRAMES
        }),
    }


# ------------------------------------------------------------
# 3. Previous joint 후보 점수
#
# first:
# pairwise_first
#
# last:
# endpoint_last
# ------------------------------------------------------------
def e_previous_score(
    row,
    order,
    anchors,
):
    first, second, third, last = (
        order
    )

    suffix = (
        f"{second}{third}{last}"
    )

    prefix = (
        f"{first}{second}{third}"
    )

    order_key = "".join(
        map(str, order)
    )

    first_log = (
        e_safe_log(
            anchors[
                "pair_first"
            ][first]
        )
        +
        e_safe_log(
            row[
                f"p_cond_fixed_first_"
                f"{first}_suffix_{suffix}"
            ]
        )
    )

    last_log = (
        e_safe_log(
            anchors[
                "endpoint_last"
            ][last]
        )
        +
        e_safe_log(
            row[
                f"p_cond_fixed_last_"
                f"{last}_prefix_{prefix}"
            ]
        )
    )

    return float(
        0.495 * first_log
        +
        0.405 * last_log
        +
        0.10
        * e_safe_log(
            row[
                f"p_order_{order_key}"
            ]
        )
    )


# ------------------------------------------------------------
# 4. 비대칭 anchor 최고 후보 점수
#
# first:
# pairwise_first 100%
#
# last:
# pairwise_last 45%
# + endpoint_last 55%
# ------------------------------------------------------------
def e_asym_score(
    row,
    order,
    anchors,
):
    first, second, third, last = (
        order
    )

    suffix = (
        f"{second}{third}{last}"
    )

    prefix = (
        f"{first}{second}{third}"
    )

    order_key = "".join(
        map(str, order)
    )

    mixed_last = e_normalize({
        frame: (
            0.45
            * anchors[
                "pair_last"
            ][frame]
            +
            0.55
            * anchors[
                "endpoint_last"
            ][frame]
        )
        for frame in E_FRAMES
    })

    first_log = (
        e_safe_log(
            anchors[
                "pair_first"
            ][first]
        )
        +
        e_safe_log(
            row[
                f"p_cond_fixed_first_"
                f"{first}_suffix_{suffix}"
            ]
        )
    )

    last_log = (
        e_safe_log(
            mixed_last[last]
        )
        +
        e_safe_log(
            row[
                f"p_cond_fixed_last_"
                f"{last}_prefix_{prefix}"
            ]
        )
    )

    return float(
        0.495 * first_log
        +
        0.405 * last_log
        +
        0.10
        * e_safe_log(
            row[
                f"p_order_{order_key}"
            ]
        )
    )


# ------------------------------------------------------------
# 5. True mixed recursive 후보 점수
# ------------------------------------------------------------
def e_recursive_score(
    row,
    order,
):
    result = mr_score_candidate(
        row=row,
        candidate_order=order,
        config=e_recursive_config,
        return_details=False,
    )

    return float(
        result["score"]
    )


# ------------------------------------------------------------
# 6. Full24 후보 점수
# ------------------------------------------------------------
def e_full24_score(
    row,
    order,
):
    order_key = "".join(
        map(str, order)
    )

    return e_safe_log(
        row[
            f"p_order_{order_key}"
        ]
    )


# ------------------------------------------------------------
# 7. 각 샘플의 24개 후보 점수를 한 번만 계산
# ------------------------------------------------------------
e_candidate_tables = {}

e_asym_predictions = []
e_previous_predictions = []
e_recursive_predictions = []
e_full24_predictions = []

e_asym_margins = []


for _, eval_row in e_eval_df.iterrows():
    sample_id = str(
        eval_row["sample_id"]
    )

    raw_row = e_raw_lookup[
        sample_id
    ]

    anchors = e_anchor_sources(
        raw_row
    )

    candidate_rows = []

    for order in E_ORDERS:
        candidate_rows.append({
            "candidate_order": order,

            "prediction": (
                e_order_text(order)
            ),

            "previous_score": (
                e_previous_score(
                    raw_row,
                    order,
                    anchors,
                )
            ),

            "asym_score": (
                e_asym_score(
                    raw_row,
                    order,
                    anchors,
                )
            ),

            "recursive_score": (
                e_recursive_score(
                    raw_row,
                    order,
                )
            ),

            "full24_score": (
                e_full24_score(
                    raw_row,
                    order,
                )
            ),
        })

    table = pd.DataFrame(
        candidate_rows
    )

    for source in [
        "previous",
        "asym",
        "recursive",
        "full24",
    ]:
        table[
            f"{source}_z"
        ] = e_zscore(
            table[
                f"{source}_score"
            ]
        )

        table[
            f"{source}_borda"
        ] = e_borda(
            table[
                f"{source}_score"
            ]
        )

    # asym joint 기준으로 정렬
    table = (
        table
        .sort_values(
            "asym_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    e_candidate_tables[
        sample_id
    ] = table

    e_asym_predictions.append(
        table.iloc[0][
            "prediction"
        ]
    )

    e_previous_predictions.append(
        e_argmax_prediction(
            table,
            "previous_score",
        )
    )

    e_recursive_predictions.append(
        e_argmax_prediction(
            table,
            "recursive_score",
        )
    )

    e_full24_predictions.append(
        e_argmax_prediction(
            table,
            "full24_score",
        )
    )

    e_asym_margins.append(
        float(
            table.iloc[0][
                "asym_score"
            ]
            -
            table.iloc[1][
                "asym_score"
            ]
        )
    )


e_asym_predictions = pd.Series(
    e_asym_predictions,
    index=e_eval_df.index,
)

e_previous_predictions = pd.Series(
    e_previous_predictions,
    index=e_eval_df.index,
)

e_recursive_predictions = pd.Series(
    e_recursive_predictions,
    index=e_eval_df.index,
)

e_full24_predictions = pd.Series(
    e_full24_predictions,
    index=e_eval_df.index,
)

e_asym_margins = np.asarray(
    e_asym_margins,
    dtype=np.float64,
)


# ------------------------------------------------------------
# 8. 평가 함수
# ------------------------------------------------------------
def e_evaluate(
    predictions,
    name,
):
    predictions = pd.Series(
        predictions,
        index=e_eval_df.index,
    )

    exact = (
        predictions
        ==
        e_eval_df[
            "gold_order"
        ]
    ).astype(int)

    changed = (
        predictions
        !=
        e_eval_df[
            "baseline_prediction"
        ]
    )

    wrong_to_right = (
        changed
        &
        (
            e_eval_df[
                "baseline_exact"
            ] == 0
        )
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (
            e_eval_df[
                "baseline_exact"
            ] == 1
        )
        &
        (exact == 0)
    )

    return {
        "strategy": name,

        "exact": float(
            exact.mean()
        ),

        "correct_count": int(
            exact.sum()
        ),

        "changed_count": int(
            changed.sum()
        ),

        "wrong_to_right": int(
            wrong_to_right.sum()
        ),

        "right_to_wrong": int(
            right_to_wrong.sum()
        ),

        "net_change": int(
            wrong_to_right.sum()
            -
            right_to_wrong.sum()
        ),

        "canonical_1234_count": int(
            (
                predictions
                == "1 2 3 4"
            ).sum()
        ),
    }


def e_hybrid(predictions):
    use_decoder = (
        e_eval_df[
            "full_order_normalized_entropy"
        ]
        >= E_ENTROPY_THRESHOLD
    )

    return pd.Series(
        np.where(
            use_decoder,
            predictions,
            e_eval_df[
                "baseline_prediction"
            ],
        ),
        index=e_eval_df.index,
    )


def e_append_result(
    rows,
    predictions,
    name,
    extra=None,
):
    global_result = e_evaluate(
        predictions,
        name,
    )

    hybrid_result = e_evaluate(
        e_hybrid(predictions),
        f"{name}_hybrid",
    )

    row = {
        "strategy": name,

        "global_exact": (
            global_result["exact"]
        ),

        "global_correct": (
            global_result[
                "correct_count"
            ]
        ),

        "global_wrong_to_right": (
            global_result[
                "wrong_to_right"
            ]
        ),

        "global_right_to_wrong": (
            global_result[
                "right_to_wrong"
            ]
        ),

        "global_net": (
            global_result[
                "net_change"
            ]
        ),

        "hybrid_exact": (
            hybrid_result["exact"]
        ),

        "hybrid_correct": (
            hybrid_result[
                "correct_count"
            ]
        ),

        "hybrid_wrong_to_right": (
            hybrid_result[
                "wrong_to_right"
            ]
        ),

        "hybrid_right_to_wrong": (
            hybrid_result[
                "right_to_wrong"
            ]
        ),

        "hybrid_net": (
            hybrid_result[
                "net_change"
            ]
        ),

        "canonical_1234_count": (
            global_result[
                "canonical_1234_count"
            ]
        ),
    }

    if extra:
        row.update(extra)

    rows.append(row)


# ------------------------------------------------------------
# 9. 기존 결과 재현 확인
# ------------------------------------------------------------
print("재현 확인")

print(
    "asym:",
    e_evaluate(
        e_asym_predictions,
        "asym",
    )["exact"],
    "/ hybrid:",
    e_evaluate(
        e_hybrid(
            e_asym_predictions
        ),
        "asym_hybrid",
    )["exact"],
    "(expected 0.445 / 0.450)",
)

print(
    "previous:",
    e_evaluate(
        e_previous_predictions,
        "previous",
    )["exact"],
    "/ hybrid:",
    e_evaluate(
        e_hybrid(
            e_previous_predictions
        ),
        "previous_hybrid",
    )["exact"],
    "(expected 0.435 / 0.445)",
)

print(
    "recursive:",
    e_evaluate(
        e_recursive_predictions,
        "recursive",
    )["exact"],
    "(expected about 0.395)",
)

print(
    "full24:",
    e_evaluate(
        e_full24_predictions,
        "full24",
    )["exact"],
    "(expected 0.405)",
)


# ============================================================
# 실험 1
# Asym Top-K + recursive 재랭킹
# ============================================================
topk_rows = []
topk_cache = {}


margin_quantiles = [
    0.25,
    0.50,
    0.75,
    1.00,
]


margin_thresholds = {
    quantile: float(
        np.quantile(
            e_asym_margins,
            quantile,
        )
    )
    for quantile
    in margin_quantiles
}


for top_k in [
    2,
    3,
    4,
    5,
]:
    for recursive_weight in [
        0.05,
        0.10,
        0.20,
        0.30,
        0.50,
        0.75,
        1.00,
    ]:
        for margin_quantile in (
            margin_quantiles
        ):
            threshold = (
                margin_thresholds[
                    margin_quantile
                ]
            )

            predictions = []

            for position, (
                _,
                eval_row,
            ) in enumerate(
                e_eval_df.iterrows()
            ):
                sample_id = str(
                    eval_row[
                        "sample_id"
                    ]
                )

                table = (
                    e_candidate_tables[
                        sample_id
                    ]
                )

                # asym joint의 1위와 2위 차이가
                # 큰 샘플은 건드리지 않음
                if (
                    e_asym_margins[
                        position
                    ]
                    > threshold
                ):
                    predictions.append(
                        table.iloc[0][
                            "prediction"
                        ]
                    )

                    continue

                top_candidates = (
                    table
                    .head(top_k)
                    .copy()
                )

                # 점수 범위 차이를 줄이기 위해
                # 24개 후보 기준 z-score 결합
                top_candidates[
                    "rerank_score"
                ] = (
                    top_candidates[
                        "asym_z"
                    ]
                    +
                    recursive_weight
                    * top_candidates[
                        "recursive_z"
                    ]
                )

                best_index = (
                    top_candidates[
                        "rerank_score"
                    ].idxmax()
                )

                predictions.append(
                    top_candidates.loc[
                        best_index,
                        "prediction",
                    ]
                )

            predictions = pd.Series(
                predictions,
                index=e_eval_df.index,
            )

            name = (
                f"topk{top_k}"
                f"_rw{recursive_weight:.2f}"
                f"_mq{margin_quantile:.2f}"
            )

            topk_cache[
                name
            ] = predictions

            e_append_result(
                topk_rows,
                predictions,
                name,
                {
                    "top_k": top_k,

                    "recursive_weight": (
                        recursive_weight
                    ),

                    "margin_quantile": (
                        margin_quantile
                    ),

                    "margin_threshold": (
                        threshold
                    ),
                },
            )


topk_summary_df = (
    pd.DataFrame(topk_rows)
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\n1) Top-K recursive reranking"
)

display(
    topk_summary_df.head(30)
)


# ============================================================
# 실험 2
# Candidate rank fusion
# ============================================================
rank_rows = []
rank_cache = {}


weight_values = np.round(
    np.arange(
        0.0,
        1.01,
        0.10,
    ),
    2,
)


for w_previous in weight_values:
    for w_asym in weight_values:
        for w_recursive in weight_values:
            w_full24 = float(
                np.round(
                    1.0
                    - w_previous
                    - w_asym
                    - w_recursive,
                    2,
                )
            )

            if (
                w_full24 < 0
                or w_full24 > 1
            ):
                continue

            # 강한 두 decoder의 합은
            # 최소 50% 이상 유지
            if (
                w_previous
                + w_asym
                < 0.50
            ):
                continue

            # 상대적으로 약한 신호 제한
            if w_recursive > 0.30:
                continue

            if w_full24 > 0.20:
                continue

            predictions = []

            for _, eval_row in (
                e_eval_df.iterrows()
            ):
                sample_id = str(
                    eval_row[
                        "sample_id"
                    ]
                )

                table = (
                    e_candidate_tables[
                        sample_id
                    ]
                    .copy()
                )

                table[
                    "fusion_score"
                ] = (
                    w_previous
                    * table[
                        "previous_borda"
                    ]
                    +
                    w_asym
                    * table[
                        "asym_borda"
                    ]
                    +
                    w_recursive
                    * table[
                        "recursive_borda"
                    ]
                    +
                    w_full24
                    * table[
                        "full24_borda"
                    ]
                )

                best_index = (
                    table[
                        "fusion_score"
                    ].idxmax()
                )

                predictions.append(
                    table.loc[
                        best_index,
                        "prediction",
                    ]
                )

            predictions = pd.Series(
                predictions,
                index=e_eval_df.index,
            )

            name = (
                f"rank"
                f"_p{w_previous:.1f}"
                f"_a{w_asym:.1f}"
                f"_r{w_recursive:.1f}"
                f"_o{w_full24:.1f}"
            )

            rank_cache[
                name
            ] = predictions

            e_append_result(
                rank_rows,
                predictions,
                name,
                {
                    "weight_previous": (
                        w_previous
                    ),

                    "weight_asym": (
                        w_asym
                    ),

                    "weight_recursive": (
                        w_recursive
                    ),

                    "weight_full24": (
                        w_full24
                    ),
                },
            )


rank_summary_df = (
    pd.DataFrame(rank_rows)
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\n2) Candidate rank fusion"
)

display(
    rank_summary_df.head(30)
)


# ============================================================
# 실험 3
# 합의 / 투표
# ============================================================
def majority_or_default(
    predictions,
    default,
    minimum_votes,
):
    counts = Counter(
        predictions
    )

    max_votes = max(
        counts.values()
    )

    winners = [
        prediction
        for prediction, count
        in counts.items()
        if count == max_votes
    ]

    if max_votes < minimum_votes:
        return default

    if default in winners:
        return default

    return sorted(winners)[0]


vote_names = [
    "majority4_default_asym",
    "strict3_default_asym",
    "switch_asym_if_other_has2",
    "switch_previous_if_other_has2",
    "binary_prev_asym_both_tiebreakers",
    "binary_prev_asym_one_tiebreaker",
]


vote_lists = {
    name: []
    for name in vote_names
}


for row_index in range(
    len(e_eval_df)
):
    previous = (
        e_previous_predictions.iloc[
            row_index
        ]
    )

    asym = (
        e_asym_predictions.iloc[
            row_index
        ]
    )

    recursive = (
        e_recursive_predictions.iloc[
            row_index
        ]
    )

    full24 = (
        e_full24_predictions.iloc[
            row_index
        ]
    )

    all_votes = [
        previous,
        asym,
        recursive,
        full24,
    ]

    # 네 decoder 중 2표 이상.
    # 동률이면 asym 유지
    vote_lists[
        "majority4_default_asym"
    ].append(
        majority_or_default(
            all_votes,
            asym,
            2,
        )
    )

    # 세 표 이상 합의가 있어야 변경
    vote_lists[
        "strict3_default_asym"
    ].append(
        majority_or_default(
            all_votes,
            asym,
            3,
        )
    )

    # asym을 기본으로 두고
    # 나머지 셋 중 같은 다른 후보가
    # 두 표 이상일 때만 변경
    counts = Counter([
        previous,
        recursive,
        full24,
    ])

    alternatives = [
        prediction
        for prediction, count
        in counts.items()
        if (
            prediction != asym
            and count >= 2
        )
    ]

    if alternatives:
        selected = sorted(
            alternatives,
            key=lambda prediction: (
                -counts[prediction],
                prediction,
            ),
        )[0]

    else:
        selected = asym

    vote_lists[
        "switch_asym_if_other_has2"
    ].append(selected)

    # previous를 기본으로 두고
    # asym/recursive/full24 중
    # 같은 다른 후보가 두 표 이상일 때 변경
    counts = Counter([
        asym,
        recursive,
        full24,
    ])

    alternatives = [
        prediction
        for prediction, count
        in counts.items()
        if (
            prediction != previous
            and count >= 2
        )
    ]

    if alternatives:
        selected = sorted(
            alternatives,
            key=lambda prediction: (
                -counts[prediction],
                prediction,
            ),
        )[0]

    else:
        selected = previous

    vote_lists[
        "switch_previous_if_other_has2"
    ].append(selected)

    # previous와 asym이 다를 때
    # recursive와 full24가 둘 다 같은 쪽을
    # 지지해야 변경
    if previous == asym:
        selected = asym

    elif (
        recursive
        == full24
        == previous
    ):
        selected = previous

    elif (
        recursive
        == full24
        == asym
    ):
        selected = asym

    else:
        selected = asym

    vote_lists[
        "binary_prev_asym_both_tiebreakers"
    ].append(selected)

    # recursive/full24 중 더 많은 표를
    # previous 또는 asym에 준 쪽 선택
    # 동률이면 asym 유지
    if previous == asym:
        selected = asym

    else:
        previous_votes = (
            int(
                recursive == previous
            )
            +
            int(
                full24 == previous
            )
        )

        asym_votes = (
            int(
                recursive == asym
            )
            +
            int(
                full24 == asym
            )
        )

        if previous_votes > asym_votes:
            selected = previous

        else:
            selected = asym

    vote_lists[
        "binary_prev_asym_one_tiebreaker"
    ].append(selected)


vote_rows = []
vote_cache = {}


for name, predictions in (
    vote_lists.items()
):
    predictions = pd.Series(
        predictions,
        index=e_eval_df.index,
    )

    vote_cache[
        name
    ] = predictions

    e_append_result(
        vote_rows,
        predictions,
        name,
    )


vote_summary_df = (
    pd.DataFrame(vote_rows)
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\n3) Agreement / voting"
)

display(
    vote_summary_df
)


# ============================================================
# 10. 전체 방식 비교
# ============================================================
reference_rows = []


e_append_result(
    reference_rows,
    e_eval_df[
        "baseline_prediction"
    ],
    "baseline",
)


e_append_result(
    reference_rows,
    e_previous_predictions,
    "previous_joint",
)


e_append_result(
    reference_rows,
    e_asym_predictions,
    "asym_best_joint",
)


e_append_result(
    reference_rows,
    e_recursive_predictions,
    "true_mixed_recursive",
)


e_append_result(
    reference_rows,
    e_full24_predictions,
    "full24",
)


combined_summary_df = pd.concat(
    [
        pd.DataFrame(
            reference_rows
        ),

        topk_summary_df.head(10),

        rank_summary_df.head(10),

        vote_summary_df,
    ],
    ignore_index=True,
    sort=False,
)


combined_summary_df = (
    combined_summary_df
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\n전체 방식 비교"
)

display(
    combined_summary_df.head(40)
)


# ============================================================
# 11. 최고 실험 결과 및 변경 샘플
# ============================================================
all_caches = {}

all_caches.update(
    topk_cache
)

all_caches.update(
    rank_cache
)

all_caches.update(
    vote_cache
)


all_experiment_rows = pd.concat(
    [
        topk_summary_df,
        rank_summary_df,
        vote_summary_df,
    ],
    ignore_index=True,
    sort=False,
)


all_experiment_rows = (
    all_experiment_rows
    .sort_values(
        [
            "hybrid_exact",
            "global_exact",
            "hybrid_right_to_wrong",
            "hybrid_net",
        ],
        ascending=[
            False,
            False,
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)


best_strategy = str(
    all_experiment_rows.iloc[0][
        "strategy"
    ]
)


best_predictions = (
    all_caches[
        best_strategy
    ]
)


best_hybrid_predictions = (
    e_hybrid(
        best_predictions
    )
)


e_best_result_df = (
    e_eval_df.copy()
)


e_best_result_df[
    "previous_prediction"
] = e_previous_predictions


e_best_result_df[
    "asym_prediction"
] = e_asym_predictions


e_best_result_df[
    "recursive_prediction"
] = e_recursive_predictions


e_best_result_df[
    "full24_prediction"
] = e_full24_predictions


e_best_result_df[
    "ensemble_prediction"
] = best_predictions


e_best_result_df[
    "ensemble_hybrid_prediction"
] = best_hybrid_predictions


e_best_result_df[
    "ensemble_exact"
] = (
    e_best_result_df[
        "ensemble_prediction"
    ]
    ==
    e_best_result_df[
        "gold_order"
    ]
).astype(int)


e_best_result_df[
    "ensemble_hybrid_exact"
] = (
    e_best_result_df[
        "ensemble_hybrid_prediction"
    ]
    ==
    e_best_result_df[
        "gold_order"
    ]
).astype(int)


print(
    "\n최고 ensemble strategy:",
    best_strategy,
)


display(
    all_experiment_rows.head(15)
)


print(
    "\n최고 ensemble이 "
    "asym joint와 다르게 선택한 샘플"
)


display(
    e_best_result_df[
        e_best_result_df[
            "ensemble_prediction"
        ]
        !=
        e_best_result_df[
            "asym_prediction"
        ]
    ][
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "baseline_prediction",
            "previous_prediction",
            "asym_prediction",
            "recursive_prediction",
            "full24_prediction",
            "ensemble_prediction",
            "ensemble_hybrid_prediction",
            "baseline_exact",
            "ensemble_exact",
            "ensemble_hybrid_exact",
        ]
    ].head(100)
)

재현 확인
asym: 0.445 / hybrid: 0.45 (expected 0.445 / 0.450)
previous: 0.435 / hybrid: 0.445 (expected 0.435 / 0.445)
recursive: 0.395 (expected about 0.395)
full24: 0.405 (expected 0.405)

1) Top-K recursive reranking


,strategy,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count,top_k,recursive_weight,margin_quantile,margin_threshold
0,topk2_rw1.00_mq0.25,0.445,89,24,9,15,0.435,87,18,5,13,44,2,1.00,0.25,0.063553
1,topk3_rw1.00_mq0.25,0.445,89,24,9,15,0.435,87,18,5,13,44,3,1.00,0.25,0.063553
2,topk4_rw1.00_mq0.25,0.445,89,24,9,15,0.435,87,18,5,13,44,4,1.00,0.25,0.063553
3,topk5_rw1.00_mq0.25,0.445,89,24,9,15,0.435,87,18,5,13,44,5,1.00,0.25,0.063553
4,topk2_rw0.05_mq0.25,0.430,86,22,10,12,0.435,87,19,6,13,50,2,0.05,0.25,0.063553
5,topk2_rw0.05_mq0.50,0.430,86,22,10,12,0.435,87,19,6,13,50,2,0.05,0.50,0.196620
6,topk2_rw0.05_mq0.75,0.430,86,22,10,12,0.435,87,19,6,13,50,2,0.05,0.75,0.555887
7,topk2_rw0.05_mq1.00,0.430,86,22,10,12,0.435,87,19,6,13,50,2,0.05,1.00,1.398539
8,topk3_rw0.05_mq0.25,0.430,86,22,10,12,0.435,87,19,6,13,50,3,0.05,0.25,0.063553
9,topk3_rw0.05_mq0.50,0.430,86,22,10,12,0.435,87,19,6,13,50,3,0.05,0.50,0.196620



2) Candidate rank fusion


,strategy,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count,weight_previous,weight_asym,weight_recursive,weight_full24
0,rank_p0.1_a0.4_r0.3_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,0.1,0.4,0.3,0.2
1,rank_p0.1_a0.5_r0.2_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,54,0.1,0.5,0.2,0.2
2,rank_p0.2_a0.3_r0.3_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,0.2,0.3,0.3,0.2
3,rank_p0.2_a0.4_r0.2_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,0.2,0.4,0.2,0.2
4,rank_p0.3_a0.2_r0.3_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,0.3,0.2,0.3,0.2
5,rank_p0.0_a0.8_r0.0_o0.2,0.445,89,25,10,15,0.45,90,22,6,16,54,0.0,0.8,0.0,0.2
6,rank_p0.0_a0.8_r0.1_o0.1,0.445,89,25,10,15,0.45,90,22,6,16,53,0.0,0.8,0.1,0.1
7,rank_p0.0_a0.8_r0.2_o-0.0,0.445,89,25,10,15,0.45,90,22,6,16,54,0.0,0.8,0.2,-0.0
8,rank_p0.0_a0.9_r0.0_o0.1,0.445,89,25,10,15,0.45,90,22,6,16,54,0.0,0.9,0.0,0.1
9,rank_p0.0_a0.9_r0.1_o-0.0,0.445,89,25,10,15,0.45,90,22,6,16,54,0.0,0.9,0.1,-0.0



3) Agreement / voting


...

,strategy,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count,top_k,recursive_weight,margin_quantile,margin_threshold,weight_previous,weight_asym,weight_recursive,weight_full24
0,rank_p0.1_a0.4_r0.3_o0.2,0.450,90,26,10,16,0.450,90,22,6,16,53,NaN,NaN,NaN,NaN,0.1,0.4,0.3,0.2
1,rank_p0.1_a0.5_r0.2_o0.2,0.450,90,26,10,16,0.450,90,22,6,16,54,NaN,NaN,NaN,NaN,0.1,0.5,0.2,0.2
2,rank_p0.2_a0.3_r0.3_o0.2,0.450,90,26,10,16,0.450,90,22,6,16,53,NaN,NaN,NaN,NaN,0.2,0.3,0.3,0.2
3,rank_p0.2_a0.4_r0.2_o0.2,0.450,90,26,10,16,0.450,90,22,6,16,53,NaN,NaN,NaN,NaN,0.2,0.4,0.2,0.2
4,rank_p0.3_a0.2_r0.3_o0.2,0.450,90,26,10,16,0.450,90,22,6,16,53,NaN,NaN,NaN,NaN,0.3,0.2,0.3,0.2
5,asym_best_joint,0.445,89,25,10,15,0.450,90,22,6,16,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,rank_p0.0_a0.8_r0.0_o0.2,0.445,89,25,10,15,0.450,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.8,0.0,0.2
7,rank_p0.0_a0.8_r0.1_o0.1,0.445,89,25,10,15,0.450,90,22,6,16,53,NaN,NaN,NaN,NaN,0.0,0.8,0.1,0.1
8,rank_p0.0_a0.8_r0.2_o-0.0,0.445,89,25,10,15,0.450,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.8,0.2,-0.0
9,rank_p0.0_a0.9_r0.0_o0.1,0.445,89,25,10,15,0.450,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.9,0.0,0.1



최고 ensemble strategy: rank_p0.1_a0.4_r0.3_o0.2


,strategy,global_exact,global_correct,global_wrong_to_right,global_right_to_wrong,global_net,hybrid_exact,hybrid_correct,hybrid_wrong_to_right,hybrid_right_to_wrong,hybrid_net,canonical_1234_count,top_k,recursive_weight,margin_quantile,margin_threshold,weight_previous,weight_asym,weight_recursive,weight_full24
0,rank_p0.1_a0.4_r0.3_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,NaN,NaN,NaN,NaN,0.1,0.4,0.3,0.2
1,rank_p0.1_a0.5_r0.2_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,54,NaN,NaN,NaN,NaN,0.1,0.5,0.2,0.2
2,rank_p0.2_a0.3_r0.3_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,NaN,NaN,NaN,NaN,0.2,0.3,0.3,0.2
3,rank_p0.2_a0.4_r0.2_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,NaN,NaN,NaN,NaN,0.2,0.4,0.2,0.2
4,rank_p0.3_a0.2_r0.3_o0.2,0.450,90,26,10,16,0.45,90,22,6,16,53,NaN,NaN,NaN,NaN,0.3,0.2,0.3,0.2
5,rank_p0.0_a0.8_r0.0_o0.2,0.445,89,25,10,15,0.45,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.8,0.0,0.2
6,rank_p0.0_a0.8_r0.1_o0.1,0.445,89,25,10,15,0.45,90,22,6,16,53,NaN,NaN,NaN,NaN,0.0,0.8,0.1,0.1
7,rank_p0.0_a0.8_r0.2_o-0.0,0.445,89,25,10,15,0.45,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.8,0.2,-0.0
8,rank_p0.0_a0.9_r0.0_o0.1,0.445,89,25,10,15,0.45,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.9,0.0,0.1
9,rank_p0.0_a0.9_r0.1_o-0.0,0.445,89,25,10,15,0.45,90,22,6,16,54,NaN,NaN,NaN,NaN,0.0,0.9,0.1,-0.0



최고 ensemble이 asym joint와 다르게 선택한 샘플


,sample_id,gold_order,full_order_normalized_entropy,baseline_prediction,previous_prediction,asym_prediction,recursive_prediction,full24_prediction,ensemble_prediction,ensemble_hybrid_prediction,baseline_exact,ensemble_exact,ensemble_hybrid_exact
0,x6AwMw,2 3 4 1,0.919593,2 3 1 4,2 3 1 4,2 3 1 4,2 3 4 1,2 3 4 1,2 3 4 1,2 3 1 4,0,1,0
46,EuFm8b,2 1 3 4,0.998520,3 2 1 4,2 3 1 4,2 3 1 4,3 2 1 4,3 4 1 2,3 1 2 4,3 1 2 4,0,0,0
61,U6JoVw,2 4 3 1,0.999027,3 2 1 4,3 1 2 4,3 1 2 4,3 2 1 4,2 4 3 1,2 1 3 4,2 1 3 4,0,0,0
103,qv76Be,4 3 1 2,0.969996,1 4 3 2,1 3 4 2,1 2 3 4,1 4 3 2,1 2 3 4,1 3 4 2,1 3 4 2,0,0,0
133,PifW3P,4 1 3 2,0.997943,4 1 3 2,1 2 3 4,1 2 3 4,4 1 2 3,4 1 2 3,4 1 2 3,4 1 2 3,1,0,0
174,pDbV4M,4 2 3 1,0.996887,3 1 2 4,3 1 4 2,3 1 4 2,1 3 4 2,1 3 4 2,1 3 4 2,1 3 4 2,0,0,0
194,3jqefc,3 4 2 1,0.997835,1 3 2 4,3 1 2 4,3 1 2 4,3 1 2 4,1 2 3 4,1 2 3 4,1 2 3 4,0,0,0


In [43]:
# ============================================================
# Rank Fusion 전용 entropy gate
# + Low-entropy consensus rescue
#
# 필요한 선행 셀:
# - 3가지 ensemble 실험 셀
#
# 필요한 변수:
# - e_eval_df
# - e_candidate_tables
# - e_asym_predictions
# - e_recursive_predictions
# - e_full24_predictions
# - e_previous_predictions
#
# 추가 모델 추론 없음
# ============================================================

from collections import Counter

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 0. 필수 변수 확인
# ------------------------------------------------------------
required_names = [
    "e_eval_df",
    "e_candidate_tables",
    "e_asym_predictions",
    "e_recursive_predictions",
    "e_full24_predictions",
    "e_previous_predictions",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise RuntimeError(
        "먼저 이전 3가지 ensemble 실험 셀을 실행하세요.\n"
        f"누락 변수: {missing_names}"
    )


CR_EPS = 1e-12


# ------------------------------------------------------------
# 1. 사용할 rank fusion 설정
#
# 둘 다 Eval200에서 global 0.450을 기록한 설정
#
# conservative:
# recursive 비중이 더 작음
#
# balanced:
# recursive 비중이 조금 더 큼
# ------------------------------------------------------------
CR_RANK_CONFIGS = {
    "rf_conservative": {
        "previous": 0.10,
        "asym": 0.50,
        "recursive": 0.20,
        "full24": 0.20,
    },

    "rf_balanced": {
        "previous": 0.10,
        "asym": 0.40,
        "recursive": 0.30,
        "full24": 0.20,
    },
}


# ------------------------------------------------------------
# 2. entropy threshold 탐색 범위
#
# 기존 0.945 포함
# ------------------------------------------------------------
CR_ENTROPY_THRESHOLDS = np.round(
    np.arange(
        0.900,
        0.991,
        0.005,
    ),
    3,
)

if 0.945 not in CR_ENTROPY_THRESHOLDS:
    CR_ENTROPY_THRESHOLDS = np.sort(
        np.append(
            CR_ENTROPY_THRESHOLDS,
            0.945,
        )
    )


# ------------------------------------------------------------
# 3. Consensus rescue 설정
#
# strict_all3:
# full24 == recursive == rank fusion
#
# recursive_full24:
# full24 == recursive
#
# full24_supported:
# full24가 recursive 또는 rank fusion 중 하나와 합의
# ------------------------------------------------------------
CR_RESCUE_MODES = [
    "strict_all3",
    "recursive_full24",
    "full24_supported",
]


# alternative가 base decoder의 몇 위 안에 있어야 하는가
CR_TOP_K_VALUES = [
    2,
    3,
    4,
    5,
]


# full24 top1-top2 margin 하한
#
# q=0.00:
# 사실상 margin 조건 없음
#
# q=0.75:
# full24 확신도가 상위 25%인 샘플에서만 rescue
CR_MARGIN_QUANTILES = [
    0.00,
    0.25,
    0.50,
    0.75,
]


# ------------------------------------------------------------
# 4. 공통 평가 함수
# ------------------------------------------------------------
def cr_evaluate(predictions):
    predictions = pd.Series(
        predictions,
        index=e_eval_df.index,
    )

    exact = (
        predictions
        == e_eval_df["gold_order"]
    ).astype(int)

    changed = (
        predictions
        != e_eval_df[
            "baseline_prediction"
        ]
    )

    wrong_to_right = (
        changed
        &
        (
            e_eval_df[
                "baseline_exact"
            ] == 0
        )
        &
        (exact == 1)
    )

    right_to_wrong = (
        changed
        &
        (
            e_eval_df[
                "baseline_exact"
            ] == 1
        )
        &
        (exact == 0)
    )

    return {
        "exact": float(
            exact.mean()
        ),

        "correct": int(
            exact.sum()
        ),

        "changed": int(
            changed.sum()
        ),

        "wrong_to_right": int(
            wrong_to_right.sum()
        ),

        "right_to_wrong": int(
            right_to_wrong.sum()
        ),

        "net": int(
            wrong_to_right.sum()
            - right_to_wrong.sum()
        ),

        "canonical_1234_count": int(
            (
                predictions
                == "1 2 3 4"
            ).sum()
        ),
    }


# ------------------------------------------------------------
# 5. Rank fusion 예측 및 후보 순위 계산
# ------------------------------------------------------------
cr_rank_predictions = {}

cr_rank_maps = {}

cr_rank_margins = {}


for config_name, weights in (
    CR_RANK_CONFIGS.items()
):
    predictions = []

    rank_maps = []

    margins = []

    for _, eval_row in (
        e_eval_df.iterrows()
    ):
        sample_id = str(
            eval_row["sample_id"]
        )

        table = (
            e_candidate_tables[
                sample_id
            ]
            .copy()
        )

        table[
            "rank_fusion_score"
        ] = (
            weights["previous"]
            * table["previous_borda"]
            +
            weights["asym"]
            * table["asym_borda"]
            +
            weights["recursive"]
            * table["recursive_borda"]
            +
            weights["full24"]
            * table["full24_borda"]
        )

        ranked = (
            table
            .sort_values(
                "rank_fusion_score",
                ascending=False,
            )
            .reset_index(drop=True)
        )

        predictions.append(
            ranked.iloc[0][
                "prediction"
            ]
        )

        margins.append(
            float(
                ranked.iloc[0][
                    "rank_fusion_score"
                ]
                -
                ranked.iloc[1][
                    "rank_fusion_score"
                ]
            )
        )

        rank_maps.append({
            prediction: rank + 1
            for rank, prediction
            in enumerate(
                ranked["prediction"]
            )
        })

    cr_rank_predictions[
        config_name
    ] = pd.Series(
        predictions,
        index=e_eval_df.index,
    )

    cr_rank_maps[
        config_name
    ] = rank_maps

    cr_rank_margins[
        config_name
    ] = np.asarray(
        margins,
        dtype=np.float64,
    )


# ------------------------------------------------------------
# 6. Asym joint의 후보 순위도 준비
# ------------------------------------------------------------
cr_asym_rank_maps = []


for _, eval_row in e_eval_df.iterrows():
    sample_id = str(
        eval_row["sample_id"]
    )

    table = (
        e_candidate_tables[
            sample_id
        ]
        .sort_values(
            "asym_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    cr_asym_rank_maps.append({
        prediction: rank + 1
        for rank, prediction
        in enumerate(
            table["prediction"]
        )
    })


# ------------------------------------------------------------
# 7. Full24의 top1-top2 log-score margin
#
# full24_score는 log probability이므로
# 차이가 클수록 full24가 1위 후보를 더 강하게 지지
# ------------------------------------------------------------
cr_full24_margins = []


for _, eval_row in e_eval_df.iterrows():
    sample_id = str(
        eval_row["sample_id"]
    )

    table = (
        e_candidate_tables[
            sample_id
        ]
        .sort_values(
            "full24_score",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    margin = float(
        table.iloc[0]["full24_score"]
        -
        table.iloc[1]["full24_score"]
    )

    cr_full24_margins.append(
        margin
    )


cr_full24_margins = np.asarray(
    cr_full24_margins,
    dtype=np.float64,
)


cr_full24_margin_thresholds = {
    quantile: float(
        np.quantile(
            cr_full24_margins,
            quantile,
        )
    )
    for quantile
    in CR_MARGIN_QUANTILES
}


print(
    "Full24 margin quantile thresholds"
)

display(
    pd.DataFrame([
        {
            "quantile": quantile,
            "margin_threshold": threshold,
        }
        for quantile, threshold
        in cr_full24_margin_thresholds.items()
    ])
)


# ------------------------------------------------------------
# 8. Rescue alternative 결정
#
# rescue candidate는 full24 예측을 기준으로 함.
# full24가 다른 decoder와 합의할 때만 사용.
# ------------------------------------------------------------
def cr_get_rescue_candidate(
    recursive_prediction,
    full24_prediction,
    rank_prediction,
    rescue_mode,
):
    if rescue_mode == "strict_all3":
        if (
            full24_prediction
            == recursive_prediction
            == rank_prediction
        ):
            return full24_prediction

        return None

    if rescue_mode == "recursive_full24":
        if (
            full24_prediction
            == recursive_prediction
        ):
            return full24_prediction

        return None

    if rescue_mode == "full24_supported":
        if (
            full24_prediction
            == recursive_prediction
            or
            full24_prediction
            == rank_prediction
        ):
            return full24_prediction

        return None

    raise ValueError(
        f"알 수 없는 rescue_mode: "
        f"{rescue_mode}"
    )


# ------------------------------------------------------------
# 9. 단일 전략 실행 함수
#
# 높은 entropy:
# base decoder 사용
#
# 낮은 entropy:
# 기본적으로 baseline 사용
#
# 단, consensus rescue 조건을 만족하면
# 낮은 entropy에서도 alternative 사용
# ------------------------------------------------------------
def cr_build_predictions(
    base_name,
    rank_config_name,
    entropy_threshold,
    rescue_mode,
    top_k,
    margin_quantile,
):
    if base_name == "asym":
        base_predictions = (
            e_asym_predictions
        )

        base_rank_maps = (
            cr_asym_rank_maps
        )

    elif base_name in cr_rank_predictions:
        base_predictions = (
            cr_rank_predictions[
                base_name
            ]
        )

        base_rank_maps = (
            cr_rank_maps[
                base_name
            ]
        )

    else:
        raise ValueError(
            f"알 수 없는 base_name: "
            f"{base_name}"
        )

    support_rank_predictions = (
        cr_rank_predictions[
            rank_config_name
        ]
    )

    margin_threshold = (
        cr_full24_margin_thresholds[
            margin_quantile
        ]
    )

    final_predictions = []

    rescued_flags = []

    for row_index in range(
        len(e_eval_df)
    ):
        entropy = float(
            e_eval_df.iloc[
                row_index
            ][
                "full_order_normalized_entropy"
            ]
        )

        baseline_prediction = (
            e_eval_df.iloc[
                row_index
            ][
                "baseline_prediction"
            ]
        )

        base_prediction = (
            base_predictions.iloc[
                row_index
            ]
        )

        recursive_prediction = (
            e_recursive_predictions.iloc[
                row_index
            ]
        )

        full24_prediction = (
            e_full24_predictions.iloc[
                row_index
            ]
        )

        rank_prediction = (
            support_rank_predictions.iloc[
                row_index
            ]
        )

        # 높은 entropy에서는 decoder 사용
        if entropy >= entropy_threshold:
            final_predictions.append(
                base_prediction
            )

            rescued_flags.append(
                False
            )

            continue

        # 낮은 entropy에서는 baseline이 기본
        selected_prediction = (
            baseline_prediction
        )

        rescued = False

        alternative = (
            cr_get_rescue_candidate(
                recursive_prediction=(
                    recursive_prediction
                ),
                full24_prediction=(
                    full24_prediction
                ),
                rank_prediction=(
                    rank_prediction
                ),
                rescue_mode=(
                    rescue_mode
                ),
            )
        )

        if alternative is not None:
            alternative_rank = (
                base_rank_maps[
                    row_index
                ].get(
                    alternative,
                    999,
                )
            )

            full24_margin = (
                cr_full24_margins[
                    row_index
                ]
            )

            # alternative가 base decoder에서도
            # 완전히 동떨어진 후보가 아니어야 함
            top_k_pass = (
                alternative_rank
                <= top_k
            )

            # full24가 충분히 확신해야 함
            margin_pass = (
                full24_margin
                >= margin_threshold
            )

            # 실제로 현재 baseline과 다를 때만 rescue
            different_from_baseline = (
                alternative
                != baseline_prediction
            )

            if (
                top_k_pass
                and margin_pass
                and different_from_baseline
            ):
                selected_prediction = (
                    alternative
                )

                rescued = True

        final_predictions.append(
            selected_prediction
        )

        rescued_flags.append(
            rescued
        )

    return (
        pd.Series(
            final_predictions,
            index=e_eval_df.index,
        ),
        np.asarray(
            rescued_flags,
            dtype=bool,
        ),
    )


# ------------------------------------------------------------
# 10. 전체 sweep
# ------------------------------------------------------------
cr_result_rows = []

cr_prediction_cache = {}

cr_rescue_cache = {}


base_names = [
    "asym",
    "rf_conservative",
    "rf_balanced",
]


for base_name in base_names:
    for rank_config_name in (
        CR_RANK_CONFIGS
    ):
        for entropy_threshold in (
            CR_ENTROPY_THRESHOLDS
        ):
            # 먼저 rescue 없는 threshold 성능
            if base_name == "asym":
                base_predictions = (
                    e_asym_predictions
                )
            else:
                base_predictions = (
                    cr_rank_predictions[
                        base_name
                    ]
                )

            no_rescue_predictions = (
                pd.Series(
                    np.where(
                        e_eval_df[
                            "full_order_normalized_entropy"
                        ]
                        >= entropy_threshold,
                        base_predictions,
                        e_eval_df[
                            "baseline_prediction"
                        ],
                    ),
                    index=e_eval_df.index,
                )
            )

            no_rescue_name = (
                f"{base_name}"
                f"__threshold_"
                f"{entropy_threshold:.3f}"
                "__no_rescue"
            )

            no_rescue_result = (
                cr_evaluate(
                    no_rescue_predictions
                )
            )

            cr_result_rows.append({
                "strategy": (
                    no_rescue_name
                ),

                "base": base_name,

                "support_rank": (
                    rank_config_name
                ),

                "entropy_threshold": (
                    float(
                        entropy_threshold
                    )
                ),

                "rescue_mode": (
                    "none"
                ),

                "top_k": np.nan,

                "margin_quantile": np.nan,

                "margin_threshold": np.nan,

                "rescue_count": 0,

                **no_rescue_result,
            })

            cr_prediction_cache[
                no_rescue_name
            ] = no_rescue_predictions

            cr_rescue_cache[
                no_rescue_name
            ] = np.zeros(
                len(e_eval_df),
                dtype=bool,
            )

            # Consensus rescue 실험
            for rescue_mode in (
                CR_RESCUE_MODES
            ):
                for top_k in (
                    CR_TOP_K_VALUES
                ):
                    for margin_quantile in (
                        CR_MARGIN_QUANTILES
                    ):
                        (
                            predictions,
                            rescued_flags,
                        ) = cr_build_predictions(
                            base_name=(
                                base_name
                            ),
                            rank_config_name=(
                                rank_config_name
                            ),
                            entropy_threshold=(
                                float(
                                    entropy_threshold
                                )
                            ),
                            rescue_mode=(
                                rescue_mode
                            ),
                            top_k=top_k,
                            margin_quantile=(
                                margin_quantile
                            ),
                        )

                        strategy_name = (
                            f"{base_name}"
                            f"__support_"
                            f"{rank_config_name}"
                            f"__threshold_"
                            f"{entropy_threshold:.3f}"
                            f"__{rescue_mode}"
                            f"__top{top_k}"
                            f"__mq"
                            f"{margin_quantile:.2f}"
                        )

                        result = cr_evaluate(
                            predictions
                        )

                        cr_result_rows.append({
                            "strategy": (
                                strategy_name
                            ),

                            "base": base_name,

                            "support_rank": (
                                rank_config_name
                            ),

                            "entropy_threshold": (
                                float(
                                    entropy_threshold
                                )
                            ),

                            "rescue_mode": (
                                rescue_mode
                            ),

                            "top_k": top_k,

                            "margin_quantile": (
                                margin_quantile
                            ),

                            "margin_threshold": (
                                cr_full24_margin_thresholds[
                                    margin_quantile
                                ]
                            ),

                            "rescue_count": int(
                                rescued_flags.sum()
                            ),

                            **result,
                        })

                        cr_prediction_cache[
                            strategy_name
                        ] = predictions

                        cr_rescue_cache[
                            strategy_name
                        ] = rescued_flags


# ------------------------------------------------------------
# 11. 결과 정렬
# ------------------------------------------------------------
cr_summary_df = (
    pd.DataFrame(
        cr_result_rows
    )
    .drop_duplicates(
        subset=["strategy"]
    )
    .sort_values(
        [
            "exact",
            "right_to_wrong",
            "wrong_to_right",
            "rescue_count",
        ],
        ascending=[
            False,
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nRank fusion gate + consensus rescue 상위 결과"
)

display(
    cr_summary_df.head(50)
)


# ------------------------------------------------------------
# 12. 기존 0.945 asym hybrid와 비교
# ------------------------------------------------------------
cr_reference_predictions = pd.Series(
    np.where(
        e_eval_df[
            "full_order_normalized_entropy"
        ]
        >= 0.945,
        e_asym_predictions,
        e_eval_df[
            "baseline_prediction"
        ],
    ),
    index=e_eval_df.index,
)


cr_reference_result = cr_evaluate(
    cr_reference_predictions
)


print(
    "\n기존 asym hybrid threshold=0.945"
)

print(
    cr_reference_result
)


# ------------------------------------------------------------
# 13. 최고 전략 확인
# ------------------------------------------------------------
cr_best_row = cr_summary_df.iloc[0]

cr_best_strategy = str(
    cr_best_row["strategy"]
)

cr_best_predictions = (
    cr_prediction_cache[
        cr_best_strategy
    ]
)

cr_best_rescued_flags = (
    cr_rescue_cache[
        cr_best_strategy
    ]
)


print(
    "\n최고 전략:",
    cr_best_strategy
)

print(
    "최고 exact:",
    cr_best_row["exact"]
)

print(
    "correct:",
    cr_best_row["correct"]
)

print(
    "wrong_to_right:",
    cr_best_row[
        "wrong_to_right"
    ]
)

print(
    "right_to_wrong:",
    cr_best_row[
        "right_to_wrong"
    ]
)

print(
    "rescue_count:",
    cr_best_row[
        "rescue_count"
    ]
)


# ------------------------------------------------------------
# 14. 최고 전략 상세 결과
# ------------------------------------------------------------
cr_best_result_df = (
    e_eval_df.copy()
)

cr_best_result_df[
    "previous_prediction"
] = e_previous_predictions

cr_best_result_df[
    "asym_prediction"
] = e_asym_predictions

cr_best_result_df[
    "recursive_prediction"
] = e_recursive_predictions

cr_best_result_df[
    "full24_prediction"
] = e_full24_predictions

cr_best_result_df[
    "rf_conservative_prediction"
] = cr_rank_predictions[
    "rf_conservative"
]

cr_best_result_df[
    "rf_balanced_prediction"
] = cr_rank_predictions[
    "rf_balanced"
]

cr_best_result_df[
    "reference_asym_hybrid"
] = cr_reference_predictions

cr_best_result_df[
    "final_prediction"
] = cr_best_predictions

cr_best_result_df[
    "rescued"
] = cr_best_rescued_flags

cr_best_result_df[
    "full24_margin"
] = cr_full24_margins

cr_best_result_df[
    "reference_exact"
] = (
    cr_best_result_df[
        "reference_asym_hybrid"
    ]
    ==
    cr_best_result_df[
        "gold_order"
    ]
).astype(int)

cr_best_result_df[
    "final_exact"
] = (
    cr_best_result_df[
        "final_prediction"
    ]
    ==
    cr_best_result_df[
        "gold_order"
    ]
).astype(int)


print(
    "\n실제로 rescue가 발생한 샘플"
)

display(
    cr_best_result_df[
        cr_best_result_df[
            "rescued"
        ]
    ][
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "baseline_prediction",
            "previous_prediction",
            "asym_prediction",
            "recursive_prediction",
            "full24_prediction",
            "rf_conservative_prediction",
            "rf_balanced_prediction",
            "reference_asym_hybrid",
            "final_prediction",
            "full24_margin",
            "reference_exact",
            "final_exact",
        ]
    ]
)


print(
    "\n기존 0.945 asym hybrid와 "
    "최종 예측이 달라진 모든 샘플"
)

display(
    cr_best_result_df[
        cr_best_result_df[
            "final_prediction"
        ]
        !=
        cr_best_result_df[
            "reference_asym_hybrid"
        ]
    ][
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "baseline_prediction",
            "asym_prediction",
            "recursive_prediction",
            "full24_prediction",
            "rf_conservative_prediction",
            "rf_balanced_prediction",
            "reference_asym_hybrid",
            "final_prediction",
            "rescued",
            "reference_exact",
            "final_exact",
        ]
    ].head(100)
)

Full24 margin quantile thresholds


,quantile,margin_threshold
0,0.00,0.000089
1,0.25,0.058045
2,0.50,0.152849
3,0.75,0.427067



Rank fusion gate + consensus rescue 상위 결과


,strategy,base,support_rank,entropy_threshold,rescue_mode,top_k,margin_quantile,margin_threshold,rescue_count,exact,correct,changed,wrong_to_right,right_to_wrong,net,canonical_1234_count
0,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,2.0,0.50,0.152849,2,0.455,91,72,23,6,17,54
1,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,2.0,0.75,0.427067,2,0.455,91,72,23,6,17,54
2,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,3.0,0.50,0.152849,2,0.455,91,72,23,6,17,54
3,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,3.0,0.75,0.427067,2,0.455,91,72,23,6,17,54
4,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,4.0,0.50,0.152849,2,0.455,91,72,23,6,17,54
5,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,4.0,0.75,0.427067,2,0.455,91,72,23,6,17,54
6,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,5.0,0.50,0.152849,2,0.455,91,72,23,6,17,54
7,asym__support_rf_conservative__threshold_0.940...,asym,rf_conservative,0.940,full24_supported,5.0,0.75,0.427067,2,0.455,91,72,23,6,17,54
8,asym__support_rf_conservative__threshold_0.945...,asym,rf_conservative,0.945,full24_supported,2.0,0.50,0.152849,2,0.455,91,72,23,6,17,54
9,asym__support_rf_conservative__threshold_0.945...,asym,rf_conservative,0.945,full24_supported,2.0,0.75,0.427067,2,0.455,91,72,23,6,17,54



기존 asym hybrid threshold=0.945
{'exact': 0.45, 'correct': 90, 'changed': 70, 'wrong_to_right': 22, 'right_to_wrong': 6, 'net': 16, 'canonical_1234_count': 54}

최고 전략: asym__support_rf_conservative__threshold_0.940__full24_supported__top2__mq0.50
최고 exact: 0.455
correct: 91
wrong_to_right: 23
right_to_wrong: 6
rescue_count: 2

실제로 rescue가 발생한 샘플


,sample_id,gold_order,full_order_normalized_entropy,baseline_prediction,previous_prediction,asym_prediction,recursive_prediction,full24_prediction,rf_conservative_prediction,rf_balanced_prediction,reference_asym_hybrid,final_prediction,full24_margin,reference_exact,final_exact
77,YzyxNa,3 2 4 1,0.908301,3 4 2 1,3 2 4 1,3 2 4 1,3 4 2 1,3 2 4 1,3 2 4 1,3 2 4 1,3 4 2 1,3 2 4 1,0.476206,0,1
124,I8IHnt,3 4 1 2,0.913792,3 2 4 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,3 2 4 1,3 4 2 1,0.953158,0,0



기존 0.945 asym hybrid와 최종 예측이 달라진 모든 샘플


,sample_id,gold_order,full_order_normalized_entropy,baseline_prediction,asym_prediction,recursive_prediction,full24_prediction,rf_conservative_prediction,rf_balanced_prediction,reference_asym_hybrid,final_prediction,rescued,reference_exact,final_exact
77,YzyxNa,3 2 4 1,0.908301,3 4 2 1,3 2 4 1,3 4 2 1,3 2 4 1,3 2 4 1,3 2 4 1,3 4 2 1,3 2 4 1,True,0,1
124,I8IHnt,3 4 1 2,0.913792,3 2 4 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,3 2 4 1,3 4 2 1,True,0,0


In [44]:
# ============================================================
# 최종 보수적 rescue 전략 선택 + 동작 검증
#
# 선택:
# - asym base
# - entropy threshold 0.945
# - rf_conservative support
# - full24_supported
# - asym Top-2
# - full24 margin quantile 0.75
# ============================================================

import numpy as np
import pandas as pd

FINAL_STRATEGY = (
    "asym"
    "__support_rf_conservative"
    "__threshold_0.945"
    "__full24_supported"
    "__top2"
    "__mq0.75"
)

if FINAL_STRATEGY not in cr_prediction_cache:
    raise KeyError(
        "선택한 전략이 cr_prediction_cache에 없습니다.\n"
        f"찾는 전략: {FINAL_STRATEGY}\n\n"
        "사용 가능한 유사 전략:\n"
        + "\n".join([
            name
            for name in cr_prediction_cache
            if (
                "threshold_0.945" in name
                and "full24_supported" in name
                and "top2" in name
            )
        ][:20])
    )

final_predictions = (
    cr_prediction_cache[
        FINAL_STRATEGY
    ].copy()
)

final_rescued_flags = np.asarray(
    cr_rescue_cache[
        FINAL_STRATEGY
    ],
    dtype=bool,
)

final_result = cr_evaluate(
    final_predictions
)

print("선택 전략:")
print(FINAL_STRATEGY)

print("\n성능:")
print(final_result)

assert final_result["correct"] == 91
assert np.isclose(
    final_result["exact"],
    0.455,
)

validation_rows = []

for position in range(
    len(e_eval_df)
):
    if not final_rescued_flags[position]:
        continue

    sample_id = str(
        e_eval_df.iloc[position][
            "sample_id"
        ]
    )

    gold = e_eval_df.iloc[position][
        "gold_order"
    ]

    entropy = float(
        e_eval_df.iloc[position][
            "full_order_normalized_entropy"
        ]
    )

    baseline = e_eval_df.iloc[position][
        "baseline_prediction"
    ]

    asym = e_asym_predictions.iloc[
        position
    ]

    recursive = (
        e_recursive_predictions.iloc[
            position
        ]
    )

    full24 = (
        e_full24_predictions.iloc[
            position
        ]
    )

    rank_prediction = (
        cr_rank_predictions[
            "rf_conservative"
        ].iloc[position]
    )

    final = final_predictions.iloc[
        position
    ]

    alternative = (
        cr_get_rescue_candidate(
            recursive_prediction=recursive,
            full24_prediction=full24,
            rank_prediction=rank_prediction,
            rescue_mode=(
                "full24_supported"
            ),
        )
    )

    asym_rank = (
        cr_asym_rank_maps[
            position
        ].get(
            alternative,
            999,
        )
        if alternative is not None
        else 999
    )

    full24_margin = float(
        cr_full24_margins[position]
    )

    margin_threshold = float(
        cr_full24_margin_thresholds[
            0.75
        ]
    )

    validation_rows.append({
        "position": position,
        "sample_id": sample_id,
        "gold": gold,
        "entropy": entropy,
        "baseline": baseline,
        "asym": asym,
        "recursive": recursive,
        "full24": full24,
        "rank_fusion": rank_prediction,
        "rescue_candidate": alternative,
        "asym_rank_of_candidate": (
            asym_rank
        ),
        "full24_margin": (
            full24_margin
        ),
        "required_margin": (
            margin_threshold
        ),
        "final": final,
        "final_equals_candidate": (
            final == alternative
        ),
        "final_exact": (
            final == gold
        ),
    })

validation_df = pd.DataFrame(
    validation_rows
)

print("\nRescue 동작 검증:")
display(validation_df)

if len(validation_df) == 0:
    raise AssertionError(
        "rescue된 샘플이 없습니다."
    )

if not validation_df[
    "final_equals_candidate"
].all():
    display(
        validation_df[
            ~validation_df[
                "final_equals_candidate"
            ]
        ]
    )

    raise AssertionError(
        "rescued=True인데 final_prediction이 "
        "rescue candidate와 다른 행이 있습니다."
    )

print(
    "\n✅ 모든 rescue 행에서 "
    "final_prediction == rescue_candidate"
)

print(
    "✅ 최종 성능:",
    final_result["correct"],
    "/",
    len(e_eval_df),
    "=",
    final_result["exact"],
)

선택 전략:
asym__support_rf_conservative__threshold_0.945__full24_supported__top2__mq0.75

성능:
{'exact': 0.455, 'correct': 91, 'changed': 72, 'wrong_to_right': 23, 'right_to_wrong': 6, 'net': 17, 'canonical_1234_count': 54}

Rescue 동작 검증:


,position,sample_id,gold,entropy,baseline,asym,recursive,full24,rank_fusion,rescue_candidate,asym_rank_of_candidate,full24_margin,required_margin,final,final_equals_candidate,final_exact
0,77,YzyxNa,3 2 4 1,0.908301,3 4 2 1,3 2 4 1,3 4 2 1,3 2 4 1,3 2 4 1,3 2 4 1,1,0.476206,0.427067,3 2 4 1,True,True
1,124,I8IHnt,3 4 1 2,0.913792,3 2 4 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,3 4 2 1,1,0.953158,0.427067,3 4 2 1,True,False



✅ 모든 rescue 행에서 final_prediction == rescue_candidate
✅ 최종 성능: 91 / 200 = 0.455


In [45]:
# ============================================================
# Eval200:
# entropy hybrid vs baseline-default conservative consensus
#
# 선행 셀:
# - rank fusion + consensus rescue 셀
# ============================================================

import numpy as np
import pandas as pd


required_vars = [
    "e_eval_df",
    "e_asym_predictions",
    "e_recursive_predictions",
    "e_full24_predictions",
    "cr_rank_predictions",
    "cr_asym_rank_maps",
    "cr_full24_margins",
    "cr_full24_margin_thresholds",
    "cr_build_predictions",
    "cr_evaluate",
]

missing = [
    name
    for name in required_vars
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"먼저 이전 셀을 실행하세요. 누락: {missing}"
    )


baseline_predictions = (
    e_eval_df["baseline_prediction"]
    .copy()
)

rank_predictions = (
    cr_rank_predictions[
        "rf_conservative"
    ]
)

margin_threshold = float(
    cr_full24_margin_thresholds[
        0.75
    ]
)


# ------------------------------------------------------------
# 결과 저장
# ------------------------------------------------------------
comparison_rows = []
prediction_cache = {}


def add_strategy(
    name,
    predictions,
    strategy_type,
):
    predictions = pd.Series(
        predictions,
        index=e_eval_df.index,
    )

    result = cr_evaluate(
        predictions
    )

    # 현재 baseline과 비교
    baseline_correct = (
        baseline_predictions
        ==
        e_eval_df["gold_order"]
    )

    new_correct = (
        predictions
        ==
        e_eval_df["gold_order"]
    )

    changed = (
        predictions
        != baseline_predictions
    )

    changed_wrong_to_right = (
        changed
        &
        (~baseline_correct)
        &
        new_correct
    )

    changed_right_to_wrong = (
        changed
        &
        baseline_correct
        &
        (~new_correct)
    )

    comparison_rows.append({
        "strategy": name,
        "type": strategy_type,
        "exact": result["exact"],
        "correct": result["correct"],
        "changed_from_baseline": int(
            changed.sum()
        ),
        "changed_rate": float(
            changed.mean()
        ),
        "wrong_to_right": int(
            changed_wrong_to_right.sum()
        ),
        "right_to_wrong": int(
            changed_right_to_wrong.sum()
        ),
        "net_gain_vs_baseline": int(
            changed_wrong_to_right.sum()
            -
            changed_right_to_wrong.sum()
        ),
        "canonical_1234_count": (
            result[
                "canonical_1234_count"
            ]
        ),
    })

    prediction_cache[
        name
    ] = predictions


# ------------------------------------------------------------
# 0. Baseline
# ------------------------------------------------------------
add_strategy(
    "baseline",
    baseline_predictions,
    "reference",
)


# ------------------------------------------------------------
# 1. Entropy threshold + rescue
# ------------------------------------------------------------
for threshold in [
    0.940,
    0.945,
    0.950,
    0.955,
    0.960,
]:
    predictions, rescued = (
        cr_build_predictions(
            base_name="asym",
            rank_config_name=(
                "rf_conservative"
            ),
            entropy_threshold=threshold,
            rescue_mode=(
                "full24_supported"
            ),
            top_k=2,
            margin_quantile=0.75,
        )
    )

    name = (
        f"entropy_{threshold:.3f}"
        "_asym_plus_rescue"
    )

    add_strategy(
        name,
        predictions,
        "entropy_hybrid",
    )

    comparison_rows[-1][
        "rescue_count"
    ] = int(
        rescued.sum()
    )

    comparison_rows[-1][
        "asym_gate_count"
    ] = int(
        (
            e_eval_df[
                "full_order_normalized_entropy"
            ]
            >= threshold
        ).sum()
    )


# ------------------------------------------------------------
# 2. Baseline-default consensus
#
# entropy를 사용하지 않고 baseline 유지.
# 조건을 만족할 때만 full24 후보로 변경.
# ------------------------------------------------------------
def build_baseline_consensus(
    mode,
    top_k=2,
    margin_quantile=0.75,
):
    required_margin = float(
        cr_full24_margin_thresholds[
            margin_quantile
        ]
    )

    final_predictions = []

    changed_flags = []

    for position in range(
        len(e_eval_df)
    ):
        baseline = (
            baseline_predictions.iloc[
                position
            ]
        )

        asym = (
            e_asym_predictions.iloc[
                position
            ]
        )

        recursive = (
            e_recursive_predictions.iloc[
                position
            ]
        )

        full24 = (
            e_full24_predictions.iloc[
                position
            ]
        )

        rank = (
            rank_predictions.iloc[
                position
            ]
        )

        candidate = None

        # 가장 엄격:
        # full24, recursive, rank fusion
        # 셋이 모두 같은 후보
        if mode == "all_three":
            if (
                full24
                == recursive
                == rank
            ):
                candidate = full24

        # full24 + rank fusion 합의
        elif mode == "full24_rank":
            if full24 == rank:
                candidate = full24

        # full24 + recursive 합의
        elif mode == "full24_recursive":
            if full24 == recursive:
                candidate = full24

        # 둘 중 하나만 full24를 지지해도 허용
        elif mode == "full24_supported":
            if (
                full24 == recursive
                or full24 == rank
            ):
                candidate = full24

        else:
            raise ValueError(mode)

        selected = baseline
        changed = False

        if candidate is not None:
            asym_rank = (
                cr_asym_rank_maps[
                    position
                ].get(
                    candidate,
                    999,
                )
            )

            margin = float(
                cr_full24_margins[
                    position
                ]
            )

            if (
                candidate != baseline
                and asym_rank <= top_k
                and margin
                >= required_margin
            ):
                selected = candidate
                changed = True

        final_predictions.append(
            selected
        )

        changed_flags.append(
            changed
        )

    return (
        pd.Series(
            final_predictions,
            index=e_eval_df.index,
        ),
        np.asarray(
            changed_flags,
            dtype=bool,
        ),
    )


for mode in [
    "all_three",
    "full24_rank",
    "full24_recursive",
    "full24_supported",
]:
    predictions, changed_flags = (
        build_baseline_consensus(
            mode=mode,
            top_k=2,
            margin_quantile=0.75,
        )
    )

    name = (
        f"baseline_default"
        f"__{mode}"
        "__top2"
        "__mq0.75"
    )

    add_strategy(
        name,
        predictions,
        "baseline_consensus",
    )

    comparison_rows[-1][
        "consensus_change_count"
    ] = int(
        changed_flags.sum()
    )


# ------------------------------------------------------------
# 3. 결과 표
# ------------------------------------------------------------
comparison_df = (
    pd.DataFrame(
        comparison_rows
    )
    .sort_values(
        [
            "exact",
            "right_to_wrong",
            "wrong_to_right",
            "changed_from_baseline",
        ],
        ascending=[
            False,
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "Eval200 전체 비교"
)

display(
    comparison_df
)


# ------------------------------------------------------------
# 4. Threshold별로 추가/제외되는 샘플 확인
# ------------------------------------------------------------
threshold_detail_rows = []


for threshold in [
    0.945,
    0.950,
    0.955,
    0.960,
]:
    name = (
        f"entropy_{threshold:.3f}"
        "_asym_plus_rescue"
    )

    predictions = (
        prediction_cache[name]
    )

    exact = (
        predictions
        ==
        e_eval_df["gold_order"]
    )

    threshold_detail_rows.append({
        "threshold": threshold,
        "correct": int(
            exact.sum()
        ),
        "exact": float(
            exact.mean()
        ),
        "changed_from_baseline": int(
            (
                predictions
                != baseline_predictions
            ).sum()
        ),
        "asym_gate_count": int(
            (
                e_eval_df[
                    "full_order_normalized_entropy"
                ]
                >= threshold
            ).sum()
        ),
    })


threshold_detail_df = (
    pd.DataFrame(
        threshold_detail_rows
    )
)


print(
    "\nThreshold 변화 비교"
)

display(
    threshold_detail_df
)


# ------------------------------------------------------------
# 5. 0.945와 0.950이 실제로 다른 샘플
# ------------------------------------------------------------
pred_0945 = prediction_cache[
    "entropy_0.945_asym_plus_rescue"
]

pred_0950 = prediction_cache[
    "entropy_0.950_asym_plus_rescue"
]


different_mask = (
    pred_0945
    != pred_0950
)


difference_df = (
    e_eval_df[
        [
            "sample_id",
            "gold_order",
            "full_order_normalized_entropy",
            "baseline_prediction",
        ]
    ]
    .copy()
)


difference_df[
    "asym_prediction"
] = e_asym_predictions


difference_df[
    "prediction_0945"
] = pred_0945


difference_df[
    "prediction_0950"
] = pred_0950


difference_df[
    "exact_0945"
] = (
    pred_0945
    ==
    e_eval_df["gold_order"]
).astype(int)


difference_df[
    "exact_0950"
] = (
    pred_0950
    ==
    e_eval_df["gold_order"]
).astype(int)


print(
    "\n0.945와 0.950에서 "
    "결과가 달라지는 Eval 샘플"
)

display(
    difference_df[
        different_mask
    ].sort_values(
        "full_order_normalized_entropy"
    )
)

Eval200 전체 비교


,strategy,type,exact,correct,changed_from_baseline,changed_rate,wrong_to_right,right_to_wrong,net_gain_vs_baseline,canonical_1234_count,rescue_count,asym_gate_count,consensus_change_count
0,entropy_0.940_asym_plus_rescue,entropy_hybrid,0.455,91,72,0.360,23,6,17,54,2.0,138.0,NaN
1,entropy_0.945_asym_plus_rescue,entropy_hybrid,0.455,91,72,0.360,23,6,17,54,2.0,128.0,NaN
2,entropy_0.950_asym_plus_rescue,entropy_hybrid,0.450,90,69,0.345,21,5,16,53,3.0,122.0,NaN
3,entropy_0.955_asym_plus_rescue,entropy_hybrid,0.445,89,66,0.330,19,4,15,52,4.0,111.0,NaN
4,entropy_0.960_asym_plus_rescue,entropy_hybrid,0.440,88,64,0.320,18,4,14,52,5.0,105.0,NaN
5,baseline_default__full24_rank__top2__mq0.75,baseline_consensus,0.390,78,6,0.030,4,0,4,30,NaN,NaN,6.0
6,baseline_default__full24_supported__top2__mq0.75,baseline_consensus,0.390,78,7,0.035,4,0,4,30,NaN,NaN,7.0
7,baseline_default__all_three__top2__mq0.75,baseline_consensus,0.380,76,3,0.015,2,0,2,30,NaN,NaN,3.0
8,baseline_default__full24_recursive__top2__mq0.75,baseline_consensus,0.380,76,4,0.020,2,0,2,30,NaN,NaN,4.0
9,baseline,reference,0.370,74,0,0.000,0,0,0,28,NaN,NaN,NaN



Threshold 변화 비교


,threshold,correct,exact,changed_from_baseline,asym_gate_count
0,0.945,91,0.455,72,128
1,0.950,90,0.450,69,122
2,0.955,89,0.445,66,111
3,0.960,88,0.440,64,105



0.945와 0.950에서 결과가 달라지는 Eval 샘플


,sample_id,gold_order,full_order_normalized_entropy,baseline_prediction,asym_prediction,prediction_0945,prediction_0950,exact_0945,exact_0950
50,zZ6ACe,2 4 3 1,0.946662,4 2 3 1,2 4 3 1,2 4 3 1,4 2 3 1,1,0
29,xEKSQr,1 3 4 2,0.947270,1 3 4 2,4 3 1 2,4 3 1 2,1 3 4 2,0,1
33,g7HN0l,1 2 3 4,0.947954,1 3 2 4,1 2 3 4,1 2 3 4,1 3 2 4,1,0


# Meta diagnostics and second-stage fallback experiments

아래 셀은 **기존 저장 확률과 `e_candidate_tables`만 재사용**하므로 모델 재추론이 없습니다.

실행 순서:

1. 기존 노트북의 True Mixed Recursive / ensemble / rank-fusion / Eval200 final 셀까지 실행
2. `META 0`부터 순서대로 실행
3. 먼저 oracle 결과로 selector 개선 여지가 충분한지 확인
4. 수동 repair와 OOF selector 결과는 `wrong_to_right`, `right_to_wrong`, `net` 기준으로 비교

`META_CURRENT_FINAL_NAME` 기본값은 `entropy_0.950_asym_plus_rescue`입니다. 다른 최종안을 기준으로 비교할 때는 첫 셀의 문자열만 변경하세요.


In [ ]:
# ============================================================
# META 0. 공통 준비
# - 기존 추론 결과를 7개 decoder + 24개 후보 표로 표준화
# - 추가 모델 추론 없음
#
# 선행 셀:
#   True Mixed Recursive Joint Decoder
#   3가지 ensemble 실험
#   Rank Fusion 전용 entropy gate
#   Eval200 0.950 final 전략 셀
# ============================================================

import itertools
import math
import re
from collections import Counter

import numpy as np
import pandas as pd


META_EPS = 1e-12
META_FRAMES = (1, 2, 3, 4)
META_ORDERS = list(itertools.permutations(META_FRAMES))

# 사용자가 말한 "현재 0.950 final"을 우선 선택한다.
# 다른 최종안을 기준으로 보고 싶으면 이 문자열만 바꾸면 된다.
META_CURRENT_FINAL_NAME = "entropy_0.950_asym_plus_rescue"


required = [
    "e_eval_df",
    "e_candidate_tables",
    "e_previous_predictions",
    "e_asym_predictions",
    "e_recursive_predictions",
    "e_full24_predictions",
    "cr_rank_predictions",
    "CR_RANK_CONFIGS",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "먼저 기존 ensemble / rank-fusion 셀을 실행하세요. "
        f"누락 변수: {missing}"
    )


def meta_order_tuple(value):
    """list / tuple / '1 2 3 4' / '[1,2,3,4]'를 tuple로 통일."""
    if isinstance(value, tuple):
        result = tuple(int(x) for x in value)
    elif isinstance(value, list):
        result = tuple(int(x) for x in value)
    else:
        numbers = re.findall(r"\d+", str(value))
        result = tuple(int(x) for x in numbers)

    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"잘못된 order 표현: {value}")
    return result


def meta_order_text(value):
    return " ".join(map(str, meta_order_tuple(value)))


def meta_kendall_distance(order_a, order_b):
    a = meta_order_tuple(order_a)
    b = meta_order_tuple(order_b)
    pos_b = {frame: index for index, frame in enumerate(b)}
    distance = 0
    for i in range(4):
        for j in range(i + 1, 4):
            if pos_b[a[i]] > pos_b[a[j]]:
                distance += 1
    return distance


def meta_safe_float(value, default=np.nan):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return float(default)
    return value if np.isfinite(value) else float(default)


def meta_row_value(row, column, default=np.nan):
    if column not in row.index:
        return default
    return meta_safe_float(row[column], default=default)


def meta_rank_series(values):
    return (
        pd.Series(values)
        .rank(method="min", ascending=False)
        .astype(int)
        .to_numpy()
    )


# ------------------------------------------------------------
# 1. Eval frame와 현재 final 선택
# ------------------------------------------------------------
meta_eval_df = e_eval_df.reset_index(drop=True).copy()
for column in [
    "gold_order",
    "baseline_prediction",
    "previous_joint_with_full_prediction",
]:
    if column in meta_eval_df.columns:
        meta_eval_df[column] = meta_eval_df[column].map(meta_order_text)

meta_eval_df["sample_id"] = meta_eval_df["sample_id"].astype(str)

if (
    "prediction_cache" in globals()
    and META_CURRENT_FINAL_NAME in prediction_cache
):
    meta_current_final = (
        pd.Series(prediction_cache[META_CURRENT_FINAL_NAME])
        .reset_index(drop=True)
        .map(meta_order_text)
    )
    meta_current_final_source = (
        f"prediction_cache[{META_CURRENT_FINAL_NAME!r}]"
    )
elif "final_predictions" in globals():
    meta_current_final = (
        pd.Series(final_predictions)
        .reset_index(drop=True)
        .map(meta_order_text)
    )
    meta_current_final_source = "final_predictions"
else:
    meta_current_final = meta_eval_df["baseline_prediction"].copy()
    meta_current_final_source = "baseline_prediction (fallback)"

if len(meta_current_final) != len(meta_eval_df):
    raise ValueError(
        "현재 final prediction 길이와 Eval 길이가 다릅니다: "
        f"{len(meta_current_final)} vs {len(meta_eval_df)}"
    )

meta_eval_df["current_final_prediction"] = meta_current_final
meta_eval_df["current_final_exact"] = (
    meta_eval_df["current_final_prediction"]
    == meta_eval_df["gold_order"]
).astype(int)


# ------------------------------------------------------------
# 2. 7개 decoder Top-1 예측
# ------------------------------------------------------------
meta_decoder_predictions = pd.DataFrame({
    "baseline": meta_eval_df["baseline_prediction"],
    "previous": pd.Series(e_previous_predictions).reset_index(drop=True),
    "asym": pd.Series(e_asym_predictions).reset_index(drop=True),
    "recursive": pd.Series(e_recursive_predictions).reset_index(drop=True),
    "full24": pd.Series(e_full24_predictions).reset_index(drop=True),
    "rf_conservative": (
        pd.Series(cr_rank_predictions["rf_conservative"])
        .reset_index(drop=True)
    ),
    "rf_balanced": (
        pd.Series(cr_rank_predictions["rf_balanced"])
        .reset_index(drop=True)
    ),
}).apply(lambda column: column.map(meta_order_text))

META_DECODERS = list(meta_decoder_predictions.columns)
META_RANKED_DECODERS = [
    "previous",
    "asym",
    "recursive",
    "full24",
    "rf_conservative",
    "rf_balanced",
]


# ------------------------------------------------------------
# 3. raw probability frame
# ------------------------------------------------------------
if "e_source_df" in globals():
    meta_source_df = e_source_df.copy()
elif "raw_df_200" in globals():
    meta_source_df = raw_df_200.copy()
else:
    raise RuntimeError("e_source_df 또는 raw_df_200이 필요합니다.")

meta_source_df["sample_id"] = meta_source_df["sample_id"].astype(str)
meta_raw_lookup = {
    str(row["sample_id"]): row
    for _, row in meta_source_df.iterrows()
}


# ------------------------------------------------------------
# 4. 후보표에 RF 점수·순위까지 추가
# ------------------------------------------------------------
meta_candidate_tables = {}
meta_decoder_margins = {
    name: np.zeros(len(meta_eval_df), dtype=np.float64)
    for name in META_RANKED_DECODERS
}

for position, eval_row in meta_eval_df.iterrows():
    sample_id = str(eval_row["sample_id"])
    table = e_candidate_tables[sample_id].copy()
    table["prediction"] = table["prediction"].map(meta_order_text)

    for rf_name, weights in CR_RANK_CONFIGS.items():
        table[f"{rf_name}_score"] = (
            weights["previous"] * table["previous_borda"]
            + weights["asym"] * table["asym_borda"]
            + weights["recursive"] * table["recursive_borda"]
            + weights["full24"] * table["full24_borda"]
        )
        # RF score 자체가 이미 0~1 Borda 결합값이다.
        table[f"{rf_name}_borda"] = table[f"{rf_name}_score"]

    for decoder in META_RANKED_DECODERS:
        score_column = f"{decoder}_score"
        if score_column not in table.columns:
            raise KeyError(
                f"{sample_id}: 후보표에 {score_column}이 없습니다."
            )

        table[f"{decoder}_rank"] = meta_rank_series(
            table[score_column].to_numpy(dtype=np.float64)
        )

        sorted_scores = np.sort(
            table[score_column].to_numpy(dtype=np.float64)
        )[::-1]
        meta_decoder_margins[decoder][position] = float(
            sorted_scores[0] - sorted_scores[1]
        )

    baseline = meta_decoder_predictions.loc[position, "baseline"]
    table["baseline_rank"] = np.where(
        table["prediction"] == baseline,
        1,
        25,
    )

    meta_candidate_tables[sample_id] = table.reset_index(drop=True)


print("현재 final 기준:", meta_current_final_source)
print(
    "현재 final exact:",
    int(meta_eval_df["current_final_exact"].sum()),
    "/",
    len(meta_eval_df),
    "=",
    float(meta_eval_df["current_final_exact"].mean()),
)
print("7개 decoder:", META_DECODERS)
print("후보표 sample 수:", len(meta_candidate_tables))


In [ ]:
# ============================================================
# META 1. 가장 먼저 볼 것
# - 7-decoder Top-1 oracle
# - decoder Top-K union oracle
# - 현재 final 오류 중 alternative 존재 여부
# - decoder 정확도와 상보성 행렬
# ============================================================

gold = meta_eval_df["gold_order"]
current = meta_eval_df["current_final_prediction"]
current_correct = current == gold

top1_gold_exists = (
    meta_decoder_predictions.eq(gold, axis=0).any(axis=1)
)

oracle_detail_rows = []
topk_masks = {1: [], 2: [], 3: []}

for position, eval_row in meta_eval_df.iterrows():
    sample_id = str(eval_row["sample_id"])
    gold_order = eval_row["gold_order"]
    table = meta_candidate_tables[sample_id]

    top1_predictions = set(
        meta_decoder_predictions.loc[position].tolist()
    )

    oracle_detail_rows.append({
        "sample_id": sample_id,
        "gold_order": gold_order,
        "current_final": current.iloc[position],
        "current_final_exact": int(current.iloc[position] == gold_order),
        "top1_oracle_hit": int(gold_order in top1_predictions),
        "correct_top1_decoders": ", ".join([
            decoder
            for decoder in META_DECODERS
            if meta_decoder_predictions.loc[position, decoder] == gold_order
        ]),
        "unique_top1_count": len(top1_predictions),
    })

    for top_k in [1, 2, 3]:
        # baseline은 후보 ranking이 없으므로 Top-1만 넣고,
        # 나머지 6개 decoder는 각자의 Top-K를 합친다.
        candidate_union = {
            meta_decoder_predictions.loc[position, "baseline"]
        }
        for decoder in META_RANKED_DECODERS:
            ranked = (
                table.sort_values(
                    f"{decoder}_score",
                    ascending=False,
                )
                .head(top_k)["prediction"]
                .tolist()
            )
            candidate_union.update(ranked)

        topk_masks[top_k].append(
            gold_order in candidate_union
        )

meta_oracle_detail_df = pd.DataFrame(oracle_detail_rows)

oracle_summary_rows = [{
    "metric": "current_final",
    "correct": int(current_correct.sum()),
    "exact": float(current_correct.mean()),
}, {
    "metric": "7_decoder_top1_oracle",
    "correct": int(top1_gold_exists.sum()),
    "exact": float(top1_gold_exists.mean()),
}, {
    "metric": "current_wrong_but_other_top1_is_gold",
    "correct": int(((~current_correct) & top1_gold_exists).sum()),
    "exact": float(((~current_correct) & top1_gold_exists).mean()),
}, {
    "metric": "current_wrong_and_no_decoder_top1_is_gold",
    "correct": int(((~current_correct) & (~top1_gold_exists)).sum()),
    "exact": float(((~current_correct) & (~top1_gold_exists)).mean()),
}]

for top_k in [1, 2, 3]:
    mask = np.asarray(topk_masks[top_k], dtype=bool)
    oracle_summary_rows.append({
        "metric": f"baseline_top1_plus_6decoder_top{top_k}_union_oracle",
        "correct": int(mask.sum()),
        "exact": float(mask.mean()),
    })

# 개별 Top-3 recall
for decoder in ["asym", "rf_conservative", "rf_balanced"]:
    hit = []
    for position, eval_row in meta_eval_df.iterrows():
        sample_id = str(eval_row["sample_id"])
        table = meta_candidate_tables[sample_id]
        top3 = set(
            table.sort_values(
                f"{decoder}_score",
                ascending=False,
            ).head(3)["prediction"]
        )
        hit.append(eval_row["gold_order"] in top3)

    oracle_summary_rows.append({
        "metric": f"{decoder}_top3_contains_gold",
        "correct": int(np.sum(hit)),
        "exact": float(np.mean(hit)),
    })

meta_oracle_summary_df = pd.DataFrame(oracle_summary_rows)

print("Oracle 요약")
display(meta_oracle_summary_df)


# ------------------------------------------------------------
# Decoder 개별 정확도
# ------------------------------------------------------------
decoder_accuracy_rows = []
for decoder in META_DECODERS:
    exact = (
        meta_decoder_predictions[decoder]
        == gold
    )
    decoder_accuracy_rows.append({
        "decoder": decoder,
        "correct": int(exact.sum()),
        "exact": float(exact.mean()),
        "unique_correct_vs_current_final": int(
            (
                exact
                & (~current_correct)
            ).sum()
        ),
    })

meta_decoder_accuracy_df = (
    pd.DataFrame(decoder_accuracy_rows)
    .sort_values(["exact", "unique_correct_vs_current_final"], ascending=False)
    .reset_index(drop=True)
)

print("\nDecoder 정확도")
display(meta_decoder_accuracy_df)


# ------------------------------------------------------------
# Decoder A/B 상보성
# ------------------------------------------------------------
complement_rows = []
for decoder_a, decoder_b in itertools.combinations(META_DECODERS, 2):
    a_correct = meta_decoder_predictions[decoder_a] == gold
    b_correct = meta_decoder_predictions[decoder_b] == gold

    complement_rows.append({
        "decoder_a": decoder_a,
        "decoder_b": decoder_b,
        "a_only": int((a_correct & ~b_correct).sum()),
        "b_only": int((~a_correct & b_correct).sum()),
        "both_correct": int((a_correct & b_correct).sum()),
        "both_wrong": int((~a_correct & ~b_correct).sum()),
        "pair_oracle_correct": int((a_correct | b_correct).sum()),
        "pair_oracle_exact": float((a_correct | b_correct).mean()),
    })

meta_decoder_complementarity_df = (
    pd.DataFrame(complement_rows)
    .sort_values(
        ["pair_oracle_exact", "both_wrong", "a_only", "b_only"],
        ascending=[False, True, False, False],
    )
    .reset_index(drop=True)
)

print("\nDecoder 상보성 상위 조합")
display(meta_decoder_complementarity_df.head(25))


# 현재 final이 틀렸지만 다른 Top-1이 정답인 실제 샘플
print("\n현재 final 오류 중 Top-1 alternative가 존재하는 샘플")
display(
    meta_oracle_detail_df[
        (meta_oracle_detail_df["current_final_exact"] == 0)
        & (meta_oracle_detail_df["top1_oracle_hit"] == 1)
    ].sort_values(
        ["unique_top1_count", "sample_id"],
        ascending=[True, True],
    )
)

if "OUTPUT_RUN_DIR" in globals():
    meta_oracle_summary_df.to_csv(
        OUTPUT_RUN_DIR / "meta_oracle_summary.csv",
        index=False,
    )
    meta_decoder_accuracy_df.to_csv(
        OUTPUT_RUN_DIR / "meta_decoder_accuracy.csv",
        index=False,
    )
    meta_decoder_complementarity_df.to_csv(
        OUTPUT_RUN_DIR / "meta_decoder_complementarity.csv",
        index=False,
    )
    meta_oracle_detail_df.to_csv(
        OUTPUT_RUN_DIR / "meta_oracle_detail.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 2. Endpoint disagreement + decoder distance
# - 독립/endpoint/pairwise/fused branch endpoint
# - 7개 decoder endpoint
# - 누가 누구와 불일치하는지
# - Kendall distance와 분열 정도
# ============================================================

def meta_int_prediction(value, default=None):
    try:
        result = int(float(value))
    except (TypeError, ValueError):
        return default
    return result if result in META_FRAMES else default


def meta_majority(values, default):
    values = [value for value in values if value in META_FRAMES]
    if not values:
        return default

    counts = Counter(values)
    max_count = max(counts.values())
    winners = sorted([
        value
        for value, count in counts.items()
        if count == max_count
    ])
    return default if default in winners else winners[0]


def meta_vote_pattern(values):
    counts = Counter(values)
    return ":".join(
        map(str, sorted(counts.values(), reverse=True))
    )


endpoint_rows = []

for position, eval_row in meta_eval_df.iterrows():
    sample_id = str(eval_row["sample_id"])
    raw_row = meta_raw_lookup[sample_id]

    decoder_orders = {
        decoder: meta_order_tuple(
            meta_decoder_predictions.loc[position, decoder]
        )
        for decoder in META_DECODERS
    }

    decoder_first = {
        decoder: order[0]
        for decoder, order in decoder_orders.items()
    }
    decoder_last = {
        decoder: order[-1]
        for decoder, order in decoder_orders.items()
    }

    current_tuple = meta_order_tuple(
        eval_row["current_final_prediction"]
    )

    # raw branch endpoint.
    # order_pos1/4 = full24 marginal,
    # endpoint = 독립 endpoint branch,
    # pairwise = pairwise endpoint branch,
    # base = 기존 10:30:60 fused endpoint.
    branch_first = {
        "order_marginal": meta_int_prediction(
            raw_row.get("order_pos1_prediction")
        ),
        "endpoint": meta_int_prediction(
            raw_row.get("endpoint_first_prediction")
        ),
        "pairwise": meta_int_prediction(
            raw_row.get("pairwise_first_prediction")
        ),
        "baseline_fused": meta_int_prediction(
            raw_row.get("base_first_prediction")
        ),
    }
    branch_last = {
        "order_marginal": meta_int_prediction(
            raw_row.get("order_pos4_prediction")
        ),
        "endpoint": meta_int_prediction(
            raw_row.get("endpoint_last_prediction")
        ),
        "pairwise": meta_int_prediction(
            raw_row.get("pairwise_last_prediction")
        ),
        "baseline_fused": meta_int_prediction(
            raw_row.get("base_last_prediction")
        ),
    }

    all_first_sources = {
        **{f"branch_{k}": v for k, v in branch_first.items()},
        **{f"decoder_{k}": v for k, v in decoder_first.items()},
    }
    all_last_sources = {
        **{f"branch_{k}": v for k, v in branch_last.items()},
        **{f"decoder_{k}": v for k, v in decoder_last.items()},
    }

    valid_first = [
        value for value in all_first_sources.values()
        if value in META_FRAMES
    ]
    valid_last = [
        value for value in all_last_sources.values()
        if value in META_FRAMES
    ]

    decoder_first_values = list(decoder_first.values())
    decoder_last_values = list(decoder_last.values())

    majority_first = meta_majority(
        decoder_first_values,
        default=current_tuple[0],
    )
    majority_last = meta_majority(
        decoder_last_values,
        default=current_tuple[-1],
    )

    first_counts = Counter(decoder_first_values)
    last_counts = Counter(decoder_last_values)

    order_distances = [
        meta_kendall_distance(
            decoder_orders[a],
            decoder_orders[b],
        )
        for a, b in itertools.combinations(META_DECODERS, 2)
    ]

    full_order_counts = Counter(
        meta_decoder_predictions.loc[position].tolist()
    )
    max_full_votes = max(full_order_counts.values())
    full_winners = sorted([
        prediction
        for prediction, count in full_order_counts.items()
        if count == max_full_votes
    ])
    majority_order = (
        eval_row["current_final_prediction"]
        if eval_row["current_final_prediction"] in full_winners
        else full_winners[0]
    )

    endpoint_rows.append({
        "sample_id": sample_id,
        "gold_order": eval_row["gold_order"],
        "current_final_prediction": eval_row["current_final_prediction"],
        "current_final_exact": eval_row["current_final_exact"],

        "decoder_unique_prediction_count": len(full_order_counts),
        "decoder_majority_prediction": majority_order,
        "decoder_majority_vote_count": max_full_votes,
        "current_vs_decoder_majority_kendall": meta_kendall_distance(
            eval_row["current_final_prediction"],
            majority_order,
        ),

        "decoder_first_unique_count": len(first_counts),
        "decoder_last_unique_count": len(last_counts),
        "decoder_first_majority": majority_first,
        "decoder_last_majority": majority_last,
        "decoder_first_majority_share": max(first_counts.values()) / 7.0,
        "decoder_last_majority_share": max(last_counts.values()) / 7.0,
        "decoder_first_vote_pattern": meta_vote_pattern(decoder_first_values),
        "decoder_last_vote_pattern": meta_vote_pattern(decoder_last_values),

        "all_source_first_unique_count": len(set(valid_first)),
        "all_source_last_unique_count": len(set(valid_last)),
        "all_source_first_majority_share": (
            max(Counter(valid_first).values()) / len(valid_first)
            if valid_first else np.nan
        ),
        "all_source_last_majority_share": (
            max(Counter(valid_last).values()) / len(valid_last)
            if valid_last else np.nan
        ),

        "baseline_first_is_minority": int(
            first_counts[decoder_first["baseline"]]
            < max(first_counts.values())
        ),
        "baseline_last_is_minority": int(
            last_counts[decoder_last["baseline"]]
            < max(last_counts.values())
        ),
        "baseline_both_endpoints_minority": int(
            (
                first_counts[decoder_first["baseline"]]
                < max(first_counts.values())
            )
            and (
                last_counts[decoder_last["baseline"]]
                < max(last_counts.values())
            )
        ),

        "baseline_only_diff_first": int(
            len(set([
                decoder_first[name]
                for name in META_DECODERS
                if name != "baseline"
            ])) == 1
            and decoder_first["baseline"]
            != decoder_first["asym"]
        ),
        "baseline_only_diff_last": int(
            len(set([
                decoder_last[name]
                for name in META_DECODERS
                if name != "baseline"
            ])) == 1
            and decoder_last["baseline"]
            != decoder_last["asym"]
        ),

        "asym_full24_rf_cons_first_agree_baseline_diff": int(
            decoder_first["asym"]
            == decoder_first["full24"]
            == decoder_first["rf_conservative"]
            != decoder_first["baseline"]
        ),
        "asym_full24_rf_cons_last_agree_baseline_diff": int(
            decoder_last["asym"]
            == decoder_last["full24"]
            == decoder_last["rf_conservative"]
            != decoder_last["baseline"]
        ),

        "decoder_distance_mean": float(np.mean(order_distances)),
        "decoder_distance_max": int(np.max(order_distances)),
        "decoder_distance_zero_pair_rate": float(
            np.mean(np.asarray(order_distances) == 0)
        ),

        "branch_order_first": branch_first["order_marginal"],
        "branch_endpoint_first": branch_first["endpoint"],
        "branch_pairwise_first": branch_first["pairwise"],
        "branch_fused_first": branch_first["baseline_fused"],
        "branch_order_last": branch_last["order_marginal"],
        "branch_endpoint_last": branch_last["endpoint"],
        "branch_pairwise_last": branch_last["pairwise"],
        "branch_fused_last": branch_last["baseline_fused"],
    })

meta_endpoint_df = pd.DataFrame(endpoint_rows)


# ------------------------------------------------------------
# Bucket별 현재 final 정확도
# ------------------------------------------------------------
def meta_bucket_accuracy(frame, column):
    return (
        frame.groupby(column, dropna=False)
        .agg(
            sample_count=("sample_id", "size"),
            current_final_correct=("current_final_exact", "sum"),
            current_final_exact=("current_final_exact", "mean"),
        )
        .reset_index()
        .sort_values(column)
    )


bucket_columns = [
    "decoder_unique_prediction_count",
    "decoder_first_unique_count",
    "decoder_last_unique_count",
    "decoder_first_vote_pattern",
    "decoder_last_vote_pattern",
    "baseline_first_is_minority",
    "baseline_last_is_minority",
    "baseline_both_endpoints_minority",
    "decoder_distance_max",
]

meta_bucket_tables = {}
for column in bucket_columns:
    meta_bucket_tables[column] = meta_bucket_accuracy(
        meta_endpoint_df,
        column,
    )
    print(f"\nBucket: {column}")
    display(meta_bucket_tables[column])


# 현재 final 오류를 잘 농축하는 신호를 한 표로
signal_columns = [
    "baseline_first_is_minority",
    "baseline_last_is_minority",
    "baseline_both_endpoints_minority",
    "baseline_only_diff_first",
    "baseline_only_diff_last",
    "asym_full24_rf_cons_first_agree_baseline_diff",
    "asym_full24_rf_cons_last_agree_baseline_diff",
]

signal_rows = []
for signal in signal_columns:
    mask = meta_endpoint_df[signal].astype(bool)
    signal_rows.append({
        "signal": signal,
        "trigger_count": int(mask.sum()),
        "current_final_correct_in_trigger": int(
            meta_endpoint_df.loc[mask, "current_final_exact"].sum()
        ),
        "current_final_accuracy_in_trigger": float(
            meta_endpoint_df.loc[mask, "current_final_exact"].mean()
        ) if mask.any() else np.nan,
        "current_final_error_recall": float(
            (
                mask
                & (meta_endpoint_df["current_final_exact"] == 0)
            ).sum()
            /
            max(
                1,
                (meta_endpoint_df["current_final_exact"] == 0).sum(),
            )
        ),
    })

meta_endpoint_signal_df = (
    pd.DataFrame(signal_rows)
    .sort_values(
        ["current_final_accuracy_in_trigger", "trigger_count"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

print("\nEndpoint disagreement 신호")
display(meta_endpoint_signal_df)

if "OUTPUT_RUN_DIR" in globals():
    meta_endpoint_df.to_csv(
        OUTPUT_RUN_DIR / "meta_endpoint_distance_features.csv",
        index=False,
    )
    meta_endpoint_signal_df.to_csv(
        OUTPUT_RUN_DIR / "meta_endpoint_signal_summary.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 3. 24개 후보 support profile
# - decoder rank / Top-K 포함 수
# - endpoint / conditional / pairwise support
# - pairwise cycle
# - current final과 best alternative 비교
# ============================================================

def meta_probability(row, column, default=META_EPS):
    value = meta_row_value(row, column, default=default)
    if not np.isfinite(value):
        return float(default)
    return float(np.clip(value, META_EPS, 1.0))


def meta_pair_before_probability(row, earlier, later):
    direct = meta_probability(
        row,
        f"p_pair_{earlier}_before_{later}",
        default=np.nan,
    )
    reverse = meta_probability(
        row,
        f"p_pair_{later}_before_{earlier}",
        default=np.nan,
    )

    if np.isfinite(direct) and np.isfinite(reverse):
        total = direct + reverse
        return direct / total if total > 0 else 0.5
    if np.isfinite(direct):
        return direct
    if np.isfinite(reverse):
        return 1.0 - reverse
    return 0.5


def meta_pairwise_cycle_count(row):
    count = 0
    for triple in itertools.combinations(META_FRAMES, 3):
        a, b, c = triple
        # 각 pair의 winner 방향으로 그래프를 만든다.
        before = {}
        for x, y in itertools.combinations(triple, 2):
            p = meta_pair_before_probability(row, x, y)
            if p >= 0.5:
                before[(x, y)] = True
            else:
                before[(y, x)] = True

        if (
            ((a, b) in before and (b, c) in before and (c, a) in before)
            or
            ((a, c) in before and (c, b) in before and (b, a) in before)
        ):
            count += 1
    return count


def meta_geometric_mean(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=np.float64),
        META_EPS,
        1.0,
    )
    return float(np.exp(np.mean(np.log(probabilities))))


candidate_profile_frames = []
sample_feature_rows = []
repair_seed_rows = []

for position, eval_row in meta_eval_df.iterrows():
    sample_id = str(eval_row["sample_id"])
    raw_row = meta_raw_lookup[sample_id]
    table = meta_candidate_tables[sample_id].copy()

    decoder_top1 = set(
        meta_decoder_predictions.loc[position].tolist()
    )
    decoder_topk = {}
    for top_k in [2, 3]:
        decoder_topk[top_k] = {
            decoder: set(
                table.sort_values(
                    f"{decoder}_score",
                    ascending=False,
                )
                .head(top_k)["prediction"]
            )
            for decoder in META_RANKED_DECODERS
        }

    cycle_count = meta_pairwise_cycle_count(raw_row)

    profile_rows = []
    for _, candidate_row in table.iterrows():
        prediction = meta_order_text(candidate_row["prediction"])
        first, second, third, last = meta_order_tuple(prediction)
        suffix = f"{second}{third}{last}"
        prefix = f"{first}{second}{third}"

        ranks = {
            decoder: int(candidate_row[f"{decoder}_rank"])
            for decoder in META_RANKED_DECODERS
        }
        bordas = {
            decoder: float(candidate_row[f"{decoder}_borda"])
            for decoder in META_RANKED_DECODERS
        }

        top1_vote_count = int(
            sum(
                meta_decoder_predictions.loc[position, decoder] == prediction
                for decoder in META_DECODERS
            )
        )
        top2_inclusion_count = int(
            sum(
                prediction in decoder_topk[2][decoder]
                for decoder in META_RANKED_DECODERS
            )
            + int(
                meta_decoder_predictions.loc[position, "baseline"]
                == prediction
            )
        )
        top3_inclusion_count = int(
            sum(
                prediction in decoder_topk[3][decoder]
                for decoder in META_RANKED_DECODERS
            )
            + int(
                meta_decoder_predictions.loc[position, "baseline"]
                == prediction
            )
        )

        first_source_values = [
            meta_endpoint_df.loc[position, "branch_order_first"],
            meta_endpoint_df.loc[position, "branch_endpoint_first"],
            meta_endpoint_df.loc[position, "branch_pairwise_first"],
            meta_endpoint_df.loc[position, "branch_fused_first"],
        ] + [
            meta_order_tuple(
                meta_decoder_predictions.loc[position, decoder]
            )[0]
            for decoder in META_DECODERS
        ]

        last_source_values = [
            meta_endpoint_df.loc[position, "branch_order_last"],
            meta_endpoint_df.loc[position, "branch_endpoint_last"],
            meta_endpoint_df.loc[position, "branch_pairwise_last"],
            meta_endpoint_df.loc[position, "branch_fused_last"],
        ] + [
            meta_order_tuple(
                meta_decoder_predictions.loc[position, decoder]
            )[-1]
            for decoder in META_DECODERS
        ]

        valid_first_sources = [
            value for value in first_source_values
            if value in META_FRAMES
        ]
        valid_last_sources = [
            value for value in last_source_values
            if value in META_FRAMES
        ]

        first_endpoint_agreement = (
            np.mean([
                value == first
                for value in valid_first_sources
            ])
            if valid_first_sources else 0.0
        )
        last_endpoint_agreement = (
            np.mean([
                value == last
                for value in valid_last_sources
            ])
            if valid_last_sources else 0.0
        )

        conditional_forward = meta_probability(
            raw_row,
            f"p_cond_fixed_first_{first}_suffix_{suffix}",
        )
        conditional_backward = meta_probability(
            raw_row,
            f"p_cond_fixed_last_{last}_prefix_{prefix}",
        )
        conditional_support = meta_geometric_mean([
            conditional_forward,
            conditional_backward,
        ])

        pair_probabilities = []
        adjacent_pair_probabilities = []
        nonadjacent_pair_probabilities = []

        for i in range(4):
            for j in range(i + 1, 4):
                probability = meta_pair_before_probability(
                    raw_row,
                    (first, second, third, last)[i],
                    (first, second, third, last)[j],
                )
                pair_probabilities.append(probability)
                if j == i + 1:
                    adjacent_pair_probabilities.append(probability)
                else:
                    nonadjacent_pair_probabilities.append(probability)

        pairwise_mean = float(np.mean(pair_probabilities))
        pairwise_min = float(np.min(pair_probabilities))
        pairwise_agree_count = int(
            np.sum(np.asarray(pair_probabilities) >= 0.5)
        )

        avg_rank = float(np.mean(list(ranks.values())))
        worst_rank = int(np.max(list(ranks.values())))
        best_rank = int(np.min(list(ranks.values())))
        avg_borda = float(np.mean(list(bordas.values())))

        # 고정 최적식이 아니라 단순 진단용 heuristic.
        heuristic_support = (
            2.0 * top1_vote_count
            + 1.0 * top3_inclusion_count
            + 2.0 * avg_borda
            + 0.75 * first_endpoint_agreement
            + 0.75 * last_endpoint_agreement
            + 1.0 * conditional_support
            + 1.0 * pairwise_mean
            - 0.5 * ((worst_rank - 1) / 23.0)
        )

        profile_rows.append({
            "sample_id": sample_id,
            "candidate_prediction": prediction,
            "candidate_is_gold": int(
                prediction == eval_row["gold_order"]
            ),
            "is_current_final": int(
                prediction == eval_row["current_final_prediction"]
            ),
            "kendall_to_current_final": meta_kendall_distance(
                prediction,
                eval_row["current_final_prediction"],
            ),

            "top1_vote_count": top1_vote_count,
            "top2_inclusion_count": top2_inclusion_count,
            "top3_inclusion_count": top3_inclusion_count,

            **{
                f"{decoder}_rank": ranks[decoder]
                for decoder in META_RANKED_DECODERS
            },
            **{
                f"{decoder}_borda": bordas[decoder]
                for decoder in META_RANKED_DECODERS
            },

            "avg_rank": avg_rank,
            "worst_rank": worst_rank,
            "best_rank": best_rank,
            "avg_borda": avg_borda,

            "first_endpoint_agreement": first_endpoint_agreement,
            "last_endpoint_agreement": last_endpoint_agreement,
            "endpoint_agreement_mean": float(
                0.5 * (
                    first_endpoint_agreement
                    + last_endpoint_agreement
                )
            ),

            "conditional_forward_probability": conditional_forward,
            "conditional_backward_probability": conditional_backward,
            "conditional_support": conditional_support,

            "pairwise_mean_probability": pairwise_mean,
            "pairwise_min_probability": pairwise_min,
            "pairwise_agree_count": pairwise_agree_count,
            "adjacent_pairwise_mean": float(
                np.mean(adjacent_pair_probabilities)
            ),
            "nonadjacent_pairwise_mean": float(
                np.mean(nonadjacent_pair_probabilities)
            ),
            "pairwise_cycle_count": cycle_count,

            "heuristic_support": heuristic_support,
        })

    profile_df = pd.DataFrame(profile_rows)
    candidate_profile_frames.append(profile_df)

    current_profile = profile_df[
        profile_df["is_current_final"] == 1
    ].iloc[0]

    best_support_profile = (
        profile_df
        .sort_values(
            [
                "heuristic_support",
                "top1_vote_count",
                "avg_rank",
            ],
            ascending=[False, False, True],
        )
        .iloc[0]
    )

    alternative_profile = (
        profile_df[
            profile_df["is_current_final"] == 0
        ]
        .sort_values(
            [
                "heuristic_support",
                "top1_vote_count",
                "avg_rank",
            ],
            ascending=[False, False, True],
        )
        .iloc[0]
    )

    sample_features = {
        "sample_id": sample_id,
        "gold_order": eval_row["gold_order"],
        "current_final_prediction": eval_row["current_final_prediction"],
        "current_final_exact": eval_row["current_final_exact"],
        "y_error": int(eval_row["current_final_exact"] == 0),
        "full_order_normalized_entropy": float(
            eval_row["full_order_normalized_entropy"]
        ),

        "decoder_unique_prediction_count": int(
            meta_endpoint_df.loc[
                position,
                "decoder_unique_prediction_count",
            ]
        ),
        "decoder_first_unique_count": int(
            meta_endpoint_df.loc[
                position,
                "decoder_first_unique_count",
            ]
        ),
        "decoder_last_unique_count": int(
            meta_endpoint_df.loc[
                position,
                "decoder_last_unique_count",
            ]
        ),
        "decoder_first_majority_share": float(
            meta_endpoint_df.loc[
                position,
                "decoder_first_majority_share",
            ]
        ),
        "decoder_last_majority_share": float(
            meta_endpoint_df.loc[
                position,
                "decoder_last_majority_share",
            ]
        ),
        "decoder_distance_mean": float(
            meta_endpoint_df.loc[
                position,
                "decoder_distance_mean",
            ]
        ),
        "decoder_distance_max": int(
            meta_endpoint_df.loc[
                position,
                "decoder_distance_max",
            ]
        ),
        "current_vs_decoder_majority_kendall": int(
            meta_endpoint_df.loc[
                position,
                "current_vs_decoder_majority_kendall",
            ]
        ),

        "pairwise_cycle_count": cycle_count,

        "current_avg_rank": float(current_profile["avg_rank"]),
        "current_worst_rank": int(current_profile["worst_rank"]),
        "current_best_rank": int(current_profile["best_rank"]),
        "current_top1_vote_count": int(
            current_profile["top1_vote_count"]
        ),
        "current_top3_inclusion_count": int(
            current_profile["top3_inclusion_count"]
        ),
        "current_endpoint_agreement": float(
            current_profile["endpoint_agreement_mean"]
        ),
        "current_conditional_support": float(
            current_profile["conditional_support"]
        ),
        "current_pairwise_mean": float(
            current_profile["pairwise_mean_probability"]
        ),
        "current_pairwise_min": float(
            current_profile["pairwise_min_probability"]
        ),
        "current_pairwise_agree_count": int(
            current_profile["pairwise_agree_count"]
        ),
        "current_heuristic_support": float(
            current_profile["heuristic_support"]
        ),

        "best_support_prediction": str(
            best_support_profile["candidate_prediction"]
        ),
        "best_alternative_prediction": str(
            alternative_profile["candidate_prediction"]
        ),
        "best_alternative_top1_vote_count": int(
            alternative_profile["top1_vote_count"]
        ),
        "best_alternative_top3_inclusion_count": int(
            alternative_profile["top3_inclusion_count"]
        ),
        "best_alternative_avg_rank": float(
            alternative_profile["avg_rank"]
        ),
        "best_alternative_worst_rank": int(
            alternative_profile["worst_rank"]
        ),
        "best_alternative_kendall": int(
            alternative_profile["kendall_to_current_final"]
        ),
        "best_alternative_heuristic_support": float(
            alternative_profile["heuristic_support"]
        ),
        "alternative_minus_current_support": float(
            alternative_profile["heuristic_support"]
            - current_profile["heuristic_support"]
        ),
        "current_minus_alternative_avg_rank": float(
            current_profile["avg_rank"]
            - alternative_profile["avg_rank"]
        ),
    }

    for decoder in META_RANKED_DECODERS:
        sample_features[f"{decoder}_margin"] = float(
            meta_decoder_margins[decoder][position]
        )
        sample_features[f"current_{decoder}_rank"] = int(
            current_profile[f"{decoder}_rank"]
        )
        sample_features[f"alternative_{decoder}_rank"] = int(
            alternative_profile[f"{decoder}_rank"]
        )

    sample_feature_rows.append(sample_features)

meta_candidate_profile_df = pd.concat(
    candidate_profile_frames,
    ignore_index=True,
)
meta_feature_df = pd.DataFrame(sample_feature_rows)

print("Candidate profile shape:", meta_candidate_profile_df.shape)
print("Sample feature shape:", meta_feature_df.shape)

print("\n현재 final이 틀린 샘플 중 support gap이 큰 순서")
display(
    meta_feature_df[
        meta_feature_df["current_final_exact"] == 0
    ].sort_values(
        "alternative_minus_current_support",
        ascending=False,
    ).head(40)
)

print("\n현재 final이 맞지만 alternative가 강하게 보이는 위험 샘플")
display(
    meta_feature_df[
        meta_feature_df["current_final_exact"] == 1
    ].sort_values(
        "alternative_minus_current_support",
        ascending=False,
    ).head(30)
)

if "OUTPUT_RUN_DIR" in globals():
    meta_candidate_profile_df.to_csv(
        OUTPUT_RUN_DIR / "meta_candidate_support_profiles.csv",
        index=False,
    )
    meta_feature_df.to_csv(
        OUTPUT_RUN_DIR / "meta_sample_error_features.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 4. 구조적 repair 후보 + 수동 gate sweep
# A. endpoint-majority repair
# B. first-only / last-only repair
# C. middle-only repair
# D. adjacent-swap repair
# E. decoder-consensus / average-rank / minimax-rank
#
# 모든 비교 기준은 baseline이 아니라 "현재 final".
# ============================================================

def meta_pick_profile(profile_df, mask=None, sort_mode="support"):
    subset = profile_df if mask is None else profile_df[mask]
    if subset.empty:
        return None

    if sort_mode == "support":
        sort_columns = [
            "heuristic_support",
            "top1_vote_count",
            "avg_rank",
        ]
        ascending = [False, False, True]
    elif sort_mode == "average_rank":
        sort_columns = [
            "avg_rank",
            "worst_rank",
            "heuristic_support",
        ]
        ascending = [True, True, False]
    elif sort_mode == "minimax_rank":
        sort_columns = [
            "worst_rank",
            "avg_rank",
            "heuristic_support",
        ]
        ascending = [True, True, False]
    else:
        raise ValueError(sort_mode)

    return (
        subset.sort_values(
            sort_columns,
            ascending=ascending,
        )
        .iloc[0]
    )


repair_rows = []

for position, feature_row in meta_feature_df.iterrows():
    sample_id = str(feature_row["sample_id"])
    current_prediction = feature_row["current_final_prediction"]
    current_order = meta_order_tuple(current_prediction)
    profile_df = meta_candidate_profile_df[
        meta_candidate_profile_df["sample_id"] == sample_id
    ].copy()

    majority_first = int(
        meta_endpoint_df.loc[position, "decoder_first_majority"]
    )
    majority_last = int(
        meta_endpoint_df.loc[position, "decoder_last_majority"]
    )

    profile_orders = profile_df[
        "candidate_prediction"
    ].map(meta_order_tuple)

    endpoint_mask = profile_orders.map(
        lambda order: (
            order[0] == majority_first
            and order[-1] == majority_last
        )
    )
    first_only_mask = profile_orders.map(
        lambda order: order[0] == majority_first
    )
    last_only_mask = profile_orders.map(
        lambda order: order[-1] == majority_last
    )
    middle_only_mask = profile_orders.map(
        lambda order: (
            order[0] == current_order[0]
            and order[-1] == current_order[-1]
        )
    )

    adjacent_candidates = {current_prediction}
    for swap_index in range(3):
        swapped = list(current_order)
        swapped[swap_index], swapped[swap_index + 1] = (
            swapped[swap_index + 1],
            swapped[swap_index],
        )
        adjacent_candidates.add(meta_order_text(swapped))

    adjacent_mask = profile_df[
        "candidate_prediction"
    ].isin(adjacent_candidates)

    picks = {
        "decoder_consensus_support": meta_pick_profile(
            profile_df,
            sort_mode="support",
        ),
        "average_rank": meta_pick_profile(
            profile_df,
            sort_mode="average_rank",
        ),
        "minimax_rank": meta_pick_profile(
            profile_df,
            sort_mode="minimax_rank",
        ),
        "endpoint_majority": meta_pick_profile(
            profile_df,
            mask=endpoint_mask,
            sort_mode="support",
        ),
        "first_majority_only": meta_pick_profile(
            profile_df,
            mask=first_only_mask,
            sort_mode="support",
        ),
        "last_majority_only": meta_pick_profile(
            profile_df,
            mask=last_only_mask,
            sort_mode="support",
        ),
        "middle_only": meta_pick_profile(
            profile_df,
            mask=middle_only_mask,
            sort_mode="support",
        ),
        "adjacent_swap": meta_pick_profile(
            profile_df,
            mask=adjacent_mask,
            sort_mode="support",
        ),
    }

    current_profile = profile_df[
        profile_df["candidate_prediction"] == current_prediction
    ].iloc[0]

    row = {
        "sample_id": sample_id,
        "gold_order": feature_row["gold_order"],
        "current_final_prediction": current_prediction,
        "current_final_exact": int(
            current_prediction == feature_row["gold_order"]
        ),
        "pairwise_cycle_count": int(
            feature_row["pairwise_cycle_count"]
        ),
    }

    for repair_name, picked in picks.items():
        if picked is None:
            picked = current_profile

        row[f"{repair_name}_prediction"] = str(
            picked["candidate_prediction"]
        )
        row[f"{repair_name}_support_gain"] = float(
            picked["heuristic_support"]
            - current_profile["heuristic_support"]
        )
        row[f"{repair_name}_top1_votes"] = int(
            picked["top1_vote_count"]
        )
        row[f"{repair_name}_top3_count"] = int(
            picked["top3_inclusion_count"]
        )
        row[f"{repair_name}_avg_rank_gain"] = float(
            current_profile["avg_rank"]
            - picked["avg_rank"]
        )
        row[f"{repair_name}_kendall"] = int(
            picked["kendall_to_current_final"]
        )

    repair_rows.append(row)

meta_repair_df = pd.DataFrame(repair_rows)


def meta_evaluate_vs_current(predictions, name):
    predictions = pd.Series(predictions).reset_index(drop=True).map(
        meta_order_text
    )
    gold = meta_repair_df["gold_order"]
    current = meta_repair_df["current_final_prediction"]

    current_correct = current == gold
    new_correct = predictions == gold
    changed = predictions != current

    wrong_to_right = changed & (~current_correct) & new_correct
    right_to_wrong = changed & current_correct & (~new_correct)

    return {
        "strategy": name,
        "exact": float(new_correct.mean()),
        "correct": int(new_correct.sum()),
        "changed_from_current": int(changed.sum()),
        "wrong_to_right": int(wrong_to_right.sum()),
        "right_to_wrong": int(right_to_wrong.sum()),
        "net": int(wrong_to_right.sum() - right_to_wrong.sum()),
    }


# ------------------------------------------------------------
# 1. Gate 없이 repair 후보 자체의 oracle-like 성질 확인
# ------------------------------------------------------------
repair_names = [
    "decoder_consensus_support",
    "average_rank",
    "minimax_rank",
    "endpoint_majority",
    "first_majority_only",
    "last_majority_only",
    "middle_only",
    "adjacent_swap",
]

unconditional_rows = [
    meta_evaluate_vs_current(
        meta_repair_df["current_final_prediction"],
        "current_final",
    )
]

for repair_name in repair_names:
    unconditional_rows.append(
        meta_evaluate_vs_current(
            meta_repair_df[f"{repair_name}_prediction"],
            repair_name,
        )
    )

meta_repair_unconditional_df = (
    pd.DataFrame(unconditional_rows)
    .sort_values(
        ["exact", "right_to_wrong", "wrong_to_right"],
        ascending=[False, True, False],
    )
    .reset_index(drop=True)
)

print("Repair 후보를 무조건 적용했을 때")
display(meta_repair_unconditional_df)


# ------------------------------------------------------------
# 2. 보수적 gate sweep
#
# 조건:
# - support gain 하한
# - alternative Top-1 vote 수
# - current와의 Kendall 거리
# - pairwise cycle 수
# ------------------------------------------------------------
gate_rows = []
meta_repair_prediction_cache = {}

for repair_name in repair_names:
    gain_values = (
        meta_repair_df[f"{repair_name}_support_gain"]
        .to_numpy(dtype=np.float64)
    )
    positive_gains = gain_values[gain_values > 0]

    if len(positive_gains) == 0:
        gain_thresholds = [0.0]
    else:
        gain_thresholds = sorted(set([
            0.0,
            float(np.quantile(positive_gains, 0.25)),
            float(np.quantile(positive_gains, 0.50)),
            float(np.quantile(positive_gains, 0.75)),
        ]))

    for gain_threshold in gain_thresholds:
        for min_top1_votes in [1, 2, 3]:
            for max_kendall in [1, 2, 6]:
                for max_cycle in [0, 4]:
                    use_repair = (
                        (
                            meta_repair_df[
                                f"{repair_name}_support_gain"
                            ]
                            >= gain_threshold
                        )
                        & (
                            meta_repair_df[
                                f"{repair_name}_top1_votes"
                            ]
                            >= min_top1_votes
                        )
                        & (
                            meta_repair_df[
                                f"{repair_name}_kendall"
                            ]
                            <= max_kendall
                        )
                        & (
                            meta_repair_df[
                                "pairwise_cycle_count"
                            ]
                            <= max_cycle
                        )
                        & (
                            meta_repair_df[
                                f"{repair_name}_prediction"
                            ]
                            != meta_repair_df[
                                "current_final_prediction"
                            ]
                        )
                    )

                    predictions = pd.Series(
                        np.where(
                            use_repair,
                            meta_repair_df[
                                f"{repair_name}_prediction"
                            ],
                            meta_repair_df[
                                "current_final_prediction"
                            ],
                        )
                    )

                    name = (
                        f"{repair_name}"
                        f"__gain{gain_threshold:.4f}"
                        f"__votes{min_top1_votes}"
                        f"__kendall{max_kendall}"
                        f"__cycle{max_cycle}"
                    )

                    result = meta_evaluate_vs_current(
                        predictions,
                        name,
                    )
                    result.update({
                        "repair": repair_name,
                        "gain_threshold": gain_threshold,
                        "min_top1_votes": min_top1_votes,
                        "max_kendall": max_kendall,
                        "max_cycle": max_cycle,
                        "gate_count": int(use_repair.sum()),
                    })
                    gate_rows.append(result)
                    meta_repair_prediction_cache[name] = predictions

meta_repair_gate_sweep_df = (
    pd.DataFrame(gate_rows)
    .sort_values(
        [
            "exact",
            "right_to_wrong",
            "wrong_to_right",
            "changed_from_current",
        ],
        ascending=[False, True, False, True],
    )
    .reset_index(drop=True)
)

print("\n보수적 repair gate sweep 상위")
display(meta_repair_gate_sweep_df.head(50))


# ------------------------------------------------------------
# 3. 실제 변경 샘플 확인
# ------------------------------------------------------------
best_manual_strategy = str(
    meta_repair_gate_sweep_df.iloc[0]["strategy"]
)
best_manual_predictions = meta_repair_prediction_cache[
    best_manual_strategy
]

meta_best_manual_detail_df = meta_repair_df[
    [
        "sample_id",
        "gold_order",
        "current_final_prediction",
        "current_final_exact",
        "pairwise_cycle_count",
    ]
].copy()

meta_best_manual_detail_df["new_prediction"] = best_manual_predictions
meta_best_manual_detail_df["changed"] = (
    meta_best_manual_detail_df["new_prediction"]
    != meta_best_manual_detail_df["current_final_prediction"]
)
meta_best_manual_detail_df["new_exact"] = (
    meta_best_manual_detail_df["new_prediction"]
    == meta_best_manual_detail_df["gold_order"]
).astype(int)
meta_best_manual_detail_df["transition"] = np.select(
    [
        (
            meta_best_manual_detail_df["current_final_exact"] == 0
        )
        & (
            meta_best_manual_detail_df["new_exact"] == 1
        ),
        (
            meta_best_manual_detail_df["current_final_exact"] == 1
        )
        & (
            meta_best_manual_detail_df["new_exact"] == 0
        ),
    ],
    ["wrong_to_right", "right_to_wrong"],
    default="unchanged_or_same_correctness",
)

print("\n최상위 수동 전략:", best_manual_strategy)
display(
    meta_best_manual_detail_df[
        meta_best_manual_detail_df["changed"]
    ].sort_values(
        ["transition", "sample_id"]
    )
)

if "OUTPUT_RUN_DIR" in globals():
    meta_repair_df.to_csv(
        OUTPUT_RUN_DIR / "meta_structural_repair_candidates.csv",
        index=False,
    )
    meta_repair_unconditional_df.to_csv(
        OUTPUT_RUN_DIR / "meta_repair_unconditional_summary.csv",
        index=False,
    )
    meta_repair_gate_sweep_df.to_csv(
        OUTPUT_RUN_DIR / "meta_repair_gate_sweep.csv",
        index=False,
    )
    meta_best_manual_detail_df.to_csv(
        OUTPUT_RUN_DIR / "meta_best_manual_strategy_detail.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 5. 5-fold OOF error detector
# - target: current_final != gold
# - Logistic / shallow tree / small RF
#
# 주의:
# OOF라도 feature/threshold 후보를 Eval200에서 여러 번 고르면
# 최종 일반화 성능은 낙관적일 수 있다.
# 최종 선택 후에는 별도 validation 또는 test-like split에서 검증.
# ============================================================

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


META_RANDOM_STATE = 42

error_feature_columns = [
    "full_order_normalized_entropy",

    "decoder_unique_prediction_count",
    "decoder_first_unique_count",
    "decoder_last_unique_count",
    "decoder_first_majority_share",
    "decoder_last_majority_share",
    "decoder_distance_mean",
    "decoder_distance_max",
    "current_vs_decoder_majority_kendall",

    "pairwise_cycle_count",

    "current_avg_rank",
    "current_worst_rank",
    "current_best_rank",
    "current_top1_vote_count",
    "current_top3_inclusion_count",
    "current_endpoint_agreement",
    "current_conditional_support",
    "current_pairwise_mean",
    "current_pairwise_min",
    "current_pairwise_agree_count",
    "current_heuristic_support",

    "best_alternative_top1_vote_count",
    "best_alternative_top3_inclusion_count",
    "best_alternative_avg_rank",
    "best_alternative_worst_rank",
    "best_alternative_kendall",
    "best_alternative_heuristic_support",
    "alternative_minus_current_support",
    "current_minus_alternative_avg_rank",
] + [
    f"{decoder}_margin"
    for decoder in META_RANKED_DECODERS
] + [
    f"current_{decoder}_rank"
    for decoder in META_RANKED_DECODERS
] + [
    f"alternative_{decoder}_rank"
    for decoder in META_RANKED_DECODERS
]

X_error = meta_feature_df[error_feature_columns].copy()
y_error = meta_feature_df["y_error"].astype(int).to_numpy()

minority_count = int(
    min(
        np.sum(y_error == 0),
        np.sum(y_error == 1),
    )
)
n_splits = min(5, minority_count)
if n_splits < 2:
    raise RuntimeError(
        "Error detector용 class별 샘플 수가 너무 적습니다."
    )

cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=META_RANDOM_STATE,
)

numeric_preprocess = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

error_models = {
    "logistic": Pipeline([
        ("preprocess", clone(numeric_preprocess)),
        (
            "model",
            LogisticRegression(
                max_iter=3000,
                class_weight="balanced",
                random_state=META_RANDOM_STATE,
            ),
        ),
    ]),
    "tree_depth3": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=3,
                min_samples_leaf=10,
                class_weight="balanced",
                random_state=META_RANDOM_STATE,
            ),
        ),
    ]),
    "rf_small": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=4,
                min_samples_leaf=5,
                class_weight="balanced_subsample",
                random_state=META_RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]),
}

meta_error_oof_probabilities = {}
error_summary_rows = []
error_threshold_rows = []

for model_name, model in error_models.items():
    oof_probability = np.zeros(len(meta_feature_df), dtype=np.float64)

    for train_index, valid_index in cv.split(X_error, y_error):
        fitted = clone(model)
        fitted.fit(
            X_error.iloc[train_index],
            y_error[train_index],
        )
        oof_probability[valid_index] = fitted.predict_proba(
            X_error.iloc[valid_index]
        )[:, 1]

    meta_error_oof_probabilities[model_name] = oof_probability

    error_summary_rows.append({
        "model": model_name,
        "roc_auc": float(
            roc_auc_score(y_error, oof_probability)
        ),
        "average_precision": float(
            average_precision_score(y_error, oof_probability)
        ),
        "positive_rate": float(np.mean(y_error)),
    })

    for threshold in np.round(
        np.arange(0.20, 0.81, 0.05),
        2,
    ):
        predicted_error = oof_probability >= threshold
        true_error = y_error == 1

        tp = int((predicted_error & true_error).sum())
        fp = int((predicted_error & ~true_error).sum())
        fn = int((~predicted_error & true_error).sum())
        tn = int((~predicted_error & ~true_error).sum())

        error_threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "flagged_count": int(predicted_error.sum()),
            "error_recall": tp / max(1, tp + fn),
            "error_precision": tp / max(1, tp + fp),
            "false_positive_count": fp,
            "true_positive_count": tp,
            "true_negative_count": tn,
            "false_negative_count": fn,
        })

meta_error_detector_summary_df = (
    pd.DataFrame(error_summary_rows)
    .sort_values(
        ["average_precision", "roc_auc"],
        ascending=False,
    )
    .reset_index(drop=True)
)

meta_error_threshold_df = (
    pd.DataFrame(error_threshold_rows)
    .sort_values(
        ["model", "threshold"]
    )
    .reset_index(drop=True)
)

print("OOF error detector 요약")
display(meta_error_detector_summary_df)

print("\nThreshold별 error recall / false positive")
display(meta_error_threshold_df)

for model_name, probabilities in (
    meta_error_oof_probabilities.items()
):
    meta_feature_df[f"oof_error_probability_{model_name}"] = probabilities

if "OUTPUT_RUN_DIR" in globals():
    meta_error_detector_summary_df.to_csv(
        OUTPUT_RUN_DIR / "meta_oof_error_detector_summary.csv",
        index=False,
    )
    meta_error_threshold_df.to_csv(
        OUTPUT_RUN_DIR / "meta_oof_error_detector_thresholds.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 6. Candidate-level OOF alternative selector
# - 한 행 = sample_id × candidate (200 × 24)
# - fold는 반드시 sample_id 기준 GroupKFold
# - selector 단독 + error detector gate 결합 평가
# ============================================================

from sklearn.model_selection import GroupKFold


candidate_feature_columns = [
    "top1_vote_count",
    "top2_inclusion_count",
    "top3_inclusion_count",

    *[
        f"{decoder}_rank"
        for decoder in META_RANKED_DECODERS
    ],
    *[
        f"{decoder}_borda"
        for decoder in META_RANKED_DECODERS
    ],

    "avg_rank",
    "worst_rank",
    "best_rank",
    "avg_borda",

    "first_endpoint_agreement",
    "last_endpoint_agreement",
    "endpoint_agreement_mean",

    "conditional_forward_probability",
    "conditional_backward_probability",
    "conditional_support",

    "pairwise_mean_probability",
    "pairwise_min_probability",
    "pairwise_agree_count",
    "adjacent_pairwise_mean",
    "nonadjacent_pairwise_mean",
    "pairwise_cycle_count",

    "kendall_to_current_final",
    "is_current_final",
    "heuristic_support",
]

candidate_frame = meta_candidate_profile_df.copy()
X_candidate = candidate_frame[candidate_feature_columns]
y_candidate = candidate_frame["candidate_is_gold"].astype(int).to_numpy()
groups = candidate_frame["sample_id"].astype(str).to_numpy()

unique_groups = np.unique(groups)
n_group_splits = min(5, len(unique_groups))
if n_group_splits < 2:
    raise RuntimeError("GroupKFold에 필요한 sample 수가 부족합니다.")

group_cv = GroupKFold(n_splits=n_group_splits)

candidate_models = {
    "candidate_logistic": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                random_state=META_RANDOM_STATE,
            ),
        ),
    ]),
    "candidate_rf": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                max_depth=6,
                min_samples_leaf=4,
                class_weight="balanced_subsample",
                random_state=META_RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]),
}

meta_candidate_oof_probabilities = {}
meta_candidate_oof_predictions = {}
selector_summary_rows = []

for model_name, model in candidate_models.items():
    oof_probability = np.zeros(len(candidate_frame), dtype=np.float64)

    for train_index, valid_index in group_cv.split(
        X_candidate,
        y_candidate,
        groups,
    ):
        fitted = clone(model)
        fitted.fit(
            X_candidate.iloc[train_index],
            y_candidate[train_index],
        )
        oof_probability[valid_index] = fitted.predict_proba(
            X_candidate.iloc[valid_index]
        )[:, 1]

    meta_candidate_oof_probabilities[model_name] = oof_probability
    candidate_frame[f"oof_gold_probability_{model_name}"] = (
        oof_probability
    )

    selected_rows = (
        candidate_frame
        .sort_values(
            [
                "sample_id",
                f"oof_gold_probability_{model_name}",
                "heuristic_support",
            ],
            ascending=[True, False, False],
        )
        .groupby("sample_id", as_index=False)
        .head(1)
    )

    selected_lookup = dict(zip(
        selected_rows["sample_id"],
        selected_rows["candidate_prediction"],
    ))

    selected_predictions = meta_feature_df[
        "sample_id"
    ].map(selected_lookup)

    meta_candidate_oof_predictions[
        model_name
    ] = selected_predictions

    selector_result = meta_evaluate_vs_current(
        selected_predictions,
        model_name,
    )
    selector_result["candidate_row_average_precision"] = float(
        average_precision_score(
            y_candidate,
            oof_probability,
        )
    )
    selector_summary_rows.append(selector_result)

meta_candidate_selector_summary_df = (
    pd.DataFrame(selector_summary_rows)
    .sort_values(
        ["exact", "right_to_wrong", "wrong_to_right"],
        ascending=[False, True, False],
    )
    .reset_index(drop=True)
)

print("OOF candidate selector 단독 성능")
display(meta_candidate_selector_summary_df)


# ------------------------------------------------------------
# Error detector + alternative selector 결합
# ------------------------------------------------------------
combined_rows = []
meta_oof_final_prediction_cache = {}

for detector_name, error_probability in (
    meta_error_oof_probabilities.items()
):
    for selector_name, selector_predictions in (
        meta_candidate_oof_predictions.items()
    ):
        selector_predictions = pd.Series(
            selector_predictions
        ).reset_index(drop=True)

        for threshold in np.round(
            np.arange(0.20, 0.81, 0.05),
            2,
        ):
            use_selector = (
                error_probability >= threshold
            ) & (
                selector_predictions
                != meta_feature_df[
                    "current_final_prediction"
                ]
            )

            final_prediction = pd.Series(
                np.where(
                    use_selector,
                    selector_predictions,
                    meta_feature_df[
                        "current_final_prediction"
                    ],
                )
            )

            name = (
                f"{detector_name}"
                f"__{selector_name}"
                f"__gate{threshold:.2f}"
            )

            result = meta_evaluate_vs_current(
                final_prediction,
                name,
            )
            result.update({
                "error_detector": detector_name,
                "candidate_selector": selector_name,
                "error_threshold": threshold,
                "gate_count": int(use_selector.sum()),
            })
            combined_rows.append(result)
            meta_oof_final_prediction_cache[name] = final_prediction

meta_oof_combined_summary_df = (
    pd.DataFrame(combined_rows)
    .sort_values(
        [
            "exact",
            "right_to_wrong",
            "wrong_to_right",
            "changed_from_current",
        ],
        ascending=[False, True, False, True],
    )
    .reset_index(drop=True)
)

print("\nOOF error detector + candidate selector")
display(meta_oof_combined_summary_df.head(50))


# ------------------------------------------------------------
# 최상위 OOF 조합의 changed sample
# ------------------------------------------------------------
best_oof_strategy = str(
    meta_oof_combined_summary_df.iloc[0]["strategy"]
)
best_oof_predictions = meta_oof_final_prediction_cache[
    best_oof_strategy
]

meta_best_oof_detail_df = meta_feature_df[
    [
        "sample_id",
        "gold_order",
        "current_final_prediction",
        "current_final_exact",
    ]
].copy()
meta_best_oof_detail_df["new_prediction"] = best_oof_predictions
meta_best_oof_detail_df["changed"] = (
    meta_best_oof_detail_df["new_prediction"]
    != meta_best_oof_detail_df["current_final_prediction"]
)
meta_best_oof_detail_df["new_exact"] = (
    meta_best_oof_detail_df["new_prediction"]
    == meta_best_oof_detail_df["gold_order"]
).astype(int)
meta_best_oof_detail_df["transition"] = np.select(
    [
        (
            meta_best_oof_detail_df["current_final_exact"] == 0
        )
        & (
            meta_best_oof_detail_df["new_exact"] == 1
        ),
        (
            meta_best_oof_detail_df["current_final_exact"] == 1
        )
        & (
            meta_best_oof_detail_df["new_exact"] == 0
        ),
    ],
    ["wrong_to_right", "right_to_wrong"],
    default="unchanged_or_same_correctness",
)

print("\n최상위 OOF 조합:", best_oof_strategy)
display(
    meta_best_oof_detail_df[
        meta_best_oof_detail_df["changed"]
    ].sort_values(
        ["transition", "sample_id"]
    )
)

if "OUTPUT_RUN_DIR" in globals():
    candidate_frame.to_csv(
        OUTPUT_RUN_DIR / "meta_candidate_oof_probabilities.csv",
        index=False,
    )
    meta_candidate_selector_summary_df.to_csv(
        OUTPUT_RUN_DIR / "meta_candidate_selector_summary.csv",
        index=False,
    )
    meta_oof_combined_summary_df.to_csv(
        OUTPUT_RUN_DIR / "meta_oof_combined_summary.csv",
        index=False,
    )
    meta_best_oof_detail_df.to_csv(
        OUTPUT_RUN_DIR / "meta_best_oof_strategy_detail.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 7A. Current final vs Top-3 consensus alternative pair table
# - 각 sample마다 현재 final + Top-3 합집합 최강 alternative 1개만 비교
# - META 3 실행 후 사용 가능
# ============================================================

import numpy as np
import pandas as pd

_pair_required = [
    "meta_feature_df",
    "meta_candidate_profile_df",
    "meta_endpoint_df",
    "META_RANKED_DECODERS",
]
_pair_missing = [name for name in _pair_required if name not in globals()]
if _pair_missing:
    raise RuntimeError(
        "먼저 META 0~3 셀을 실행하세요. 누락 변수: "
        f"{_pair_missing}"
    )

# 후보별 비교에 사용할 profile feature.
META_PAIR_PROFILE_FEATURES = [
    "top1_vote_count",
    "top2_inclusion_count",
    "top3_inclusion_count",
    "avg_rank",
    "worst_rank",
    "best_rank",
    "avg_borda",
    "first_endpoint_agreement",
    "last_endpoint_agreement",
    "endpoint_agreement_mean",
    "conditional_forward_probability",
    "conditional_backward_probability",
    "conditional_support",
    "pairwise_mean_probability",
    "pairwise_min_probability",
    "pairwise_agree_count",
    "adjacent_pairwise_mean",
    "nonadjacent_pairwise_mean",
    "heuristic_support",
] + [
    f"{decoder}_rank" for decoder in META_RANKED_DECODERS
] + [
    f"{decoder}_borda" for decoder in META_RANKED_DECODERS
]

profile = meta_candidate_profile_df.copy()
profile["sample_id"] = profile["sample_id"].astype(str)

# 현재 final profile: sample당 정확히 1개여야 한다.
current_profile = profile[profile["is_current_final"] == 1].copy()
current_count = current_profile.groupby("sample_id").size()
if not (current_count == 1).all():
    raise RuntimeError("sample마다 current final profile이 정확히 1개가 아닙니다.")

# Top-3 합집합에 한 번이라도 들어온, current가 아닌 후보만 alternative pool로 사용.
# 정답 label은 후보 선택에 전혀 사용하지 않는다.
alternative_pool = profile[
    (profile["is_current_final"] == 0)
    & (profile["top3_inclusion_count"] >= 1)
].copy()

# decoder consensus를 먼저 보고, 동률이면 평균 rank와 support로 결정.
alternative_profile = (
    alternative_pool
    .sort_values(
        [
            "sample_id",
            "top1_vote_count",
            "top3_inclusion_count",
            "avg_rank",
            "worst_rank",
            "heuristic_support",
            "candidate_prediction",
        ],
        ascending=[True, False, False, True, True, False, True],
    )
    .groupby("sample_id", as_index=False)
    .head(1)
    .copy()
)

if alternative_profile["sample_id"].nunique() != meta_feature_df["sample_id"].nunique():
    missing_ids = sorted(
        set(meta_feature_df["sample_id"].astype(str))
        - set(alternative_profile["sample_id"].astype(str))
    )
    raise RuntimeError(f"alternative를 만들지 못한 sample: {missing_ids[:10]}")


def _pair_profile_view(frame, prefix):
    selected = frame[
        ["sample_id", "candidate_prediction", *META_PAIR_PROFILE_FEATURES]
    ].copy()
    return selected.rename(columns={
        "candidate_prediction": f"{prefix}_prediction",
        **{
            column: f"{prefix}__{column}"
            for column in META_PAIR_PROFILE_FEATURES
        },
    })


meta_pair_df = meta_feature_df.copy()
meta_pair_df["sample_id"] = meta_pair_df["sample_id"].astype(str)
meta_pair_df = meta_pair_df.merge(
    _pair_profile_view(current_profile, "current"),
    on="sample_id",
    how="left",
    validate="one_to_one",
)
meta_pair_df = meta_pair_df.merge(
    _pair_profile_view(alternative_profile, "alternative"),
    on="sample_id",
    how="left",
    validate="one_to_one",
)

# profile에서 읽은 current와 실제 current final이 같은지 검증.
if not (
    meta_pair_df["current_prediction"]
    == meta_pair_df["current_final_prediction"]
).all():
    raise RuntimeError("current profile과 current_final_prediction이 다릅니다.")

meta_pair_df["current_exact"] = (
    meta_pair_df["current_final_prediction"]
    == meta_pair_df["gold_order"]
).astype(int)
meta_pair_df["alternative_exact"] = (
    meta_pair_df["alternative_prediction"]
    == meta_pair_df["gold_order"]
).astype(int)

# 서로 다른 두 후보이므로 둘 다 맞을 수는 없다.
meta_pair_df["pair_decisive"] = (
    meta_pair_df["current_exact"]
    != meta_pair_df["alternative_exact"]
).astype(int)
meta_pair_df["pair_target_choose_alternative"] = (
    meta_pair_df["alternative_exact"] == 1
).astype(int)
meta_pair_df["two_candidate_oracle_exact"] = (
    (meta_pair_df["current_exact"] == 1)
    | (meta_pair_df["alternative_exact"] == 1)
).astype(int)

# endpoint 불일치 개수: 0, 1, 2.
meta_pair_df["decoder_endpoint_disagree_count"] = (
    (meta_pair_df["decoder_first_unique_count"] > 1).astype(int)
    + (meta_pair_df["decoder_last_unique_count"] > 1).astype(int)
)

# higher-is-better feature는 alternative-current,
# rank는 작을수록 좋으므로 current-alternative를 gain으로 둔다.
PAIR_RANK_FEATURES = {
    "avg_rank",
    "worst_rank",
    "best_rank",
    *[f"{decoder}_rank" for decoder in META_RANKED_DECODERS],
}

pair_delta_columns = []
for column in META_PAIR_PROFILE_FEATURES:
    current_column = f"current__{column}"
    alternative_column = f"alternative__{column}"

    if column in PAIR_RANK_FEATURES:
        delta_column = f"rank_gain__{column}"
        meta_pair_df[delta_column] = (
            meta_pair_df[current_column]
            - meta_pair_df[alternative_column]
        )
    else:
        delta_column = f"delta__{column}"
        meta_pair_df[delta_column] = (
            meta_pair_df[alternative_column]
            - meta_pair_df[current_column]
        )
    pair_delta_columns.append(delta_column)

# 이전 분석에서 얻은 보수적 구조 gate를 그대로 feature와 비교 기준으로 둔다.
meta_pair_df["recommended_structural_gate"] = (
    (meta_pair_df["decoder_endpoint_disagree_count"] > 0)
    & (meta_pair_df["alternative__top1_vote_count"] >= 2)
    & (meta_pair_df["alternative__top3_inclusion_count"] >= 3)
    & (meta_pair_df["rank_gain__avg_rank"] >= 0.0)
)

META_PAIR_CONTEXT_FEATURES = [
    "full_order_normalized_entropy",
    "decoder_unique_prediction_count",
    "decoder_first_unique_count",
    "decoder_last_unique_count",
    "decoder_first_majority_share",
    "decoder_last_majority_share",
    "decoder_distance_mean",
    "decoder_distance_max",
    "current_vs_decoder_majority_kendall",
    "pairwise_cycle_count",
    "decoder_endpoint_disagree_count",
] + [
    f"{decoder}_margin" for decoder in META_RANKED_DECODERS
]

# absolute profile + 두 후보 차이를 함께 준다.
META_PAIR_FEATURE_COLUMNS = (
    META_PAIR_CONTEXT_FEATURES
    + [f"current__{column}" for column in META_PAIR_PROFILE_FEATURES]
    + [f"alternative__{column}" for column in META_PAIR_PROFILE_FEATURES]
    + pair_delta_columns
)
META_PAIR_FEATURE_COLUMNS = list(dict.fromkeys(META_PAIR_FEATURE_COLUMNS))

pair_case = np.select(
    [
        (meta_pair_df["current_exact"] == 1),
        (meta_pair_df["alternative_exact"] == 1),
    ],
    ["current_only_correct", "alternative_only_correct"],
    default="neither_correct",
)
meta_pair_df["pair_case"] = pair_case

print("Current exact:", int(meta_pair_df["current_exact"].sum()), "/", len(meta_pair_df))
print(
    "Current/alternative 2-candidate oracle:",
    int(meta_pair_df["two_candidate_oracle_exact"].sum()),
    "/",
    len(meta_pair_df),
    "=",
    float(meta_pair_df["two_candidate_oracle_exact"].mean()),
)
print("Pair cases")
display(meta_pair_df["pair_case"].value_counts().rename_axis("case").reset_index(name="count"))
print("Recommended structural gate trigger:", int(meta_pair_df["recommended_structural_gate"].sum()))

display(
    meta_pair_df[
        [
            "sample_id",
            "gold_order",
            "current_final_prediction",
            "alternative_prediction",
            "pair_case",
            "decoder_endpoint_disagree_count",
            "alternative__top1_vote_count",
            "alternative__top3_inclusion_count",
            "rank_gain__avg_rank",
            "recommended_structural_gate",
        ]
    ].head(20)
)


In [ ]:
# ============================================================
# META 7B. 5-fold OOF binary pair selector
# - 학습 target: current와 alternative 중 어느 쪽이 gold인가
# - 둘 다 틀린 sample은 학습 label에서 제외하고, 검증/평가에는 포함
# - probability threshold와 구조 gate를 진단
# ============================================================

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

META_PAIR_RANDOM_STATE = 42

X_pair = meta_pair_df[META_PAIR_FEATURE_COLUMNS].copy()
y_pair = meta_pair_df["pair_target_choose_alternative"].astype(int).to_numpy()
decisive_mask = meta_pair_df["pair_decisive"].astype(bool).to_numpy()

# fold는 전체 sample 기준으로 나눈다.
# 현재 final correct/error를 stratify하여 각 fold의 난이도 차이를 줄인다.
stratify_y = meta_pair_df["current_exact"].astype(int).to_numpy()
minority_count = int(min(np.sum(stratify_y == 0), np.sum(stratify_y == 1)))
n_splits = min(5, minority_count)
if n_splits < 2:
    raise RuntimeError("OOF split에 필요한 class별 sample 수가 부족합니다.")

pair_cv = StratifiedKFold(
    n_splits=n_splits,
    shuffle=True,
    random_state=META_PAIR_RANDOM_STATE,
)
META_PAIR_OUTER_SPLITS = list(pair_cv.split(X_pair, stratify_y))

meta_pair_df["pair_oof_fold"] = -1
for fold_id, (_, valid_index) in enumerate(META_PAIR_OUTER_SPLITS):
    meta_pair_df.loc[valid_index, "pair_oof_fold"] = fold_id

META_PAIR_MODELS = {
    "pair_logistic": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                max_iter=4000,
                class_weight="balanced",
                C=0.5,
                random_state=META_PAIR_RANDOM_STATE,
            ),
        ),
    ]),
    "pair_tree_depth2": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            DecisionTreeClassifier(
                max_depth=2,
                min_samples_leaf=12,
                class_weight="balanced",
                random_state=META_PAIR_RANDOM_STATE,
            ),
        ),
    ]),
    "pair_rf_small": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        (
            "model",
            RandomForestClassifier(
                n_estimators=400,
                max_depth=4,
                min_samples_leaf=6,
                max_features="sqrt",
                class_weight="balanced_subsample",
                random_state=META_PAIR_RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]),
}


def _fit_pair_model(model, train_index):
    # current 또는 alternative 중 하나가 맞는 decisive sample만 학습한다.
    fit_index = np.asarray(train_index)[decisive_mask[np.asarray(train_index)]]
    fit_y = y_pair[fit_index]

    if len(fit_index) == 0 or len(np.unique(fit_y)) < 2:
        return None, float(np.mean(fit_y)) if len(fit_y) else 0.0

    fitted = clone(model)
    fitted.fit(X_pair.iloc[fit_index], fit_y)
    return fitted, None


def _predict_pair_probability(model, train_index, valid_index):
    fitted, constant_probability = _fit_pair_model(model, train_index)
    if fitted is None:
        return np.full(len(valid_index), constant_probability, dtype=np.float64)
    return fitted.predict_proba(X_pair.iloc[valid_index])[:, 1]


meta_pair_oof_probabilities = {}
for model_name, model in META_PAIR_MODELS.items():
    oof_probability = np.zeros(len(meta_pair_df), dtype=np.float64)

    for train_index, valid_index in META_PAIR_OUTER_SPLITS:
        oof_probability[valid_index] = _predict_pair_probability(
            model,
            train_index,
            valid_index,
        )

    meta_pair_oof_probabilities[model_name] = oof_probability
    meta_pair_df[f"oof_choose_alternative_probability__{model_name}"] = oof_probability


def _evaluate_pair_switch(use_alternative, strategy_name):
    use_alternative = np.asarray(use_alternative, dtype=bool)
    current = meta_pair_df["current_final_prediction"].astype(str)
    alternative = meta_pair_df["alternative_prediction"].astype(str)
    gold = meta_pair_df["gold_order"].astype(str)

    prediction = pd.Series(
        np.where(use_alternative, alternative, current),
        index=meta_pair_df.index,
    )
    current_correct = current.eq(gold)
    new_correct = prediction.eq(gold)
    changed = prediction.ne(current)

    wrong_to_right = changed & (~current_correct) & new_correct
    right_to_wrong = changed & current_correct & (~new_correct)

    fold_nets = []
    fold_exacts = []
    for fold_id in sorted(meta_pair_df["pair_oof_fold"].unique()):
        fold_mask = meta_pair_df["pair_oof_fold"].eq(fold_id)
        fold_w2r = int((wrong_to_right & fold_mask).sum())
        fold_r2w = int((right_to_wrong & fold_mask).sum())
        fold_nets.append(fold_w2r - fold_r2w)
        fold_exacts.append(float(new_correct[fold_mask].mean()))

    return {
        "strategy": strategy_name,
        "exact": float(new_correct.mean()),
        "correct": int(new_correct.sum()),
        "changed": int(changed.sum()),
        "wrong_to_right": int(wrong_to_right.sum()),
        "right_to_wrong": int(right_to_wrong.sum()),
        "net": int(wrong_to_right.sum() - right_to_wrong.sum()),
        "fold_exact_mean": float(np.mean(fold_exacts)),
        "fold_exact_std": float(np.std(fold_exacts)),
        "fold_min_net": int(np.min(fold_nets)),
        "fold_net_values": fold_nets,
    }, prediction


pair_thresholds = np.round(np.arange(0.50, 0.91, 0.05), 2)
gate_masks = {
    "no_structure_gate": np.ones(len(meta_pair_df), dtype=bool),
    "endpoint_disagree": (
        meta_pair_df["decoder_endpoint_disagree_count"].to_numpy() > 0
    ),
    "recommended_structure": (
        meta_pair_df["recommended_structural_gate"].to_numpy(dtype=bool)
    ),
}

pair_sweep_rows = []
meta_pair_prediction_cache = {}

for model_name, probability in meta_pair_oof_probabilities.items():
    for gate_name, gate_mask in gate_masks.items():
        for threshold in pair_thresholds:
            use_alternative = (
                gate_mask
                & (probability >= threshold)
                & (
                    meta_pair_df["alternative_prediction"].to_numpy()
                    != meta_pair_df["current_final_prediction"].to_numpy()
                )
            )
            strategy = f"{model_name}__{gate_name}__p{threshold:.2f}"
            result, prediction = _evaluate_pair_switch(use_alternative, strategy)
            result.update({
                "model": model_name,
                "gate": gate_name,
                "pair_threshold": float(threshold),
                "gate_count": int(use_alternative.sum()),
            })
            pair_sweep_rows.append(result)
            meta_pair_prediction_cache[strategy] = prediction

# META 5의 OOF error detector를 이미 실행했다면 결합 결과도 같이 본다.
if "meta_error_oof_probabilities" in globals():
    for model_name, probability in meta_pair_oof_probabilities.items():
        for error_name, error_probability in meta_error_oof_probabilities.items():
            error_probability = np.asarray(error_probability, dtype=np.float64)
            for pair_threshold in [0.55, 0.65, 0.75]:
                for error_threshold in [0.50, 0.60, 0.70]:
                    use_alternative = (
                        meta_pair_df["recommended_structural_gate"].to_numpy(dtype=bool)
                        & (probability >= pair_threshold)
                        & (error_probability >= error_threshold)
                    )
                    strategy = (
                        f"{model_name}__error_{error_name}"
                        f"__pair{pair_threshold:.2f}__error{error_threshold:.2f}"
                    )
                    result, prediction = _evaluate_pair_switch(use_alternative, strategy)
                    result.update({
                        "model": model_name,
                        "gate": f"recommended+error_{error_name}",
                        "pair_threshold": float(pair_threshold),
                        "error_threshold": float(error_threshold),
                        "gate_count": int(use_alternative.sum()),
                    })
                    pair_sweep_rows.append(result)
                    meta_pair_prediction_cache[strategy] = prediction

meta_pair_oof_sweep_df = (
    pd.DataFrame(pair_sweep_rows)
    .sort_values(
        [
            "exact",
            "fold_min_net",
            "right_to_wrong",
            "wrong_to_right",
            "changed",
            "pair_threshold",
        ],
        ascending=[False, False, True, False, True, False],
    )
    .reset_index(drop=True)
)

# fold마다 손해가 없었던 결과만 따로 본다.
meta_pair_oof_stable_df = (
    meta_pair_oof_sweep_df[
        meta_pair_oof_sweep_df["fold_min_net"] >= 0
    ]
    .copy()
    .reset_index(drop=True)
)

print("OOF pair selector 상위 결과")
display(meta_pair_oof_sweep_df.head(40))
print("\nFold별 net이 모두 0 이상인 결과")
display(meta_pair_oof_stable_df.head(30))

if "OUTPUT_RUN_DIR" in globals():
    meta_pair_df.to_csv(
        OUTPUT_RUN_DIR / "meta_binary_pair_features_oof.csv",
        index=False,
    )
    meta_pair_oof_sweep_df.to_csv(
        OUTPUT_RUN_DIR / "meta_binary_pair_selector_sweep.csv",
        index=False,
    )
    meta_pair_oof_stable_df.to_csv(
        OUTPUT_RUN_DIR / "meta_binary_pair_selector_stable.csv",
        index=False,
    )


In [ ]:
# ============================================================
# META 7C. Nested OOF threshold validation
# - 7B의 전체 OOF threshold sweep는 threshold를 Eval200 전체에서 고르므로 낙관적일 수 있음
# - 각 outer fold의 train 부분에서 inner OOF로 threshold를 선택한 뒤
#   outer validation에 적용하여 더 엄격하게 확인
# - recommended_structural_gate는 고정
# ============================================================

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold

NESTED_PAIR_THRESHOLDS = [
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90, 1.01,  # 1.01은 변경 안 함 선택지
]


def _subset_switch_stats(index, probability, threshold):
    index = np.asarray(index, dtype=int)
    use_alt = (
        meta_pair_df.loc[index, "recommended_structural_gate"].to_numpy(dtype=bool)
        & (np.asarray(probability) >= threshold)
    )

    current_correct = meta_pair_df.loc[index, "current_exact"].to_numpy(dtype=bool)
    alternative_correct = meta_pair_df.loc[index, "alternative_exact"].to_numpy(dtype=bool)

    wrong_to_right = use_alt & (~current_correct) & alternative_correct
    right_to_wrong = use_alt & current_correct & (~alternative_correct)

    return {
        "threshold": float(threshold),
        "changed": int(use_alt.sum()),
        "wrong_to_right": int(wrong_to_right.sum()),
        "right_to_wrong": int(right_to_wrong.sum()),
        "net": int(wrong_to_right.sum() - right_to_wrong.sum()),
    }


def _inner_oof_probabilities(model, outer_train_index, random_state):
    outer_train_index = np.asarray(outer_train_index, dtype=int)
    inner_stratify = meta_pair_df.loc[
        outer_train_index,
        "current_exact",
    ].astype(int).to_numpy()

    minority = int(min(
        np.sum(inner_stratify == 0),
        np.sum(inner_stratify == 1),
    ))
    inner_splits = min(4, minority)
    if inner_splits < 2:
        return np.zeros(len(outer_train_index), dtype=np.float64)

    inner_cv = StratifiedKFold(
        n_splits=inner_splits,
        shuffle=True,
        random_state=random_state,
    )
    inner_oof = np.zeros(len(outer_train_index), dtype=np.float64)

    for inner_train_local, inner_valid_local in inner_cv.split(
        X_pair.iloc[outer_train_index],
        inner_stratify,
    ):
        inner_train_global = outer_train_index[inner_train_local]
        inner_valid_global = outer_train_index[inner_valid_local]
        inner_oof[inner_valid_local] = _predict_pair_probability(
            model,
            inner_train_global,
            inner_valid_global,
        )

    return inner_oof


nested_summary_rows = []
meta_pair_nested_prediction_cache = {}
meta_pair_nested_threshold_rows = []

for model_offset, (model_name, model) in enumerate(META_PAIR_MODELS.items()):
    nested_use_alternative = np.zeros(len(meta_pair_df), dtype=bool)
    nested_probability = np.zeros(len(meta_pair_df), dtype=np.float64)

    for fold_id, (outer_train_index, outer_valid_index) in enumerate(META_PAIR_OUTER_SPLITS):
        inner_probability = _inner_oof_probabilities(
            model,
            outer_train_index,
            random_state=META_PAIR_RANDOM_STATE + 100 * model_offset + fold_id,
        )

        threshold_candidates = []
        for threshold in NESTED_PAIR_THRESHOLDS:
            stats = _subset_switch_stats(
                outer_train_index,
                inner_probability,
                threshold,
            )
            threshold_candidates.append(stats)

        # train 내부에서 net 최대 -> R2W 최소 -> 변경 최소 -> 더 높은 threshold 순.
        threshold_table = (
            pd.DataFrame(threshold_candidates)
            .sort_values(
                ["net", "right_to_wrong", "changed", "threshold"],
                ascending=[False, True, True, False],
            )
            .reset_index(drop=True)
        )
        selected_threshold = float(threshold_table.iloc[0]["threshold"])

        outer_probability = _predict_pair_probability(
            model,
            outer_train_index,
            outer_valid_index,
        )
        nested_probability[outer_valid_index] = outer_probability
        nested_use_alternative[outer_valid_index] = (
            meta_pair_df.loc[
                outer_valid_index,
                "recommended_structural_gate",
            ].to_numpy(dtype=bool)
            & (outer_probability >= selected_threshold)
        )

        meta_pair_nested_threshold_rows.append({
            "model": model_name,
            "outer_fold": fold_id,
            "selected_threshold": selected_threshold,
            "inner_best_net": int(threshold_table.iloc[0]["net"]),
            "inner_best_wrong_to_right": int(
                threshold_table.iloc[0]["wrong_to_right"]
            ),
            "inner_best_right_to_wrong": int(
                threshold_table.iloc[0]["right_to_wrong"]
            ),
            "inner_best_changed": int(threshold_table.iloc[0]["changed"]),
            "outer_gate_count": int(nested_use_alternative[outer_valid_index].sum()),
        })

    strategy = f"nested_{model_name}__recommended_structure"
    result, prediction = _evaluate_pair_switch(
        nested_use_alternative,
        strategy,
    )
    result.update({
        "model": model_name,
        "gate": "nested_recommended_structure",
        "gate_count": int(nested_use_alternative.sum()),
    })
    nested_summary_rows.append(result)
    meta_pair_nested_prediction_cache[strategy] = prediction
    meta_pair_df[
        f"nested_choose_alternative_probability__{model_name}"
    ] = nested_probability

meta_pair_nested_summary_df = (
    pd.DataFrame(nested_summary_rows)
    .sort_values(
        ["exact", "fold_min_net", "right_to_wrong", "changed"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)
meta_pair_nested_threshold_df = pd.DataFrame(meta_pair_nested_threshold_rows)

print("Nested OOF binary pair selector")
display(meta_pair_nested_summary_df)
print("\nOuter fold별 inner-selected threshold")
display(meta_pair_nested_threshold_df)

best_nested_strategy = str(meta_pair_nested_summary_df.iloc[0]["strategy"])
best_nested_prediction = meta_pair_nested_prediction_cache[best_nested_strategy]

meta_pair_nested_detail_df = meta_pair_df[
    [
        "sample_id",
        "gold_order",
        "current_final_prediction",
        "alternative_prediction",
        "current_exact",
        "alternative_exact",
        "pair_oof_fold",
        "decoder_endpoint_disagree_count",
        "alternative__top1_vote_count",
        "alternative__top3_inclusion_count",
        "rank_gain__avg_rank",
        "recommended_structural_gate",
    ]
].copy()
meta_pair_nested_detail_df["new_prediction"] = best_nested_prediction
meta_pair_nested_detail_df["changed"] = (
    meta_pair_nested_detail_df["new_prediction"]
    != meta_pair_nested_detail_df["current_final_prediction"]
)
meta_pair_nested_detail_df["new_exact"] = (
    meta_pair_nested_detail_df["new_prediction"]
    == meta_pair_nested_detail_df["gold_order"]
).astype(int)
meta_pair_nested_detail_df["transition"] = np.select(
    [
        (
            meta_pair_nested_detail_df["current_exact"].eq(0)
            & meta_pair_nested_detail_df["new_exact"].eq(1)
        ),
        (
            meta_pair_nested_detail_df["current_exact"].eq(1)
            & meta_pair_nested_detail_df["new_exact"].eq(0)
        ),
    ],
    ["wrong_to_right", "right_to_wrong"],
    default="unchanged_or_same_correctness",
)

print("\nBest nested strategy:", best_nested_strategy)
display(
    meta_pair_nested_detail_df[
        meta_pair_nested_detail_df["changed"]
    ].sort_values(["transition", "pair_oof_fold", "sample_id"])
)

if "OUTPUT_RUN_DIR" in globals():
    meta_pair_nested_summary_df.to_csv(
        OUTPUT_RUN_DIR / "meta_binary_pair_nested_summary.csv",
        index=False,
    )
    meta_pair_nested_threshold_df.to_csv(
        OUTPUT_RUN_DIR / "meta_binary_pair_nested_thresholds.csv",
        index=False,
    )
    meta_pair_nested_detail_df.to_csv(
        OUTPUT_RUN_DIR / "meta_binary_pair_nested_best_detail.csv",
        index=False,
    )
